In [ ]:
///- --package . --timeout 2400000
open xml_codec

prototype to_text t : t -> string

inl text_node forall t {to_text}. (name : string) (v : t) : xml.xml_node =
    xml.element name [] (xml.text_children (to_text v))

inl optional_text_nodes forall t {to_text}. (name : string) (v : option t) : list xml.xml_node =
    xml.option_list (optionm.map (text_node name) v)

inl repeated_text_nodes forall t {to_text}. (name : string) (vs : list t) : list xml.xml_node =
    listm.map (text_node name) vs

nominal lexical t = { value : t; text : string }

instance to_text lexical t = fun x => x.text

inl lexical_of forall t. (text : string) (value : option t) : option (lexical t) =
    optionm.map (fun v => lexical { value = v; text = text }) value

inl whole_of (text : string) : option i64 =
    if sm'.index_of "." text >= 0i32
    then None
    else optionm.map (fun (d : decimal.decimal) => d.units) (decimal.parse text)

inl whole_at_least (least : i64) (text : string) : option i64 =
    match whole_of text with
    | Some n => if n >= least then Some n else None
    | None => None

inl boolean_of (text : string) : option bool =
    match xml.collapsed_space text with
    | "true" => Some true
    | "1" => Some true
    | "false" => Some false
    | "0" => Some false
    | _ => None

type xsd_decimal = lexical decimal.decimal

inl xsd_decimal_of_string (s : string) : option xsd_decimal = lexical_of s (decimal.parse s)

inl xsd_decimal_example (_ : i32) : xsd_decimal = lexical { value = decimal.of_int 1i64; text = "1" }

inl xsd_decimal_example_text (_ : i32) : string = "1"

type xsd_integer = lexical i64

inl xsd_integer_of_string (s : string) : option xsd_integer = lexical_of s (whole_of s)

inl xsd_integer_example (_ : i32) : xsd_integer = lexical { value = 1i64; text = "1" }

inl xsd_integer_example_text (_ : i32) : string = "1"

type xsd_positive_integer = lexical i64

inl xsd_positive_integer_of_string (s : string) : option xsd_positive_integer = lexical_of s (whole_at_least 1i64 s)

inl xsd_positive_integer_example (_ : i32) : xsd_positive_integer = lexical { value = 1i64; text = "1" }

inl xsd_positive_integer_example_text (_ : i32) : string = "1"

nominal unsigned = ()
nominal signed = ()

union staged s a =
    | Pending :: forall a. string -> staged unsigned a
    | Present :: forall a. a -> staged signed a

union stage s =
    | Unsigned :: stage unsigned
    | Signed :: stage signed

prototype stage_of s : () -> stage s
instance stage_of unsigned = fun () => Unsigned
instance stage_of signed = fun () => Signed

inl staged_read forall s a.
    (witness : stage s) (read : list xml_codec.decode_error -> option a * list xml_codec.decode_error)
    (errors : list xml_codec.decode_error)
    : option (staged s a) * list xml_codec.decode_error
    =
    match witness with
    | Unsigned => Some (Pending "unsigned"), errors
    | Signed =>
        inl value, errors = read errors
        optionm.map (fun v => Present v) value, errors

inl signed_value forall a. (x : staged signed a) : a =
    match x with
    | Present v => v

inl staged_list forall s a b. (f : a -> b) (x : staged s a) : list b =
    match x with
    | Pending _ => []
    | Present v => [ f v ]

type x_fecha_hora_type = string

inl x_fecha_hora_type_is_valid (s : string) : bool = true

inl x_fecha_hora_type_example (_ : i32) : string = "2026-01-01T00:00:00"

inl x_fecha_hora_type_example_text (k : i32) : string = x_fecha_hora_type_example k

type x_fecha_type = string

inl x_fecha_type_is_valid (s : string) : bool = true

inl x_fecha_type_example (_ : i32) : string = "2026-01-01"

inl x_fecha_type_example_text (k : i32) : string = x_fecha_type_example k

union x_tip_mon_type =
    | TipMonType_BOLIVAR
    | TipMonType_BOLIVIANO
    | TipMonType_CHELIN
    | TipMonType_CORONA_DIN
    | TipMonType_CORONA_NOR
    | TipMonType_CORONA_SC
    | TipMonType_CRUZEIRO_REAL
    | TipMonType_DIRHAM
    | TipMonType_DOLAR_AUST
    | TipMonType_DOLAR_CAN
    | TipMonType_DOLAR_HK
    | TipMonType_DOLAR_NZ
    | TipMonType_DOLAR_SIN
    | TipMonType_DOLAR_TAI
    | TipMonType_DOLAR_USA
    | TipMonType_DRACMA
    | TipMonType_ESCUDO
    | TipMonType_EURO
    | TipMonType_FLORIN
    | TipMonType_FRANCO_BEL
    | TipMonType_FRANCO_FR
    | TipMonType_FRANCO_SZ
    | TipMonType_GUARANI
    | TipMonType_LIBRA_EST
    | TipMonType_LIRA
    | TipMonType_MARCO_AL
    | TipMonType_MARCO_FIN
    | TipMonType_NUEVO_SOL
    | TipMonType_OTRAS_MONEDAS
    | TipMonType_PESETA
    | TipMonType_PESO
    | TipMonType_PESO_CL
    | TipMonType_PESO_COL
    | TipMonType_PESO_MEX
    | TipMonType_PESO_URUG
    | TipMonType_RAND
    | TipMonType_RENMINBI
    | TipMonType_RUPIA
    | TipMonType_SUCRE
    | TipMonType_YEN

inl x_tip_mon_type_of_string (s : string) : option x_tip_mon_type =
    match s with
    | "BOLIVAR" => Some TipMonType_BOLIVAR
    | "BOLIVIANO" => Some TipMonType_BOLIVIANO
    | "CHELIN" => Some TipMonType_CHELIN
    | "CORONA DIN" => Some TipMonType_CORONA_DIN
    | "CORONA NOR" => Some TipMonType_CORONA_NOR
    | "CORONA SC" => Some TipMonType_CORONA_SC
    | "CRUZEIRO REAL" => Some TipMonType_CRUZEIRO_REAL
    | "DIRHAM" => Some TipMonType_DIRHAM
    | "DOLAR AUST" => Some TipMonType_DOLAR_AUST
    | "DOLAR CAN" => Some TipMonType_DOLAR_CAN
    | "DOLAR HK" => Some TipMonType_DOLAR_HK
    | "DOLAR NZ" => Some TipMonType_DOLAR_NZ
    | "DOLAR SIN" => Some TipMonType_DOLAR_SIN
    | "DOLAR TAI" => Some TipMonType_DOLAR_TAI
    | "DOLAR USA" => Some TipMonType_DOLAR_USA
    | "DRACMA" => Some TipMonType_DRACMA
    | "ESCUDO" => Some TipMonType_ESCUDO
    | "EURO" => Some TipMonType_EURO
    | "FLORIN" => Some TipMonType_FLORIN
    | "FRANCO BEL" => Some TipMonType_FRANCO_BEL
    | "FRANCO FR" => Some TipMonType_FRANCO_FR
    | "FRANCO SZ" => Some TipMonType_FRANCO_SZ
    | "GUARANI" => Some TipMonType_GUARANI
    | "LIBRA EST" => Some TipMonType_LIBRA_EST
    | "LIRA" => Some TipMonType_LIRA
    | "MARCO AL" => Some TipMonType_MARCO_AL
    | "MARCO FIN" => Some TipMonType_MARCO_FIN
    | "NUEVO SOL" => Some TipMonType_NUEVO_SOL
    | "OTRAS MONEDAS" => Some TipMonType_OTRAS_MONEDAS
    | "PESETA" => Some TipMonType_PESETA
    | "PESO" => Some TipMonType_PESO
    | "PESO CL" => Some TipMonType_PESO_CL
    | "PESO COL" => Some TipMonType_PESO_COL
    | "PESO MEX" => Some TipMonType_PESO_MEX
    | "PESO URUG" => Some TipMonType_PESO_URUG
    | "RAND" => Some TipMonType_RAND
    | "RENMINBI" => Some TipMonType_RENMINBI
    | "RUPIA" => Some TipMonType_RUPIA
    | "SUCRE" => Some TipMonType_SUCRE
    | "YEN" => Some TipMonType_YEN
    | _ => None

inl x_tip_mon_type_to_string (x : x_tip_mon_type) : string =
    match x with
    | TipMonType_BOLIVAR => "BOLIVAR"
    | TipMonType_BOLIVIANO => "BOLIVIANO"
    | TipMonType_CHELIN => "CHELIN"
    | TipMonType_CORONA_DIN => "CORONA DIN"
    | TipMonType_CORONA_NOR => "CORONA NOR"
    | TipMonType_CORONA_SC => "CORONA SC"
    | TipMonType_CRUZEIRO_REAL => "CRUZEIRO REAL"
    | TipMonType_DIRHAM => "DIRHAM"
    | TipMonType_DOLAR_AUST => "DOLAR AUST"
    | TipMonType_DOLAR_CAN => "DOLAR CAN"
    | TipMonType_DOLAR_HK => "DOLAR HK"
    | TipMonType_DOLAR_NZ => "DOLAR NZ"
    | TipMonType_DOLAR_SIN => "DOLAR SIN"
    | TipMonType_DOLAR_TAI => "DOLAR TAI"
    | TipMonType_DOLAR_USA => "DOLAR USA"
    | TipMonType_DRACMA => "DRACMA"
    | TipMonType_ESCUDO => "ESCUDO"
    | TipMonType_EURO => "EURO"
    | TipMonType_FLORIN => "FLORIN"
    | TipMonType_FRANCO_BEL => "FRANCO BEL"
    | TipMonType_FRANCO_FR => "FRANCO FR"
    | TipMonType_FRANCO_SZ => "FRANCO SZ"
    | TipMonType_GUARANI => "GUARANI"
    | TipMonType_LIBRA_EST => "LIBRA EST"
    | TipMonType_LIRA => "LIRA"
    | TipMonType_MARCO_AL => "MARCO AL"
    | TipMonType_MARCO_FIN => "MARCO FIN"
    | TipMonType_NUEVO_SOL => "NUEVO SOL"
    | TipMonType_OTRAS_MONEDAS => "OTRAS MONEDAS"
    | TipMonType_PESETA => "PESETA"
    | TipMonType_PESO => "PESO"
    | TipMonType_PESO_CL => "PESO CL"
    | TipMonType_PESO_COL => "PESO COL"
    | TipMonType_PESO_MEX => "PESO MEX"
    | TipMonType_PESO_URUG => "PESO URUG"
    | TipMonType_RAND => "RAND"
    | TipMonType_RENMINBI => "RENMINBI"
    | TipMonType_RUPIA => "RUPIA"
    | TipMonType_SUCRE => "SUCRE"
    | TipMonType_YEN => "YEN"

instance to_text x_tip_mon_type = fun x => x_tip_mon_type_to_string x

inl x_tip_mon_type_example (k : i32) : x_tip_mon_type =
    inl index = ((k % 8i32) + 8i32) % 8i32
    if index = 0i32
    then TipMonType_BOLIVAR
    elif index = 1i32
    then TipMonType_BOLIVIANO
    elif index = 2i32
    then TipMonType_CHELIN
    elif index = 3i32
    then TipMonType_CORONA_DIN
    elif index = 4i32
    then TipMonType_CORONA_NOR
    elif index = 5i32
    then TipMonType_CORONA_SC
    elif index = 6i32
    then TipMonType_CRUZEIRO_REAL
    else TipMonType_DIRHAM

inl x_tip_mon_type_example_text (k : i32) : string = x_tip_mon_type_to_string (x_tip_mon_type_example k)

union x_medio_pago_type =
    | MedioPagoType_CH
    | MedioPagoType_LT
    | MedioPagoType_EF
    | MedioPagoType_PE
    | MedioPagoType_TC
    | MedioPagoType_CF
    | MedioPagoType_OT

inl x_medio_pago_type_of_string (s : string) : option x_medio_pago_type =
    match s with
    | "CH" => Some MedioPagoType_CH
    | "LT" => Some MedioPagoType_LT
    | "EF" => Some MedioPagoType_EF
    | "PE" => Some MedioPagoType_PE
    | "TC" => Some MedioPagoType_TC
    | "CF" => Some MedioPagoType_CF
    | "OT" => Some MedioPagoType_OT
    | _ => None

inl x_medio_pago_type_to_string (x : x_medio_pago_type) : string =
    match x with
    | MedioPagoType_CH => "CH"
    | MedioPagoType_LT => "LT"
    | MedioPagoType_EF => "EF"
    | MedioPagoType_PE => "PE"
    | MedioPagoType_TC => "TC"
    | MedioPagoType_CF => "CF"
    | MedioPagoType_OT => "OT"

instance to_text x_medio_pago_type = fun x => x_medio_pago_type_to_string x

inl x_medio_pago_type_example (k : i32) : x_medio_pago_type =
    inl index = ((k % 7i32) + 7i32) % 7i32
    if index = 0i32
    then MedioPagoType_CH
    elif index = 1i32
    then MedioPagoType_LT
    elif index = 2i32
    then MedioPagoType_EF
    elif index = 3i32
    then MedioPagoType_PE
    elif index = 4i32
    then MedioPagoType_TC
    elif index = 5i32
    then MedioPagoType_CF
    else MedioPagoType_OT

inl x_medio_pago_type_example_text (k : i32) : string = x_medio_pago_type_to_string (x_medio_pago_type_example k)

union x_exp_type =
    | EXPType_110
    | EXPType_111
    | EXPType_112

inl x_exp_type_of_string (s : string) : option x_exp_type =
    inl s = xml.collapsed_space s
    match s with
    | "110" => Some EXPType_110
    | "111" => Some EXPType_111
    | "112" => Some EXPType_112
    | _ => None

inl x_exp_type_to_string (x : x_exp_type) : string =
    match x with
    | EXPType_110 => "110"
    | EXPType_111 => "111"
    | EXPType_112 => "112"

instance to_text x_exp_type = fun x => x_exp_type_to_string x

inl x_exp_type_example (k : i32) : x_exp_type =
    inl index = ((k % 3i32) + 3i32) % 3i32
    if index = 0i32
    then EXPType_110
    elif index = 1i32
    then EXPType_111
    else EXPType_112

inl x_exp_type_example_text (k : i32) : string = x_exp_type_to_string (x_exp_type_example k)

union x_liq_type =
    | LIQType_43

inl x_liq_type_of_string (s : string) : option x_liq_type =
    inl s = xml.collapsed_space s
    match s with
    | "43" => Some LIQType_43
    | _ => None

inl x_liq_type_to_string (x : x_liq_type) : string =
    match x with
    | LIQType_43 => "43"

instance to_text x_liq_type = fun x => x_liq_type_to_string x

inl x_liq_type_example (k : i32) : x_liq_type =
    inl index = ((k % 1i32) + 1i32) % 1i32
    LIQType_43

inl x_liq_type_example_text (k : i32) : string = x_liq_type_to_string (x_liq_type_example k)

nominal private x_nombre_type = string

inl x_nombre_type_is_valid (s : string) : bool = xml.code_point_count s <= 40

inl x_nombre_type_of_string (s : string) : option x_nombre_type =
    if x_nombre_type_is_valid s then Some (x_nombre_type s) else None

inl x_nombre_type_to_string (x : x_nombre_type) : string = match x with x_nombre_type s => s

instance to_text x_nombre_type = fun x => x_nombre_type_to_string x
inl x_nombre_type_example (_ : i32) : x_nombre_type = x_nombre_type "1"

inl x_nombre_type_example_text (k : i32) : string = x_nombre_type_to_string (x_nombre_type_example k)

nominal private x_fono_type = string

inl x_fono_type_is_valid (s : string) : bool = xml.code_point_count s <= 40

inl x_fono_type_of_string (s : string) : option x_fono_type =
    if x_fono_type_is_valid s then Some (x_fono_type s) else None

inl x_fono_type_to_string (x : x_fono_type) : string = match x with x_fono_type s => s

instance to_text x_fono_type = fun x => x_fono_type_to_string x
inl x_fono_type_example (_ : i32) : x_fono_type = x_fono_type "1"

inl x_fono_type_example_text (k : i32) : string = x_fono_type_to_string (x_fono_type_example k)

nominal private x_ciudad_type = string

inl x_ciudad_type_is_valid (s : string) : bool = xml.code_point_count s <= 20

inl x_ciudad_type_of_string (s : string) : option x_ciudad_type =
    if x_ciudad_type_is_valid s then Some (x_ciudad_type s) else None

inl x_ciudad_type_to_string (x : x_ciudad_type) : string = match x with x_ciudad_type s => s

instance to_text x_ciudad_type = fun x => x_ciudad_type_to_string x
inl x_ciudad_type_example (_ : i32) : x_ciudad_type = x_ciudad_type "1"

inl x_ciudad_type_example_text (k : i32) : string = x_ciudad_type_to_string (x_ciudad_type_example k)

nominal private x_comuna_type = string

inl x_comuna_type_is_valid (s : string) : bool = xml.code_point_count s <= 20

inl x_comuna_type_of_string (s : string) : option x_comuna_type =
    if x_comuna_type_is_valid s then Some (x_comuna_type s) else None

inl x_comuna_type_to_string (x : x_comuna_type) : string = match x with x_comuna_type s => s

instance to_text x_comuna_type = fun x => x_comuna_type_to_string x
inl x_comuna_type_example (_ : i32) : x_comuna_type = x_comuna_type "1"

inl x_comuna_type_example_text (k : i32) : string = x_comuna_type_to_string (x_comuna_type_example k)

nominal private x_direcc_type = string

inl x_direcc_type_is_valid (s : string) : bool = xml.code_point_count s <= 80

inl x_direcc_type_of_string (s : string) : option x_direcc_type =
    if x_direcc_type_is_valid s then Some (x_direcc_type s) else None

inl x_direcc_type_to_string (x : x_direcc_type) : string = match x with x_direcc_type s => s

instance to_text x_direcc_type = fun x => x_direcc_type_to_string x
inl x_direcc_type_example (_ : i32) : x_direcc_type = x_direcc_type "1"

inl x_direcc_type_example_text (k : i32) : string = x_direcc_type_to_string (x_direcc_type_example k)

nominal private x_direcc_solo_dte_type = string

inl x_direcc_solo_dte_type_is_valid (s : string) : bool = xml.code_point_count s <= 60

inl x_direcc_solo_dte_type_of_string (s : string) : option x_direcc_solo_dte_type =
    if x_direcc_solo_dte_type_is_valid s then Some (x_direcc_solo_dte_type s) else None

inl x_direcc_solo_dte_type_to_string (x : x_direcc_solo_dte_type) : string = match x with x_direcc_solo_dte_type s => s

instance to_text x_direcc_solo_dte_type = fun x => x_direcc_solo_dte_type_to_string x
inl x_direcc_solo_dte_type_example (_ : i32) : x_direcc_solo_dte_type = x_direcc_solo_dte_type "1"

inl x_direcc_solo_dte_type_example_text (k : i32) : string =
    x_direcc_solo_dte_type_to_string (x_direcc_solo_dte_type_example k)

nominal private x_rzn_soc_corta_type = string

inl x_rzn_soc_corta_type_is_valid (s : string) : bool = xml.code_point_count s <= 40 && xml.code_point_count s >= 1

inl x_rzn_soc_corta_type_of_string (s : string) : option x_rzn_soc_corta_type =
    if x_rzn_soc_corta_type_is_valid s then Some (x_rzn_soc_corta_type s) else None

inl x_rzn_soc_corta_type_to_string (x : x_rzn_soc_corta_type) : string = match x with x_rzn_soc_corta_type s => s

instance to_text x_rzn_soc_corta_type = fun x => x_rzn_soc_corta_type_to_string x
inl x_rzn_soc_corta_type_example (_ : i32) : x_rzn_soc_corta_type = x_rzn_soc_corta_type "1"

inl x_rzn_soc_corta_type_example_text (k : i32) : string =
    x_rzn_soc_corta_type_to_string (x_rzn_soc_corta_type_example k)

nominal private x_rzn_soc_larga_type = string

inl x_rzn_soc_larga_type_is_valid (s : string) : bool = xml.code_point_count s <= 100

inl x_rzn_soc_larga_type_of_string (s : string) : option x_rzn_soc_larga_type =
    if x_rzn_soc_larga_type_is_valid s then Some (x_rzn_soc_larga_type s) else None

inl x_rzn_soc_larga_type_to_string (x : x_rzn_soc_larga_type) : string = match x with x_rzn_soc_larga_type s => s

instance to_text x_rzn_soc_larga_type = fun x => x_rzn_soc_larga_type_to_string x
inl x_rzn_soc_larga_type_example (_ : i32) : x_rzn_soc_larga_type = x_rzn_soc_larga_type "1"

inl x_rzn_soc_larga_type_example_text (k : i32) : string =
    x_rzn_soc_larga_type_to_string (x_rzn_soc_larga_type_example k)

type x_nro_resol_type = string

inl x_nro_resol_type_is_valid (s : string) : bool = true

inl x_nro_resol_type_example (_ : i32) : string = "1"

inl x_nro_resol_type_example_text (k : i32) : string = x_nro_resol_type_example k

union x_dinero_porcentaje_type =
    | DineroPorcentajeType__
    | DineroPorcentajeType___

inl x_dinero_porcentaje_type_of_string (s : string) : option x_dinero_porcentaje_type =
    match s with
    | "%" => Some DineroPorcentajeType__
    | "$" => Some DineroPorcentajeType___
    | _ => None

inl x_dinero_porcentaje_type_to_string (x : x_dinero_porcentaje_type) : string =
    match x with
    | DineroPorcentajeType__ => "%"
    | DineroPorcentajeType___ => "$"

instance to_text x_dinero_porcentaje_type = fun x => x_dinero_porcentaje_type_to_string x

inl x_dinero_porcentaje_type_example (k : i32) : x_dinero_porcentaje_type =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then DineroPorcentajeType__
    else DineroPorcentajeType___

inl x_dinero_porcentaje_type_example_text (k : i32) : string =
    x_dinero_porcentaje_type_to_string (x_dinero_porcentaje_type_example k)

nominal private x_mail_type = string

inl x_mail_type_is_valid (s : string) : bool = xml.code_point_count s <= 80

inl x_mail_type_of_string (s : string) : option x_mail_type =
    if x_mail_type_is_valid s then Some (x_mail_type s) else None

inl x_mail_type_to_string (x : x_mail_type) : string = match x with x_mail_type s => s

instance to_text x_mail_type = fun x => x_mail_type_to_string x
inl x_mail_type_example (_ : i32) : x_mail_type = x_mail_type "1"

inl x_mail_type_example_text (k : i32) : string = x_mail_type_to_string (x_mail_type_example k)

union x_doc_type =
    | DocType_33
    | DocType_34
    | DocType_46
    | DocType_52
    | DocType_56
    | DocType_61

inl x_doc_type_of_string (s : string) : option x_doc_type =
    inl s = xml.collapsed_space s
    match s with
    | "33" => Some DocType_33
    | "34" => Some DocType_34
    | "46" => Some DocType_46
    | "52" => Some DocType_52
    | "56" => Some DocType_56
    | "61" => Some DocType_61
    | _ => None

inl x_doc_type_to_string (x : x_doc_type) : string =
    match x with
    | DocType_33 => "33"
    | DocType_34 => "34"
    | DocType_46 => "46"
    | DocType_52 => "52"
    | DocType_56 => "56"
    | DocType_61 => "61"

instance to_text x_doc_type = fun x => x_doc_type_to_string x

inl x_doc_type_example (k : i32) : x_doc_type =
    inl index = ((k % 6i32) + 6i32) % 6i32
    if index = 0i32
    then DocType_33
    elif index = 1i32
    then DocType_34
    elif index = 2i32
    then DocType_46
    elif index = 3i32
    then DocType_52
    elif index = 4i32
    then DocType_56
    else DocType_61

inl x_doc_type_example_text (k : i32) : string = x_doc_type_to_string (x_doc_type_example k)

type x_monto_type = string

inl x_monto_type_is_valid (s : string) : bool = true

inl x_monto_type_example (_ : i32) : string = "1"

inl x_monto_type_example_text (k : i32) : string = x_monto_type_example k

union x_imp_adic_dte_type =
    | ImpAdicDTEType_14
    | ImpAdicDTEType_15
    | ImpAdicDTEType_16
    | ImpAdicDTEType_17
    | ImpAdicDTEType_18
    | ImpAdicDTEType_19
    | ImpAdicDTEType_23
    | ImpAdicDTEType_24
    | ImpAdicDTEType_25
    | ImpAdicDTEType_26
    | ImpAdicDTEType_27
    | ImpAdicDTEType_28
    | ImpAdicDTEType_30
    | ImpAdicDTEType_31
    | ImpAdicDTEType_32
    | ImpAdicDTEType_33
    | ImpAdicDTEType_34
    | ImpAdicDTEType_35
    | ImpAdicDTEType_36
    | ImpAdicDTEType_37
    | ImpAdicDTEType_38
    | ImpAdicDTEType_39
    | ImpAdicDTEType_40
    | ImpAdicDTEType_41
    | ImpAdicDTEType_44
    | ImpAdicDTEType_45
    | ImpAdicDTEType_46
    | ImpAdicDTEType_47
    | ImpAdicDTEType_48
    | ImpAdicDTEType_49
    | ImpAdicDTEType_50
    | ImpAdicDTEType_51
    | ImpAdicDTEType_52
    | ImpAdicDTEType_53
    | ImpAdicDTEType_54
    | ImpAdicDTEType_55
    | ImpAdicDTEType_271
    | ImpAdicDTEType_301
    | ImpAdicDTEType_321
    | ImpAdicDTEType_331
    | ImpAdicDTEType_341
    | ImpAdicDTEType_361
    | ImpAdicDTEType_371
    | ImpAdicDTEType_481

inl x_imp_adic_dte_type_of_string (s : string) : option x_imp_adic_dte_type =
    match s with
    | "14" => Some ImpAdicDTEType_14
    | "15" => Some ImpAdicDTEType_15
    | "16" => Some ImpAdicDTEType_16
    | "17" => Some ImpAdicDTEType_17
    | "18" => Some ImpAdicDTEType_18
    | "19" => Some ImpAdicDTEType_19
    | "23" => Some ImpAdicDTEType_23
    | "24" => Some ImpAdicDTEType_24
    | "25" => Some ImpAdicDTEType_25
    | "26" => Some ImpAdicDTEType_26
    | "27" => Some ImpAdicDTEType_27
    | "28" => Some ImpAdicDTEType_28
    | "30" => Some ImpAdicDTEType_30
    | "31" => Some ImpAdicDTEType_31
    | "32" => Some ImpAdicDTEType_32
    | "33" => Some ImpAdicDTEType_33
    | "34" => Some ImpAdicDTEType_34
    | "35" => Some ImpAdicDTEType_35
    | "36" => Some ImpAdicDTEType_36
    | "37" => Some ImpAdicDTEType_37
    | "38" => Some ImpAdicDTEType_38
    | "39" => Some ImpAdicDTEType_39
    | "40" => Some ImpAdicDTEType_40
    | "41" => Some ImpAdicDTEType_41
    | "44" => Some ImpAdicDTEType_44
    | "45" => Some ImpAdicDTEType_45
    | "46" => Some ImpAdicDTEType_46
    | "47" => Some ImpAdicDTEType_47
    | "48" => Some ImpAdicDTEType_48
    | "49" => Some ImpAdicDTEType_49
    | "50" => Some ImpAdicDTEType_50
    | "51" => Some ImpAdicDTEType_51
    | "52" => Some ImpAdicDTEType_52
    | "53" => Some ImpAdicDTEType_53
    | "54" => Some ImpAdicDTEType_54
    | "55" => Some ImpAdicDTEType_55
    | "271" => Some ImpAdicDTEType_271
    | "301" => Some ImpAdicDTEType_301
    | "321" => Some ImpAdicDTEType_321
    | "331" => Some ImpAdicDTEType_331
    | "341" => Some ImpAdicDTEType_341
    | "361" => Some ImpAdicDTEType_361
    | "371" => Some ImpAdicDTEType_371
    | "481" => Some ImpAdicDTEType_481
    | _ => None

inl x_imp_adic_dte_type_to_string (x : x_imp_adic_dte_type) : string =
    match x with
    | ImpAdicDTEType_14 => "14"
    | ImpAdicDTEType_15 => "15"
    | ImpAdicDTEType_16 => "16"
    | ImpAdicDTEType_17 => "17"
    | ImpAdicDTEType_18 => "18"
    | ImpAdicDTEType_19 => "19"
    | ImpAdicDTEType_23 => "23"
    | ImpAdicDTEType_24 => "24"
    | ImpAdicDTEType_25 => "25"
    | ImpAdicDTEType_26 => "26"
    | ImpAdicDTEType_27 => "27"
    | ImpAdicDTEType_28 => "28"
    | ImpAdicDTEType_30 => "30"
    | ImpAdicDTEType_31 => "31"
    | ImpAdicDTEType_32 => "32"
    | ImpAdicDTEType_33 => "33"
    | ImpAdicDTEType_34 => "34"
    | ImpAdicDTEType_35 => "35"
    | ImpAdicDTEType_36 => "36"
    | ImpAdicDTEType_37 => "37"
    | ImpAdicDTEType_38 => "38"
    | ImpAdicDTEType_39 => "39"
    | ImpAdicDTEType_40 => "40"
    | ImpAdicDTEType_41 => "41"
    | ImpAdicDTEType_44 => "44"
    | ImpAdicDTEType_45 => "45"
    | ImpAdicDTEType_46 => "46"
    | ImpAdicDTEType_47 => "47"
    | ImpAdicDTEType_48 => "48"
    | ImpAdicDTEType_49 => "49"
    | ImpAdicDTEType_50 => "50"
    | ImpAdicDTEType_51 => "51"
    | ImpAdicDTEType_52 => "52"
    | ImpAdicDTEType_53 => "53"
    | ImpAdicDTEType_54 => "54"
    | ImpAdicDTEType_55 => "55"
    | ImpAdicDTEType_271 => "271"
    | ImpAdicDTEType_301 => "301"
    | ImpAdicDTEType_321 => "321"
    | ImpAdicDTEType_331 => "331"
    | ImpAdicDTEType_341 => "341"
    | ImpAdicDTEType_361 => "361"
    | ImpAdicDTEType_371 => "371"
    | ImpAdicDTEType_481 => "481"

instance to_text x_imp_adic_dte_type = fun x => x_imp_adic_dte_type_to_string x

inl x_imp_adic_dte_type_example (k : i32) : x_imp_adic_dte_type =
    inl index = ((k % 8i32) + 8i32) % 8i32
    if index = 0i32
    then ImpAdicDTEType_14
    elif index = 1i32
    then ImpAdicDTEType_15
    elif index = 2i32
    then ImpAdicDTEType_16
    elif index = 3i32
    then ImpAdicDTEType_17
    elif index = 4i32
    then ImpAdicDTEType_18
    elif index = 5i32
    then ImpAdicDTEType_19
    elif index = 6i32
    then ImpAdicDTEType_23
    else ImpAdicDTEType_24

inl x_imp_adic_dte_type_example_text (k : i32) : string = x_imp_adic_dte_type_to_string (x_imp_adic_dte_type_example k)

union x_imp_adic_type =
    | ImpAdicType_14
    | ImpAdicType_15
    | ImpAdicType_16
    | ImpAdicType_17
    | ImpAdicType_18
    | ImpAdicType_19
    | ImpAdicType_23
    | ImpAdicType_24
    | ImpAdicType_25
    | ImpAdicType_26
    | ImpAdicType_27
    | ImpAdicType_28
    | ImpAdicType_29
    | ImpAdicType_30
    | ImpAdicType_31
    | ImpAdicType_32
    | ImpAdicType_33
    | ImpAdicType_34
    | ImpAdicType_35
    | ImpAdicType_36
    | ImpAdicType_37
    | ImpAdicType_38
    | ImpAdicType_39
    | ImpAdicType_40
    | ImpAdicType_41
    | ImpAdicType_44
    | ImpAdicType_45
    | ImpAdicType_46
    | ImpAdicType_47
    | ImpAdicType_48
    | ImpAdicType_49
    | ImpAdicType_50
    | ImpAdicType_51
    | ImpAdicType_52
    | ImpAdicType_53
    | ImpAdicType_54
    | ImpAdicType_55
    | ImpAdicType_271
    | ImpAdicType_301
    | ImpAdicType_321
    | ImpAdicType_331
    | ImpAdicType_341
    | ImpAdicType_361
    | ImpAdicType_371
    | ImpAdicType_481

inl x_imp_adic_type_of_string (s : string) : option x_imp_adic_type =
    match s with
    | "14" => Some ImpAdicType_14
    | "15" => Some ImpAdicType_15
    | "16" => Some ImpAdicType_16
    | "17" => Some ImpAdicType_17
    | "18" => Some ImpAdicType_18
    | "19" => Some ImpAdicType_19
    | "23" => Some ImpAdicType_23
    | "24" => Some ImpAdicType_24
    | "25" => Some ImpAdicType_25
    | "26" => Some ImpAdicType_26
    | "27" => Some ImpAdicType_27
    | "28" => Some ImpAdicType_28
    | "29" => Some ImpAdicType_29
    | "30" => Some ImpAdicType_30
    | "31" => Some ImpAdicType_31
    | "32" => Some ImpAdicType_32
    | "33" => Some ImpAdicType_33
    | "34" => Some ImpAdicType_34
    | "35" => Some ImpAdicType_35
    | "36" => Some ImpAdicType_36
    | "37" => Some ImpAdicType_37
    | "38" => Some ImpAdicType_38
    | "39" => Some ImpAdicType_39
    | "40" => Some ImpAdicType_40
    | "41" => Some ImpAdicType_41
    | "44" => Some ImpAdicType_44
    | "45" => Some ImpAdicType_45
    | "46" => Some ImpAdicType_46
    | "47" => Some ImpAdicType_47
    | "48" => Some ImpAdicType_48
    | "49" => Some ImpAdicType_49
    | "50" => Some ImpAdicType_50
    | "51" => Some ImpAdicType_51
    | "52" => Some ImpAdicType_52
    | "53" => Some ImpAdicType_53
    | "54" => Some ImpAdicType_54
    | "55" => Some ImpAdicType_55
    | "271" => Some ImpAdicType_271
    | "301" => Some ImpAdicType_301
    | "321" => Some ImpAdicType_321
    | "331" => Some ImpAdicType_331
    | "341" => Some ImpAdicType_341
    | "361" => Some ImpAdicType_361
    | "371" => Some ImpAdicType_371
    | "481" => Some ImpAdicType_481
    | _ => None

inl x_imp_adic_type_to_string (x : x_imp_adic_type) : string =
    match x with
    | ImpAdicType_14 => "14"
    | ImpAdicType_15 => "15"
    | ImpAdicType_16 => "16"
    | ImpAdicType_17 => "17"
    | ImpAdicType_18 => "18"
    | ImpAdicType_19 => "19"
    | ImpAdicType_23 => "23"
    | ImpAdicType_24 => "24"
    | ImpAdicType_25 => "25"
    | ImpAdicType_26 => "26"
    | ImpAdicType_27 => "27"
    | ImpAdicType_28 => "28"
    | ImpAdicType_29 => "29"
    | ImpAdicType_30 => "30"
    | ImpAdicType_31 => "31"
    | ImpAdicType_32 => "32"
    | ImpAdicType_33 => "33"
    | ImpAdicType_34 => "34"
    | ImpAdicType_35 => "35"
    | ImpAdicType_36 => "36"
    | ImpAdicType_37 => "37"
    | ImpAdicType_38 => "38"
    | ImpAdicType_39 => "39"
    | ImpAdicType_40 => "40"
    | ImpAdicType_41 => "41"
    | ImpAdicType_44 => "44"
    | ImpAdicType_45 => "45"
    | ImpAdicType_46 => "46"
    | ImpAdicType_47 => "47"
    | ImpAdicType_48 => "48"
    | ImpAdicType_49 => "49"
    | ImpAdicType_50 => "50"
    | ImpAdicType_51 => "51"
    | ImpAdicType_52 => "52"
    | ImpAdicType_53 => "53"
    | ImpAdicType_54 => "54"
    | ImpAdicType_55 => "55"
    | ImpAdicType_271 => "271"
    | ImpAdicType_301 => "301"
    | ImpAdicType_321 => "321"
    | ImpAdicType_331 => "331"
    | ImpAdicType_341 => "341"
    | ImpAdicType_361 => "361"
    | ImpAdicType_371 => "371"
    | ImpAdicType_481 => "481"

instance to_text x_imp_adic_type = fun x => x_imp_adic_type_to_string x

inl x_imp_adic_type_example (k : i32) : x_imp_adic_type =
    inl index = ((k % 8i32) + 8i32) % 8i32
    if index = 0i32
    then ImpAdicType_14
    elif index = 1i32
    then ImpAdicType_15
    elif index = 2i32
    then ImpAdicType_16
    elif index = 3i32
    then ImpAdicType_17
    elif index = 4i32
    then ImpAdicType_18
    elif index = 5i32
    then ImpAdicType_19
    elif index = 6i32
    then ImpAdicType_23
    else ImpAdicType_24

inl x_imp_adic_type_example_text (k : i32) : string = x_imp_adic_type_to_string (x_imp_adic_type_example k)

type x_dec12_6_type = string

inl x_dec12_6_type_is_valid (s : string) : bool = true

inl x_dec12_6_type_example (_ : i32) : string = "1"

inl x_dec12_6_type_example_text (k : i32) : string = x_dec12_6_type_example k

type x_pct_type = string

inl x_pct_type_is_valid (s : string) : bool = true

inl x_pct_type_example (_ : i32) : string = "1"

inl x_pct_type_example_text (k : i32) : string = x_pct_type_example k

type x_dec6_4_type = string

inl x_dec6_4_type_is_valid (s : string) : bool = true

inl x_dec6_4_type_example (_ : i32) : string = "1"

inl x_dec6_4_type_example_text (k : i32) : string = x_dec6_4_type_example k

type x_dec8_4_type = string

inl x_dec8_4_type_is_valid (s : string) : bool = true

inl x_dec8_4_type_example (_ : i32) : string = "1"

inl x_dec8_4_type_example_text (k : i32) : string = x_dec8_4_type_example k

type x_dec14_4_type = string

inl x_dec14_4_type_is_valid (s : string) : bool = true

inl x_dec14_4_type_example (_ : i32) : string = "1"

inl x_dec14_4_type_example_text (k : i32) : string = x_dec14_4_type_example k

type x_dec16_2_type = string

inl x_dec16_2_type_is_valid (s : string) : bool = true

inl x_dec16_2_type_example (_ : i32) : string = "1"

inl x_dec16_2_type_example_text (k : i32) : string = x_dec16_2_type_example k

nominal private x_folio_r_type = string

inl x_folio_r_type_is_valid (s : string) : bool = xml.code_point_count s <= 18 && xml.code_point_count s >= 1

inl x_folio_r_type_of_string (s : string) : option x_folio_r_type =
    if x_folio_r_type_is_valid s then Some (x_folio_r_type s) else None

inl x_folio_r_type_to_string (x : x_folio_r_type) : string = match x with x_folio_r_type s => s

instance to_text x_folio_r_type = fun x => x_folio_r_type_to_string x
inl x_folio_r_type_example (_ : i32) : x_folio_r_type = x_folio_r_type "1"

inl x_folio_r_type_example_text (k : i32) : string = x_folio_r_type_to_string (x_folio_r_type_example k)

type x_folio_type = string

inl x_folio_type_is_valid (s : string) : bool = true

inl x_folio_type_example (_ : i32) : string = "1"

inl x_folio_type_example_text (k : i32) : string = x_folio_type_example k

type x_valor_type = string

inl x_valor_type_is_valid (s : string) : bool = true

inl x_valor_type_example (_ : i32) : string = "1"

inl x_valor_type_example_text (k : i32) : string = x_valor_type_example k

type x_mnt_imp_type = string

inl x_mnt_imp_type_is_valid (s : string) : bool = true

inl x_mnt_imp_type_example (_ : i32) : string = "1"

inl x_mnt_imp_type_example_text (k : i32) : string = x_mnt_imp_type_example k

let rec xsd_pattern_68c18fd1_16 (state : i32) (s : string) (i : i32) : bool =
    if i >= sm'.length s
    then state = 3i32
    else
        inl c = xml.code_point_at s i
        inl next = i + xml.code_point_width s i
        if state = 0i32 then
            if 48u32 <= c && c <= 57u32
            then xsd_pattern_68c18fd1_16 1i32 s next
            else false
        elif state = 1i32 then
            if c = 45u32
            then xsd_pattern_68c18fd1_16 2i32 s next
            elif 48u32 <= c && c <= 57u32
            then xsd_pattern_68c18fd1_16 1i32 s next
            else false
        elif state = 2i32 then
            if 48u32 <= c && c <= 57u32
            then xsd_pattern_68c18fd1_16 3i32 s next
            elif c = 75u32
            then xsd_pattern_68c18fd1_16 3i32 s next
            else false
        elif state = 3i32 then
            false
        else false

nominal private x_rut_type = string

inl x_rut_type_is_valid (s : string) : bool =
    (xsd_pattern_68c18fd1_16 0i32 s 0i32) && xml.code_point_count s <= 10 && xml.code_point_count s >= 3

inl x_rut_type_of_string (s : string) : option x_rut_type = if x_rut_type_is_valid s then Some (x_rut_type s) else None

inl x_rut_type_to_string (x : x_rut_type) : string = match x with x_rut_type s => s

instance to_text x_rut_type = fun x => x_rut_type_to_string x
inl x_rut_type_example (_ : i32) : x_rut_type = x_rut_type "0-0"

inl x_rut_type_example_text (k : i32) : string = x_rut_type_to_string (x_rut_type_example k)

union x_dte_facturas_type =
    | DTEFacturasType_33
    | DTEFacturasType_34
    | DTEFacturasType_46
    | DTEFacturasType_43

inl x_dte_facturas_type_of_string (s : string) : option x_dte_facturas_type =
    inl s = xml.collapsed_space s
    match s with
    | "33" => Some DTEFacturasType_33
    | "34" => Some DTEFacturasType_34
    | "46" => Some DTEFacturasType_46
    | "43" => Some DTEFacturasType_43
    | _ => None

inl x_dte_facturas_type_to_string (x : x_dte_facturas_type) : string =
    match x with
    | DTEFacturasType_33 => "33"
    | DTEFacturasType_34 => "34"
    | DTEFacturasType_46 => "46"
    | DTEFacturasType_43 => "43"

instance to_text x_dte_facturas_type = fun x => x_dte_facturas_type_to_string x

inl x_dte_facturas_type_example (k : i32) : x_dte_facturas_type =
    inl index = ((k % 4i32) + 4i32) % 4i32
    if index = 0i32
    then DTEFacturasType_33
    elif index = 1i32
    then DTEFacturasType_34
    elif index = 2i32
    then DTEFacturasType_46
    else DTEFacturasType_43

inl x_dte_facturas_type_example_text (k : i32) : string = x_dte_facturas_type_to_string (x_dte_facturas_type_example k)

type x_dte_type = x_doc_type

inl x_dte_type_of_string (s : string) : option x_dte_type = x_doc_type_of_string s

inl x_dte_type_to_string (x : x_dte_type) : string = x_doc_type_to_string x

inl x_dte_type_example (k : i32) : x_dte_type = x_doc_type_example k

inl x_dte_type_example_text (k : i32) : string = x_doc_type_example_text k

nominal x_signature_type_signed_info_canonicalization_method =
    {
        algorithm : string
    }

union x_signature_type_signed_info_signature_method_algorithm =
    | SignatureType_SignedInfo_SignatureMethod_Algorithm_rsa_sha1
    | SignatureType_SignedInfo_SignatureMethod_Algorithm_dsa_sha1

inl x_signature_type_signed_info_signature_method_algorithm_of_string
    (s : string)
    : option x_signature_type_signed_info_signature_method_algorithm
    =
    inl s = xml.collapsed_space s
    match s with
    | "http://www.w3.org/2000/09/xmldsig#rsa-sha1" => Some SignatureType_SignedInfo_SignatureMethod_Algorithm_rsa_sha1
    | "http://www.w3.org/2000/09/xmldsig#dsa-sha1" => Some SignatureType_SignedInfo_SignatureMethod_Algorithm_dsa_sha1
    | _ => None

inl x_signature_type_signed_info_signature_method_algorithm_to_string
    (x : x_signature_type_signed_info_signature_method_algorithm)
    : string
    =
    match x with
    | SignatureType_SignedInfo_SignatureMethod_Algorithm_rsa_sha1 => "http://www.w3.org/2000/09/xmldsig#rsa-sha1"
    | SignatureType_SignedInfo_SignatureMethod_Algorithm_dsa_sha1 => "http://www.w3.org/2000/09/xmldsig#dsa-sha1"

instance to_text x_signature_type_signed_info_signature_method_algorithm =
    fun x => x_signature_type_signed_info_signature_method_algorithm_to_string x

inl x_signature_type_signed_info_signature_method_algorithm_example
    (k : i32)
    : x_signature_type_signed_info_signature_method_algorithm
    =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then SignatureType_SignedInfo_SignatureMethod_Algorithm_rsa_sha1
    else SignatureType_SignedInfo_SignatureMethod_Algorithm_dsa_sha1

inl x_signature_type_signed_info_signature_method_algorithm_example_text (k : i32) : string =
    x_signature_type_signed_info_signature_method_algorithm_to_string (
        x_signature_type_signed_info_signature_method_algorithm_example k
    )

nominal x_signature_type_signed_info_signature_method =
    {
        algorithm : x_signature_type_signed_info_signature_method_algorithm
    }

nominal x_signature_type_signed_info_reference_transforms_transform =
    {
        algorithm : string
    }

nominal x_signature_type_signed_info_reference_transforms =
    {
        transform : x_signature_type_signed_info_reference_transforms_transform
    }

nominal x_signature_type_signed_info_reference_digest_method =
    {
        algorithm : string
    }

nominal x_signature_type_signed_info_reference =
    {
        transforms : option (x_signature_type_signed_info_reference_transforms)
        digest_method : x_signature_type_signed_info_reference_digest_method
        digest_value : string
        uri : string
    }

nominal x_signature_type_signed_info =
    {
        canonicalization_method : x_signature_type_signed_info_canonicalization_method
        signature_method : x_signature_type_signed_info_signature_method
        reference : x_signature_type_signed_info_reference
    }

nominal x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value =
    {
        modulus : string
        exponent : string
    }

nominal x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value =
    {
        p : string
        q : string
        g : string
        y : string
    }

union x_signature_type_key_info_key_value_choice_rsa_key_value =
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue :
        x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue :
        x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value

nominal x_signature_type_key_info_key_value =
    {
        choice_rsa_key_value : x_signature_type_key_info_key_value_choice_rsa_key_value
    }

nominal x_signature_type_key_info_x509_data =
    {
        x509_certificate : string
    }

nominal x_signature_type_key_info =
    {
        key_value : x_signature_type_key_info_key_value
        x509_data : x_signature_type_key_info_x509_data
    }

nominal x_signature_type =
    {
        signed_info : x_signature_type_signed_info
        signature_value : string
        key_info : x_signature_type_key_info
    }

union x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja =
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndNoRebaja_1

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja
    =
    inl s = xml.collapsed_space s
    match s with
    | "1" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndNoRebaja_1
    | _ => None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndNoRebaja_1 => "1"

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja
    =
    inl index = ((k % 1i32) + 1i32) % 1i32
    DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndNoRebaja_1

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example k
    )

union x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho =
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_1
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_2
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_3

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho
    =
    inl s = xml.collapsed_space s
    match s with
    | "1" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_1
    | "2" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_2
    | "3" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_3
    | _ => None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_1 => "1"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_2 => "2"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_3 => "3"

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho
    =
    inl index = ((k % 3i32) + 3i32) % 3i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_1
    elif index = 1i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_2
    else DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TipoDespacho_3

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k
    )

union x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado =
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_1
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_2
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_3
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_4
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_5
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_6
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_7
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_8
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_9

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado
    =
    inl s = xml.collapsed_space s
    match s with
    | "1" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_1
    | "2" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_2
    | "3" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_3
    | "4" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_4
    | "5" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_5
    | "6" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_6
    | "7" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_7
    | "8" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_8
    | "9" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_9
    | _ => None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_1 => "1"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_2 => "2"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_3 => "3"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_4 => "4"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_5 => "5"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_6 => "6"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_7 => "7"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_8 => "8"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_9 => "9"

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado
    =
    inl index = ((k % 8i32) + 8i32) % 8i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_1
    elif index = 1i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_2
    elif index = 2i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_3
    elif index = 3i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_4
    elif index = 4i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_5
    elif index = 5i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_6
    elif index = 6i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_7
    else DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_IndTraslado_8

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_example k
    )

union x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion =
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_N
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_T

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion
    =
    match s with
    | "N" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_N
    | "T" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_T
    | _ => None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_N => "N"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_T => "T"

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion
    =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_N
    else DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoImpresion_T

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

type x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_of_string s

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example k

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example_text k

type x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

type x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pag_exp = string

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pag_exp_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pag_exp_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pag_exp_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pag_exp_example k

nominal private x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos = string

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos
    =
    if x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos "1"

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_example k
    )

nominal x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos =
    {
        fch_pago : x_fecha_type
        mnt_pago : x_monto_type
        glosa_pagos : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos)
    }

union x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago =
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_AHORRO
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_CORRIENTE
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_VISTA

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago
    =
    match s with
    | "AHORRO" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_AHORRO
    | "CORRIENTE" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_CORRIENTE
    | "VISTA" => Some DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_VISTA
    | _ => None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_AHORRO => "AHORRO"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_CORRIENTE => "CORRIENTE"
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_VISTA => "VISTA"

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago
    =
    inl index = ((k % 3i32) + 3i32) % 3i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_AHORRO
    elif index = 1i32
    then DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_CORRIENTE
    else DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_TpoCtaPago_VISTA

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago = string

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago
    =
    if x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago "1"

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago = string

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago
    =
    if x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago "1"

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg = string

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg
    =
    if x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg "1"

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa = string

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_is_valid (s : string) : bool =
    xml.code_point_count s <= 100

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa
    =
    if x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa "1"

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_dias = string

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_dias_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_dias_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_dias_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_dias_example k

nominal x_dte_def_type_choice_documento_documento_encabezado_id_doc =
    {
        tipo_dte : x_dte_type
        folio : x_folio_type
        fch_emis : x_fecha_type
        ind_no_rebaja : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja)
        tipo_despacho : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho)
        ind_traslado : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado)
        tpo_impresion : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion)
        ind_servicio : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio)
        mnt_bruto : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto)
        fma_pago : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago)
        fma_pag_exp : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pag_exp)
        fch_cancel : option (x_fecha_type)
        mnt_cancel : option (x_monto_type)
        saldo_insol : option (x_monto_type)
        mnt_pagos : list (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos)
        periodo_desde : option (x_fecha_type)
        periodo_hasta : option (x_fecha_type)
        medio_pago : option (x_medio_pago_type)
        tpo_cta_pago : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago)
        num_cta_pago : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago)
        bco_pago : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago)
        term_pago_cdg : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg)
        term_pago_glosa : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa)
        term_pago_dias : option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_dias)
        fch_venc : option (x_fecha_type)
    }

type x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc_example (k : i32) : string =
    x_rzn_soc_larga_type_example_text k

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc_example k

nominal private x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_is_valid (s : string) : bool =
    xml.code_point_count s <= 80 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis
    =
    if x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono
    =
    if x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco_example k

union x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado =
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_1
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_2
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_3
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_4

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado
    =
    inl s = xml.collapsed_space s
    match s with
    | "1" => Some DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_1
    | "2" => Some DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_2
    | "3" => Some DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_3
    | "4" => Some DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_4
    | _ => None

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_1 => "1"
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_2 => "2"
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_3 => "3"
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_4 => "4"

instance to_text x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_to_string x

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado
    =
    inl index = ((k % 4i32) + 4i32) % 4i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_1
    elif index = 1i32
    then DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_2
    elif index = 2i32
    then DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_3
    else DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_CdgTraslado_4

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_example k
    )

nominal x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export =
    {
        cdg_traslado : option (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado)
        folio_aut : option (x_nro_resol_type)
        fch_aut : option (x_fecha_type)
    }

nominal private x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal
    =
    if x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_sii_sucur = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_sii_sucur_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_sii_sucur_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_sii_sucur_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_sii_sucur_example k

nominal private x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen
    =
    if x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_is_valid (s : string) : bool =
    xml.code_point_count s <= 60

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor
    =
    if x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor = string

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor
    =
    if x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor "1"

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_example k
    )

nominal x_dte_def_type_choice_documento_documento_encabezado_emisor =
    {
        rut_emisor : x_rut_type
        rzn_soc : x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc
        giro_emis : x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis
        telefono : list (x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono)
        correo_emisor : option (x_mail_type)
        acteco : list (x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco)
        guia_export : option (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export)
        sucursal : option (x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal)
        cdg_sii_sucur : option (x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_sii_sucur)
        dir_origen : option (x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen)
        cmna_origen : option (x_comuna_type)
        ciudad_origen : option (x_ciudad_type)
        cdg_vendedor : option (x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor)
        id_adic_emisor : option (x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor)
    }

nominal private x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep = string

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep
    =
    if x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep "1"

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id = string

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id
    =
    if x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id "1"

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad = string

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_is_valid (s : string) : bool =
    xml.code_point_count s <= 3 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad
    =
    if x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad "1"

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_example k
    )

nominal x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero =
    {
        num_id : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id)
        nacionalidad : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad)
    }

nominal private x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep = string

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep
    =
    if x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep "1"

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto = string

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_is_valid (s : string) : bool =
    xml.code_point_count s <= 80

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto
    =
    if x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto "1"

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep = string

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep
    =
    if x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep "1"

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal = string

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal
    =
    if x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal "1"

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_example k
    )

nominal x_dte_def_type_choice_documento_documento_encabezado_receptor =
    {
        rut_recep : x_rut_type
        cdg_int_recep : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep)
        rzn_soc_recep : x_rzn_soc_larga_type
        extranjero : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero)
        giro_recep : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep)
        contacto : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto)
        correo_recep : option (x_mail_type)
        dir_recep : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep)
        cmna_recep : option (x_comuna_type)
        ciudad_recep : option (x_ciudad_type)
        dir_postal : option (x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal)
        cmna_postal : option (x_comuna_type)
        ciudad_postal : option (x_ciudad_type)
    }

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_patente = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_is_valid (s : string) : bool =
    xml.code_point_count s <= 8

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_patente
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_patente s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_patente)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_patente s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_patente =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_patente
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_patente "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_is_valid (s : string) : bool =
    xml.code_point_count s <= 30

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_example k
    )

nominal x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer =
    {
        rut_chofer : x_rut_type
        nombre_chofer : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer
    }

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_mod_venta = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_mod_venta_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_mod_venta_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_mod_venta_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_mod_venta_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_clau_venta = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_clau_venta_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_clau_venta_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_clau_venta_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_clau_venta_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_via_transp = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_via_transp_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_via_transp_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_via_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_via_transp_example k

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_embarque = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_embarque_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_embarque_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_embarque_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_embarque_example k

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_desemb = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_desemb_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_desemb_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_desemb_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_desemb_example k

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_example k
    )

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tara = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tara_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tara_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tara_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tara_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_med_tara = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_med_tara_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_med_tara_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_med_tara_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_med_tara_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_bruto = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_bruto_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_bruto_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_bruto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_bruto_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_bruto = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_bruto_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_bruto_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_bruto_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_bruto_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_neto = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_neto_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_neto_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_neto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_neto_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_neto = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_neto_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_neto_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_neto_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_neto_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_items = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_items_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_items_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_items_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_items_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_bultos = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_bultos_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_bultos_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_bultos_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_bultos_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cant_bultos = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cant_bultos_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cant_bultos_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cant_bultos_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cant_bultos_example k

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 255

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 25

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_example k
    )

nominal private x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello
    =
    if x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello s)
    else None

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_to_string
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello s => s

instance to_text x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello =
    fun x => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_to_string x
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_to_string (
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_example k
    )

nominal x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos =
    {
        cod_tpo_bultos : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos)
        cant_bultos : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cant_bultos)
        marcas : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas)
        id_container : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container)
        sello : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello)
        emisor_sello : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello)
    }

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_recep = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_recep_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_recep_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_recep_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_recep_example k

type x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_destin = string

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_destin_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_destin_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_destin_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_destin_example k

nominal x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana =
    {
        cod_mod_venta : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_mod_venta)
        cod_clau_venta : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_clau_venta)
        tot_clau_venta : option (x_dec16_2_type)
        cod_via_transp : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_via_transp)
        nombre_transp : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp)
        rut_cia_transp : option (x_rut_type)
        nom_cia_transp : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp)
        id_adic_transp : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp)
        booking : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking)
        operador : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador)
        cod_pto_embarque : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_embarque)
        id_adic_pto_emb : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb)
        cod_pto_desemb : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_desemb)
        id_adic_pto_desemb : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb)
        tara : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tara)
        cod_unid_med_tara : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_med_tara)
        peso_bruto : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_bruto)
        cod_unid_peso_bruto : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_bruto)
        peso_neto : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_neto)
        cod_unid_peso_neto : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_neto)
        tot_items : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_items)
        tot_bultos : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_bultos)
        tipo_bultos : list (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos)
        mnt_flete : option (x_dec14_4_type)
        mnt_seguro : option (x_dec14_4_type)
        cod_pais_recep : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_recep)
        cod_pais_destin : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_destin)
    }

nominal x_dte_def_type_choice_documento_documento_encabezado_transporte =
    {
        patente : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_patente)
        rut_trans : option (x_rut_type)
        chofer : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer)
        dir_dest : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest)
        cmna_dest : option (x_comuna_type)
        ciudad_dest : option (x_ciudad_type)
        aduana : option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana)
    }

type x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_tasa_imp = string

inl x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_tasa_imp_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_tasa_imp_example (k : i32) : string =
    x_pct_type_example_text k

inl x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_tasa_imp_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_tasa_imp_example k

nominal x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten =
    {
        tipo_imp : x_imp_adic_dte_type
        tasa_imp : option (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_tasa_imp)
        monto_imp : x_mnt_imp_type
    }

nominal x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones =
    {
        val_com_neto : option (x_monto_type)
        val_com_exe : option (x_monto_type)
        val_com_iva : option (x_monto_type)
    }

nominal x_dte_def_type_choice_documento_documento_encabezado_totales =
    {
        mnt_neto : option (x_monto_type)
        mnt_exe : option (x_monto_type)
        mnt_base : option (x_monto_type)
        mnt_margen_com : option (x_monto_type)
        tasa_iva : option (x_pct_type)
        iva : option (x_monto_type)
        iva_prop : option (x_monto_type)
        iva_terc : option (x_monto_type)
        impto_reten : list (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten)
        iva_no_ret : option (x_mnt_imp_type)
        cred_ec : option (x_monto_type)
        grnt_dep : option (x_monto_type)
        comisiones : option (x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones)
        mnt_total : x_monto_type
        monto_nf : option (x_valor_type)
        monto_periodo : option (x_valor_type)
        saldo_anterior : option (x_valor_type)
        vlr_pagar : option (x_valor_type)
    }

nominal x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda =
    {
        tipo_imp_otr_mnda : x_imp_adic_dte_type
        tasa_imp_otr_mnda : option (x_pct_type)
        vlr_imp_otr_mnda : x_dec14_4_type
    }

nominal x_dte_def_type_choice_documento_documento_encabezado_otra_moneda =
    {
        tpo_moneda : x_tip_mon_type
        tpo_cambio : option (x_dec6_4_type)
        mnt_neto_otr_mnda : option (x_dec14_4_type)
        mnt_exe_otr_mnda : option (x_dec14_4_type)
        mnt_fae_carne_otr_mnda : option (x_dec14_4_type)
        mnt_marg_com_otr_mnda : option (x_dec14_4_type)
        iva_otr_mnda : option (x_dec14_4_type)
        imp_ret_otr_mnda : list (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda)
        iva_no_ret_otr_mnda : option (x_dec14_4_type)
        mnt_tot_otr_mnda : x_dec14_4_type
    }

nominal x_dte_def_type_choice_documento_documento_encabezado =
    {
        id_doc : x_dte_def_type_choice_documento_documento_encabezado_id_doc
        emisor : x_dte_def_type_choice_documento_documento_encabezado_emisor
        rut_mandante : option (x_rut_type)
        receptor : x_dte_def_type_choice_documento_documento_encabezado_receptor
        rut_solicita : option (x_rut_type)
        transporte : option (x_dte_def_type_choice_documento_documento_encabezado_transporte)
        totales : x_dte_def_type_choice_documento_documento_encabezado_totales
        otra_moneda : option (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda)
    }

type x_dte_def_type_choice_documento_documento_detalle_nro_lin_det = string

inl x_dte_def_type_choice_documento_documento_detalle_nro_lin_det_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_detalle_nro_lin_det_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_detalle_nro_lin_det_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_nro_lin_det_example k

nominal private x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo = string

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_is_valid (s : string) : bool =
    xml.code_point_count s <= 10

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo
    =
    if x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo =
    fun x => x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo
    =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo "1"

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_to_string (
        x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_example k
    )

nominal private x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo = string

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_is_valid (s : string) : bool =
    xml.code_point_count s <= 35

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo
    =
    if x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo =
    fun x => x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo
    =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo "1"

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_to_string (
        x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_example k
    )

nominal x_dte_def_type_choice_documento_documento_detalle_cdg_item =
    {
        tpo_codigo : x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo
        vlr_codigo : x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo
    }

union x_dte_def_type_choice_documento_documento_detalle_ind_exe =
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_1
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_2
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_3
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_4
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_5
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_6

inl x_dte_def_type_choice_documento_documento_detalle_ind_exe_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_ind_exe
    =
    inl s = xml.collapsed_space s
    match s with
    | "1" => Some DTEDefType_choice_Documento_Documento_Detalle_IndExe_1
    | "2" => Some DTEDefType_choice_Documento_Documento_Detalle_IndExe_2
    | "3" => Some DTEDefType_choice_Documento_Documento_Detalle_IndExe_3
    | "4" => Some DTEDefType_choice_Documento_Documento_Detalle_IndExe_4
    | "5" => Some DTEDefType_choice_Documento_Documento_Detalle_IndExe_5
    | "6" => Some DTEDefType_choice_Documento_Documento_Detalle_IndExe_6
    | _ => None

inl x_dte_def_type_choice_documento_documento_detalle_ind_exe_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_ind_exe)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_1 => "1"
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_2 => "2"
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_3 => "3"
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_4 => "4"
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_5 => "5"
    | DTEDefType_choice_Documento_Documento_Detalle_IndExe_6 => "6"

instance to_text x_dte_def_type_choice_documento_documento_detalle_ind_exe =
    fun x => x_dte_def_type_choice_documento_documento_detalle_ind_exe_to_string x

inl x_dte_def_type_choice_documento_documento_detalle_ind_exe_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_ind_exe
    =
    inl index = ((k % 6i32) + 6i32) % 6i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_Detalle_IndExe_1
    elif index = 1i32
    then DTEDefType_choice_Documento_Documento_Detalle_IndExe_2
    elif index = 2i32
    then DTEDefType_choice_Documento_Documento_Detalle_IndExe_3
    elif index = 3i32
    then DTEDefType_choice_Documento_Documento_Detalle_IndExe_4
    elif index = 4i32
    then DTEDefType_choice_Documento_Documento_Detalle_IndExe_5
    else DTEDefType_choice_Documento_Documento_Detalle_IndExe_6

inl x_dte_def_type_choice_documento_documento_detalle_ind_exe_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_to_string (
        x_dte_def_type_choice_documento_documento_detalle_ind_exe_example k
    )

union x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente =
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_IndAgente_R

inl x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente
    =
    match s with
    | "R" => Some DTEDefType_choice_Documento_Documento_Detalle_Retenedor_IndAgente_R
    | _ => None

inl x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_IndAgente_R => "R"

instance to_text x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente =
    fun x => x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_to_string x

inl x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente
    =
    inl index = ((k % 1i32) + 1i32) % 1i32
    DTEDefType_choice_Documento_Documento_Detalle_Retenedor_IndAgente_R

inl x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_to_string (
        x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_example k
    )

nominal x_dte_def_type_choice_documento_documento_detalle_retenedor =
    {
        ind_agente : x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente
        mnt_base_faena : option (x_monto_type)
        mnt_marg_comer : option (x_monto_type)
        prc_cons_final : option (x_monto_type)
    }

nominal private x_dte_def_type_choice_documento_documento_detalle_nmb_item = string

inl x_dte_def_type_choice_documento_documento_detalle_nmb_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 80

inl x_dte_def_type_choice_documento_documento_detalle_nmb_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_nmb_item
    =
    if x_dte_def_type_choice_documento_documento_detalle_nmb_item_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_nmb_item s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_nmb_item_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_nmb_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_nmb_item s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_nmb_item =
    fun x => x_dte_def_type_choice_documento_documento_detalle_nmb_item_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_nmb_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_nmb_item
    =
    x_dte_def_type_choice_documento_documento_detalle_nmb_item "1"

inl x_dte_def_type_choice_documento_documento_detalle_nmb_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_nmb_item_to_string (
        x_dte_def_type_choice_documento_documento_detalle_nmb_item_example k
    )

nominal private x_dte_def_type_choice_documento_documento_detalle_dsc_item = string

inl x_dte_def_type_choice_documento_documento_detalle_dsc_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 1000

inl x_dte_def_type_choice_documento_documento_detalle_dsc_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_dsc_item
    =
    if x_dte_def_type_choice_documento_documento_detalle_dsc_item_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_dsc_item s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_dsc_item_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_dsc_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_dsc_item s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_dsc_item =
    fun x => x_dte_def_type_choice_documento_documento_detalle_dsc_item_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_dsc_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_dsc_item
    =
    x_dte_def_type_choice_documento_documento_detalle_dsc_item "1"

inl x_dte_def_type_choice_documento_documento_detalle_dsc_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_dsc_item_to_string (
        x_dte_def_type_choice_documento_documento_detalle_dsc_item_example k
    )

nominal private x_dte_def_type_choice_documento_documento_detalle_unmd_ref = string

inl x_dte_def_type_choice_documento_documento_detalle_unmd_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_documento_detalle_unmd_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_unmd_ref
    =
    if x_dte_def_type_choice_documento_documento_detalle_unmd_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_unmd_ref s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_unmd_ref_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_unmd_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_unmd_ref s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_unmd_ref =
    fun x => x_dte_def_type_choice_documento_documento_detalle_unmd_ref_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_unmd_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_unmd_ref
    =
    x_dte_def_type_choice_documento_documento_detalle_unmd_ref "1"

inl x_dte_def_type_choice_documento_documento_detalle_unmd_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_unmd_ref_to_string (
        x_dte_def_type_choice_documento_documento_detalle_unmd_ref_example k
    )

nominal private x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod = string

inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_is_valid (s : string) : bool =
    xml.code_point_count s <= 35

inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod
    =
    if x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod =
    fun x => x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod
    =
    x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod "1"

inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_to_string (
        x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_example k
    )

nominal x_dte_def_type_choice_documento_documento_detalle_subcantidad =
    {
        sub_qty : x_dec12_6_type
        sub_cod : x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod
    }

nominal private x_dte_def_type_choice_documento_documento_detalle_unmd_item = string

inl x_dte_def_type_choice_documento_documento_detalle_unmd_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_documento_detalle_unmd_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_unmd_item
    =
    if x_dte_def_type_choice_documento_documento_detalle_unmd_item_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_unmd_item s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_unmd_item_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_unmd_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_unmd_item s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_unmd_item =
    fun x => x_dte_def_type_choice_documento_documento_detalle_unmd_item_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_unmd_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_unmd_item
    =
    x_dte_def_type_choice_documento_documento_detalle_unmd_item "1"

inl x_dte_def_type_choice_documento_documento_detalle_unmd_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_unmd_item_to_string (
        x_dte_def_type_choice_documento_documento_detalle_unmd_item_example k
    )

nominal private x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda = string

inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_is_valid (s : string) : bool =
    xml.code_point_count s <= 3

inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda
    =
    if x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda s)
    else None

inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_to_string
    (x : x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda s => s

instance to_text x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda =
    fun x => x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_to_string x
inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda
    =
    x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda "1"

inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_to_string (
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_example k
    )

nominal x_dte_def_type_choice_documento_documento_detalle_otr_mnda =
    {
        prc_otr_mon : x_dec12_6_type
        moneda : x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda
        fct_conv : option (x_dec6_4_type)
        dcto_otr_mnda : option (x_dec14_4_type)
        recargo_otr_mnda : option (x_dec14_4_type)
        monto_item_otr_mnda : option (x_dec14_4_type)
    }

nominal x_dte_def_type_choice_documento_documento_detalle_sub_dscto =
    {
        tipo_dscto : x_dinero_porcentaje_type
        valor_dscto : x_dec16_2_type
    }

type x_dte_def_type_choice_documento_documento_detalle_recargo_pct = string

inl x_dte_def_type_choice_documento_documento_detalle_recargo_pct_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_detalle_recargo_pct_example (k : i32) : string = x_pct_type_example_text k

inl x_dte_def_type_choice_documento_documento_detalle_recargo_pct_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_recargo_pct_example k

nominal x_dte_def_type_choice_documento_documento_detalle_sub_recargo =
    {
        tipo_recargo : x_dinero_porcentaje_type
        valor_recargo : x_dec16_2_type
    }

nominal x_dte_def_type_choice_documento_documento_detalle =
    {
        nro_lin_det : x_dte_def_type_choice_documento_documento_detalle_nro_lin_det
        cdg_item : list (x_dte_def_type_choice_documento_documento_detalle_cdg_item)
        ind_exe : option (x_dte_def_type_choice_documento_documento_detalle_ind_exe)
        retenedor : option (x_dte_def_type_choice_documento_documento_detalle_retenedor)
        nmb_item : x_dte_def_type_choice_documento_documento_detalle_nmb_item
        dsc_item : option (x_dte_def_type_choice_documento_documento_detalle_dsc_item)
        qty_ref : option (x_dec12_6_type)
        unmd_ref : option (x_dte_def_type_choice_documento_documento_detalle_unmd_ref)
        prc_ref : option (x_dec12_6_type)
        qty_item : option (x_dec12_6_type)
        subcantidad : list (x_dte_def_type_choice_documento_documento_detalle_subcantidad)
        fch_elabor : option (x_fecha_type)
        fch_vencim : option (x_fecha_type)
        unmd_item : option (x_dte_def_type_choice_documento_documento_detalle_unmd_item)
        prc_item : option (x_dec12_6_type)
        otr_mnda : option (x_dte_def_type_choice_documento_documento_detalle_otr_mnda)
        descuento_pct : option (x_pct_type)
        descuento_monto : option (x_mnt_imp_type)
        sub_dscto : list (x_dte_def_type_choice_documento_documento_detalle_sub_dscto)
        recargo_pct : option (x_dte_def_type_choice_documento_documento_detalle_recargo_pct)
        recargo_monto : option (x_mnt_imp_type)
        sub_recargo : list (x_dte_def_type_choice_documento_documento_detalle_sub_recargo)
        cod_imp_adic : list (x_imp_adic_dte_type)
        monto_item : x_monto_type
    }

type x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti = string

inl x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti_example k

nominal private x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti = string

inl x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti
    =
    if x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti s)
    else None

inl x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_to_string
    (x : x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti s => s

instance to_text x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti =
    fun x => x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_to_string x
inl x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti
    =
    x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti "1"

inl x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_to_string (
        x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_example k
    )

type x_dte_def_type_choice_documento_documento_sub_tot_info_orden_sti = string

inl x_dte_def_type_choice_documento_documento_sub_tot_info_orden_sti_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_sub_tot_info_orden_sti_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_sub_tot_info_orden_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_sub_tot_info_orden_sti_example k

type x_dte_def_type_choice_documento_documento_sub_tot_info_lineas_deta = string

inl x_dte_def_type_choice_documento_documento_sub_tot_info_lineas_deta_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_sub_tot_info_lineas_deta_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_sub_tot_info_lineas_deta_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_sub_tot_info_lineas_deta_example k

nominal x_dte_def_type_choice_documento_documento_sub_tot_info =
    {
        nro_sti : x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti
        glosa_sti : x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti
        orden_sti : option (x_dte_def_type_choice_documento_documento_sub_tot_info_orden_sti)
        sub_tot_neto_sti : option (x_dec16_2_type)
        sub_tot_ivasti : option (x_dec16_2_type)
        sub_tot_adic_sti : option (x_dec16_2_type)
        sub_tot_exe_sti : option (x_dec16_2_type)
        val_subtot_sti : option (x_dec16_2_type)
        lineas_deta : list (x_dte_def_type_choice_documento_documento_sub_tot_info_lineas_deta)
    }

union x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov =
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_D
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_R

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov
    =
    match s with
    | "D" => Some DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_D
    | "R" => Some DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_R
    | _ => None

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_to_string
    (x : x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_D => "D"
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_R => "R"

instance to_text x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov =
    fun x => x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_to_string x

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov
    =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_D
    else DTEDefType_choice_Documento_Documento_DscRcgGlobal_TpoMov_R

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_to_string (
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_example k
    )

nominal private x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr = string

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_is_valid (s : string) : bool =
    xml.code_point_count s <= 45

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr
    =
    if x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr s)
    else None

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_to_string
    (x : x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr s => s

instance to_text x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr =
    fun x => x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_to_string x
inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr "1"

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_to_string (
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_example k
    )

union x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr =
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_1
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_2

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr
    =
    inl s = xml.collapsed_space s
    match s with
    | "1" => Some DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_1
    | "2" => Some DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_2
    | _ => None

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_to_string
    (x : x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_1 => "1"
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_2 => "2"

instance to_text x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr =
    fun x => x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_to_string x

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr
    =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_1
    else DTEDefType_choice_Documento_Documento_DscRcgGlobal_IndExeDR_2

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_to_string (
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_example k
    )

nominal x_dte_def_type_choice_documento_documento_dsc_rcg_global =
    {
        nro_lin_dr : xsd_positive_integer
        tpo_mov : x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov
        glosa_dr : option (x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr)
        tpo_valor : x_dinero_porcentaje_type
        valor_dr : x_dec16_2_type
        valor_dr_otr_mnda : option (x_dec14_4_type)
        ind_exe_dr : option (x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr)
    }

type x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref = string

inl x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref_example k

nominal private x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref = string

inl x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 3 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref
    =
    if x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref s)
    else None

inl x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_to_string
    (x : x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref s => s

instance to_text x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref =
    fun x => x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_to_string x
inl x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref
    =
    x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref "1"

inl x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_to_string (
        x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_example k
    )

type x_dte_def_type_choice_documento_documento_referencia_ind_global =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja

inl x_dte_def_type_choice_documento_documento_referencia_ind_global_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_referencia_ind_global
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_of_string s

inl x_dte_def_type_choice_documento_documento_referencia_ind_global_to_string
    (x : x_dte_def_type_choice_documento_documento_referencia_ind_global)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string x

inl x_dte_def_type_choice_documento_documento_referencia_ind_global_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_referencia_ind_global
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example k

inl x_dte_def_type_choice_documento_documento_referencia_ind_global_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example_text k

type x_dte_def_type_choice_documento_documento_referencia_cod_ref =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_documento_referencia_cod_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_referencia_cod_ref
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_documento_referencia_cod_ref_to_string
    (x : x_dte_def_type_choice_documento_documento_referencia_cod_ref)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_documento_referencia_cod_ref_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_referencia_cod_ref
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_documento_referencia_cod_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

nominal private x_dte_def_type_choice_documento_documento_referencia_razon_ref = string

inl x_dte_def_type_choice_documento_documento_referencia_razon_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 90

inl x_dte_def_type_choice_documento_documento_referencia_razon_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_referencia_razon_ref
    =
    if x_dte_def_type_choice_documento_documento_referencia_razon_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_referencia_razon_ref s)
    else None

inl x_dte_def_type_choice_documento_documento_referencia_razon_ref_to_string
    (x : x_dte_def_type_choice_documento_documento_referencia_razon_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_referencia_razon_ref s => s

instance to_text x_dte_def_type_choice_documento_documento_referencia_razon_ref =
    fun x => x_dte_def_type_choice_documento_documento_referencia_razon_ref_to_string x
inl x_dte_def_type_choice_documento_documento_referencia_razon_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_referencia_razon_ref
    =
    x_dte_def_type_choice_documento_documento_referencia_razon_ref "1"

inl x_dte_def_type_choice_documento_documento_referencia_razon_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_referencia_razon_ref_to_string (
        x_dte_def_type_choice_documento_documento_referencia_razon_ref_example k
    )

nominal x_dte_def_type_choice_documento_documento_referencia =
    {
        nro_lin_ref : x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref
        tpo_doc_ref : x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref
        ind_global : option (x_dte_def_type_choice_documento_documento_referencia_ind_global)
        folio_ref : x_folio_r_type
        rut_otr : option (x_rut_type)
        fch_ref : x_fecha_type
        cod_ref : option (x_dte_def_type_choice_documento_documento_referencia_cod_ref)
        razon_ref : option (x_dte_def_type_choice_documento_documento_referencia_razon_ref)
    }

type x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com = string

inl x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com_example k

union x_dte_def_type_choice_documento_documento_comisiones_tipo_movim =
    | DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_C
    | DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_O

inl x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_comisiones_tipo_movim
    =
    match s with
    | "C" => Some DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_C
    | "O" => Some DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_O
    | _ => None

inl x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_to_string
    (x : x_dte_def_type_choice_documento_documento_comisiones_tipo_movim)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_C => "C"
    | DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_O => "O"

instance to_text x_dte_def_type_choice_documento_documento_comisiones_tipo_movim =
    fun x => x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_to_string x

inl x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_comisiones_tipo_movim
    =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_C
    else DTEDefType_choice_Documento_Documento_Comisiones_TipoMovim_O

inl x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_to_string (
        x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_example k
    )

nominal private x_dte_def_type_choice_documento_documento_comisiones_glosa = string

inl x_dte_def_type_choice_documento_documento_comisiones_glosa_is_valid (s : string) : bool =
    xml.code_point_count s <= 60 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_documento_comisiones_glosa_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_comisiones_glosa
    =
    if x_dte_def_type_choice_documento_documento_comisiones_glosa_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_comisiones_glosa s)
    else None

inl x_dte_def_type_choice_documento_documento_comisiones_glosa_to_string
    (x : x_dte_def_type_choice_documento_documento_comisiones_glosa)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_comisiones_glosa s => s

instance to_text x_dte_def_type_choice_documento_documento_comisiones_glosa =
    fun x => x_dte_def_type_choice_documento_documento_comisiones_glosa_to_string x
inl x_dte_def_type_choice_documento_documento_comisiones_glosa_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_comisiones_glosa
    =
    x_dte_def_type_choice_documento_documento_comisiones_glosa "1"

inl x_dte_def_type_choice_documento_documento_comisiones_glosa_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_comisiones_glosa_to_string (
        x_dte_def_type_choice_documento_documento_comisiones_glosa_example k
    )

nominal x_dte_def_type_choice_documento_documento_comisiones =
    {
        nro_lin_com : x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com
        tipo_movim : x_dte_def_type_choice_documento_documento_comisiones_tipo_movim
        glosa : x_dte_def_type_choice_documento_documento_comisiones_glosa
        tasa_comision : option (x_pct_type)
        val_com_neto : x_monto_type
        val_com_exe : x_monto_type
        val_com_iva : option (x_monto_type)
    }

nominal private x_dte_def_type_choice_documento_documento_ted_dd_rsr = string

inl x_dte_def_type_choice_documento_documento_ted_dd_rsr_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_ted_dd_rsr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_ted_dd_rsr
    =
    if x_dte_def_type_choice_documento_documento_ted_dd_rsr_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_ted_dd_rsr s)
    else None

inl x_dte_def_type_choice_documento_documento_ted_dd_rsr_to_string
    (x : x_dte_def_type_choice_documento_documento_ted_dd_rsr)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_ted_dd_rsr s => s

instance to_text x_dte_def_type_choice_documento_documento_ted_dd_rsr =
    fun x => x_dte_def_type_choice_documento_documento_ted_dd_rsr_to_string x
inl x_dte_def_type_choice_documento_documento_ted_dd_rsr_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_rsr
    =
    x_dte_def_type_choice_documento_documento_ted_dd_rsr "1"

inl x_dte_def_type_choice_documento_documento_ted_dd_rsr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_ted_dd_rsr_to_string (
        x_dte_def_type_choice_documento_documento_ted_dd_rsr_example k
    )

nominal private x_dte_def_type_choice_documento_documento_ted_dd_it1 = string

inl x_dte_def_type_choice_documento_documento_ted_dd_it1_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_ted_dd_it1_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_ted_dd_it1
    =
    if x_dte_def_type_choice_documento_documento_ted_dd_it1_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_ted_dd_it1 s)
    else None

inl x_dte_def_type_choice_documento_documento_ted_dd_it1_to_string
    (x : x_dte_def_type_choice_documento_documento_ted_dd_it1)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_ted_dd_it1 s => s

instance to_text x_dte_def_type_choice_documento_documento_ted_dd_it1 =
    fun x => x_dte_def_type_choice_documento_documento_ted_dd_it1_to_string x
inl x_dte_def_type_choice_documento_documento_ted_dd_it1_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_it1
    =
    x_dte_def_type_choice_documento_documento_ted_dd_it1 "1"

inl x_dte_def_type_choice_documento_documento_ted_dd_it1_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_ted_dd_it1_to_string (
        x_dte_def_type_choice_documento_documento_ted_dd_it1_example k
    )

nominal private x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs = string

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs
    =
    if x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_is_valid s
    then Some (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs s)
    else None

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_to_string
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs)
    : string
    =
    match x with x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs s => s

instance to_text x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs =
    fun x => x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_to_string x
inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_example
    (_ : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs "1"

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_to_string (
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_example k
    )

nominal x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng =
    {
        d : x_folio_type
        h : x_folio_type
    }

nominal x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk =
    {
        m : string
        e : string
    }

nominal x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk =
    {
        p : string
        q : string
        g : string
        y : string
    }

union x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk =
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK :
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK :
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk

nominal x_dte_def_type_choice_documento_documento_ted_dd_caf_da =
    {
        re : x_rut_type
        rs : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs
        td : x_dte_type
        rng : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng
        fa : x_fecha_type
        choice_rsapk : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk
        idk : xsd_integer
    }

nominal x_dte_def_type_choice_documento_documento_ted_dd_caf_frma =
    {
        base : string
        algoritmo : string
    }

nominal x_dte_def_type_choice_documento_documento_ted_dd_caf =
    {
        da : x_dte_def_type_choice_documento_documento_ted_dd_caf_da
        frma : x_dte_def_type_choice_documento_documento_ted_dd_caf_frma
        version : string
    }

nominal x_dte_def_type_choice_documento_documento_ted_dd =
    {
        re : x_rut_type
        td : x_dte_type
        f : x_folio_type
        fe : x_fecha_type
        rr : x_rut_type
        rsr : x_dte_def_type_choice_documento_documento_ted_dd_rsr
        mnt : string
        it1 : x_dte_def_type_choice_documento_documento_ted_dd_it1
        caf : x_dte_def_type_choice_documento_documento_ted_dd_caf
        tsted : x_fecha_hora_type
    }

union x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo =
    | DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withRSA
    | DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withDSA

inl x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo
    =
    match s with
    | "SHA1withRSA" => Some DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withRSA
    | "SHA1withDSA" => Some DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withDSA
    | _ => None

inl x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_to_string
    (x : x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withRSA => "SHA1withRSA"
    | DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withDSA => "SHA1withDSA"

instance to_text x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo =
    fun x => x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_to_string x

inl x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo
    =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withRSA
    else DTEDefType_choice_Documento_Documento_TED_FRMT_algoritmo_SHA1withDSA

inl x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_to_string (
        x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example k
    )

nominal x_dte_def_type_choice_documento_documento_ted_frmt =
    {
        base : string
        algoritmo : x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo
    }

nominal x_dte_def_type_choice_documento_documento_ted =
    {
        dd : x_dte_def_type_choice_documento_documento_ted_dd
        frmt : x_dte_def_type_choice_documento_documento_ted_frmt
        version : string
    }

nominal x_dte_def_type_choice_documento_documento =
    {
        encabezado : x_dte_def_type_choice_documento_documento_encabezado
        detalle : list (x_dte_def_type_choice_documento_documento_detalle)
        sub_tot_info : list (x_dte_def_type_choice_documento_documento_sub_tot_info)
        dsc_rcg_global : list (x_dte_def_type_choice_documento_documento_dsc_rcg_global)
        referencia : list (x_dte_def_type_choice_documento_documento_referencia)
        comisiones : list (x_dte_def_type_choice_documento_documento_comisiones)
        ted : x_dte_def_type_choice_documento_documento_ted
        tmst_firma : x_fecha_hora_type
        id : string
    }

type x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

type x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_of_string s

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string x

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example k

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example_text k

type x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos =
    {
        fch_pago : x_fecha_type
        mnt_pago : x_monto_type
        glosa_pagos : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos)
    }

type x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_of_string s

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_to_string x

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_example k

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_example_text k

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_is_valid (s : string) : bool =
    xml.code_point_count s <= 100

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_example k
    )

type x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_dias = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_dias_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_dias_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_dias_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_dias_example k

nominal x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc =
    {
        tipo_dte : x_liq_type
        folio : x_folio_type
        fch_emis : x_fecha_type
        ind_servicio : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio)
        mnt_bruto : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto)
        fma_pago : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago)
        fch_cancel : option (x_fecha_type)
        mnt_cancel : option (x_monto_type)
        saldo_insol : option (x_monto_type)
        mnt_pagos : list (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos)
        periodo_desde : option (x_fecha_type)
        periodo_hasta : option (x_fecha_type)
        medio_pago : option (x_medio_pago_type)
        tpo_cta_pago : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago)
        num_cta_pago : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago)
        bco_pago : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago)
        term_pago_cdg : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg)
        term_pago_glosa : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa)
        term_pago_dias : option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_dias)
        fch_venc : option (x_fecha_type)
    }

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_is_valid (s : string) : bool =
    xml.code_point_count s <= 100 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_is_valid (s : string) : bool =
    xml.code_point_count s <= 80 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_example k
    )

type x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_acteco = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_acteco_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_acteco_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_acteco_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_acteco_example k

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_example k
    )

type x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_sii_sucur = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_sii_sucur_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_sii_sucur_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_sii_sucur_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_sii_sucur_example k

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_is_valid (s : string) : bool =
    xml.code_point_count s <= 60

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_encabezado_emisor =
    {
        rut_emisor : x_rut_type
        rzn_soc : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc
        giro_emis : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis
        telefono : list (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono)
        correo_emisor : option (x_mail_type)
        acteco : list (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_acteco)
        sucursal : option (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal)
        cdg_sii_sucur : option (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_sii_sucur)
        dir_origen : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen
        cmna_origen : option (x_comuna_type)
        ciudad_origen : option (x_ciudad_type)
        cdg_vendedor : option (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor)
    }

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_is_valid (s : string) : bool =
    xml.code_point_count s <= 80

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal = string

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal
    =
    if x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal =
    fun x => x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_to_string x
inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal "1"

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_to_string (
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_encabezado_receptor =
    {
        rut_recep : x_rut_type
        cdg_int_recep : option (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep)
        rzn_soc_recep : x_rzn_soc_larga_type
        giro_recep : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep
        contacto : option (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto)
        correo_recep : option (x_mail_type)
        dir_recep : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep
        cmna_recep : x_comuna_type
        ciudad_recep : option (x_ciudad_type)
        dir_postal : option (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal)
        cmna_postal : option (x_comuna_type)
        ciudad_postal : option (x_ciudad_type)
    }

nominal x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten =
    {
        tipo_imp : x_imp_adic_dte_type
        tasa_imp : option (x_pct_type)
        monto_imp : x_mnt_imp_type
    }

nominal x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones =
    {
        val_com_neto : option (x_valor_type)
        val_com_exe : option (x_valor_type)
        val_com_iva : option (x_valor_type)
    }

nominal x_dte_def_type_choice_documento_liquidacion_encabezado_totales =
    {
        mnt_neto : option (x_valor_type)
        mnt_exe : option (x_valor_type)
        tasa_iva : option (x_pct_type)
        iva : option (x_valor_type)
        iva_prop : option (x_valor_type)
        iva_terc : option (x_valor_type)
        impto_reten : list (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten)
        comisiones : option (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones)
        mnt_total : x_valor_type
        monto_periodo : option (x_valor_type)
        saldo_anterior : option (x_valor_type)
        vlr_pagar : option (x_valor_type)
    }

nominal x_dte_def_type_choice_documento_liquidacion_encabezado =
    {
        id_doc : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc
        emisor : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor
        receptor : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor
        totales : x_dte_def_type_choice_documento_liquidacion_encabezado_totales
    }

type x_dte_def_type_choice_documento_liquidacion_detalle_nro_lin_det = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_nro_lin_det_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_detalle_nro_lin_det_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_nro_lin_det_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_nro_lin_det_example k

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_is_valid (s : string) : bool =
    xml.code_point_count s <= 10

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_is_valid (s : string) : bool =
    xml.code_point_count s <= 35

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item =
    {
        tpo_codigo : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo
        vlr_codigo : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo
    }

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 3

inl x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_example k
    )

type x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe

inl x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe
    =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_of_string s

inl x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe)
    : string
    =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_to_string x

inl x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe
    =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_example k

inl x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_example_text k

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 80

inl x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 1000

inl x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_is_valid (s : string) : bool =
    xml.code_point_count s <= 35

inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad =
    {
        sub_qty : x_dec12_6_type
        sub_cod : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod
    }

nominal private x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item = string

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item
    =
    if x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item =
    fun x => x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_to_string x
inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item "1"

inl x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_to_string (
        x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_detalle =
    {
        nro_lin_det : x_dte_def_type_choice_documento_liquidacion_detalle_nro_lin_det
        cdg_item : list (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item)
        tpo_doc_liq : x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq
        ind_exe : option (x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe)
        nmb_item : x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item
        dsc_item : option (x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item)
        qty_ref : option (x_dec12_6_type)
        unmd_ref : option (x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref)
        prc_ref : option (x_dec12_6_type)
        qty_item : option (x_dec12_6_type)
        subcantidad : list (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad)
        fch_elabor : option (x_fecha_type)
        fch_vencim : option (x_fecha_type)
        unmd_item : option (x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item)
        prc_item : option (x_dec12_6_type)
        cod_imp_adic : list (x_imp_adic_dte_type)
        monto_item : x_valor_type
    }

type x_dte_def_type_choice_documento_liquidacion_sub_tot_info_nro_sti = string

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_nro_sti_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_nro_sti_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_nro_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_nro_sti_example k

nominal private x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti = string

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti
    =
    if x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti =
    fun x => x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_to_string x
inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti
    =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti "1"

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_to_string (
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_example k
    )

type x_dte_def_type_choice_documento_liquidacion_sub_tot_info_orden_sti = string

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_orden_sti_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_orden_sti_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_orden_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_orden_sti_example k

type x_dte_def_type_choice_documento_liquidacion_sub_tot_info_lineas_deta = string

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_lineas_deta_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_lineas_deta_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_lineas_deta_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_lineas_deta_example k

nominal x_dte_def_type_choice_documento_liquidacion_sub_tot_info =
    {
        nro_sti : x_dte_def_type_choice_documento_liquidacion_sub_tot_info_nro_sti
        glosa_sti : x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti
        orden_sti : option (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_orden_sti)
        sub_tot_neto_sti : option (x_dec16_2_type)
        sub_tot_ivasti : option (x_dec16_2_type)
        sub_tot_adic_sti : option (x_dec16_2_type)
        sub_tot_exe_sti : option (x_dec16_2_type)
        val_subtot_sti : option (x_dec16_2_type)
        lineas_deta : list (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_lineas_deta)
    }

type x_dte_def_type_choice_documento_liquidacion_referencia_nro_lin_ref = string

inl x_dte_def_type_choice_documento_liquidacion_referencia_nro_lin_ref_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_referencia_nro_lin_ref_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_referencia_nro_lin_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_referencia_nro_lin_ref_example k

nominal private x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref = string

inl x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 3 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref
    =
    if x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref =
    fun x => x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_to_string x
inl x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref
    =
    x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref "1"

inl x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_to_string (
        x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_example k
    )

type x_dte_def_type_choice_documento_liquidacion_referencia_ind_global =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja

inl x_dte_def_type_choice_documento_liquidacion_referencia_ind_global_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_referencia_ind_global
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_of_string s

inl x_dte_def_type_choice_documento_liquidacion_referencia_ind_global_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_referencia_ind_global)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string x

inl x_dte_def_type_choice_documento_liquidacion_referencia_ind_global_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_referencia_ind_global
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example k

inl x_dte_def_type_choice_documento_liquidacion_referencia_ind_global_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example_text k

type x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

nominal private x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref = string

inl x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 90

inl x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref
    =
    if x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref =
    fun x => x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_to_string x
inl x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref
    =
    x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref "1"

inl x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_to_string (
        x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_referencia =
    {
        nro_lin_ref : x_dte_def_type_choice_documento_liquidacion_referencia_nro_lin_ref
        tpo_doc_ref : x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref
        ind_global : option (x_dte_def_type_choice_documento_liquidacion_referencia_ind_global)
        folio_ref : x_folio_r_type
        fch_ref : x_fecha_type
        cod_ref : option (x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref)
        razon_ref : option (x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref)
    }

type x_dte_def_type_choice_documento_liquidacion_comisiones_nro_lin_com = string

inl x_dte_def_type_choice_documento_liquidacion_comisiones_nro_lin_com_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_liquidacion_comisiones_nro_lin_com_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_liquidacion_comisiones_nro_lin_com_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_comisiones_nro_lin_com_example k

type x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim =
    x_dte_def_type_choice_documento_documento_comisiones_tipo_movim

inl x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim
    =
    x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_of_string s

inl x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim)
    : string
    =
    x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_to_string x

inl x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim
    =
    x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_example k

inl x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_example_text k

nominal private x_dte_def_type_choice_documento_liquidacion_comisiones_glosa = string

inl x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_is_valid (s : string) : bool =
    xml.code_point_count s <= 60 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_comisiones_glosa
    =
    if x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_comisiones_glosa s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_comisiones_glosa)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_comisiones_glosa s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_comisiones_glosa =
    fun x => x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_to_string x
inl x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_comisiones_glosa
    =
    x_dte_def_type_choice_documento_liquidacion_comisiones_glosa "1"

inl x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_to_string (
        x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_comisiones =
    {
        nro_lin_com : x_dte_def_type_choice_documento_liquidacion_comisiones_nro_lin_com
        tipo_movim : x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim
        glosa : x_dte_def_type_choice_documento_liquidacion_comisiones_glosa
        tasa_comision : option (x_pct_type)
        val_com_neto : x_valor_type
        val_com_exe : x_valor_type
        val_com_iva : option (x_valor_type)
    }

nominal private x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr = string

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr
    =
    if x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr =
    fun x => x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_to_string x
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr "1"

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_to_string (
        x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_example k
    )

nominal private x_dte_def_type_choice_documento_liquidacion_ted_dd_it1 = string

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_ted_dd_it1
    =
    if x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_is_valid s
    then Some (x_dte_def_type_choice_documento_liquidacion_ted_dd_it1 s)
    else None

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_it1)
    : string
    =
    match x with x_dte_def_type_choice_documento_liquidacion_ted_dd_it1 s => s

instance to_text x_dte_def_type_choice_documento_liquidacion_ted_dd_it1 =
    fun x => x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_to_string x
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_example
    (_ : i32)
    : x_dte_def_type_choice_documento_liquidacion_ted_dd_it1
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_it1 "1"

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_to_string (
        x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_example k
    )

nominal x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng =
    {
        d : x_folio_type
        h : x_folio_type
    }

nominal x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk =
    {
        m : string
        e : string
    }

nominal x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk =
    {
        p : string
        q : string
        g : string
        y : string
    }

union x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk =
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK :
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK :
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk

nominal x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da =
    {
        re : x_rut_type
        rs : x_rzn_soc_corta_type
        td : x_liq_type
        rng : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng
        fa : x_fecha_type
        choice_rsapk : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk
        idk : xsd_integer
    }

nominal x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma =
    {
        base : string
        algoritmo : string
    }

nominal x_dte_def_type_choice_documento_liquidacion_ted_dd_caf =
    {
        da : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da
        frma : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma
        version : string
    }

nominal x_dte_def_type_choice_documento_liquidacion_ted_dd =
    {
        re : x_rut_type
        td : x_liq_type
        f : x_folio_type
        fe : x_fecha_type
        rr : x_rut_type
        rsr : x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr
        mnt : x_valor_type
        it1 : x_dte_def_type_choice_documento_liquidacion_ted_dd_it1
        caf : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf
        tsted : x_fecha_hora_type
    }

type x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo

inl x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo
    =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_of_string s

inl x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo_to_string
    (x : x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo)
    : string
    =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_to_string x

inl x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo_example
    (k : i32)
    : x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo
    =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example k

inl x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example_text k

nominal x_dte_def_type_choice_documento_liquidacion_ted_frmt =
    {
        base : string
        algoritmo : x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo
    }

nominal x_dte_def_type_choice_documento_liquidacion_ted =
    {
        dd : x_dte_def_type_choice_documento_liquidacion_ted_dd
        frmt : x_dte_def_type_choice_documento_liquidacion_ted_frmt
        version : string
    }

nominal x_dte_def_type_choice_documento_liquidacion =
    {
        encabezado : x_dte_def_type_choice_documento_liquidacion_encabezado
        detalle : list (x_dte_def_type_choice_documento_liquidacion_detalle)
        sub_tot_info : list (x_dte_def_type_choice_documento_liquidacion_sub_tot_info)
        referencia : list (x_dte_def_type_choice_documento_liquidacion_referencia)
        comisiones : list (x_dte_def_type_choice_documento_liquidacion_comisiones)
        ted : x_dte_def_type_choice_documento_liquidacion_ted
        tmst_firma : x_fecha_hora_type
        id : string
    }

type x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

union x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_3
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_4
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_5

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio
    =
    inl s = xml.collapsed_space s
    match s with
    | "3" => Some DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_3
    | "4" => Some DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_4
    | "5" => Some DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_5
    | _ => None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_3 => "3"
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_4 => "4"
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_5 => "5"

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_to_string x

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio
    =
    inl index = ((k % 3i32) + 3i32) % 3i32
    if index = 0i32
    then DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_3
    elif index = 1i32
    then DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_4
    else DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_IndServicio_5

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

type x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pag_exp = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pag_exp_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pag_exp_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pag_exp_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pag_exp_example k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos =
    {
        fch_pago : x_fecha_type
        mnt_pago : x_monto_type
        glosa_pagos : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos)
    }

type x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_of_string s

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_to_string x

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_example k

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_example_text k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_is_valid (s : string) : bool =
    xml.code_point_count s <= 100

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_dias = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_dias_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_dias_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_dias_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_dias_example k

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc =
    {
        tipo_dte : x_exp_type
        folio : x_folio_type
        fch_emis : x_fecha_type
        tipo_despacho : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho)
        ind_servicio : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio)
        fma_pago : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago)
        fma_pag_exp : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pag_exp)
        fch_cancel : option (x_fecha_type)
        mnt_cancel : option (x_monto_type)
        saldo_insol : option (x_monto_type)
        mnt_pagos : list (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos)
        periodo_desde : option (x_fecha_type)
        periodo_hasta : option (x_fecha_type)
        medio_pago : option (x_medio_pago_type)
        tpo_cta_pago : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago)
        num_cta_pago : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago)
        bco_pago : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago)
        term_pago_cdg : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg)
        term_pago_glosa : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa)
        term_pago_dias : option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_dias)
        fch_venc : option (x_fecha_type)
    }

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_is_valid (s : string) : bool =
    xml.code_point_count s <= 100 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_is_valid (s : string) : bool =
    xml.code_point_count s <= 80 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_acteco = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_acteco_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_acteco_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_acteco_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_acteco_example k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_sii_sucur = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_sii_sucur_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_sii_sucur_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_sii_sucur_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_sii_sucur_example k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_is_valid (s : string) : bool =
    xml.code_point_count s <= 60

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_emisor =
    {
        rut_emisor : x_rut_type
        rzn_soc : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc
        giro_emis : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis
        telefono : list (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono)
        correo_emisor : option (x_mail_type)
        acteco : list (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_acteco)
        sucursal : option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal)
        cdg_sii_sucur : option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_sii_sucur)
        cod_adic_sucur : option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur)
        dir_origen : option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen)
        cmna_origen : option (x_comuna_type)
        ciudad_origen : option (x_ciudad_type)
        cdg_vendedor : option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor)
        id_adic_emisor : option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor)
    }

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 3 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero =
    {
        num_id : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id)
        nacionalidad : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad)
        id_adic_recep : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep)
    }

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_is_valid (s : string) : bool =
    xml.code_point_count s <= 80

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_receptor =
    {
        rut_recep : x_rut_type
        cdg_int_recep : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep)
        rzn_soc_recep : x_rzn_soc_larga_type
        extranjero : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero)
        giro_recep : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep)
        contacto : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto)
        correo_recep : option (x_mail_type)
        dir_recep : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep)
        cmna_recep : option (x_comuna_type)
        ciudad_recep : option (x_ciudad_type)
        dir_postal : option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal)
        cmna_postal : option (x_comuna_type)
        ciudad_postal : option (x_ciudad_type)
    }

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_is_valid (s : string) : bool =
    xml.code_point_count s <= 8

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 30

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer =
    {
        rut_chofer : x_rut_type
        nombre_chofer : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer
    }

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_is_valid (s : string) : bool =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_mod_venta = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_mod_venta_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_mod_venta_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_mod_venta_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_mod_venta_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_clau_venta = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_clau_venta_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_clau_venta_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_clau_venta_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_clau_venta_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_via_transp = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_via_transp_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_via_transp_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_via_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_via_transp_example k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_embarque = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_embarque_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_embarque_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_embarque_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_embarque_example k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_desemb = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_desemb_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_desemb_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_desemb_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_desemb_example k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_example k
    )

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tara = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tara_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tara_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tara_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tara_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_med_tara = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_med_tara_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_med_tara_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_med_tara_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_med_tara_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_bruto = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_bruto_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_bruto_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_bruto_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_bruto_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_bruto = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_bruto_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_bruto_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_bruto_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_bruto_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_neto = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_neto_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_neto_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_neto_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_neto_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_neto = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_neto_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_neto_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_neto_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_neto_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_items = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_items_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_items_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_items_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_items_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_bultos = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_bultos_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_bultos_example (_ : i32) : string =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_bultos_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_bultos_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cant_bultos = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cant_bultos_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cant_bultos_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cant_bultos_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cant_bultos_example k

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 255

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container =
    string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 25

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container)
    : string
    =
    match x with
    | x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container =
    fun x =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 20

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello =
    fun x => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello =
    string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_is_valid
    (s : string)
    : bool
    =
    xml.code_point_count s <= 70

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello
    =
    if x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello)
    : string
    =
    match x with
    | x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello =
    fun x =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_to_string x
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_to_string (
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos =
    {
        cod_tpo_bultos : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos)
        cant_bultos : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cant_bultos)
        marcas : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas)
        id_container : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container)
        sello : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello)
        emisor_sello : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello)
    }

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_recep = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_recep_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_recep_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_recep_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_recep_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_destin = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_destin_is_valid
    (s : string)
    : bool
    =
    true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_destin_example
    (_ : i32)
    : string
    =
    "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_destin_example_text
    (k : i32)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_destin_example k

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana =
    {
        cod_mod_venta : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_mod_venta)
        cod_clau_venta : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_clau_venta)
        tot_clau_venta : option (x_dec16_2_type)
        cod_via_transp : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_via_transp)
        nombre_transp : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp)
        rut_cia_transp : option (x_rut_type)
        nom_cia_transp : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp)
        id_adic_transp : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp)
        booking : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking)
        operador : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador)
        cod_pto_embarque : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_embarque)
        id_adic_pto_emb : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb)
        cod_pto_desemb : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_desemb)
        id_adic_pto_desemb : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb)
        tara : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tara)
        cod_unid_med_tara : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_med_tara)
        peso_bruto : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_bruto)
        cod_unid_peso_bruto : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_bruto)
        peso_neto : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_neto)
        cod_unid_peso_neto : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_neto)
        tot_items : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_items)
        tot_bultos : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_bultos)
        tipo_bultos : list (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos)
        mnt_flete : option (x_dec14_4_type)
        mnt_seguro : option (x_dec14_4_type)
        cod_pais_recep : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_recep)
        cod_pais_destin : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_destin)
    }

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_transporte =
    {
        patente : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente)
        rut_trans : option (x_rut_type)
        chofer : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer)
        dir_dest : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest)
        cmna_dest : option (x_comuna_type)
        ciudad_dest : option (x_ciudad_type)
        aduana : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana)
    }

type x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_exe = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_exe_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_exe_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_exe_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_exe_example k

type x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_total = string

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_total_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_total_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_total_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_total_example k

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_totales =
    {
        tpo_moneda : x_tip_mon_type
        mnt_exe : x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_exe
        mnt_total : x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_total
    }

nominal x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda =
    {
        tpo_moneda : x_tip_mon_type
        tpo_cambio : option (x_dec6_4_type)
        mnt_exe_otr_mnda : option (x_dec14_4_type)
        mnt_tot_otr_mnda : x_dec14_4_type
    }

nominal x_dte_def_type_choice_documento_exportaciones_encabezado =
    {
        id_doc : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc
        emisor : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor
        receptor : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor
        transporte : option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte)
        totales : x_dte_def_type_choice_documento_exportaciones_encabezado_totales
        otra_moneda : option (x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda)
    }

type x_dte_def_type_choice_documento_exportaciones_detalle_nro_lin_det = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_nro_lin_det_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_detalle_nro_lin_det_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_nro_lin_det_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_nro_lin_det_example k

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_is_valid (s : string) : bool =
    xml.code_point_count s <= 10

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_is_valid (s : string) : bool =
    xml.code_point_count s <= 35

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item =
    {
        tpo_codigo : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo
        vlr_codigo : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo
    }

type x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe

inl x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe
    =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_of_string s

inl x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe)
    : string
    =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_to_string x

inl x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe
    =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_example k

inl x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_detalle_ind_exe_example_text k

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 80

inl x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 1000

inl x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_is_valid (s : string) : bool =
    xml.code_point_count s <= 35

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_is_valid (s : string) : bool =
    xml.code_point_count s <= 10 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad =
    {
        sub_qty : x_dec12_6_type
        sub_cod : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod
        tip_cod_sub_qty : option (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty)
    }

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_is_valid (s : string) : bool =
    xml.code_point_count s <= 4

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_is_valid (s : string) : bool =
    xml.code_point_count s <= 3

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda
    =
    if x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_to_string x
inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_example k
    )

type x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_recargo_otr_mnda = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_recargo_otr_mnda_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_recargo_otr_mnda_example (k : i32) : string =
    x_dec14_4_type_example_text k

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_recargo_otr_mnda_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_recargo_otr_mnda_example k

type x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_monto_item_otr_mnda = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_monto_item_otr_mnda_is_valid (s : string) : bool =
    true

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_monto_item_otr_mnda_example (k : i32) : string =
    x_dec14_4_type_example_text k

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_monto_item_otr_mnda_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_monto_item_otr_mnda_example k

nominal x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda =
    {
        prc_otr_mon : x_dec14_4_type
        moneda : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda
        fct_conv : option (x_dec6_4_type)
        dcto_otr_mnda : option (x_dec14_4_type)
        recargo_otr_mnda : option (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_recargo_otr_mnda)
        monto_item_otr_mnda : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_monto_item_otr_mnda
    }

union x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto =
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto__
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto___

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto
    =
    match s with
    | "$" => Some DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto__
    | "%" => Some DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto___
    | _ => None

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto)
    : string
    =
    match x with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto__ => "$"
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto___ => "%"

instance to_text x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto =
    fun x => x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_to_string x

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto
    =
    inl index = ((k % 2i32) + 2i32) % 2i32
    if index = 0i32
    then DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto__
    else DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_TipoDscto___

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_to_string (
        x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto =
    {
        tipo_dscto : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto
        valor_dscto : x_dec16_2_type
    }

type x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_of_string s

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo)
    : string
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_to_string x

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_example k

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_example_text k

nominal x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo =
    {
        tipo_recargo : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo
        valor_recargo : x_dec16_2_type
    }

type x_dte_def_type_choice_documento_exportaciones_detalle_monto_item = string

inl x_dte_def_type_choice_documento_exportaciones_detalle_monto_item_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_detalle_monto_item_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_detalle_monto_item_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_detalle_monto_item_example k

nominal x_dte_def_type_choice_documento_exportaciones_detalle =
    {
        nro_lin_det : x_dte_def_type_choice_documento_exportaciones_detalle_nro_lin_det
        cdg_item : list (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item)
        ind_exe : option (x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe)
        nmb_item : x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item
        dsc_item : option (x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item)
        qty_ref : option (x_dec12_6_type)
        unmd_ref : option (x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref)
        prc_ref : option (x_dec12_6_type)
        qty_item : option (x_dec12_6_type)
        subcantidad : list (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad)
        fch_elabor : option (x_fecha_type)
        fch_vencim : option (x_fecha_type)
        unmd_item : option (x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item)
        prc_item : option (x_dec12_6_type)
        otr_mnda : option (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda)
        descuento_pct : option (x_pct_type)
        descuento_monto : option (x_mnt_imp_type)
        sub_dscto : list (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto)
        recargo_pct : option (x_pct_type)
        recargo_monto : option (x_mnt_imp_type)
        sub_recargo : list (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo)
        monto_item : x_dte_def_type_choice_documento_exportaciones_detalle_monto_item
    }

type x_dte_def_type_choice_documento_exportaciones_sub_tot_info_nro_sti = string

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_nro_sti_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_nro_sti_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_nro_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_nro_sti_example k

nominal private x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti = string

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_is_valid (s : string) : bool =
    xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti
    =
    if x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti =
    fun x => x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_to_string x
inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti
    =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti "1"

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_to_string (
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_example k
    )

type x_dte_def_type_choice_documento_exportaciones_sub_tot_info_orden_sti = string

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_orden_sti_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_orden_sti_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_orden_sti_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_orden_sti_example k

type x_dte_def_type_choice_documento_exportaciones_sub_tot_info_lineas_deta = string

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_lineas_deta_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_lineas_deta_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_lineas_deta_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_lineas_deta_example k

nominal x_dte_def_type_choice_documento_exportaciones_sub_tot_info =
    {
        nro_sti : x_dte_def_type_choice_documento_exportaciones_sub_tot_info_nro_sti
        glosa_sti : x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti
        orden_sti : option (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_orden_sti)
        sub_tot_neto_sti : option (x_dec16_2_type)
        sub_tot_ivasti : option (x_dec16_2_type)
        sub_tot_adic_sti : option (x_dec16_2_type)
        sub_tot_exe_sti : option (x_dec16_2_type)
        val_subtot_sti : option (x_dec16_2_type)
        lineas_deta : list (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_lineas_deta)
    }

type x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_of_string s

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov)
    : string
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_to_string x

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_example k

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_example_text k

nominal private x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr = string

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_is_valid (s : string) : bool =
    xml.code_point_count s <= 45

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr
    =
    if x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr =
    fun x => x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_to_string x
inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr
    =
    x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr "1"

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_to_string (
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_example k
    )

type x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor = x_dinero_porcentaje_type

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor
    =
    x_dinero_porcentaje_type_of_string s

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor)
    : string
    =
    x_dinero_porcentaje_type_to_string x

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor
    =
    x_dinero_porcentaje_type_example k

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor_example_text (k : i32) : string =
    x_dinero_porcentaje_type_example_text k

type x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_of_string s

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr)
    : string
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_to_string x

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_example k

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_example_text k

nominal x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global =
    {
        nro_lin_dr : xsd_positive_integer
        tpo_mov : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov
        glosa_dr : option (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr)
        tpo_valor : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor
        valor_dr : x_dec16_2_type
        valor_dr_otr_mnda : option (x_dec14_4_type)
        ind_exe_dr : option (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr)
    }

type x_dte_def_type_choice_documento_exportaciones_referencia_nro_lin_ref = string

inl x_dte_def_type_choice_documento_exportaciones_referencia_nro_lin_ref_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_referencia_nro_lin_ref_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_referencia_nro_lin_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_referencia_nro_lin_ref_example k

nominal private x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref = string

inl x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 3 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref
    =
    if x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref =
    fun x => x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_to_string x
inl x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref
    =
    x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref "1"

inl x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_to_string (
        x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_example k
    )

type x_dte_def_type_choice_documento_exportaciones_referencia_ind_global =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja

inl x_dte_def_type_choice_documento_exportaciones_referencia_ind_global_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_referencia_ind_global
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_of_string s

inl x_dte_def_type_choice_documento_exportaciones_referencia_ind_global_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_referencia_ind_global)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_to_string x

inl x_dte_def_type_choice_documento_exportaciones_referencia_ind_global_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_referencia_ind_global
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example k

inl x_dte_def_type_choice_documento_exportaciones_referencia_ind_global_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_example_text k

nominal private x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr = string

inl x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_is_valid (s : string) : bool =
    xml.code_point_count s <= 20 && xml.code_point_count s >= 1

inl x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr
    =
    if x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr =
    fun x => x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_to_string x
inl x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr
    =
    x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr "1"

inl x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_to_string (
        x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_example k
    )

type x_dte_def_type_choice_documento_exportaciones_referencia_fch_ref = string

inl x_dte_def_type_choice_documento_exportaciones_referencia_fch_ref_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_referencia_fch_ref_example (_ : i32) : string = "2026-01-01"

inl x_dte_def_type_choice_documento_exportaciones_referencia_fch_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_referencia_fch_ref_example k

type x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho

inl x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string s

inl x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref)
    : string
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_to_string x

inl x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example k

inl x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_example_text k

nominal private x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref = string

inl x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_is_valid (s : string) : bool =
    xml.code_point_count s <= 90

inl x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref
    =
    if x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref =
    fun x => x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_to_string x
inl x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref
    =
    x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref "1"

inl x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_to_string (
        x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_referencia =
    {
        nro_lin_ref : x_dte_def_type_choice_documento_exportaciones_referencia_nro_lin_ref
        tpo_doc_ref : x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref
        ind_global : option (x_dte_def_type_choice_documento_exportaciones_referencia_ind_global)
        folio_ref : x_folio_r_type
        rut_otr : option (x_rut_type)
        id_adic_otr : option (x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr)
        fch_ref : x_dte_def_type_choice_documento_exportaciones_referencia_fch_ref
        cod_ref : option (x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref)
        razon_ref : option (x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref)
    }

nominal private x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr = string

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr
    =
    if x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr =
    fun x => x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_to_string x
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr "1"

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_to_string (
        x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_example k
    )

type x_dte_def_type_choice_documento_exportaciones_ted_dd_mnt = string

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_mnt_is_valid (s : string) : bool = true

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_mnt_example (_ : i32) : string = "1"

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_mnt_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_mnt_example k

nominal private x_dte_def_type_choice_documento_exportaciones_ted_dd_it1 = string

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_ted_dd_it1
    =
    if x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_ted_dd_it1 s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_it1)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_ted_dd_it1 s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_ted_dd_it1 =
    fun x => x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_to_string x
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_ted_dd_it1
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_it1 "1"

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_to_string (
        x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_example k
    )

nominal private x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs = string

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_is_valid (s : string) : bool =
    xml.code_point_count s >= 1 && xml.code_point_count s <= 40

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs
    =
    if x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_is_valid s
    then Some (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs s)
    else None

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs)
    : string
    =
    match x with x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs s => s

instance to_text x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs =
    fun x => x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_to_string x
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_example
    (_ : i32)
    : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs "1"

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_to_string (
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_example k
    )

nominal x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng =
    {
        d : x_folio_type
        h : x_folio_type
    }

nominal x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk =
    {
        m : string
        e : string
    }

nominal x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk =
    {
        p : string
        q : string
        g : string
        y : string
    }

union x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk =
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK :
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK :
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk

nominal x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da =
    {
        re : x_rut_type
        rs : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs
        td : x_exp_type
        rng : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng
        fa : x_fecha_type
        choice_rsapk : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk
        idk : xsd_integer
    }

nominal x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma =
    {
        base : string
        algoritmo : string
    }

nominal x_dte_def_type_choice_documento_exportaciones_ted_dd_caf =
    {
        da : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da
        frma : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma
        version : string
    }

nominal x_dte_def_type_choice_documento_exportaciones_ted_dd =
    {
        re : x_rut_type
        td : x_exp_type
        f : x_folio_type
        fe : x_fecha_type
        rr : x_rut_type
        rsr : x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr
        mnt : x_dte_def_type_choice_documento_exportaciones_ted_dd_mnt
        it1 : x_dte_def_type_choice_documento_exportaciones_ted_dd_it1
        caf : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf
        tsted : x_fecha_hora_type
    }

type x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo

inl x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo_of_string
    (s : string)
    : option x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo
    =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_of_string s

inl x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo_to_string
    (x : x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo)
    : string
    =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_to_string x

inl x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo_example
    (k : i32)
    : x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo
    =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example k

inl x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo_example_text (k : i32) : string =
    x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example_text k

nominal x_dte_def_type_choice_documento_exportaciones_ted_frmt =
    {
        base : string
        algoritmo : x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo
    }

nominal x_dte_def_type_choice_documento_exportaciones_ted =
    {
        dd : x_dte_def_type_choice_documento_exportaciones_ted_dd
        frmt : x_dte_def_type_choice_documento_exportaciones_ted_frmt
        version : string
    }

nominal x_dte_def_type_choice_documento_exportaciones =
    {
        encabezado : x_dte_def_type_choice_documento_exportaciones_encabezado
        detalle : list (x_dte_def_type_choice_documento_exportaciones_detalle)
        sub_tot_info : list (x_dte_def_type_choice_documento_exportaciones_sub_tot_info)
        dsc_rcg_global : list (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global)
        referencia : list (x_dte_def_type_choice_documento_exportaciones_referencia)
        ted : x_dte_def_type_choice_documento_exportaciones_ted
        tmst_firma : x_fecha_hora_type
        id : string
    }

union x_dte_def_type_choice_documento =
    | DTEDefType_choice_Documento_Documento : x_dte_def_type_choice_documento_documento
    | DTEDefType_choice_Documento_Liquidacion : x_dte_def_type_choice_documento_liquidacion
    | DTEDefType_choice_Documento_Exportaciones : x_dte_def_type_choice_documento_exportaciones

nominal x_dte_def_type s =
    {
        choice_documento : x_dte_def_type_choice_documento
        signature : staged s x_signature_type
        version : xsd_decimal
    }

union x_dte_def_type_choice_documento_exportaciones_field r =
    | DTEDefType_choice_Documento_Exportaciones_field_Encabezado ::
        x_dte_def_type_choice_documento_exportaciones_field (x_dte_def_type_choice_documento_exportaciones_encabezado)
    | DTEDefType_choice_Documento_Exportaciones_field_Detalle ::
        x_dte_def_type_choice_documento_exportaciones_field
            (list (x_dte_def_type_choice_documento_exportaciones_detalle))
    | DTEDefType_choice_Documento_Exportaciones_field_SubTotInfo ::
        x_dte_def_type_choice_documento_exportaciones_field
            (list (x_dte_def_type_choice_documento_exportaciones_sub_tot_info))
    | DTEDefType_choice_Documento_Exportaciones_field_DscRcgGlobal ::
        x_dte_def_type_choice_documento_exportaciones_field
            (list (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global))
    | DTEDefType_choice_Documento_Exportaciones_field_Referencia ::
        x_dte_def_type_choice_documento_exportaciones_field
            (list (x_dte_def_type_choice_documento_exportaciones_referencia))
    | DTEDefType_choice_Documento_Exportaciones_field_TED ::
        x_dte_def_type_choice_documento_exportaciones_field (x_dte_def_type_choice_documento_exportaciones_ted)
    | DTEDefType_choice_Documento_Exportaciones_field_TmstFirma ::
        x_dte_def_type_choice_documento_exportaciones_field (x_fecha_hora_type)

inl x_dte_def_type_choice_documento_exportaciones_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_field r) (x : x_dte_def_type_choice_documento_exportaciones)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_field_Encabezado => x.encabezado
    | DTEDefType_choice_Documento_Exportaciones_field_Detalle => x.detalle
    | DTEDefType_choice_Documento_Exportaciones_field_SubTotInfo => x.sub_tot_info
    | DTEDefType_choice_Documento_Exportaciones_field_DscRcgGlobal => x.dsc_rcg_global
    | DTEDefType_choice_Documento_Exportaciones_field_Referencia => x.referencia
    | DTEDefType_choice_Documento_Exportaciones_field_TED => x.ted
    | DTEDefType_choice_Documento_Exportaciones_field_TmstFirma => x.tmst_firma

inl x_dte_def_type_choice_documento_exportaciones_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_field_Encabezado
        exists DTEDefType_choice_Documento_Exportaciones_field_Detalle
        exists DTEDefType_choice_Documento_Exportaciones_field_SubTotInfo
        exists DTEDefType_choice_Documento_Exportaciones_field_DscRcgGlobal
        exists DTEDefType_choice_Documento_Exportaciones_field_Referencia
        exists DTEDefType_choice_Documento_Exportaciones_field_TED
        exists DTEDefType_choice_Documento_Exportaciones_field_TmstFirma
    ]
inl x_dte_def_type_choice_documento_exportaciones_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_field_Encabezado => { tag = "Encabezado"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_field_Detalle => { tag = "Detalle"; min = 1i32; max = Some 60i32 }
    | DTEDefType_choice_Documento_Exportaciones_field_SubTotInfo => { tag = "SubTotInfo"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Exportaciones_field_DscRcgGlobal =>
        { tag = "DscRcgGlobal"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Exportaciones_field_Referencia => { tag = "Referencia"; min = 0i32; max = Some 40i32 }
    | DTEDefType_choice_Documento_Exportaciones_field_TED => { tag = "TED"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_field_TmstFirma => { tag = "TmstFirma"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_ted_field r =
    | DTEDefType_choice_Documento_Exportaciones_TED_field_DD ::
        x_dte_def_type_choice_documento_exportaciones_ted_field (x_dte_def_type_choice_documento_exportaciones_ted_dd)
    | DTEDefType_choice_Documento_Exportaciones_TED_field_FRMT ::
        x_dte_def_type_choice_documento_exportaciones_ted_field (x_dte_def_type_choice_documento_exportaciones_ted_frmt)

inl x_dte_def_type_choice_documento_exportaciones_ted_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_ted)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_field_DD => x.dd
    | DTEDefType_choice_Documento_Exportaciones_TED_field_FRMT => x.frmt

inl x_dte_def_type_choice_documento_exportaciones_ted_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_TED_field_DD
        exists DTEDefType_choice_Documento_Exportaciones_TED_field_FRMT
    ]
inl x_dte_def_type_choice_documento_exportaciones_ted_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_field_DD => { tag = "DD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_field_FRMT => { tag = "FRMT"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_ted_dd_field r =
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RE ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field (x_rut_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TD ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field (x_exp_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_F ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field (x_folio_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_FE ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field (x_fecha_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RR ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field (x_rut_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RSR ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_MNT ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_mnt)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_IT1 ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_it1)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_CAF ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TSTED ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_field (x_fecha_hora_type)

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RE => x.re
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TD => x.td
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_F => x.f
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_FE => x.fe
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RR => x.rr
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RSR => x.rsr
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_MNT => x.mnt
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_IT1 => x.it1
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_CAF => x.caf
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TSTED => x.tsted

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RE
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TD
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_F
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_FE
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RR
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RSR
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_MNT
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_IT1
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_CAF
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TSTED
    ]
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RE => { tag = "RE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TD => { tag = "TD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_F => { tag = "F"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_FE => { tag = "FE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RR => { tag = "RR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RSR => { tag = "RSR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_MNT => { tag = "MNT"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_IT1 => { tag = "IT1"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_CAF => { tag = "CAF"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TSTED => { tag = "TSTED"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field r =
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_DA ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_FRMA ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma)

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_DA => x.da
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_FRMA => x.frma

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_DA
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_FRMA
    ]
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_DA => { tag = "DA"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_FRMA => { tag = "FRMA"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field r =
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RE ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field (x_rut_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RS ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_TD ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field (x_exp_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RNG ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_FA ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field (x_fecha_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_choice_rsapk ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_IDK ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field (xsd_integer)

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RE => x.re
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RS => x.rs
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_TD => x.td
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RNG => x.rng
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_FA => x.fa
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_choice_rsapk => x.choice_rsapk
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_IDK => x.idk

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RE
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RS
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_TD
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RNG
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_FA
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_choice_rsapk
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_IDK
    ]
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RE => { tag = "RE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RS => { tag = "RS"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_TD => { tag = "TD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RNG => { tag = "RNG"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_FA => { tag = "FA"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_choice_rsapk =>
        { tag = "choice_rsapk"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_IDK => { tag = "IDK"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field r =
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field (string)

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P => x.p
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q => x.q
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G => x.g
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y => x.y

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y
    ]
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P =>
        { tag = "P"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q =>
        { tag = "Q"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G =>
        { tag = "G"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y =>
        { tag = "Y"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field r =
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field (string)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field (string)

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M => x.m
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E => x.e

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E
    ]
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M =>
        { tag = "M"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E =>
        { tag = "E"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field r =
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_D ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field (x_folio_type)
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_H ::
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field (x_folio_type)

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_D => x.d
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_H => x.h

inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_D
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_H
    ]
inl x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_D => { tag = "D"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_H => { tag = "H"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_referencia_field r =
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_NroLinRef ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field
            (x_dte_def_type_choice_documento_exportaciones_referencia_nro_lin_ref)
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_TpoDocRef ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field
            (x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref)
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IndGlobal ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field
            (option (x_dte_def_type_choice_documento_exportaciones_referencia_ind_global))
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FolioRef ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field (x_folio_r_type)
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RUTOtr ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IdAdicOtr ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field
            (option (x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr))
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FchRef ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field
            (x_dte_def_type_choice_documento_exportaciones_referencia_fch_ref)
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_CodRef ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field
            (option (x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref))
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RazonRef ::
        x_dte_def_type_choice_documento_exportaciones_referencia_field
            (option (x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref))

inl x_dte_def_type_choice_documento_exportaciones_referencia_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_referencia_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_referencia)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_NroLinRef => x.nro_lin_ref
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_TpoDocRef => x.tpo_doc_ref
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IndGlobal => x.ind_global
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FolioRef => x.folio_ref
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RUTOtr => x.rut_otr
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IdAdicOtr => x.id_adic_otr
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FchRef => x.fch_ref
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_CodRef => x.cod_ref
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RazonRef => x.razon_ref

inl x_dte_def_type_choice_documento_exportaciones_referencia_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_referencia_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_NroLinRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_TpoDocRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_IndGlobal
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_FolioRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_RUTOtr
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_IdAdicOtr
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_FchRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_CodRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_RazonRef
    ]
inl x_dte_def_type_choice_documento_exportaciones_referencia_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_referencia_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_NroLinRef =>
        { tag = "NroLinRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_TpoDocRef =>
        { tag = "TpoDocRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IndGlobal =>
        { tag = "IndGlobal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FolioRef =>
        { tag = "FolioRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RUTOtr =>
        { tag = "RUTOtr"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IdAdicOtr =>
        { tag = "IdAdicOtr"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FchRef =>
        { tag = "FchRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_CodRef =>
        { tag = "CodRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RazonRef =>
        { tag = "RazonRef"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field r =
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_NroLinDR ::
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field (xsd_positive_integer)
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoMov ::
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field
            (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov)
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_GlosaDR ::
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field
            (option (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr))
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoValor ::
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field
            (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor)
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDR ::
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field (x_dec16_2_type)
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDROtrMnda ::
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_IndExeDR ::
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field
            (option (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr))

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_NroLinDR => x.nro_lin_dr
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoMov => x.tpo_mov
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_GlosaDR => x.glosa_dr
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoValor => x.tpo_valor
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDR => x.valor_dr
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDROtrMnda => x.valor_dr_otr_mnda
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_IndExeDR => x.ind_exe_dr

inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_NroLinDR
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoMov
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_GlosaDR
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoValor
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDR
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDROtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_IndExeDR
    ]
inl x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_NroLinDR =>
        { tag = "NroLinDR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoMov =>
        { tag = "TpoMov"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_GlosaDR =>
        { tag = "GlosaDR"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoValor =>
        { tag = "TpoValor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDR =>
        { tag = "ValorDR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDROtrMnda =>
        { tag = "ValorDROtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_IndExeDR =>
        { tag = "IndExeDR"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field r =
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_NroSTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field
            (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_nro_sti)
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_GlosaSTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field
            (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti)
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_OrdenSTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field
            (option (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_orden_sti))
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotNetoSTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotIVASTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotAdicSTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotExeSTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_ValSubtotSTI ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_LineasDeta ::
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field
            (list (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_lineas_deta))

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_sub_tot_info)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_NroSTI => x.nro_sti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_GlosaSTI => x.glosa_sti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_OrdenSTI => x.orden_sti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotNetoSTI => x.sub_tot_neto_sti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotIVASTI => x.sub_tot_ivasti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotAdicSTI => x.sub_tot_adic_sti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotExeSTI => x.sub_tot_exe_sti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_ValSubtotSTI => x.val_subtot_sti
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_LineasDeta => x.lineas_deta

inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_NroSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_GlosaSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_OrdenSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotNetoSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotIVASTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotAdicSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotExeSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_ValSubtotSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_LineasDeta
    ]
inl x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_NroSTI =>
        { tag = "NroSTI"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_GlosaSTI =>
        { tag = "GlosaSTI"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_OrdenSTI =>
        { tag = "OrdenSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotNetoSTI =>
        { tag = "SubTotNetoSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotIVASTI =>
        { tag = "SubTotIVASTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotAdicSTI =>
        { tag = "SubTotAdicSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotExeSTI =>
        { tag = "SubTotExeSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_ValSubtotSTI =>
        { tag = "ValSubtotSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_LineasDeta =>
        { tag = "LineasDeta"; min = 0i32; max = Some 60i32 }

union x_dte_def_type_choice_documento_exportaciones_detalle_field r =
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NroLinDet ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_nro_lin_det)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_CdgItem ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (list (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_IndExe ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (option (x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NmbItem ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DscItem ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (option (x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyRef ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdRef ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (option (x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcRef ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyItem ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_Subcantidad ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (list (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchElabor ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchVencim ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdItem ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (option (x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcItem ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_OtrMnda ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (option (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoPct ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoMonto ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_mnt_imp_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubDscto ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (list (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoPct ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoMonto ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field (option (x_mnt_imp_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubRecargo ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (list (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_MontoItem ::
        x_dte_def_type_choice_documento_exportaciones_detalle_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_monto_item)

inl x_dte_def_type_choice_documento_exportaciones_detalle_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_detalle)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NroLinDet => x.nro_lin_det
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_CdgItem => x.cdg_item
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_IndExe => x.ind_exe
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NmbItem => x.nmb_item
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DscItem => x.dsc_item
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyRef => x.qty_ref
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdRef => x.unmd_ref
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcRef => x.prc_ref
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyItem => x.qty_item
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_Subcantidad => x.subcantidad
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchElabor => x.fch_elabor
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchVencim => x.fch_vencim
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdItem => x.unmd_item
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcItem => x.prc_item
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_OtrMnda => x.otr_mnda
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoPct => x.descuento_pct
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoMonto => x.descuento_monto
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubDscto => x.sub_dscto
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoPct => x.recargo_pct
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoMonto => x.recargo_monto
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubRecargo => x.sub_recargo
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_MontoItem => x.monto_item

inl x_dte_def_type_choice_documento_exportaciones_detalle_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_NroLinDet
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_CdgItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_IndExe
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_NmbItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_DscItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyRef
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdRef
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcRef
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_Subcantidad
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchElabor
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchVencim
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_OtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoPct
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoMonto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubDscto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoPct
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoMonto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubRecargo
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_MontoItem
    ]
inl x_dte_def_type_choice_documento_exportaciones_detalle_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NroLinDet =>
        { tag = "NroLinDet"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_CdgItem =>
        { tag = "CdgItem"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_IndExe => { tag = "IndExe"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NmbItem =>
        { tag = "NmbItem"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DscItem =>
        { tag = "DscItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyRef => { tag = "QtyRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdRef =>
        { tag = "UnmdRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcRef => { tag = "PrcRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyItem =>
        { tag = "QtyItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_Subcantidad =>
        { tag = "Subcantidad"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchElabor =>
        { tag = "FchElabor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchVencim =>
        { tag = "FchVencim"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdItem =>
        { tag = "UnmdItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcItem =>
        { tag = "PrcItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_OtrMnda =>
        { tag = "OtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoPct =>
        { tag = "DescuentoPct"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoMonto =>
        { tag = "DescuentoMonto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubDscto =>
        { tag = "SubDscto"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoPct =>
        { tag = "RecargoPct"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoMonto =>
        { tag = "RecargoMonto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubRecargo =>
        { tag = "SubRecargo"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_field_MontoItem =>
        { tag = "MontoItem"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field r =
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_TipoRecargo ::
        x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_ValorRecargo ::
        x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field (x_dec16_2_type)

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_TipoRecargo => x.tipo_recargo
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_ValorRecargo => x.valor_recargo

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_TipoRecargo
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_ValorRecargo
    ]
inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_TipoRecargo =>
        { tag = "TipoRecargo"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_ValorRecargo =>
        { tag = "ValorRecargo"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field r =
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_TipoDscto ::
        x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_ValorDscto ::
        x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field (x_dec16_2_type)

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_TipoDscto => x.tipo_dscto
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_ValorDscto => x.valor_dscto

inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_TipoDscto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_ValorDscto
    ]
inl x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_TipoDscto =>
        { tag = "TipoDscto"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_ValorDscto =>
        { tag = "ValorDscto"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field r =
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_PrcOtrMon ::
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field (x_dec14_4_type)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_Moneda ::
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_FctConv ::
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field (option (x_dec6_4_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_DctoOtrMnda ::
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_RecargoOtrMnda ::
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field
            (option (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_recargo_otr_mnda))
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_MontoItemOtrMnda ::
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_monto_item_otr_mnda)

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_PrcOtrMon => x.prc_otr_mon
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_Moneda => x.moneda
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_FctConv => x.fct_conv
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_DctoOtrMnda => x.dcto_otr_mnda
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_RecargoOtrMnda => x.recargo_otr_mnda
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_MontoItemOtrMnda => x.monto_item_otr_mnda

inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_PrcOtrMon
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_Moneda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_FctConv
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_DctoOtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_RecargoOtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_MontoItemOtrMnda
    ]
inl x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_PrcOtrMon =>
        { tag = "PrcOtrMon"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_Moneda =>
        { tag = "Moneda"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_FctConv =>
        { tag = "FctConv"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_DctoOtrMnda =>
        { tag = "DctoOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_RecargoOtrMnda =>
        { tag = "RecargoOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_MontoItemOtrMnda =>
        { tag = "MontoItemOtrMnda"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field r =
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubQty ::
        x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field (x_dec12_6_type)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubCod ::
        x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_TipCodSubQty ::
        x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field
            (option (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty))

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubQty => x.sub_qty
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubCod => x.sub_cod
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_TipCodSubQty => x.tip_cod_sub_qty

inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubQty
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubCod
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_TipCodSubQty
    ]
inl x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubQty =>
        { tag = "SubQty"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubCod =>
        { tag = "SubCod"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_TipCodSubQty =>
        { tag = "TipCodSubQty"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field r =
    | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_TpoCodigo ::
        x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo)
    | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_VlrCodigo ::
        x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field
            (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo)

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_TpoCodigo => x.tpo_codigo
    | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_VlrCodigo => x.vlr_codigo

inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_TpoCodigo
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_VlrCodigo
    ]
inl x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_TpoCodigo =>
        { tag = "TpoCodigo"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_VlrCodigo =>
        { tag = "VlrCodigo"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_IdDoc ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Emisor ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Receptor ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Transporte ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Totales ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_totales)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_OtraMoneda ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_IdDoc => x.id_doc
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Emisor => x.emisor
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Receptor => x.receptor
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Transporte => x.transporte
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Totales => x.totales
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_OtraMoneda => x.otra_moneda

inl x_dte_def_type_choice_documento_exportaciones_encabezado_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_IdDoc
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Emisor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Receptor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Transporte
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Totales
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_OtraMoneda
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_IdDoc => { tag = "IdDoc"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Emisor =>
        { tag = "Emisor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Receptor =>
        { tag = "Receptor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Transporte =>
        { tag = "Transporte"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Totales =>
        { tag = "Totales"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_OtraMoneda =>
        { tag = "OtraMoneda"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoMoneda ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field (x_tip_mon_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoCambio ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field (option (x_dec6_4_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntExeOtrMnda ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntTotOtrMnda ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field (x_dec14_4_type)

inl x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoMoneda => x.tpo_moneda
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoCambio => x.tpo_cambio
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntExeOtrMnda => x.mnt_exe_otr_mnda
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntTotOtrMnda => x.mnt_tot_otr_mnda

inl x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoMoneda
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoCambio
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntExeOtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntTotOtrMnda
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoMoneda =>
        { tag = "TpoMoneda"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoCambio =>
        { tag = "TpoCambio"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntExeOtrMnda =>
        { tag = "MntExeOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntTotOtrMnda =>
        { tag = "MntTotOtrMnda"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_TpoMoneda ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field (x_tip_mon_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntExe ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_exe)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntTotal ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_totales_mnt_total)

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_totales)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_TpoMoneda => x.tpo_moneda
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntExe => x.mnt_exe
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntTotal => x.mnt_total

inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_TpoMoneda
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntExe
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntTotal
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_TpoMoneda =>
        { tag = "TpoMoneda"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntExe =>
        { tag = "MntExe"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntTotal =>
        { tag = "MntTotal"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Patente ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_RUTTrans ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Chofer ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_DirDest ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CmnaDest ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CiudadDest ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Aduana ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Patente => x.patente
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_RUTTrans => x.rut_trans
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Chofer => x.chofer
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_DirDest => x.dir_dest
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CmnaDest => x.cmna_dest
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CiudadDest => x.ciudad_dest
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Aduana => x.aduana

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Patente
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_RUTTrans
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Chofer
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_DirDest
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CmnaDest
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CiudadDest
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Aduana
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Patente =>
        { tag = "Patente"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_RUTTrans =>
        { tag = "RUTTrans"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Chofer =>
        { tag = "Chofer"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_DirDest =>
        { tag = "DirDest"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CmnaDest =>
        { tag = "CmnaDest"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CiudadDest =>
        { tag = "CiudadDest"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Aduana =>
        { tag = "Aduana"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodModVenta ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_mod_venta))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodClauVenta ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_clau_venta))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotClauVenta ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodViaTransp ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_via_transp))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NombreTransp ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_RUTCiaTransp ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NomCiaTransp ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicTransp ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Booking ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Operador ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoEmbarque ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_embarque))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoDesemb ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pto_desemb))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Tara ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tara))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidMedTara ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_med_tara))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoBruto ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_bruto))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_bruto))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoNeto ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_peso_neto))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_unid_peso_neto))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotItems ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_items))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotBultos ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tot_bultos))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TipoBultos ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (list (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntFlete ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntSeguro ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_recep))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisDestin ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_cod_pais_destin))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodModVenta => x.cod_mod_venta
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodClauVenta => x.cod_clau_venta
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotClauVenta => x.tot_clau_venta
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodViaTransp => x.cod_via_transp
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NombreTransp => x.nombre_transp
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_RUTCiaTransp => x.rut_cia_transp
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NomCiaTransp => x.nom_cia_transp
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicTransp => x.id_adic_transp
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Booking => x.booking
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Operador => x.operador
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoEmbarque => x.cod_pto_embarque
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb => x.id_adic_pto_emb
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoDesemb => x.cod_pto_desemb
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb =>
        x.id_adic_pto_desemb
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Tara => x.tara
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidMedTara => x.cod_unid_med_tara
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoBruto => x.peso_bruto
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto =>
        x.cod_unid_peso_bruto
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoNeto => x.peso_neto
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto =>
        x.cod_unid_peso_neto
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotItems => x.tot_items
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotBultos => x.tot_bultos
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TipoBultos => x.tipo_bultos
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntFlete => x.mnt_flete
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntSeguro => x.mnt_seguro
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisRecep => x.cod_pais_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisDestin => x.cod_pais_destin

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodModVenta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodClauVenta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotClauVenta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodViaTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NombreTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_RUTCiaTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NomCiaTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Booking
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Operador
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoEmbarque
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoDesemb
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Tara
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidMedTara
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoBruto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoNeto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotItems
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TipoBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntFlete
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntSeguro
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisDestin
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodModVenta =>
        { tag = "CodModVenta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodClauVenta =>
        { tag = "CodClauVenta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotClauVenta =>
        { tag = "TotClauVenta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodViaTransp =>
        { tag = "CodViaTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NombreTransp =>
        { tag = "NombreTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_RUTCiaTransp =>
        { tag = "RUTCiaTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NomCiaTransp =>
        { tag = "NomCiaTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicTransp =>
        { tag = "IdAdicTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Booking =>
        { tag = "Booking"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Operador =>
        { tag = "Operador"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoEmbarque =>
        { tag = "CodPtoEmbarque"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb =>
        { tag = "IdAdicPtoEmb"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoDesemb =>
        { tag = "CodPtoDesemb"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb =>
        { tag = "IdAdicPtoDesemb"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Tara =>
        { tag = "Tara"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidMedTara =>
        { tag = "CodUnidMedTara"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoBruto =>
        { tag = "PesoBruto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto =>
        { tag = "CodUnidPesoBruto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoNeto =>
        { tag = "PesoNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto =>
        { tag = "CodUnidPesoNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotItems =>
        { tag = "TotItems"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotBultos =>
        { tag = "TotBultos"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TipoBultos =>
        { tag = "TipoBultos"; min = 0i32; max = Some 10i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntFlete =>
        { tag = "MntFlete"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntSeguro =>
        { tag = "MntSeguro"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisRecep =>
        { tag = "CodPaisRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisDestin =>
        { tag = "CodPaisDestin"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field
            (
                option
                    (
                        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos
                    )
            )
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field
            (
                option
                    (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_cant_bultos)
            )
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field
            (
                option
                    (
                        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container
                    )
            )
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Sello ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field
            (
                option
                    (
                        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello
                    )
            )

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos =>
        x.cod_tpo_bultos
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos =>
        x.cant_bultos
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas => x.marcas
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer =>
        x.id_container
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Sello => x.sello
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello =>
        x.emisor_sello

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Sello
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos =>
        { tag = "CodTpoBultos"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos =>
        { tag = "CantBultos"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas =>
        { tag = "Marcas"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer =>
        { tag = "IdContainer"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Sello =>
        { tag = "Sello"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello =>
        { tag = "EmisorSello"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_RUTChofer ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field (x_rut_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_NombreChofer ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer)

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_RUTChofer => x.rut_chofer
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_NombreChofer => x.nombre_chofer

inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_RUTChofer
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_NombreChofer
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_RUTChofer =>
        { tag = "RUTChofer"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_NombreChofer =>
        { tag = "NombreChofer"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RUTRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field (x_rut_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CdgIntRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RznSocRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field (x_rzn_soc_larga_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Extranjero ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_GiroRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Contacto ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CorreoRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field (option (x_mail_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirPostal ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaPostal ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadPostal ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field (option (x_ciudad_type))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RUTRecep => x.rut_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CdgIntRecep => x.cdg_int_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RznSocRecep => x.rzn_soc_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Extranjero => x.extranjero
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_GiroRecep => x.giro_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Contacto => x.contacto
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CorreoRecep => x.correo_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirRecep => x.dir_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaRecep => x.cmna_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadRecep => x.ciudad_recep
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirPostal => x.dir_postal
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaPostal => x.cmna_postal
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadPostal => x.ciudad_postal

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RUTRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CdgIntRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RznSocRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Extranjero
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_GiroRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Contacto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CorreoRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirPostal
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaPostal
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadPostal
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RUTRecep =>
        { tag = "RUTRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CdgIntRecep =>
        { tag = "CdgIntRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RznSocRecep =>
        { tag = "RznSocRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Extranjero =>
        { tag = "Extranjero"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_GiroRecep =>
        { tag = "GiroRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Contacto =>
        { tag = "Contacto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CorreoRecep =>
        { tag = "CorreoRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirRecep =>
        { tag = "DirRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaRecep =>
        { tag = "CmnaRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadRecep =>
        { tag = "CiudadRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirPostal =>
        { tag = "DirPostal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaPostal =>
        { tag = "CmnaPostal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadPostal =>
        { tag = "CiudadPostal"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_NumId ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_Nacionalidad ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_IdAdicRecep ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_NumId => x.num_id
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_Nacionalidad => x.nacionalidad
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_IdAdicRecep => x.id_adic_recep

inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_NumId
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_Nacionalidad
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_IdAdicRecep
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_NumId =>
        { tag = "NumId"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_Nacionalidad =>
        { tag = "Nacionalidad"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_IdAdicRecep =>
        { tag = "IdAdicRecep"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RUTEmisor ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field (x_rut_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RznSoc ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_GiroEmis ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Telefono ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (list (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CorreoEmisor ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field (option (x_mail_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Acteco ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (list (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_acteco))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Sucursal ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgSIISucur ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_sii_sucur))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CodAdicSucur ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_DirOrigen ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CmnaOrigen ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CiudadOrigen ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgVendedor ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_IdAdicEmisor ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RUTEmisor => x.rut_emisor
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RznSoc => x.rzn_soc
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_GiroEmis => x.giro_emis
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Telefono => x.telefono
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CorreoEmisor => x.correo_emisor
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Acteco => x.acteco
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Sucursal => x.sucursal
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgSIISucur => x.cdg_sii_sucur
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CodAdicSucur => x.cod_adic_sucur
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_DirOrigen => x.dir_origen
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CmnaOrigen => x.cmna_origen
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CiudadOrigen => x.ciudad_origen
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgVendedor => x.cdg_vendedor
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_IdAdicEmisor => x.id_adic_emisor

inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RUTEmisor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RznSoc
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_GiroEmis
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Telefono
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CorreoEmisor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Acteco
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Sucursal
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgSIISucur
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CodAdicSucur
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_DirOrigen
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CmnaOrigen
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CiudadOrigen
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgVendedor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_IdAdicEmisor
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RUTEmisor =>
        { tag = "RUTEmisor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RznSoc =>
        { tag = "RznSoc"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_GiroEmis =>
        { tag = "GiroEmis"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Telefono =>
        { tag = "Telefono"; min = 0i32; max = Some 2i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CorreoEmisor =>
        { tag = "CorreoEmisor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Acteco =>
        { tag = "Acteco"; min = 1i32; max = Some 4i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Sucursal =>
        { tag = "Sucursal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgSIISucur =>
        { tag = "CdgSIISucur"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CodAdicSucur =>
        { tag = "CodAdicSucur"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_DirOrigen =>
        { tag = "DirOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CmnaOrigen =>
        { tag = "CmnaOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CiudadOrigen =>
        { tag = "CiudadOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgVendedor =>
        { tag = "CdgVendedor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_IdAdicEmisor =>
        { tag = "IdAdicEmisor"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDTE ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (x_exp_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_Folio ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (x_folio_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchEmis ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (x_fecha_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDespacho ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_IndServicio ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPago ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPagExp ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pag_exp))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchCancel ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntCancel ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_SaldoInsol ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntPagos ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (list (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoDesde ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoHasta ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MedioPago ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (option (x_medio_pago_type))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TpoCtaPago ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_NumCtaPago ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_BcoPago ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoCdg ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoGlosa ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoDias ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_dias))
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchVenc ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field (option (x_fecha_type))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDTE => x.tipo_dte
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_Folio => x.folio
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchEmis => x.fch_emis
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDespacho => x.tipo_despacho
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_IndServicio => x.ind_servicio
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPago => x.fma_pago
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPagExp => x.fma_pag_exp
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchCancel => x.fch_cancel
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntCancel => x.mnt_cancel
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_SaldoInsol => x.saldo_insol
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntPagos => x.mnt_pagos
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoDesde => x.periodo_desde
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoHasta => x.periodo_hasta
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MedioPago => x.medio_pago
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TpoCtaPago => x.tpo_cta_pago
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_NumCtaPago => x.num_cta_pago
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_BcoPago => x.bco_pago
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoCdg => x.term_pago_cdg
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoGlosa => x.term_pago_glosa
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoDias => x.term_pago_dias
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchVenc => x.fch_venc

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDTE
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_Folio
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchEmis
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDespacho
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_IndServicio
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPagExp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchCancel
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntCancel
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_SaldoInsol
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntPagos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoDesde
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoHasta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MedioPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TpoCtaPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_NumCtaPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_BcoPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoCdg
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoGlosa
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoDias
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchVenc
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDTE =>
        { tag = "TipoDTE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_Folio =>
        { tag = "Folio"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchEmis =>
        { tag = "FchEmis"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDespacho =>
        { tag = "TipoDespacho"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_IndServicio =>
        { tag = "IndServicio"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPago =>
        { tag = "FmaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPagExp =>
        { tag = "FmaPagExp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchCancel =>
        { tag = "FchCancel"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntCancel =>
        { tag = "MntCancel"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_SaldoInsol =>
        { tag = "SaldoInsol"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntPagos =>
        { tag = "MntPagos"; min = 0i32; max = Some 30i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoDesde =>
        { tag = "PeriodoDesde"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoHasta =>
        { tag = "PeriodoHasta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MedioPago =>
        { tag = "MedioPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TpoCtaPago =>
        { tag = "TpoCtaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_NumCtaPago =>
        { tag = "NumCtaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_BcoPago =>
        { tag = "BcoPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoCdg =>
        { tag = "TermPagoCdg"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoGlosa =>
        { tag = "TermPagoGlosa"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoDias =>
        { tag = "TermPagoDias"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchVenc =>
        { tag = "FchVenc"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field r =
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_FchPago ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field (x_fecha_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_MntPago ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field (x_monto_type)
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_GlosaPagos ::
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field
            (option (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos))

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_get forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field r)
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_FchPago => x.fch_pago
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_MntPago => x.mnt_pago
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_GlosaPagos => x.glosa_pagos

inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field r)
    =
    [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_FchPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_MntPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_GlosaPagos
    ]
inl x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field_meta forall r.
    (f : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_FchPago =>
        { tag = "FchPago"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_MntPago =>
        { tag = "MntPago"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_GlosaPagos =>
        { tag = "GlosaPagos"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_field r =
    | DTEDefType_choice_Documento_Liquidacion_field_Encabezado ::
        x_dte_def_type_choice_documento_liquidacion_field (x_dte_def_type_choice_documento_liquidacion_encabezado)
    | DTEDefType_choice_Documento_Liquidacion_field_Detalle ::
        x_dte_def_type_choice_documento_liquidacion_field (list (x_dte_def_type_choice_documento_liquidacion_detalle))
    | DTEDefType_choice_Documento_Liquidacion_field_SubTotInfo ::
        x_dte_def_type_choice_documento_liquidacion_field
            (list (x_dte_def_type_choice_documento_liquidacion_sub_tot_info))
    | DTEDefType_choice_Documento_Liquidacion_field_Referencia ::
        x_dte_def_type_choice_documento_liquidacion_field
            (list (x_dte_def_type_choice_documento_liquidacion_referencia))
    | DTEDefType_choice_Documento_Liquidacion_field_Comisiones ::
        x_dte_def_type_choice_documento_liquidacion_field
            (list (x_dte_def_type_choice_documento_liquidacion_comisiones))
    | DTEDefType_choice_Documento_Liquidacion_field_TED ::
        x_dte_def_type_choice_documento_liquidacion_field (x_dte_def_type_choice_documento_liquidacion_ted)
    | DTEDefType_choice_Documento_Liquidacion_field_TmstFirma ::
        x_dte_def_type_choice_documento_liquidacion_field (x_fecha_hora_type)

inl x_dte_def_type_choice_documento_liquidacion_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_field r) (x : x_dte_def_type_choice_documento_liquidacion)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_field_Encabezado => x.encabezado
    | DTEDefType_choice_Documento_Liquidacion_field_Detalle => x.detalle
    | DTEDefType_choice_Documento_Liquidacion_field_SubTotInfo => x.sub_tot_info
    | DTEDefType_choice_Documento_Liquidacion_field_Referencia => x.referencia
    | DTEDefType_choice_Documento_Liquidacion_field_Comisiones => x.comisiones
    | DTEDefType_choice_Documento_Liquidacion_field_TED => x.ted
    | DTEDefType_choice_Documento_Liquidacion_field_TmstFirma => x.tmst_firma

inl x_dte_def_type_choice_documento_liquidacion_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_field_Encabezado
        exists DTEDefType_choice_Documento_Liquidacion_field_Detalle
        exists DTEDefType_choice_Documento_Liquidacion_field_SubTotInfo
        exists DTEDefType_choice_Documento_Liquidacion_field_Referencia
        exists DTEDefType_choice_Documento_Liquidacion_field_Comisiones
        exists DTEDefType_choice_Documento_Liquidacion_field_TED
        exists DTEDefType_choice_Documento_Liquidacion_field_TmstFirma
    ]
inl x_dte_def_type_choice_documento_liquidacion_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_field_Encabezado => { tag = "Encabezado"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_field_Detalle => { tag = "Detalle"; min = 1i32; max = Some 60i32 }
    | DTEDefType_choice_Documento_Liquidacion_field_SubTotInfo => { tag = "SubTotInfo"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Liquidacion_field_Referencia => { tag = "Referencia"; min = 0i32; max = Some 40i32 }
    | DTEDefType_choice_Documento_Liquidacion_field_Comisiones => { tag = "Comisiones"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Liquidacion_field_TED => { tag = "TED"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_field_TmstFirma => { tag = "TmstFirma"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_ted_field r =
    | DTEDefType_choice_Documento_Liquidacion_TED_field_DD ::
        x_dte_def_type_choice_documento_liquidacion_ted_field (x_dte_def_type_choice_documento_liquidacion_ted_dd)
    | DTEDefType_choice_Documento_Liquidacion_TED_field_FRMT ::
        x_dte_def_type_choice_documento_liquidacion_ted_field (x_dte_def_type_choice_documento_liquidacion_ted_frmt)

inl x_dte_def_type_choice_documento_liquidacion_ted_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_field r) (x : x_dte_def_type_choice_documento_liquidacion_ted)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_field_DD => x.dd
    | DTEDefType_choice_Documento_Liquidacion_TED_field_FRMT => x.frmt

inl x_dte_def_type_choice_documento_liquidacion_ted_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_TED_field_DD
        exists DTEDefType_choice_Documento_Liquidacion_TED_field_FRMT
    ]
inl x_dte_def_type_choice_documento_liquidacion_ted_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_field_DD => { tag = "DD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_field_FRMT => { tag = "FRMT"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_ted_dd_field r =
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RE ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field (x_rut_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TD ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field (x_liq_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_F ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field (x_folio_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_FE ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field (x_fecha_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RR ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field (x_rut_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RSR ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_MNT ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field (x_valor_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_IT1 ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_it1)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_CAF ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TSTED ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_field (x_fecha_hora_type)

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RE => x.re
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TD => x.td
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_F => x.f
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_FE => x.fe
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RR => x.rr
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RSR => x.rsr
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_MNT => x.mnt
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_IT1 => x.it1
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_CAF => x.caf
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TSTED => x.tsted

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RE
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TD
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_F
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_FE
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RR
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RSR
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_MNT
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_IT1
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_CAF
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TSTED
    ]
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RE => { tag = "RE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TD => { tag = "TD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_F => { tag = "F"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_FE => { tag = "FE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RR => { tag = "RR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RSR => { tag = "RSR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_MNT => { tag = "MNT"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_IT1 => { tag = "IT1"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_CAF => { tag = "CAF"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TSTED => { tag = "TSTED"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field r =
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_DA ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_FRMA ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma)

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_DA => x.da
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_FRMA => x.frma

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_DA
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_FRMA
    ]
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_DA => { tag = "DA"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_FRMA => { tag = "FRMA"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field r =
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RE ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field (x_rut_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RS ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field (x_rzn_soc_corta_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_TD ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field (x_liq_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RNG ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_FA ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field (x_fecha_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_choice_rsapk ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_IDK ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field (xsd_integer)

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RE => x.re
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RS => x.rs
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_TD => x.td
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RNG => x.rng
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_FA => x.fa
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_choice_rsapk => x.choice_rsapk
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_IDK => x.idk

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RE
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RS
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_TD
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RNG
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_FA
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_choice_rsapk
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_IDK
    ]
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RE => { tag = "RE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RS => { tag = "RS"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_TD => { tag = "TD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RNG => { tag = "RNG"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_FA => { tag = "FA"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_choice_rsapk =>
        { tag = "choice_rsapk"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_IDK => { tag = "IDK"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field r =
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field (string)

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P => x.p
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q => x.q
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G => x.g
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y => x.y

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y
    ]
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P =>
        { tag = "P"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q =>
        { tag = "Q"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G =>
        { tag = "G"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y =>
        { tag = "Y"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field r =
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field (string)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field (string)

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M => x.m
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E => x.e

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E
    ]
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M =>
        { tag = "M"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E =>
        { tag = "E"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field r =
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_D ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field (x_folio_type)
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_H ::
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field (x_folio_type)

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_D => x.d
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_H => x.h

inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_D
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_H
    ]
inl x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_D => { tag = "D"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_H => { tag = "H"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_comisiones_field r =
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_NroLinCom ::
        x_dte_def_type_choice_documento_liquidacion_comisiones_field
            (x_dte_def_type_choice_documento_liquidacion_comisiones_nro_lin_com)
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TipoMovim ::
        x_dte_def_type_choice_documento_liquidacion_comisiones_field
            (x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim)
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_Glosa ::
        x_dte_def_type_choice_documento_liquidacion_comisiones_field
            (x_dte_def_type_choice_documento_liquidacion_comisiones_glosa)
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TasaComision ::
        x_dte_def_type_choice_documento_liquidacion_comisiones_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComNeto ::
        x_dte_def_type_choice_documento_liquidacion_comisiones_field (x_valor_type)
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComExe ::
        x_dte_def_type_choice_documento_liquidacion_comisiones_field (x_valor_type)
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComIVA ::
        x_dte_def_type_choice_documento_liquidacion_comisiones_field (option (x_valor_type))

inl x_dte_def_type_choice_documento_liquidacion_comisiones_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_comisiones_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_comisiones)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_NroLinCom => x.nro_lin_com
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TipoMovim => x.tipo_movim
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_Glosa => x.glosa
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TasaComision => x.tasa_comision
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComNeto => x.val_com_neto
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComExe => x.val_com_exe
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComIVA => x.val_com_iva

inl x_dte_def_type_choice_documento_liquidacion_comisiones_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_comisiones_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_NroLinCom
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TipoMovim
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_Glosa
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TasaComision
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComIVA
    ]
inl x_dte_def_type_choice_documento_liquidacion_comisiones_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_comisiones_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_NroLinCom =>
        { tag = "NroLinCom"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TipoMovim =>
        { tag = "TipoMovim"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_Glosa => { tag = "Glosa"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TasaComision =>
        { tag = "TasaComision"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComNeto =>
        { tag = "ValComNeto"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComExe =>
        { tag = "ValComExe"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComIVA =>
        { tag = "ValComIVA"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_referencia_field r =
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_NroLinRef ::
        x_dte_def_type_choice_documento_liquidacion_referencia_field
            (x_dte_def_type_choice_documento_liquidacion_referencia_nro_lin_ref)
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_TpoDocRef ::
        x_dte_def_type_choice_documento_liquidacion_referencia_field
            (x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref)
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_IndGlobal ::
        x_dte_def_type_choice_documento_liquidacion_referencia_field
            (option (x_dte_def_type_choice_documento_liquidacion_referencia_ind_global))
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FolioRef ::
        x_dte_def_type_choice_documento_liquidacion_referencia_field (x_folio_r_type)
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FchRef ::
        x_dte_def_type_choice_documento_liquidacion_referencia_field (x_fecha_type)
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_CodRef ::
        x_dte_def_type_choice_documento_liquidacion_referencia_field
            (option (x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref))
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_RazonRef ::
        x_dte_def_type_choice_documento_liquidacion_referencia_field
            (option (x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref))

inl x_dte_def_type_choice_documento_liquidacion_referencia_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_referencia_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_referencia)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_NroLinRef => x.nro_lin_ref
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_TpoDocRef => x.tpo_doc_ref
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_IndGlobal => x.ind_global
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FolioRef => x.folio_ref
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FchRef => x.fch_ref
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_CodRef => x.cod_ref
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_RazonRef => x.razon_ref

inl x_dte_def_type_choice_documento_liquidacion_referencia_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_referencia_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_NroLinRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_TpoDocRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_IndGlobal
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_FolioRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_FchRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_CodRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_RazonRef
    ]
inl x_dte_def_type_choice_documento_liquidacion_referencia_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_referencia_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_NroLinRef =>
        { tag = "NroLinRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_TpoDocRef =>
        { tag = "TpoDocRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_IndGlobal =>
        { tag = "IndGlobal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FolioRef =>
        { tag = "FolioRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FchRef => { tag = "FchRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_CodRef => { tag = "CodRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Referencia_field_RazonRef =>
        { tag = "RazonRef"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field r =
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_NroSTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field
            (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_nro_sti)
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_GlosaSTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field
            (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti)
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_OrdenSTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field
            (option (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_orden_sti))
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotNetoSTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotIVASTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotAdicSTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotExeSTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_ValSubtotSTI ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_LineasDeta ::
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field
            (list (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_lineas_deta))

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_sub_tot_info)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_NroSTI => x.nro_sti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_GlosaSTI => x.glosa_sti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_OrdenSTI => x.orden_sti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotNetoSTI => x.sub_tot_neto_sti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotIVASTI => x.sub_tot_ivasti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotAdicSTI => x.sub_tot_adic_sti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotExeSTI => x.sub_tot_exe_sti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_ValSubtotSTI => x.val_subtot_sti
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_LineasDeta => x.lineas_deta

inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_NroSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_GlosaSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_OrdenSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotNetoSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotIVASTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotAdicSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotExeSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_ValSubtotSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_LineasDeta
    ]
inl x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_NroSTI => { tag = "NroSTI"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_GlosaSTI =>
        { tag = "GlosaSTI"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_OrdenSTI =>
        { tag = "OrdenSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotNetoSTI =>
        { tag = "SubTotNetoSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotIVASTI =>
        { tag = "SubTotIVASTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotAdicSTI =>
        { tag = "SubTotAdicSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotExeSTI =>
        { tag = "SubTotExeSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_ValSubtotSTI =>
        { tag = "ValSubtotSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_LineasDeta =>
        { tag = "LineasDeta"; min = 0i32; max = Some 60i32 }

union x_dte_def_type_choice_documento_liquidacion_detalle_field r =
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NroLinDet ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (x_dte_def_type_choice_documento_liquidacion_detalle_nro_lin_det)
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CdgItem ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (list (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_TpoDocLiq ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq)
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_IndExe ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (option (x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NmbItem ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item)
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_DscItem ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (option (x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyRef ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdRef ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (option (x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcRef ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyItem ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_Subcantidad ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (list (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchElabor ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchVencim ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdItem ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field
            (option (x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcItem ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CodImpAdic ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (list (x_imp_adic_dte_type))
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_MontoItem ::
        x_dte_def_type_choice_documento_liquidacion_detalle_field (x_valor_type)

inl x_dte_def_type_choice_documento_liquidacion_detalle_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_detalle_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_detalle)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NroLinDet => x.nro_lin_det
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CdgItem => x.cdg_item
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_TpoDocLiq => x.tpo_doc_liq
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_IndExe => x.ind_exe
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NmbItem => x.nmb_item
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_DscItem => x.dsc_item
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyRef => x.qty_ref
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdRef => x.unmd_ref
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcRef => x.prc_ref
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyItem => x.qty_item
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_Subcantidad => x.subcantidad
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchElabor => x.fch_elabor
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchVencim => x.fch_vencim
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdItem => x.unmd_item
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcItem => x.prc_item
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CodImpAdic => x.cod_imp_adic
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_MontoItem => x.monto_item

inl x_dte_def_type_choice_documento_liquidacion_detalle_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_detalle_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_NroLinDet
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_CdgItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_TpoDocLiq
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_IndExe
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_NmbItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_DscItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyRef
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdRef
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcRef
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_Subcantidad
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchElabor
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchVencim
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_CodImpAdic
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_MontoItem
    ]
inl x_dte_def_type_choice_documento_liquidacion_detalle_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_detalle_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NroLinDet =>
        { tag = "NroLinDet"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CdgItem => { tag = "CdgItem"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_TpoDocLiq =>
        { tag = "TpoDocLiq"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_IndExe => { tag = "IndExe"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NmbItem => { tag = "NmbItem"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_DscItem => { tag = "DscItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyRef => { tag = "QtyRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdRef => { tag = "UnmdRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcRef => { tag = "PrcRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyItem => { tag = "QtyItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_Subcantidad =>
        { tag = "Subcantidad"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchElabor =>
        { tag = "FchElabor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchVencim =>
        { tag = "FchVencim"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdItem =>
        { tag = "UnmdItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcItem => { tag = "PrcItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CodImpAdic =>
        { tag = "CodImpAdic"; min = 0i32; max = Some 2i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_field_MontoItem =>
        { tag = "MontoItem"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field r =
    | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubQty ::
        x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field (x_dec12_6_type)
    | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubCod ::
        x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field
            (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod)

inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubQty => x.sub_qty
    | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubCod => x.sub_cod

inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubQty
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubCod
    ]
inl x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubQty =>
        { tag = "SubQty"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubCod =>
        { tag = "SubCod"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field r =
    | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_TpoCodigo ::
        x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field
            (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo)
    | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_VlrCodigo ::
        x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field
            (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo)

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_TpoCodigo => x.tpo_codigo
    | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_VlrCodigo => x.vlr_codigo

inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_TpoCodigo
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_VlrCodigo
    ]
inl x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_TpoCodigo =>
        { tag = "TpoCodigo"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_VlrCodigo =>
        { tag = "VlrCodigo"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_IdDoc ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Emisor ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Receptor ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Totales ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_totales)

inl x_dte_def_type_choice_documento_liquidacion_encabezado_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_IdDoc => x.id_doc
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Emisor => x.emisor
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Receptor => x.receptor
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Totales => x.totales

inl x_dte_def_type_choice_documento_liquidacion_encabezado_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_IdDoc
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Emisor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Receptor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Totales
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_IdDoc => { tag = "IdDoc"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Emisor => { tag = "Emisor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Receptor =>
        { tag = "Receptor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Totales =>
        { tag = "Totales"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntNeto ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntExe ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_TasaIVA ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVA ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVAProp ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVATerc ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_ImptoReten ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field
            (list (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_Comisiones ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntTotal ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (x_valor_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MontoPeriodo ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_SaldoAnterior ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_VlrPagar ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field (option (x_valor_type))

inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntNeto => x.mnt_neto
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntExe => x.mnt_exe
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_TasaIVA => x.tasa_iva
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVA => x.iva
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVAProp => x.iva_prop
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVATerc => x.iva_terc
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_ImptoReten => x.impto_reten
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_Comisiones => x.comisiones
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntTotal => x.mnt_total
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MontoPeriodo => x.monto_periodo
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_SaldoAnterior => x.saldo_anterior
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_VlrPagar => x.vlr_pagar

inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntNeto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntExe
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_TasaIVA
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVA
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVAProp
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVATerc
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_ImptoReten
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_Comisiones
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntTotal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MontoPeriodo
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_SaldoAnterior
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_VlrPagar
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntNeto =>
        { tag = "MntNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntExe =>
        { tag = "MntExe"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_TasaIVA =>
        { tag = "TasaIVA"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVA =>
        { tag = "IVA"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVAProp =>
        { tag = "IVAProp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVATerc =>
        { tag = "IVATerc"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_ImptoReten =>
        { tag = "ImptoReten"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_Comisiones =>
        { tag = "Comisiones"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntTotal =>
        { tag = "MntTotal"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MontoPeriodo =>
        { tag = "MontoPeriodo"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_SaldoAnterior =>
        { tag = "SaldoAnterior"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_VlrPagar =>
        { tag = "VlrPagar"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComNeto ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComExe ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComIVA ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field (option (x_valor_type))

inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComNeto => x.val_com_neto
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComExe => x.val_com_exe
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComIVA => x.val_com_iva

inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComIVA
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComNeto =>
        { tag = "ValComNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComExe =>
        { tag = "ValComExe"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComIVA =>
        { tag = "ValComIVA"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TipoImp ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field (x_imp_adic_dte_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TasaImp ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_MontoImp ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field (x_mnt_imp_type)

inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TipoImp => x.tipo_imp
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TasaImp => x.tasa_imp
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_MontoImp => x.monto_imp

inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TipoImp
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TasaImp
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_MontoImp
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TipoImp =>
        { tag = "TipoImp"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TasaImp =>
        { tag = "TasaImp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_MontoImp =>
        { tag = "MontoImp"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RUTRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field (x_rut_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CdgIntRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RznSocRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field (x_rzn_soc_larga_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_GiroRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_Contacto ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CorreoRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field (option (x_mail_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field (x_comuna_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadRecep ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirPostal ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaPostal ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadPostal ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field (option (x_ciudad_type))

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RUTRecep => x.rut_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CdgIntRecep => x.cdg_int_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RznSocRecep => x.rzn_soc_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_GiroRecep => x.giro_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_Contacto => x.contacto
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CorreoRecep => x.correo_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirRecep => x.dir_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaRecep => x.cmna_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadRecep => x.ciudad_recep
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirPostal => x.dir_postal
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaPostal => x.cmna_postal
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadPostal => x.ciudad_postal

inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RUTRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CdgIntRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RznSocRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_GiroRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_Contacto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CorreoRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirPostal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaPostal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadPostal
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RUTRecep =>
        { tag = "RUTRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CdgIntRecep =>
        { tag = "CdgIntRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RznSocRecep =>
        { tag = "RznSocRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_GiroRecep =>
        { tag = "GiroRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_Contacto =>
        { tag = "Contacto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CorreoRecep =>
        { tag = "CorreoRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirRecep =>
        { tag = "DirRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaRecep =>
        { tag = "CmnaRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadRecep =>
        { tag = "CiudadRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirPostal =>
        { tag = "DirPostal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaPostal =>
        { tag = "CmnaPostal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadPostal =>
        { tag = "CiudadPostal"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RUTEmisor ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field (x_rut_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RznSoc ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_GiroEmis ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Telefono ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (list (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CorreoEmisor ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field (option (x_mail_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Acteco ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (list (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_acteco))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Sucursal ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgSIISucur ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_sii_sucur))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_DirOrigen ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CmnaOrigen ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CiudadOrigen ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgVendedor ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor))

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RUTEmisor => x.rut_emisor
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RznSoc => x.rzn_soc
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_GiroEmis => x.giro_emis
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Telefono => x.telefono
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CorreoEmisor => x.correo_emisor
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Acteco => x.acteco
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Sucursal => x.sucursal
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgSIISucur => x.cdg_sii_sucur
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_DirOrigen => x.dir_origen
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CmnaOrigen => x.cmna_origen
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CiudadOrigen => x.ciudad_origen
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgVendedor => x.cdg_vendedor

inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RUTEmisor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RznSoc
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_GiroEmis
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Telefono
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CorreoEmisor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Acteco
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Sucursal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgSIISucur
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_DirOrigen
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CmnaOrigen
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CiudadOrigen
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgVendedor
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RUTEmisor =>
        { tag = "RUTEmisor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RznSoc =>
        { tag = "RznSoc"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_GiroEmis =>
        { tag = "GiroEmis"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Telefono =>
        { tag = "Telefono"; min = 0i32; max = Some 2i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CorreoEmisor =>
        { tag = "CorreoEmisor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Acteco =>
        { tag = "Acteco"; min = 1i32; max = Some 4i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Sucursal =>
        { tag = "Sucursal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgSIISucur =>
        { tag = "CdgSIISucur"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_DirOrigen =>
        { tag = "DirOrigen"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CmnaOrigen =>
        { tag = "CmnaOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CiudadOrigen =>
        { tag = "CiudadOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgVendedor =>
        { tag = "CdgVendedor"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TipoDTE ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (x_liq_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_Folio ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (x_folio_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchEmis ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (x_fecha_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_IndServicio ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntBruto ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FmaPago ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchCancel ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntCancel ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_SaldoInsol ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntPagos ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (list (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoDesde ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoHasta ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MedioPago ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (option (x_medio_pago_type))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TpoCtaPago ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_NumCtaPago ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_BcoPago ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoCdg ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoGlosa ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoDias ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_dias))
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchVenc ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field (option (x_fecha_type))

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TipoDTE => x.tipo_dte
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_Folio => x.folio
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchEmis => x.fch_emis
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_IndServicio => x.ind_servicio
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntBruto => x.mnt_bruto
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FmaPago => x.fma_pago
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchCancel => x.fch_cancel
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntCancel => x.mnt_cancel
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_SaldoInsol => x.saldo_insol
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntPagos => x.mnt_pagos
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoDesde => x.periodo_desde
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoHasta => x.periodo_hasta
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MedioPago => x.medio_pago
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TpoCtaPago => x.tpo_cta_pago
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_NumCtaPago => x.num_cta_pago
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_BcoPago => x.bco_pago
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoCdg => x.term_pago_cdg
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoGlosa => x.term_pago_glosa
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoDias => x.term_pago_dias
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchVenc => x.fch_venc

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TipoDTE
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_Folio
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchEmis
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_IndServicio
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntBruto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FmaPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchCancel
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntCancel
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_SaldoInsol
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntPagos
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoDesde
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoHasta
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MedioPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TpoCtaPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_NumCtaPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_BcoPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoCdg
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoGlosa
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoDias
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchVenc
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TipoDTE =>
        { tag = "TipoDTE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_Folio =>
        { tag = "Folio"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchEmis =>
        { tag = "FchEmis"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_IndServicio =>
        { tag = "IndServicio"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntBruto =>
        { tag = "MntBruto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FmaPago =>
        { tag = "FmaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchCancel =>
        { tag = "FchCancel"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntCancel =>
        { tag = "MntCancel"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_SaldoInsol =>
        { tag = "SaldoInsol"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntPagos =>
        { tag = "MntPagos"; min = 0i32; max = Some 30i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoDesde =>
        { tag = "PeriodoDesde"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoHasta =>
        { tag = "PeriodoHasta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MedioPago =>
        { tag = "MedioPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TpoCtaPago =>
        { tag = "TpoCtaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_NumCtaPago =>
        { tag = "NumCtaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_BcoPago =>
        { tag = "BcoPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoCdg =>
        { tag = "TermPagoCdg"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoGlosa =>
        { tag = "TermPagoGlosa"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoDias =>
        { tag = "TermPagoDias"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchVenc =>
        { tag = "FchVenc"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field r =
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_FchPago ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field (x_fecha_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_MntPago ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field (x_monto_type)
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_GlosaPagos ::
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field
            (option (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos))

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_get forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field r)
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_FchPago => x.fch_pago
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_MntPago => x.mnt_pago
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_GlosaPagos => x.glosa_pagos

inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field r)
    =
    [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_FchPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_MntPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_GlosaPagos
    ]
inl x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field_meta forall r.
    (f : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_FchPago =>
        { tag = "FchPago"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_MntPago =>
        { tag = "MntPago"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_GlosaPagos =>
        { tag = "GlosaPagos"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_field r =
    | DTEDefType_choice_Documento_Documento_field_Encabezado ::
        x_dte_def_type_choice_documento_documento_field (x_dte_def_type_choice_documento_documento_encabezado)
    | DTEDefType_choice_Documento_Documento_field_Detalle ::
        x_dte_def_type_choice_documento_documento_field (list (x_dte_def_type_choice_documento_documento_detalle))
    | DTEDefType_choice_Documento_Documento_field_SubTotInfo ::
        x_dte_def_type_choice_documento_documento_field (list (x_dte_def_type_choice_documento_documento_sub_tot_info))
    | DTEDefType_choice_Documento_Documento_field_DscRcgGlobal ::
        x_dte_def_type_choice_documento_documento_field
            (list (x_dte_def_type_choice_documento_documento_dsc_rcg_global))
    | DTEDefType_choice_Documento_Documento_field_Referencia ::
        x_dte_def_type_choice_documento_documento_field (list (x_dte_def_type_choice_documento_documento_referencia))
    | DTEDefType_choice_Documento_Documento_field_Comisiones ::
        x_dte_def_type_choice_documento_documento_field (list (x_dte_def_type_choice_documento_documento_comisiones))
    | DTEDefType_choice_Documento_Documento_field_TED ::
        x_dte_def_type_choice_documento_documento_field (x_dte_def_type_choice_documento_documento_ted)
    | DTEDefType_choice_Documento_Documento_field_TmstFirma ::
        x_dte_def_type_choice_documento_documento_field (x_fecha_hora_type)

inl x_dte_def_type_choice_documento_documento_get forall r.
    (f : x_dte_def_type_choice_documento_documento_field r) (x : x_dte_def_type_choice_documento_documento)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_field_Encabezado => x.encabezado
    | DTEDefType_choice_Documento_Documento_field_Detalle => x.detalle
    | DTEDefType_choice_Documento_Documento_field_SubTotInfo => x.sub_tot_info
    | DTEDefType_choice_Documento_Documento_field_DscRcgGlobal => x.dsc_rcg_global
    | DTEDefType_choice_Documento_Documento_field_Referencia => x.referencia
    | DTEDefType_choice_Documento_Documento_field_Comisiones => x.comisiones
    | DTEDefType_choice_Documento_Documento_field_TED => x.ted
    | DTEDefType_choice_Documento_Documento_field_TmstFirma => x.tmst_firma

inl x_dte_def_type_choice_documento_documento_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_field_Encabezado
        exists DTEDefType_choice_Documento_Documento_field_Detalle
        exists DTEDefType_choice_Documento_Documento_field_SubTotInfo
        exists DTEDefType_choice_Documento_Documento_field_DscRcgGlobal
        exists DTEDefType_choice_Documento_Documento_field_Referencia
        exists DTEDefType_choice_Documento_Documento_field_Comisiones
        exists DTEDefType_choice_Documento_Documento_field_TED
        exists DTEDefType_choice_Documento_Documento_field_TmstFirma
    ]
inl x_dte_def_type_choice_documento_documento_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_field_Encabezado => { tag = "Encabezado"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_field_Detalle => { tag = "Detalle"; min = 1i32; max = Some 60i32 }
    | DTEDefType_choice_Documento_Documento_field_SubTotInfo => { tag = "SubTotInfo"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Documento_field_DscRcgGlobal => { tag = "DscRcgGlobal"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Documento_field_Referencia => { tag = "Referencia"; min = 0i32; max = Some 40i32 }
    | DTEDefType_choice_Documento_Documento_field_Comisiones => { tag = "Comisiones"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Documento_field_TED => { tag = "TED"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_field_TmstFirma => { tag = "TmstFirma"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_ted_field r =
    | DTEDefType_choice_Documento_Documento_TED_field_DD ::
        x_dte_def_type_choice_documento_documento_ted_field (x_dte_def_type_choice_documento_documento_ted_dd)
    | DTEDefType_choice_Documento_Documento_TED_field_FRMT ::
        x_dte_def_type_choice_documento_documento_ted_field (x_dte_def_type_choice_documento_documento_ted_frmt)

inl x_dte_def_type_choice_documento_documento_ted_get forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_field r) (x : x_dte_def_type_choice_documento_documento_ted)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_field_DD => x.dd
    | DTEDefType_choice_Documento_Documento_TED_field_FRMT => x.frmt

inl x_dte_def_type_choice_documento_documento_ted_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_ted_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_TED_field_DD
        exists DTEDefType_choice_Documento_Documento_TED_field_FRMT
    ]
inl x_dte_def_type_choice_documento_documento_ted_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_field_DD => { tag = "DD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_field_FRMT => { tag = "FRMT"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_ted_dd_field r =
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RE ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_rut_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_TD ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_dte_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_F ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_folio_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_FE ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_fecha_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RR ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_rut_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RSR ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_dte_def_type_choice_documento_documento_ted_dd_rsr)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_MNT ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (string)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_IT1 ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_dte_def_type_choice_documento_documento_ted_dd_it1)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_CAF ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_dte_def_type_choice_documento_documento_ted_dd_caf)
    | DTEDefType_choice_Documento_Documento_TED_DD_field_TSTED ::
        x_dte_def_type_choice_documento_documento_ted_dd_field (x_fecha_hora_type)

inl x_dte_def_type_choice_documento_documento_ted_dd_get forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_field r)
    (x : x_dte_def_type_choice_documento_documento_ted_dd)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RE => x.re
    | DTEDefType_choice_Documento_Documento_TED_DD_field_TD => x.td
    | DTEDefType_choice_Documento_Documento_TED_DD_field_F => x.f
    | DTEDefType_choice_Documento_Documento_TED_DD_field_FE => x.fe
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RR => x.rr
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RSR => x.rsr
    | DTEDefType_choice_Documento_Documento_TED_DD_field_MNT => x.mnt
    | DTEDefType_choice_Documento_Documento_TED_DD_field_IT1 => x.it1
    | DTEDefType_choice_Documento_Documento_TED_DD_field_CAF => x.caf
    | DTEDefType_choice_Documento_Documento_TED_DD_field_TSTED => x.tsted

inl x_dte_def_type_choice_documento_documento_ted_dd_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_RE
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_TD
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_F
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_FE
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_RR
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_RSR
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_MNT
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_IT1
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_CAF
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_TSTED
    ]
inl x_dte_def_type_choice_documento_documento_ted_dd_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RE => { tag = "RE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_TD => { tag = "TD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_F => { tag = "F"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_FE => { tag = "FE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RR => { tag = "RR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_RSR => { tag = "RSR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_MNT => { tag = "MNT"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_IT1 => { tag = "IT1"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_CAF => { tag = "CAF"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_field_TSTED => { tag = "TSTED"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_ted_dd_caf_field r =
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_DA ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_field
            (x_dte_def_type_choice_documento_documento_ted_dd_caf_da)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_FRMA ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_field
            (x_dte_def_type_choice_documento_documento_ted_dd_caf_frma)

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_get forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_field r)
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_DA => x.da
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_FRMA => x.frma

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_DA
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_FRMA
    ]
inl x_dte_def_type_choice_documento_documento_ted_dd_caf_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_DA => { tag = "DA"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_FRMA => { tag = "FRMA"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field r =
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RE ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field (x_rut_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RS ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_TD ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field (x_dte_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RNG ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_FA ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field (x_fecha_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_choice_rsapk ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field
            (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_IDK ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field (xsd_integer)

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_get forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field r)
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RE => x.re
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RS => x.rs
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_TD => x.td
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RNG => x.rng
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_FA => x.fa
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_choice_rsapk => x.choice_rsapk
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_IDK => x.idk

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RE
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RS
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_TD
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RNG
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_FA
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_choice_rsapk
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_IDK
    ]
inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RE => { tag = "RE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RS => { tag = "RS"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_TD => { tag = "TD"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RNG => { tag = "RNG"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_FA => { tag = "FA"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_choice_rsapk =>
        { tag = "choice_rsapk"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_IDK => { tag = "IDK"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field r =
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field (string)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field (string)

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_get forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P => x.p
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q => x.q
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G => x.g
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y => x.y

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y
    ]
inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P =>
        { tag = "P"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q =>
        { tag = "Q"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G =>
        { tag = "G"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y =>
        { tag = "Y"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field r =
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field (string)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field (string)

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_get forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M => x.m
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E => x.e

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E
    ]
inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M =>
        { tag = "M"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E =>
        { tag = "E"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field r =
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_D ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field (x_folio_type)
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_H ::
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field (x_folio_type)

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_get forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field r)
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_D => x.d
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_H => x.h

inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_D
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_H
    ]
inl x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_D => { tag = "D"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_H => { tag = "H"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_comisiones_field r =
    | DTEDefType_choice_Documento_Documento_Comisiones_field_NroLinCom ::
        x_dte_def_type_choice_documento_documento_comisiones_field
            (x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com)
    | DTEDefType_choice_Documento_Documento_Comisiones_field_TipoMovim ::
        x_dte_def_type_choice_documento_documento_comisiones_field
            (x_dte_def_type_choice_documento_documento_comisiones_tipo_movim)
    | DTEDefType_choice_Documento_Documento_Comisiones_field_Glosa ::
        x_dte_def_type_choice_documento_documento_comisiones_field
            (x_dte_def_type_choice_documento_documento_comisiones_glosa)
    | DTEDefType_choice_Documento_Documento_Comisiones_field_TasaComision ::
        x_dte_def_type_choice_documento_documento_comisiones_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComNeto ::
        x_dte_def_type_choice_documento_documento_comisiones_field (x_monto_type)
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComExe ::
        x_dte_def_type_choice_documento_documento_comisiones_field (x_monto_type)
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComIVA ::
        x_dte_def_type_choice_documento_documento_comisiones_field (option (x_monto_type))

inl x_dte_def_type_choice_documento_documento_comisiones_get forall r.
    (f : x_dte_def_type_choice_documento_documento_comisiones_field r)
    (x : x_dte_def_type_choice_documento_documento_comisiones)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Comisiones_field_NroLinCom => x.nro_lin_com
    | DTEDefType_choice_Documento_Documento_Comisiones_field_TipoMovim => x.tipo_movim
    | DTEDefType_choice_Documento_Documento_Comisiones_field_Glosa => x.glosa
    | DTEDefType_choice_Documento_Documento_Comisiones_field_TasaComision => x.tasa_comision
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComNeto => x.val_com_neto
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComExe => x.val_com_exe
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComIVA => x.val_com_iva

inl x_dte_def_type_choice_documento_documento_comisiones_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_comisiones_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_NroLinCom
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_TipoMovim
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_Glosa
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_TasaComision
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_ValComIVA
    ]
inl x_dte_def_type_choice_documento_documento_comisiones_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_comisiones_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Comisiones_field_NroLinCom =>
        { tag = "NroLinCom"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Comisiones_field_TipoMovim =>
        { tag = "TipoMovim"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Comisiones_field_Glosa => { tag = "Glosa"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Comisiones_field_TasaComision =>
        { tag = "TasaComision"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComNeto =>
        { tag = "ValComNeto"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComExe =>
        { tag = "ValComExe"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComIVA =>
        { tag = "ValComIVA"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_referencia_field r =
    | DTEDefType_choice_Documento_Documento_Referencia_field_NroLinRef ::
        x_dte_def_type_choice_documento_documento_referencia_field
            (x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref)
    | DTEDefType_choice_Documento_Documento_Referencia_field_TpoDocRef ::
        x_dte_def_type_choice_documento_documento_referencia_field
            (x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref)
    | DTEDefType_choice_Documento_Documento_Referencia_field_IndGlobal ::
        x_dte_def_type_choice_documento_documento_referencia_field
            (option (x_dte_def_type_choice_documento_documento_referencia_ind_global))
    | DTEDefType_choice_Documento_Documento_Referencia_field_FolioRef ::
        x_dte_def_type_choice_documento_documento_referencia_field (x_folio_r_type)
    | DTEDefType_choice_Documento_Documento_Referencia_field_RUTOtr ::
        x_dte_def_type_choice_documento_documento_referencia_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Documento_Referencia_field_FchRef ::
        x_dte_def_type_choice_documento_documento_referencia_field (x_fecha_type)
    | DTEDefType_choice_Documento_Documento_Referencia_field_CodRef ::
        x_dte_def_type_choice_documento_documento_referencia_field
            (option (x_dte_def_type_choice_documento_documento_referencia_cod_ref))
    | DTEDefType_choice_Documento_Documento_Referencia_field_RazonRef ::
        x_dte_def_type_choice_documento_documento_referencia_field
            (option (x_dte_def_type_choice_documento_documento_referencia_razon_ref))

inl x_dte_def_type_choice_documento_documento_referencia_get forall r.
    (f : x_dte_def_type_choice_documento_documento_referencia_field r)
    (x : x_dte_def_type_choice_documento_documento_referencia)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Referencia_field_NroLinRef => x.nro_lin_ref
    | DTEDefType_choice_Documento_Documento_Referencia_field_TpoDocRef => x.tpo_doc_ref
    | DTEDefType_choice_Documento_Documento_Referencia_field_IndGlobal => x.ind_global
    | DTEDefType_choice_Documento_Documento_Referencia_field_FolioRef => x.folio_ref
    | DTEDefType_choice_Documento_Documento_Referencia_field_RUTOtr => x.rut_otr
    | DTEDefType_choice_Documento_Documento_Referencia_field_FchRef => x.fch_ref
    | DTEDefType_choice_Documento_Documento_Referencia_field_CodRef => x.cod_ref
    | DTEDefType_choice_Documento_Documento_Referencia_field_RazonRef => x.razon_ref

inl x_dte_def_type_choice_documento_documento_referencia_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_referencia_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Referencia_field_NroLinRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_TpoDocRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_IndGlobal
        exists DTEDefType_choice_Documento_Documento_Referencia_field_FolioRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_RUTOtr
        exists DTEDefType_choice_Documento_Documento_Referencia_field_FchRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_CodRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_RazonRef
    ]
inl x_dte_def_type_choice_documento_documento_referencia_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_referencia_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Referencia_field_NroLinRef =>
        { tag = "NroLinRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Referencia_field_TpoDocRef =>
        { tag = "TpoDocRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Referencia_field_IndGlobal =>
        { tag = "IndGlobal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Referencia_field_FolioRef =>
        { tag = "FolioRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Referencia_field_RUTOtr => { tag = "RUTOtr"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Referencia_field_FchRef => { tag = "FchRef"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Referencia_field_CodRef => { tag = "CodRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Referencia_field_RazonRef =>
        { tag = "RazonRef"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_dsc_rcg_global_field r =
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_NroLinDR ::
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_field (xsd_positive_integer)
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoMov ::
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_field
            (x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov)
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_GlosaDR ::
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_field
            (option (x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr))
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoValor ::
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_field (x_dinero_porcentaje_type)
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDR ::
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_field (x_dec16_2_type)
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDROtrMnda ::
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_IndExeDR ::
        x_dte_def_type_choice_documento_documento_dsc_rcg_global_field
            (option (x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr))

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_get forall r.
    (f : x_dte_def_type_choice_documento_documento_dsc_rcg_global_field r)
    (x : x_dte_def_type_choice_documento_documento_dsc_rcg_global)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_NroLinDR => x.nro_lin_dr
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoMov => x.tpo_mov
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_GlosaDR => x.glosa_dr
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoValor => x.tpo_valor
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDR => x.valor_dr
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDROtrMnda => x.valor_dr_otr_mnda
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_IndExeDR => x.ind_exe_dr

inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_dsc_rcg_global_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_NroLinDR
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoMov
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_GlosaDR
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoValor
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDR
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDROtrMnda
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_IndExeDR
    ]
inl x_dte_def_type_choice_documento_documento_dsc_rcg_global_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_dsc_rcg_global_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_NroLinDR =>
        { tag = "NroLinDR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoMov => { tag = "TpoMov"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_GlosaDR =>
        { tag = "GlosaDR"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoValor =>
        { tag = "TpoValor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDR =>
        { tag = "ValorDR"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDROtrMnda =>
        { tag = "ValorDROtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_IndExeDR =>
        { tag = "IndExeDR"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_sub_tot_info_field r =
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_NroSTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field
            (x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti)
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_GlosaSTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field
            (x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti)
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_OrdenSTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field
            (option (x_dte_def_type_choice_documento_documento_sub_tot_info_orden_sti))
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotNetoSTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotIVASTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotAdicSTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotExeSTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_ValSubtotSTI ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_LineasDeta ::
        x_dte_def_type_choice_documento_documento_sub_tot_info_field
            (list (x_dte_def_type_choice_documento_documento_sub_tot_info_lineas_deta))

inl x_dte_def_type_choice_documento_documento_sub_tot_info_get forall r.
    (f : x_dte_def_type_choice_documento_documento_sub_tot_info_field r)
    (x : x_dte_def_type_choice_documento_documento_sub_tot_info)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_NroSTI => x.nro_sti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_GlosaSTI => x.glosa_sti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_OrdenSTI => x.orden_sti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotNetoSTI => x.sub_tot_neto_sti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotIVASTI => x.sub_tot_ivasti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotAdicSTI => x.sub_tot_adic_sti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotExeSTI => x.sub_tot_exe_sti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_ValSubtotSTI => x.val_subtot_sti
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_LineasDeta => x.lineas_deta

inl x_dte_def_type_choice_documento_documento_sub_tot_info_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_sub_tot_info_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_NroSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_GlosaSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_OrdenSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotNetoSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotIVASTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotAdicSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotExeSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_ValSubtotSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_LineasDeta
    ]
inl x_dte_def_type_choice_documento_documento_sub_tot_info_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_sub_tot_info_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_NroSTI => { tag = "NroSTI"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_GlosaSTI =>
        { tag = "GlosaSTI"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_OrdenSTI =>
        { tag = "OrdenSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotNetoSTI =>
        { tag = "SubTotNetoSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotIVASTI =>
        { tag = "SubTotIVASTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotAdicSTI =>
        { tag = "SubTotAdicSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotExeSTI =>
        { tag = "SubTotExeSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_ValSubtotSTI =>
        { tag = "ValSubtotSTI"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_SubTotInfo_field_LineasDeta =>
        { tag = "LineasDeta"; min = 0i32; max = Some 60i32 }

union x_dte_def_type_choice_documento_documento_detalle_field r =
    | DTEDefType_choice_Documento_Documento_Detalle_field_NroLinDet ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (x_dte_def_type_choice_documento_documento_detalle_nro_lin_det)
    | DTEDefType_choice_Documento_Documento_Detalle_field_CdgItem ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (list (x_dte_def_type_choice_documento_documento_detalle_cdg_item))
    | DTEDefType_choice_Documento_Documento_Detalle_field_IndExe ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (option (x_dte_def_type_choice_documento_documento_detalle_ind_exe))
    | DTEDefType_choice_Documento_Documento_Detalle_field_Retenedor ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (option (x_dte_def_type_choice_documento_documento_detalle_retenedor))
    | DTEDefType_choice_Documento_Documento_Detalle_field_NmbItem ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (x_dte_def_type_choice_documento_documento_detalle_nmb_item)
    | DTEDefType_choice_Documento_Documento_Detalle_field_DscItem ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (option (x_dte_def_type_choice_documento_documento_detalle_dsc_item))
    | DTEDefType_choice_Documento_Documento_Detalle_field_QtyRef ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdRef ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (option (x_dte_def_type_choice_documento_documento_detalle_unmd_ref))
    | DTEDefType_choice_Documento_Documento_Detalle_field_PrcRef ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_QtyItem ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_Subcantidad ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (list (x_dte_def_type_choice_documento_documento_detalle_subcantidad))
    | DTEDefType_choice_Documento_Documento_Detalle_field_FchElabor ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_FchVencim ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdItem ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (option (x_dte_def_type_choice_documento_documento_detalle_unmd_item))
    | DTEDefType_choice_Documento_Documento_Detalle_field_PrcItem ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_dec12_6_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_OtrMnda ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (option (x_dte_def_type_choice_documento_documento_detalle_otr_mnda))
    | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoPct ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoMonto ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_mnt_imp_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_SubDscto ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (list (x_dte_def_type_choice_documento_documento_detalle_sub_dscto))
    | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoPct ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (option (x_dte_def_type_choice_documento_documento_detalle_recargo_pct))
    | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoMonto ::
        x_dte_def_type_choice_documento_documento_detalle_field (option (x_mnt_imp_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_SubRecargo ::
        x_dte_def_type_choice_documento_documento_detalle_field
            (list (x_dte_def_type_choice_documento_documento_detalle_sub_recargo))
    | DTEDefType_choice_Documento_Documento_Detalle_field_CodImpAdic ::
        x_dte_def_type_choice_documento_documento_detalle_field (list (x_imp_adic_dte_type))
    | DTEDefType_choice_Documento_Documento_Detalle_field_MontoItem ::
        x_dte_def_type_choice_documento_documento_detalle_field (x_monto_type)

inl x_dte_def_type_choice_documento_documento_detalle_get forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_field r)
    (x : x_dte_def_type_choice_documento_documento_detalle)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_field_NroLinDet => x.nro_lin_det
    | DTEDefType_choice_Documento_Documento_Detalle_field_CdgItem => x.cdg_item
    | DTEDefType_choice_Documento_Documento_Detalle_field_IndExe => x.ind_exe
    | DTEDefType_choice_Documento_Documento_Detalle_field_Retenedor => x.retenedor
    | DTEDefType_choice_Documento_Documento_Detalle_field_NmbItem => x.nmb_item
    | DTEDefType_choice_Documento_Documento_Detalle_field_DscItem => x.dsc_item
    | DTEDefType_choice_Documento_Documento_Detalle_field_QtyRef => x.qty_ref
    | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdRef => x.unmd_ref
    | DTEDefType_choice_Documento_Documento_Detalle_field_PrcRef => x.prc_ref
    | DTEDefType_choice_Documento_Documento_Detalle_field_QtyItem => x.qty_item
    | DTEDefType_choice_Documento_Documento_Detalle_field_Subcantidad => x.subcantidad
    | DTEDefType_choice_Documento_Documento_Detalle_field_FchElabor => x.fch_elabor
    | DTEDefType_choice_Documento_Documento_Detalle_field_FchVencim => x.fch_vencim
    | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdItem => x.unmd_item
    | DTEDefType_choice_Documento_Documento_Detalle_field_PrcItem => x.prc_item
    | DTEDefType_choice_Documento_Documento_Detalle_field_OtrMnda => x.otr_mnda
    | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoPct => x.descuento_pct
    | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoMonto => x.descuento_monto
    | DTEDefType_choice_Documento_Documento_Detalle_field_SubDscto => x.sub_dscto
    | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoPct => x.recargo_pct
    | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoMonto => x.recargo_monto
    | DTEDefType_choice_Documento_Documento_Detalle_field_SubRecargo => x.sub_recargo
    | DTEDefType_choice_Documento_Documento_Detalle_field_CodImpAdic => x.cod_imp_adic
    | DTEDefType_choice_Documento_Documento_Detalle_field_MontoItem => x.monto_item

inl x_dte_def_type_choice_documento_documento_detalle_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_detalle_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Detalle_field_NroLinDet
        exists DTEDefType_choice_Documento_Documento_Detalle_field_CdgItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_IndExe
        exists DTEDefType_choice_Documento_Documento_Detalle_field_Retenedor
        exists DTEDefType_choice_Documento_Documento_Detalle_field_NmbItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_DscItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_QtyRef
        exists DTEDefType_choice_Documento_Documento_Detalle_field_UnmdRef
        exists DTEDefType_choice_Documento_Documento_Detalle_field_PrcRef
        exists DTEDefType_choice_Documento_Documento_Detalle_field_QtyItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_Subcantidad
        exists DTEDefType_choice_Documento_Documento_Detalle_field_FchElabor
        exists DTEDefType_choice_Documento_Documento_Detalle_field_FchVencim
        exists DTEDefType_choice_Documento_Documento_Detalle_field_UnmdItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_PrcItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_OtrMnda
        exists DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoPct
        exists DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoMonto
        exists DTEDefType_choice_Documento_Documento_Detalle_field_SubDscto
        exists DTEDefType_choice_Documento_Documento_Detalle_field_RecargoPct
        exists DTEDefType_choice_Documento_Documento_Detalle_field_RecargoMonto
        exists DTEDefType_choice_Documento_Documento_Detalle_field_SubRecargo
        exists DTEDefType_choice_Documento_Documento_Detalle_field_CodImpAdic
        exists DTEDefType_choice_Documento_Documento_Detalle_field_MontoItem
    ]
inl x_dte_def_type_choice_documento_documento_detalle_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_field_NroLinDet =>
        { tag = "NroLinDet"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_CdgItem => { tag = "CdgItem"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_IndExe => { tag = "IndExe"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_Retenedor =>
        { tag = "Retenedor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_NmbItem => { tag = "NmbItem"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_DscItem => { tag = "DscItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_QtyRef => { tag = "QtyRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdRef => { tag = "UnmdRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_PrcRef => { tag = "PrcRef"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_QtyItem => { tag = "QtyItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_Subcantidad =>
        { tag = "Subcantidad"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_FchElabor =>
        { tag = "FchElabor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_FchVencim =>
        { tag = "FchVencim"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdItem => { tag = "UnmdItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_PrcItem => { tag = "PrcItem"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_OtrMnda => { tag = "OtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoPct =>
        { tag = "DescuentoPct"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoMonto =>
        { tag = "DescuentoMonto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_SubDscto => { tag = "SubDscto"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoPct =>
        { tag = "RecargoPct"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoMonto =>
        { tag = "RecargoMonto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_SubRecargo =>
        { tag = "SubRecargo"; min = 0i32; max = Some 5i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_CodImpAdic =>
        { tag = "CodImpAdic"; min = 0i32; max = Some 2i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_field_MontoItem =>
        { tag = "MontoItem"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field r =
    | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_TipoRecargo ::
        x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field (x_dinero_porcentaje_type)
    | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_ValorRecargo ::
        x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field (x_dec16_2_type)

inl x_dte_def_type_choice_documento_documento_detalle_sub_recargo_get forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field r)
    (x : x_dte_def_type_choice_documento_documento_detalle_sub_recargo)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_TipoRecargo => x.tipo_recargo
    | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_ValorRecargo => x.valor_recargo

inl x_dte_def_type_choice_documento_documento_detalle_sub_recargo_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_TipoRecargo
        exists DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_ValorRecargo
    ]
inl x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_TipoRecargo =>
        { tag = "TipoRecargo"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_ValorRecargo =>
        { tag = "ValorRecargo"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field r =
    | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_TipoDscto ::
        x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field (x_dinero_porcentaje_type)
    | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_ValorDscto ::
        x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field (x_dec16_2_type)

inl x_dte_def_type_choice_documento_documento_detalle_sub_dscto_get forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field r)
    (x : x_dte_def_type_choice_documento_documento_detalle_sub_dscto)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_TipoDscto => x.tipo_dscto
    | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_ValorDscto => x.valor_dscto

inl x_dte_def_type_choice_documento_documento_detalle_sub_dscto_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_TipoDscto
        exists DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_ValorDscto
    ]
inl x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_TipoDscto =>
        { tag = "TipoDscto"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_ValorDscto =>
        { tag = "ValorDscto"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field r =
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_PrcOtrMon ::
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field (x_dec12_6_type)
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_Moneda ::
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field
            (x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda)
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_FctConv ::
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field (option (x_dec6_4_type))
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_DctoOtrMnda ::
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_RecargoOtrMnda ::
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_MontoItemOtrMnda ::
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field (option (x_dec14_4_type))

inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_get forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field r)
    (x : x_dte_def_type_choice_documento_documento_detalle_otr_mnda)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_PrcOtrMon => x.prc_otr_mon
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_Moneda => x.moneda
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_FctConv => x.fct_conv
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_DctoOtrMnda => x.dcto_otr_mnda
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_RecargoOtrMnda => x.recargo_otr_mnda
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_MontoItemOtrMnda => x.monto_item_otr_mnda

inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_PrcOtrMon
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_Moneda
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_FctConv
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_DctoOtrMnda
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_RecargoOtrMnda
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_MontoItemOtrMnda
    ]
inl x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_PrcOtrMon =>
        { tag = "PrcOtrMon"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_Moneda =>
        { tag = "Moneda"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_FctConv =>
        { tag = "FctConv"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_DctoOtrMnda =>
        { tag = "DctoOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_RecargoOtrMnda =>
        { tag = "RecargoOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_MontoItemOtrMnda =>
        { tag = "MontoItemOtrMnda"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_detalle_subcantidad_field r =
    | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubQty ::
        x_dte_def_type_choice_documento_documento_detalle_subcantidad_field (x_dec12_6_type)
    | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubCod ::
        x_dte_def_type_choice_documento_documento_detalle_subcantidad_field
            (x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod)

inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_get forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_subcantidad_field r)
    (x : x_dte_def_type_choice_documento_documento_detalle_subcantidad)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubQty => x.sub_qty
    | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubCod => x.sub_cod

inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_detalle_subcantidad_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubQty
        exists DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubCod
    ]
inl x_dte_def_type_choice_documento_documento_detalle_subcantidad_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_subcantidad_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubQty =>
        { tag = "SubQty"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubCod =>
        { tag = "SubCod"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_detalle_retenedor_field r =
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_IndAgente ::
        x_dte_def_type_choice_documento_documento_detalle_retenedor_field
            (x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente)
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntBaseFaena ::
        x_dte_def_type_choice_documento_documento_detalle_retenedor_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntMargComer ::
        x_dte_def_type_choice_documento_documento_detalle_retenedor_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_PrcConsFinal ::
        x_dte_def_type_choice_documento_documento_detalle_retenedor_field (option (x_monto_type))

inl x_dte_def_type_choice_documento_documento_detalle_retenedor_get forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_retenedor_field r)
    (x : x_dte_def_type_choice_documento_documento_detalle_retenedor)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_IndAgente => x.ind_agente
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntBaseFaena => x.mnt_base_faena
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntMargComer => x.mnt_marg_comer
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_PrcConsFinal => x.prc_cons_final

inl x_dte_def_type_choice_documento_documento_detalle_retenedor_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_detalle_retenedor_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_IndAgente
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntBaseFaena
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntMargComer
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_PrcConsFinal
    ]
inl x_dte_def_type_choice_documento_documento_detalle_retenedor_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_retenedor_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_IndAgente =>
        { tag = "IndAgente"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntBaseFaena =>
        { tag = "MntBaseFaena"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntMargComer =>
        { tag = "MntMargComer"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_PrcConsFinal =>
        { tag = "PrcConsFinal"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_detalle_cdg_item_field r =
    | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_TpoCodigo ::
        x_dte_def_type_choice_documento_documento_detalle_cdg_item_field
            (x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo)
    | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_VlrCodigo ::
        x_dte_def_type_choice_documento_documento_detalle_cdg_item_field
            (x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo)

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_get forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_cdg_item_field r)
    (x : x_dte_def_type_choice_documento_documento_detalle_cdg_item)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_TpoCodigo => x.tpo_codigo
    | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_VlrCodigo => x.vlr_codigo

inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_detalle_cdg_item_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_TpoCodigo
        exists DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_VlrCodigo
    ]
inl x_dte_def_type_choice_documento_documento_detalle_cdg_item_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_detalle_cdg_item_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_TpoCodigo =>
        { tag = "TpoCodigo"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_VlrCodigo =>
        { tag = "VlrCodigo"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_field_IdDoc ::
        x_dte_def_type_choice_documento_documento_encabezado_field
            (x_dte_def_type_choice_documento_documento_encabezado_id_doc)
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Emisor ::
        x_dte_def_type_choice_documento_documento_encabezado_field
            (x_dte_def_type_choice_documento_documento_encabezado_emisor)
    | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTMandante ::
        x_dte_def_type_choice_documento_documento_encabezado_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Receptor ::
        x_dte_def_type_choice_documento_documento_encabezado_field
            (x_dte_def_type_choice_documento_documento_encabezado_receptor)
    | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTSolicita ::
        x_dte_def_type_choice_documento_documento_encabezado_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Transporte ::
        x_dte_def_type_choice_documento_documento_encabezado_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte))
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Totales ::
        x_dte_def_type_choice_documento_documento_encabezado_field
            (x_dte_def_type_choice_documento_documento_encabezado_totales)
    | DTEDefType_choice_Documento_Documento_Encabezado_field_OtraMoneda ::
        x_dte_def_type_choice_documento_documento_encabezado_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda))

inl x_dte_def_type_choice_documento_documento_encabezado_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_field_IdDoc => x.id_doc
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Emisor => x.emisor
    | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTMandante => x.rut_mandante
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Receptor => x.receptor
    | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTSolicita => x.rut_solicita
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Transporte => x.transporte
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Totales => x.totales
    | DTEDefType_choice_Documento_Documento_Encabezado_field_OtraMoneda => x.otra_moneda

inl x_dte_def_type_choice_documento_documento_encabezado_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_IdDoc
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Emisor
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_RUTMandante
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Receptor
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_RUTSolicita
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Transporte
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Totales
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_OtraMoneda
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_field_IdDoc => { tag = "IdDoc"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Emisor => { tag = "Emisor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTMandante =>
        { tag = "RUTMandante"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Receptor =>
        { tag = "Receptor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTSolicita =>
        { tag = "RUTSolicita"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Transporte =>
        { tag = "Transporte"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_field_Totales => { tag = "Totales"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_field_OtraMoneda =>
        { tag = "OtraMoneda"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoMoneda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (x_tip_mon_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoCambio ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (option (x_dec6_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntNetoOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntExeOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntFaeCarneOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntMargComOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVAOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_ImpRetOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field
            (list (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVANoRetOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntTotOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field (x_dec14_4_type)

inl x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoMoneda => x.tpo_moneda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoCambio => x.tpo_cambio
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntNetoOtrMnda => x.mnt_neto_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntExeOtrMnda => x.mnt_exe_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntFaeCarneOtrMnda => x.mnt_fae_carne_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntMargComOtrMnda => x.mnt_marg_com_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVAOtrMnda => x.iva_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_ImpRetOtrMnda => x.imp_ret_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVANoRetOtrMnda => x.iva_no_ret_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntTotOtrMnda => x.mnt_tot_otr_mnda

inl x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoMoneda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoCambio
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntNetoOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntExeOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntFaeCarneOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntMargComOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVAOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_ImpRetOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVANoRetOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntTotOtrMnda
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoMoneda =>
        { tag = "TpoMoneda"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoCambio =>
        { tag = "TpoCambio"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntNetoOtrMnda =>
        { tag = "MntNetoOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntExeOtrMnda =>
        { tag = "MntExeOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntFaeCarneOtrMnda =>
        { tag = "MntFaeCarneOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntMargComOtrMnda =>
        { tag = "MntMargComOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVAOtrMnda =>
        { tag = "IVAOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_ImpRetOtrMnda =>
        { tag = "ImpRetOtrMnda"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVANoRetOtrMnda =>
        { tag = "IVANoRetOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntTotOtrMnda =>
        { tag = "MntTotOtrMnda"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TipoImpOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field (x_imp_adic_dte_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TasaImpOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_VlrImpOtrMnda ::
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field (x_dec14_4_type)

inl x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TipoImpOtrMnda =>
        x.tipo_imp_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TasaImpOtrMnda =>
        x.tasa_imp_otr_mnda
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_VlrImpOtrMnda =>
        x.vlr_imp_otr_mnda

inl x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TipoImpOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TasaImpOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_VlrImpOtrMnda
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TipoImpOtrMnda =>
        { tag = "TipoImpOtrMnda"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TasaImpOtrMnda =>
        { tag = "TasaImpOtrMnda"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_VlrImpOtrMnda =>
        { tag = "VlrImpOtrMnda"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_totales_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntNeto ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntExe ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntBase ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntMargenCom ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_TasaIVA ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_pct_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVA ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVAProp ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVATerc ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_ImptoReten ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field
            (list (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVANoRet ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_mnt_imp_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_CredEC ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_GrntDep ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_Comisiones ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntTotal ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (x_monto_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoNF ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoPeriodo ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_SaldoAnterior ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_valor_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_VlrPagar ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_field (option (x_valor_type))

inl x_dte_def_type_choice_documento_documento_encabezado_totales_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_totales_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntNeto => x.mnt_neto
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntExe => x.mnt_exe
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntBase => x.mnt_base
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntMargenCom => x.mnt_margen_com
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_TasaIVA => x.tasa_iva
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVA => x.iva
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVAProp => x.iva_prop
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVATerc => x.iva_terc
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_ImptoReten => x.impto_reten
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVANoRet => x.iva_no_ret
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_CredEC => x.cred_ec
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_GrntDep => x.grnt_dep
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_Comisiones => x.comisiones
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntTotal => x.mnt_total
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoNF => x.monto_nf
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoPeriodo => x.monto_periodo
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_SaldoAnterior => x.saldo_anterior
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_VlrPagar => x.vlr_pagar

inl x_dte_def_type_choice_documento_documento_encabezado_totales_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_totales_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntExe
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntBase
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntMargenCom
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_TasaIVA
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVA
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVAProp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVATerc
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_ImptoReten
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVANoRet
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_CredEC
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_GrntDep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_Comisiones
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntTotal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoNF
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoPeriodo
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_SaldoAnterior
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_VlrPagar
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_totales_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_totales_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntNeto =>
        { tag = "MntNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntExe =>
        { tag = "MntExe"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntBase =>
        { tag = "MntBase"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntMargenCom =>
        { tag = "MntMargenCom"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_TasaIVA =>
        { tag = "TasaIVA"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVA => { tag = "IVA"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVAProp =>
        { tag = "IVAProp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVATerc =>
        { tag = "IVATerc"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_ImptoReten =>
        { tag = "ImptoReten"; min = 0i32; max = Some 20i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVANoRet =>
        { tag = "IVANoRet"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_CredEC =>
        { tag = "CredEC"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_GrntDep =>
        { tag = "GrntDep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_Comisiones =>
        { tag = "Comisiones"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntTotal =>
        { tag = "MntTotal"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoNF =>
        { tag = "MontoNF"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoPeriodo =>
        { tag = "MontoPeriodo"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_SaldoAnterior =>
        { tag = "SaldoAnterior"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_VlrPagar =>
        { tag = "VlrPagar"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComNeto ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComExe ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComIVA ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field (option (x_monto_type))

inl x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComNeto => x.val_com_neto
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComExe => x.val_com_exe
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComIVA => x.val_com_iva

inl x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComIVA
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComNeto =>
        { tag = "ValComNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComExe =>
        { tag = "ValComExe"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComIVA =>
        { tag = "ValComIVA"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TipoImp ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field (x_imp_adic_dte_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TasaImp ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_tasa_imp))
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_MontoImp ::
        x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field (x_mnt_imp_type)

inl x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TipoImp => x.tipo_imp
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TasaImp => x.tasa_imp
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_MontoImp => x.monto_imp

inl x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TipoImp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TasaImp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_MontoImp
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TipoImp =>
        { tag = "TipoImp"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TasaImp =>
        { tag = "TasaImp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_MontoImp =>
        { tag = "MontoImp"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_transporte_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Patente ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_patente))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_RUTTrans ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Chofer ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_DirDest ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CmnaDest ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CiudadDest ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Aduana ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana))

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Patente => x.patente
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_RUTTrans => x.rut_trans
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Chofer => x.chofer
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_DirDest => x.dir_dest
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CmnaDest => x.cmna_dest
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CiudadDest => x.ciudad_dest
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Aduana => x.aduana

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Patente
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_RUTTrans
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Chofer
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_DirDest
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CmnaDest
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CiudadDest
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Aduana
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Patente =>
        { tag = "Patente"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_RUTTrans =>
        { tag = "RUTTrans"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Chofer =>
        { tag = "Chofer"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_DirDest =>
        { tag = "DirDest"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CmnaDest =>
        { tag = "CmnaDest"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CiudadDest =>
        { tag = "CiudadDest"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Aduana =>
        { tag = "Aduana"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodModVenta ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_mod_venta))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodClauVenta ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_clau_venta))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotClauVenta ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field (option (x_dec16_2_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodViaTransp ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_via_transp))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NombreTransp ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_RUTCiaTransp ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field (option (x_rut_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NomCiaTransp ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicTransp ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Booking ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Operador ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoEmbarque ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_embarque))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoDesemb ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pto_desemb))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Tara ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tara))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidMedTara ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_med_tara))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoBruto ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_bruto))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_bruto))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoNeto ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_peso_neto))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_unid_peso_neto))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotItems ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_items))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotBultos ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tot_bultos))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TipoBultos ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (list (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntFlete ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntSeguro ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field (option (x_dec14_4_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_recep))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisDestin ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_cod_pais_destin))

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodModVenta => x.cod_mod_venta
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodClauVenta => x.cod_clau_venta
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotClauVenta => x.tot_clau_venta
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodViaTransp => x.cod_via_transp
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NombreTransp => x.nombre_transp
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_RUTCiaTransp => x.rut_cia_transp
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NomCiaTransp => x.nom_cia_transp
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicTransp => x.id_adic_transp
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Booking => x.booking
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Operador => x.operador
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoEmbarque => x.cod_pto_embarque
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb => x.id_adic_pto_emb
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoDesemb => x.cod_pto_desemb
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb => x.id_adic_pto_desemb
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Tara => x.tara
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidMedTara => x.cod_unid_med_tara
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoBruto => x.peso_bruto
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto => x.cod_unid_peso_bruto
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoNeto => x.peso_neto
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto => x.cod_unid_peso_neto
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotItems => x.tot_items
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotBultos => x.tot_bultos
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TipoBultos => x.tipo_bultos
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntFlete => x.mnt_flete
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntSeguro => x.mnt_seguro
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisRecep => x.cod_pais_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisDestin => x.cod_pais_destin

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodModVenta
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodClauVenta
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotClauVenta
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodViaTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NombreTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_RUTCiaTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NomCiaTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Booking
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Operador
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoEmbarque
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoDesemb
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Tara
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidMedTara
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoBruto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotItems
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TipoBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntFlete
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntSeguro
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisDestin
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodModVenta =>
        { tag = "CodModVenta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodClauVenta =>
        { tag = "CodClauVenta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotClauVenta =>
        { tag = "TotClauVenta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodViaTransp =>
        { tag = "CodViaTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NombreTransp =>
        { tag = "NombreTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_RUTCiaTransp =>
        { tag = "RUTCiaTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NomCiaTransp =>
        { tag = "NomCiaTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicTransp =>
        { tag = "IdAdicTransp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Booking =>
        { tag = "Booking"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Operador =>
        { tag = "Operador"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoEmbarque =>
        { tag = "CodPtoEmbarque"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb =>
        { tag = "IdAdicPtoEmb"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoDesemb =>
        { tag = "CodPtoDesemb"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb =>
        { tag = "IdAdicPtoDesemb"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Tara =>
        { tag = "Tara"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidMedTara =>
        { tag = "CodUnidMedTara"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoBruto =>
        { tag = "PesoBruto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto =>
        { tag = "CodUnidPesoBruto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoNeto =>
        { tag = "PesoNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto =>
        { tag = "CodUnidPesoNeto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotItems =>
        { tag = "TotItems"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotBultos =>
        { tag = "TotBultos"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TipoBultos =>
        { tag = "TipoBultos"; min = 0i32; max = Some 10i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntFlete =>
        { tag = "MntFlete"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntSeguro =>
        { tag = "MntSeguro"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisRecep =>
        { tag = "CodPaisRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisDestin =>
        { tag = "CodPaisDestin"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cod_tpo_bultos))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_cant_bultos))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Sello ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello))
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello))

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos =>
        x.cod_tpo_bultos
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos => x.cant_bultos
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas => x.marcas
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer => x.id_container
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Sello => x.sello
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello => x.emisor_sello

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Sello
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos =>
        { tag = "CodTpoBultos"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos =>
        { tag = "CantBultos"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas =>
        { tag = "Marcas"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer =>
        { tag = "IdContainer"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Sello =>
        { tag = "Sello"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello =>
        { tag = "EmisorSello"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_RUTChofer ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field (x_rut_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_NombreChofer ::
        x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field
            (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer)

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_RUTChofer => x.rut_chofer
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_NombreChofer => x.nombre_chofer

inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_RUTChofer
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_NombreChofer
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_RUTChofer =>
        { tag = "RUTChofer"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_NombreChofer =>
        { tag = "NombreChofer"; min = 1i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_receptor_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RUTRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field (x_rut_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CdgIntRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RznSocRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field (x_rzn_soc_larga_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Extranjero ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_GiroRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Contacto ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CorreoRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field (option (x_mail_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadRecep ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirPostal ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaPostal ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadPostal ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_field (option (x_ciudad_type))

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_receptor_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RUTRecep => x.rut_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CdgIntRecep => x.cdg_int_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RznSocRecep => x.rzn_soc_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Extranjero => x.extranjero
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_GiroRecep => x.giro_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Contacto => x.contacto
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CorreoRecep => x.correo_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirRecep => x.dir_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaRecep => x.cmna_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadRecep => x.ciudad_recep
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirPostal => x.dir_postal
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaPostal => x.cmna_postal
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadPostal => x.ciudad_postal

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_receptor_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RUTRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CdgIntRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RznSocRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Extranjero
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_GiroRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Contacto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CorreoRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirPostal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaPostal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadPostal
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_receptor_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RUTRecep =>
        { tag = "RUTRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CdgIntRecep =>
        { tag = "CdgIntRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RznSocRecep =>
        { tag = "RznSocRecep"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Extranjero =>
        { tag = "Extranjero"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_GiroRecep =>
        { tag = "GiroRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Contacto =>
        { tag = "Contacto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CorreoRecep =>
        { tag = "CorreoRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirRecep =>
        { tag = "DirRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaRecep =>
        { tag = "CmnaRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadRecep =>
        { tag = "CiudadRecep"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirPostal =>
        { tag = "DirPostal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaPostal =>
        { tag = "CmnaPostal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadPostal =>
        { tag = "CiudadPostal"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_NumId ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id))
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_Nacionalidad ::
        x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad))

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_NumId => x.num_id
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_Nacionalidad => x.nacionalidad

inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_NumId
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_Nacionalidad
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_NumId =>
        { tag = "NumId"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_Nacionalidad =>
        { tag = "Nacionalidad"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_emisor_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RUTEmisor ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field (x_rut_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RznSoc ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc)
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GiroEmis ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis)
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Telefono ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (list (x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CorreoEmisor ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field (option (x_mail_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Acteco ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (list (x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GuiaExport ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Sucursal ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgSIISucur ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_sii_sucur))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_DirOrigen ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CmnaOrigen ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field (option (x_comuna_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CiudadOrigen ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field (option (x_ciudad_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgVendedor ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_IdAdicEmisor ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor))

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_emisor_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RUTEmisor => x.rut_emisor
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RznSoc => x.rzn_soc
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GiroEmis => x.giro_emis
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Telefono => x.telefono
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CorreoEmisor => x.correo_emisor
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Acteco => x.acteco
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GuiaExport => x.guia_export
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Sucursal => x.sucursal
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgSIISucur => x.cdg_sii_sucur
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_DirOrigen => x.dir_origen
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CmnaOrigen => x.cmna_origen
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CiudadOrigen => x.ciudad_origen
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgVendedor => x.cdg_vendedor
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_IdAdicEmisor => x.id_adic_emisor

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_emisor_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RUTEmisor
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RznSoc
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GiroEmis
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Telefono
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CorreoEmisor
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Acteco
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GuiaExport
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Sucursal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgSIISucur
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_DirOrigen
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CmnaOrigen
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CiudadOrigen
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgVendedor
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_IdAdicEmisor
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_emisor_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RUTEmisor =>
        { tag = "RUTEmisor"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RznSoc =>
        { tag = "RznSoc"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GiroEmis =>
        { tag = "GiroEmis"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Telefono =>
        { tag = "Telefono"; min = 0i32; max = Some 2i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CorreoEmisor =>
        { tag = "CorreoEmisor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Acteco =>
        { tag = "Acteco"; min = 1i32; max = Some 4i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GuiaExport =>
        { tag = "GuiaExport"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Sucursal =>
        { tag = "Sucursal"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgSIISucur =>
        { tag = "CdgSIISucur"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_DirOrigen =>
        { tag = "DirOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CmnaOrigen =>
        { tag = "CmnaOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CiudadOrigen =>
        { tag = "CiudadOrigen"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgVendedor =>
        { tag = "CdgVendedor"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_IdAdicEmisor =>
        { tag = "IdAdicEmisor"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_CdgTraslado ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FolioAut ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field (option (x_nro_resol_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FchAut ::
        x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field (option (x_fecha_type))

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_CdgTraslado => x.cdg_traslado
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FolioAut => x.folio_aut
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FchAut => x.fch_aut

inl x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_CdgTraslado
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FolioAut
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FchAut
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_CdgTraslado =>
        { tag = "CdgTraslado"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FolioAut =>
        { tag = "FolioAut"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FchAut =>
        { tag = "FchAut"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_id_doc_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDTE ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (x_dte_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_Folio ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (x_folio_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchEmis ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (x_fecha_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndNoRebaja ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDespacho ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndTraslado ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoImpresion ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndServicio ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntBruto ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPago ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPagExp ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pag_exp))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchCancel ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntCancel ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_SaldoInsol ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (option (x_monto_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntPagos ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (list (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoDesde ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoHasta ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (option (x_fecha_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MedioPago ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (option (x_medio_pago_type))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoCtaPago ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_NumCtaPago ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_BcoPago ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoCdg ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoGlosa ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoDias ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_dias))
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchVenc ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_field (option (x_fecha_type))

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_id_doc_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDTE => x.tipo_dte
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_Folio => x.folio
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchEmis => x.fch_emis
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndNoRebaja => x.ind_no_rebaja
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDespacho => x.tipo_despacho
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndTraslado => x.ind_traslado
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoImpresion => x.tpo_impresion
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndServicio => x.ind_servicio
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntBruto => x.mnt_bruto
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPago => x.fma_pago
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPagExp => x.fma_pag_exp
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchCancel => x.fch_cancel
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntCancel => x.mnt_cancel
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_SaldoInsol => x.saldo_insol
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntPagos => x.mnt_pagos
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoDesde => x.periodo_desde
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoHasta => x.periodo_hasta
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MedioPago => x.medio_pago
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoCtaPago => x.tpo_cta_pago
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_NumCtaPago => x.num_cta_pago
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_BcoPago => x.bco_pago
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoCdg => x.term_pago_cdg
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoGlosa => x.term_pago_glosa
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoDias => x.term_pago_dias
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchVenc => x.fch_venc

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_id_doc_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDTE
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_Folio
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchEmis
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndNoRebaja
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDespacho
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndTraslado
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoImpresion
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndServicio
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntBruto
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPagExp
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchCancel
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntCancel
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_SaldoInsol
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntPagos
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoDesde
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoHasta
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MedioPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoCtaPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_NumCtaPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_BcoPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoCdg
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoGlosa
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoDias
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchVenc
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_id_doc_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDTE =>
        { tag = "TipoDTE"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_Folio =>
        { tag = "Folio"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchEmis =>
        { tag = "FchEmis"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndNoRebaja =>
        { tag = "IndNoRebaja"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDespacho =>
        { tag = "TipoDespacho"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndTraslado =>
        { tag = "IndTraslado"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoImpresion =>
        { tag = "TpoImpresion"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndServicio =>
        { tag = "IndServicio"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntBruto =>
        { tag = "MntBruto"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPago =>
        { tag = "FmaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPagExp =>
        { tag = "FmaPagExp"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchCancel =>
        { tag = "FchCancel"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntCancel =>
        { tag = "MntCancel"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_SaldoInsol =>
        { tag = "SaldoInsol"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntPagos =>
        { tag = "MntPagos"; min = 0i32; max = Some 30i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoDesde =>
        { tag = "PeriodoDesde"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoHasta =>
        { tag = "PeriodoHasta"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MedioPago =>
        { tag = "MedioPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoCtaPago =>
        { tag = "TpoCtaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_NumCtaPago =>
        { tag = "NumCtaPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_BcoPago =>
        { tag = "BcoPago"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoCdg =>
        { tag = "TermPagoCdg"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoGlosa =>
        { tag = "TermPagoGlosa"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoDias =>
        { tag = "TermPagoDias"; min = 0i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchVenc =>
        { tag = "FchVenc"; min = 0i32; max = Some 1i32 }

union x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field r =
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_FchPago ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field (x_fecha_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_MntPago ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field (x_monto_type)
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_GlosaPagos ::
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field
            (option (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos))

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_get forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field r)
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos)
    : r
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_FchPago => x.fch_pago
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_MntPago => x.mnt_pago
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_GlosaPagos => x.glosa_pagos

inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_fields
    ()
    : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field r)
    =
    [
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_FchPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_MntPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_GlosaPagos
    ]
inl x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field_meta forall r.
    (f : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field r)
    : xml_codec.field_meta
    =
    match f with
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_FchPago =>
        { tag = "FchPago"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_MntPago =>
        { tag = "MntPago"; min = 1i32; max = Some 1i32 }
    | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_GlosaPagos =>
        { tag = "GlosaPagos"; min = 0i32; max = Some 1i32 }

union x_signature_type_field r =
    | SignatureType_field_SignedInfo :: x_signature_type_field (x_signature_type_signed_info)
    | SignatureType_field_SignatureValue :: x_signature_type_field (string)
    | SignatureType_field_KeyInfo :: x_signature_type_field (x_signature_type_key_info)

inl x_signature_type_get forall r. (f : x_signature_type_field r) (x : x_signature_type) : r =
    match f with
    | SignatureType_field_SignedInfo => x.signed_info
    | SignatureType_field_SignatureValue => x.signature_value
    | SignatureType_field_KeyInfo => x.key_info

inl x_signature_type_fields () : list (exists r. x_signature_type_field r) =
    [
        exists SignatureType_field_SignedInfo
        exists SignatureType_field_SignatureValue
        exists SignatureType_field_KeyInfo
    ]
inl x_signature_type_field_meta forall r. (f : x_signature_type_field r) : xml_codec.field_meta =
    match f with
    | SignatureType_field_SignedInfo => { tag = "SignedInfo"; min = 1i32; max = Some 1i32 }
    | SignatureType_field_SignatureValue => { tag = "SignatureValue"; min = 1i32; max = Some 1i32 }
    | SignatureType_field_KeyInfo => { tag = "KeyInfo"; min = 1i32; max = Some 1i32 }

union x_signature_type_key_info_field r =
    | SignatureType_KeyInfo_field_KeyValue :: x_signature_type_key_info_field (x_signature_type_key_info_key_value)
    | SignatureType_KeyInfo_field_X509Data :: x_signature_type_key_info_field (x_signature_type_key_info_x509_data)

inl x_signature_type_key_info_get forall r.
    (f : x_signature_type_key_info_field r) (x : x_signature_type_key_info)
    : r
    =
    match f with
    | SignatureType_KeyInfo_field_KeyValue => x.key_value
    | SignatureType_KeyInfo_field_X509Data => x.x509_data

inl x_signature_type_key_info_fields () : list (exists r. x_signature_type_key_info_field r) =
    [ exists SignatureType_KeyInfo_field_KeyValue; exists SignatureType_KeyInfo_field_X509Data ]
inl x_signature_type_key_info_field_meta forall r. (f : x_signature_type_key_info_field r) : xml_codec.field_meta =
    match f with
    | SignatureType_KeyInfo_field_KeyValue => { tag = "KeyValue"; min = 1i32; max = Some 1i32 }
    | SignatureType_KeyInfo_field_X509Data => { tag = "X509Data"; min = 1i32; max = Some 1i32 }

union x_signature_type_key_info_x509_data_field r =
    | SignatureType_KeyInfo_X509Data_field_X509Certificate :: x_signature_type_key_info_x509_data_field (string)

inl x_signature_type_key_info_x509_data_get forall r.
    (f : x_signature_type_key_info_x509_data_field r) (x : x_signature_type_key_info_x509_data)
    : r
    =
    match f with
    | SignatureType_KeyInfo_X509Data_field_X509Certificate => x.x509_certificate

inl x_signature_type_key_info_x509_data_fields () : list (exists r. x_signature_type_key_info_x509_data_field r) =
    [ exists SignatureType_KeyInfo_X509Data_field_X509Certificate ]
inl x_signature_type_key_info_x509_data_field_meta forall r.
    (f : x_signature_type_key_info_x509_data_field r)
    : xml_codec.field_meta
    =
    match f with
    | SignatureType_KeyInfo_X509Data_field_X509Certificate => { tag = "X509Certificate"; min = 1i32; max = Some 1i32 }

union x_signature_type_key_info_key_value_field r =
    | SignatureType_KeyInfo_KeyValue_field_choice_rsa_key_value ::
        x_signature_type_key_info_key_value_field (x_signature_type_key_info_key_value_choice_rsa_key_value)

inl x_signature_type_key_info_key_value_get forall r.
    (f : x_signature_type_key_info_key_value_field r) (x : x_signature_type_key_info_key_value)
    : r
    =
    match f with
    | SignatureType_KeyInfo_KeyValue_field_choice_rsa_key_value => x.choice_rsa_key_value

inl x_signature_type_key_info_key_value_fields () : list (exists r. x_signature_type_key_info_key_value_field r) =
    [ exists SignatureType_KeyInfo_KeyValue_field_choice_rsa_key_value ]
inl x_signature_type_key_info_key_value_field_meta forall r.
    (f : x_signature_type_key_info_key_value_field r)
    : xml_codec.field_meta
    =
    match f with
    | SignatureType_KeyInfo_KeyValue_field_choice_rsa_key_value =>
        { tag = "choice_rsa_key_value"; min = 1i32; max = Some 1i32 }

union x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field r =
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_P ::
        x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field (string)
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Q ::
        x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field (string)
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_G ::
        x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field (string)
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Y ::
        x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field (string)

inl x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_get forall r.
    (f : x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field r)
    (x : x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value)
    : r
    =
    match f with
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_P => x.p
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Q => x.q
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_G => x.g
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Y => x.y

inl x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_fields
    ()
    : list (exists r. x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field r)
    =
    [
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_P
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Q
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_G
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Y
    ]
inl x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field_meta forall r.
    (f : x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field r)
    : xml_codec.field_meta
    =
    match f with
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_P =>
        { tag = "P"; min = 1i32; max = Some 1i32 }
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Q =>
        { tag = "Q"; min = 1i32; max = Some 1i32 }
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_G =>
        { tag = "G"; min = 1i32; max = Some 1i32 }
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Y =>
        { tag = "Y"; min = 1i32; max = Some 1i32 }

union x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field r =
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Modulus ::
        x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field (string)
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Exponent ::
        x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field (string)

inl x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_get forall r.
    (f : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field r)
    (x : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value)
    : r
    =
    match f with
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Modulus => x.modulus
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Exponent => x.exponent

inl x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_fields
    ()
    : list (exists r. x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field r)
    =
    [
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Modulus
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Exponent
    ]
inl x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field_meta forall r.
    (f : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field r)
    : xml_codec.field_meta
    =
    match f with
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Modulus =>
        { tag = "Modulus"; min = 1i32; max = Some 1i32 }
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Exponent =>
        { tag = "Exponent"; min = 1i32; max = Some 1i32 }

union x_signature_type_signed_info_field r =
    | SignatureType_SignedInfo_field_CanonicalizationMethod ::
        x_signature_type_signed_info_field (x_signature_type_signed_info_canonicalization_method)
    | SignatureType_SignedInfo_field_SignatureMethod ::
        x_signature_type_signed_info_field (x_signature_type_signed_info_signature_method)
    | SignatureType_SignedInfo_field_Reference ::
        x_signature_type_signed_info_field (x_signature_type_signed_info_reference)

inl x_signature_type_signed_info_get forall r.
    (f : x_signature_type_signed_info_field r) (x : x_signature_type_signed_info)
    : r
    =
    match f with
    | SignatureType_SignedInfo_field_CanonicalizationMethod => x.canonicalization_method
    | SignatureType_SignedInfo_field_SignatureMethod => x.signature_method
    | SignatureType_SignedInfo_field_Reference => x.reference

inl x_signature_type_signed_info_fields () : list (exists r. x_signature_type_signed_info_field r) =
    [
        exists SignatureType_SignedInfo_field_CanonicalizationMethod
        exists SignatureType_SignedInfo_field_SignatureMethod
        exists SignatureType_SignedInfo_field_Reference
    ]
inl x_signature_type_signed_info_field_meta forall r.
    (f : x_signature_type_signed_info_field r)
    : xml_codec.field_meta
    =
    match f with
    | SignatureType_SignedInfo_field_CanonicalizationMethod =>
        { tag = "CanonicalizationMethod"; min = 1i32; max = Some 1i32 }
    | SignatureType_SignedInfo_field_SignatureMethod => { tag = "SignatureMethod"; min = 1i32; max = Some 1i32 }
    | SignatureType_SignedInfo_field_Reference => { tag = "Reference"; min = 1i32; max = Some 1i32 }

union x_signature_type_signed_info_reference_field r =
    | SignatureType_SignedInfo_Reference_field_Transforms ::
        x_signature_type_signed_info_reference_field (option (x_signature_type_signed_info_reference_transforms))
    | SignatureType_SignedInfo_Reference_field_DigestMethod ::
        x_signature_type_signed_info_reference_field (x_signature_type_signed_info_reference_digest_method)
    | SignatureType_SignedInfo_Reference_field_DigestValue :: x_signature_type_signed_info_reference_field (string)

inl x_signature_type_signed_info_reference_get forall r.
    (f : x_signature_type_signed_info_reference_field r) (x : x_signature_type_signed_info_reference)
    : r
    =
    match f with
    | SignatureType_SignedInfo_Reference_field_Transforms => x.transforms
    | SignatureType_SignedInfo_Reference_field_DigestMethod => x.digest_method
    | SignatureType_SignedInfo_Reference_field_DigestValue => x.digest_value

inl x_signature_type_signed_info_reference_fields () : list (exists r. x_signature_type_signed_info_reference_field r) =
    [
        exists SignatureType_SignedInfo_Reference_field_Transforms
        exists SignatureType_SignedInfo_Reference_field_DigestMethod
        exists SignatureType_SignedInfo_Reference_field_DigestValue
    ]
inl x_signature_type_signed_info_reference_field_meta forall r.
    (f : x_signature_type_signed_info_reference_field r)
    : xml_codec.field_meta
    =
    match f with
    | SignatureType_SignedInfo_Reference_field_Transforms => { tag = "Transforms"; min = 0i32; max = Some 1i32 }
    | SignatureType_SignedInfo_Reference_field_DigestMethod => { tag = "DigestMethod"; min = 1i32; max = Some 1i32 }
    | SignatureType_SignedInfo_Reference_field_DigestValue => { tag = "DigestValue"; min = 1i32; max = Some 1i32 }

union x_signature_type_signed_info_reference_transforms_field r =
    | SignatureType_SignedInfo_Reference_Transforms_field_Transform ::
        x_signature_type_signed_info_reference_transforms_field
            (x_signature_type_signed_info_reference_transforms_transform)

inl x_signature_type_signed_info_reference_transforms_get forall r.
    (f : x_signature_type_signed_info_reference_transforms_field r)
    (x : x_signature_type_signed_info_reference_transforms)
    : r
    =
    match f with
    | SignatureType_SignedInfo_Reference_Transforms_field_Transform => x.transform

inl x_signature_type_signed_info_reference_transforms_fields
    ()
    : list (exists r. x_signature_type_signed_info_reference_transforms_field r)
    =
    [ exists SignatureType_SignedInfo_Reference_Transforms_field_Transform ]
inl x_signature_type_signed_info_reference_transforms_field_meta forall r.
    (f : x_signature_type_signed_info_reference_transforms_field r)
    : xml_codec.field_meta
    =
    match f with
    | SignatureType_SignedInfo_Reference_Transforms_field_Transform =>
        { tag = "Transform"; min = 1i32; max = Some 1i32 }

let rec x_dte_def_type_of_children forall s.
    (witness : stage s) (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type s) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_choice_documento, errors =
        xml_codec.taken (x_dte_def_type_choice_documento_of_children attributes children) errors
    inl o_signature, errors =
        staged_read witness
            fun errors => xml_codec.read_required "ds:Signature" x_signature_type_of_xml children errors
            errors
    inl o_version, errors = xml_codec.attribute_parsed xsd_decimal_of_string "version" attributes errors
    inl errors = xml_codec.fixed_checked "version" "1.0" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type {
            choice_documento = xml_codec.present o_choice_documento
            signature = xml_codec.present o_signature
            version = xml_codec.present o_version
        }

and let x_dte_def_type_of_xml forall s.
    (witness : stage s) (c : xml.xml_node)
    : result (x_dte_def_type s) (list xml_codec.decode_error)
    =
    x_dte_def_type_of_children witness (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento) (list xml_codec.decode_error)
    =
    if false
    then xml_codec.no_alternative "x_dte_def_type_choice_documento"
    elif xml.has_child "Documento" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Documento v
            (xml_codec.read_child "Documento" x_dte_def_type_choice_documento_documento_of_xml children)
    elif xml.has_child "Liquidacion" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Liquidacion v
            (xml_codec.read_child "Liquidacion" x_dte_def_type_choice_documento_liquidacion_of_xml children)
    elif xml.has_child "Exportaciones" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Exportaciones v
            (xml_codec.read_child "Exportaciones" x_dte_def_type_choice_documento_exportaciones_of_xml children)
    else xml_codec.no_alternative "x_dte_def_type_choice_documento"

and let x_dte_def_type_choice_documento_exportaciones_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_encabezado, errors =
        xml_codec.read_required
            "Encabezado"
            x_dte_def_type_choice_documento_exportaciones_encabezado_of_xml
            children
            errors
    inl v_detalle, errors =
        xml_codec.read_repeated "Detalle" x_dte_def_type_choice_documento_exportaciones_detalle_of_xml children errors
    inl v_sub_tot_info, errors =
        xml_codec.read_repeated
            "SubTotInfo"
            x_dte_def_type_choice_documento_exportaciones_sub_tot_info_of_xml
            children
            errors
    inl v_dsc_rcg_global, errors =
        xml_codec.read_repeated
            "DscRcgGlobal"
            x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_of_xml
            children
            errors
    inl v_referencia, errors =
        xml_codec.read_repeated
            "Referencia"
            x_dte_def_type_choice_documento_exportaciones_referencia_of_xml
            children
            errors
    inl o_ted, errors =
        xml_codec.read_required "TED" x_dte_def_type_choice_documento_exportaciones_ted_of_xml children errors
    inl o_tmst_firma, errors = xml_codec.read_required "TmstFirma" xml_codec.read_text children errors
    inl v_id, errors = xml_codec.attribute_required "ID" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones {
            encabezado = xml_codec.present o_encabezado
            detalle = v_detalle
            sub_tot_info = v_sub_tot_info
            dsc_rcg_global = v_dsc_rcg_global
            referencia = v_referencia
            ted = xml_codec.present o_ted
            tmst_firma = xml_codec.present o_tmst_firma
            id = v_id
        }

and let x_dte_def_type_choice_documento_exportaciones_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_dd, errors =
        xml_codec.read_required "DD" x_dte_def_type_choice_documento_exportaciones_ted_dd_of_xml children errors
    inl o_frmt, errors =
        xml_codec.read_required "FRMT" x_dte_def_type_choice_documento_exportaciones_ted_frmt_of_xml children errors
    inl v_version, errors = xml_codec.attribute_required "version" attributes errors
    inl errors = xml_codec.fixed_checked "version" "1.0" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted {
            dd = xml_codec.present o_dd
            frmt = xml_codec.present o_frmt
            version = v_version
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_frmt_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_frmt) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_base, errors = xml.text_in children, errors
    inl o_algoritmo, errors =
        xml_codec.attribute_parsed x_dte_def_type_choice_documento_exportaciones_ted_frmt_algoritmo_of_string
            "algoritmo"
            attributes
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_frmt {
            base = v_base
            algoritmo = xml_codec.present o_algoritmo
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_frmt_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_frmt) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_frmt_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_re, errors = xml_codec.read_required "RE" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_td, errors = xml_codec.read_required "TD" (xml_codec.read_value x_exp_type_of_string) children errors
    inl o_f, errors = xml_codec.read_required "F" xml_codec.read_text children errors
    inl o_fe, errors = xml_codec.read_required "FE" xml_codec.read_text children errors
    inl o_rr, errors = xml_codec.read_required "RR" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rsr, errors =
        xml_codec.read_required
            "RSR"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_ted_dd_rsr_of_string)
            children
            errors
    inl o_mnt, errors = xml_codec.read_required "MNT" xml_codec.read_text children errors
    inl o_it1, errors =
        xml_codec.read_required
            "IT1"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_ted_dd_it1_of_string)
            children
            errors
    inl o_caf, errors =
        xml_codec.read_required "CAF" x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_of_xml children errors
    inl o_tsted, errors = xml_codec.read_required "TSTED" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_dd {
            re = xml_codec.present o_re
            td = xml_codec.present o_td
            f = xml_codec.present o_f
            fe = xml_codec.present o_fe
            rr = xml_codec.present o_rr
            rsr = xml_codec.present o_rsr
            mnt = xml_codec.present o_mnt
            it1 = xml_codec.present o_it1
            caf = xml_codec.present o_caf
            tsted = xml_codec.present o_tsted
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_da, errors =
        xml_codec.read_required "DA" x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_of_xml children errors
    inl o_frma, errors =
        xml_codec.read_required
            "FRMA"
            x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_of_xml
            children
            errors
    inl v_version, errors = xml_codec.attribute_required "version" attributes errors
    inl errors = xml_codec.fixed_checked "version" "1.0" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf {
            da = xml_codec.present o_da
            frma = xml_codec.present o_frma
            version = v_version
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_base, errors = xml.text_in children, errors
    inl v_algoritmo, errors = xml_codec.attribute_required "algoritmo" attributes errors
    inl errors = xml_codec.fixed_checked "algoritmo" "SHA1withRSA" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma { base = v_base; algoritmo = v_algoritmo }

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_re, errors = xml_codec.read_required "RE" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rs, errors =
        xml_codec.read_required
            "RS"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rs_of_string)
            children
            errors
    inl o_td, errors = xml_codec.read_required "TD" (xml_codec.read_value x_exp_type_of_string) children errors
    inl o_rng, errors =
        xml_codec.read_required
            "RNG"
            x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_of_xml
            children
            errors
    inl o_fa, errors = xml_codec.read_required "FA" xml_codec.read_text children errors
    inl o_choice_rsapk, errors =
        xml_codec.taken
            (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_of_children attributes children)
            errors
    inl o_idk, errors = xml_codec.read_required "IDK" (xml_codec.read_value xsd_integer_of_string) children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da {
            re = xml_codec.present o_re
            rs = xml_codec.present o_rs
            td = xml_codec.present o_td
            rng = xml_codec.present o_rng
            fa = xml_codec.present o_fa
            choice_rsapk = xml_codec.present o_choice_rsapk
            idk = xml_codec.present o_idk
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk) (list xml_codec.decode_error)
    =
    if false
    then xml_codec.no_alternative "x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk"
    elif xml.has_child "RSAPK" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK v
            (
                xml_codec.read_child
                    "RSAPK"
                    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
                    children
            )
    elif xml.has_child "DSAPK" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK v
            (
                xml_codec.read_child
                    "DSAPK"
                    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
                    children
            )
    else xml_codec.no_alternative "x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk"

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_p, errors = xml_codec.read_required "P" xml_codec.read_text children errors
    inl o_q, errors = xml_codec.read_required "Q" xml_codec.read_text children errors
    inl o_g, errors = xml_codec.read_required "G" xml_codec.read_text children errors
    inl o_y, errors = xml_codec.read_required "Y" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk {
            p = xml_codec.present o_p
            q = xml_codec.present o_q
            g = xml_codec.present o_g
            y = xml_codec.present o_y
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_m, errors = xml_codec.read_required "M" xml_codec.read_text children errors
    inl o_e, errors = xml_codec.read_required "E" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk {
            m = xml_codec.present o_m
            e = xml_codec.present o_e
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_d, errors = xml_codec.read_required "D" xml_codec.read_text children errors
    inl o_h, errors = xml_codec.read_required "H" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng {
            d = xml_codec.present o_d
            h = xml_codec.present o_h
        }

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_referencia_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_referencia) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_ref, errors = xml_codec.read_required "NroLinRef" xml_codec.read_text children errors
    inl o_tpo_doc_ref, errors =
        xml_codec.read_required
            "TpoDocRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_referencia_tpo_doc_ref_of_string)
            children
            errors
    inl v_ind_global, errors =
        xml_codec.read_optional
            "IndGlobal"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_referencia_ind_global_of_string)
            children
            errors
    inl o_folio_ref, errors =
        xml_codec.read_required "FolioRef" (xml_codec.read_value x_folio_r_type_of_string) children errors
    inl v_rut_otr, errors = xml_codec.read_optional "RUTOtr" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_id_adic_otr, errors =
        xml_codec.read_optional
            "IdAdicOtr"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_referencia_id_adic_otr_of_string)
            children
            errors
    inl o_fch_ref, errors = xml_codec.read_required "FchRef" xml_codec.read_text children errors
    inl v_cod_ref, errors =
        xml_codec.read_optional
            "CodRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_referencia_cod_ref_of_string)
            children
            errors
    inl v_razon_ref, errors =
        xml_codec.read_optional
            "RazonRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_referencia_razon_ref_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_referencia {
            nro_lin_ref = xml_codec.present o_nro_lin_ref
            tpo_doc_ref = xml_codec.present o_tpo_doc_ref
            ind_global = v_ind_global
            folio_ref = xml_codec.present o_folio_ref
            rut_otr = v_rut_otr
            id_adic_otr = v_id_adic_otr
            fch_ref = xml_codec.present o_fch_ref
            cod_ref = v_cod_ref
            razon_ref = v_razon_ref
        }

and let x_dte_def_type_choice_documento_exportaciones_referencia_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_referencia) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_referencia_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_dr, errors =
        xml_codec.read_required "NroLinDR" (xml_codec.read_value xsd_positive_integer_of_string) children errors
    inl o_tpo_mov, errors =
        xml_codec.read_required
            "TpoMov"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_mov_of_string)
            children
            errors
    inl v_glosa_dr, errors =
        xml_codec.read_optional
            "GlosaDR"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_glosa_dr_of_string)
            children
            errors
    inl o_tpo_valor, errors =
        xml_codec.read_required
            "TpoValor"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_tpo_valor_of_string)
            children
            errors
    inl o_valor_dr, errors = xml_codec.read_required "ValorDR" xml_codec.read_text children errors
    inl v_valor_dr_otr_mnda, errors = xml_codec.read_optional "ValorDROtrMnda" xml_codec.read_text children errors
    inl v_ind_exe_dr, errors =
        xml_codec.read_optional
            "IndExeDR"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_ind_exe_dr_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global {
            nro_lin_dr = xml_codec.present o_nro_lin_dr
            tpo_mov = xml_codec.present o_tpo_mov
            glosa_dr = v_glosa_dr
            tpo_valor = xml_codec.present o_tpo_valor
            valor_dr = xml_codec.present o_valor_dr
            valor_dr_otr_mnda = v_valor_dr_otr_mnda
            ind_exe_dr = v_ind_exe_dr
        }

and let x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_sub_tot_info_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_sub_tot_info) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_sti, errors = xml_codec.read_required "NroSTI" xml_codec.read_text children errors
    inl o_glosa_sti, errors =
        xml_codec.read_required
            "GlosaSTI"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_sub_tot_info_glosa_sti_of_string)
            children
            errors
    inl v_orden_sti, errors = xml_codec.read_optional "OrdenSTI" xml_codec.read_text children errors
    inl v_sub_tot_neto_sti, errors = xml_codec.read_optional "SubTotNetoSTI" xml_codec.read_text children errors
    inl v_sub_tot_ivasti, errors = xml_codec.read_optional "SubTotIVASTI" xml_codec.read_text children errors
    inl v_sub_tot_adic_sti, errors = xml_codec.read_optional "SubTotAdicSTI" xml_codec.read_text children errors
    inl v_sub_tot_exe_sti, errors = xml_codec.read_optional "SubTotExeSTI" xml_codec.read_text children errors
    inl v_val_subtot_sti, errors = xml_codec.read_optional "ValSubtotSTI" xml_codec.read_text children errors
    inl v_lineas_deta, errors = xml_codec.read_repeated "LineasDeta" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_sub_tot_info {
            nro_sti = xml_codec.present o_nro_sti
            glosa_sti = xml_codec.present o_glosa_sti
            orden_sti = v_orden_sti
            sub_tot_neto_sti = v_sub_tot_neto_sti
            sub_tot_ivasti = v_sub_tot_ivasti
            sub_tot_adic_sti = v_sub_tot_adic_sti
            sub_tot_exe_sti = v_sub_tot_exe_sti
            val_subtot_sti = v_val_subtot_sti
            lineas_deta = v_lineas_deta
        }

and let x_dte_def_type_choice_documento_exportaciones_sub_tot_info_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_sub_tot_info) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_detalle_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_det, errors = xml_codec.read_required "NroLinDet" xml_codec.read_text children errors
    inl v_cdg_item, errors =
        xml_codec.read_repeated
            "CdgItem"
            x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_of_xml
            children
            errors
    inl v_ind_exe, errors =
        xml_codec.read_optional
            "IndExe"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_ind_exe_of_string)
            children
            errors
    inl o_nmb_item, errors =
        xml_codec.read_required
            "NmbItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_nmb_item_of_string)
            children
            errors
    inl v_dsc_item, errors =
        xml_codec.read_optional
            "DscItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_dsc_item_of_string)
            children
            errors
    inl v_qty_ref, errors = xml_codec.read_optional "QtyRef" xml_codec.read_text children errors
    inl v_unmd_ref, errors =
        xml_codec.read_optional
            "UnmdRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_unmd_ref_of_string)
            children
            errors
    inl v_prc_ref, errors = xml_codec.read_optional "PrcRef" xml_codec.read_text children errors
    inl v_qty_item, errors = xml_codec.read_optional "QtyItem" xml_codec.read_text children errors
    inl v_subcantidad, errors =
        xml_codec.read_repeated
            "Subcantidad"
            x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_of_xml
            children
            errors
    inl v_fch_elabor, errors = xml_codec.read_optional "FchElabor" xml_codec.read_text children errors
    inl v_fch_vencim, errors = xml_codec.read_optional "FchVencim" xml_codec.read_text children errors
    inl v_unmd_item, errors =
        xml_codec.read_optional
            "UnmdItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_unmd_item_of_string)
            children
            errors
    inl v_prc_item, errors = xml_codec.read_optional "PrcItem" xml_codec.read_text children errors
    inl v_otr_mnda, errors =
        xml_codec.read_optional
            "OtrMnda"
            x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_of_xml
            children
            errors
    inl v_descuento_pct, errors = xml_codec.read_optional "DescuentoPct" xml_codec.read_text children errors
    inl v_descuento_monto, errors = xml_codec.read_optional "DescuentoMonto" xml_codec.read_text children errors
    inl v_sub_dscto, errors =
        xml_codec.read_repeated
            "SubDscto"
            x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_of_xml
            children
            errors
    inl v_recargo_pct, errors = xml_codec.read_optional "RecargoPct" xml_codec.read_text children errors
    inl v_recargo_monto, errors = xml_codec.read_optional "RecargoMonto" xml_codec.read_text children errors
    inl v_sub_recargo, errors =
        xml_codec.read_repeated
            "SubRecargo"
            x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_of_xml
            children
            errors
    inl o_monto_item, errors = xml_codec.read_required "MontoItem" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_detalle {
            nro_lin_det = xml_codec.present o_nro_lin_det
            cdg_item = v_cdg_item
            ind_exe = v_ind_exe
            nmb_item = xml_codec.present o_nmb_item
            dsc_item = v_dsc_item
            qty_ref = v_qty_ref
            unmd_ref = v_unmd_ref
            prc_ref = v_prc_ref
            qty_item = v_qty_item
            subcantidad = v_subcantidad
            fch_elabor = v_fch_elabor
            fch_vencim = v_fch_vencim
            unmd_item = v_unmd_item
            prc_item = v_prc_item
            otr_mnda = v_otr_mnda
            descuento_pct = v_descuento_pct
            descuento_monto = v_descuento_monto
            sub_dscto = v_sub_dscto
            recargo_pct = v_recargo_pct
            recargo_monto = v_recargo_monto
            sub_recargo = v_sub_recargo
            monto_item = xml_codec.present o_monto_item
        }

and let x_dte_def_type_choice_documento_exportaciones_detalle_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_recargo, errors =
        xml_codec.read_required
            "TipoRecargo"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_tipo_recargo_of_string
            )
            children
            errors
    inl o_valor_recargo, errors = xml_codec.read_required "ValorRecargo" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo {
            tipo_recargo = xml_codec.present o_tipo_recargo
            valor_recargo = xml_codec.present o_valor_recargo
        }

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_dscto, errors =
        xml_codec.read_required
            "TipoDscto"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_tipo_dscto_of_string)
            children
            errors
    inl o_valor_dscto, errors = xml_codec.read_required "ValorDscto" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto {
            tipo_dscto = xml_codec.present o_tipo_dscto
            valor_dscto = xml_codec.present o_valor_dscto
        }

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_prc_otr_mon, errors = xml_codec.read_required "PrcOtrMon" xml_codec.read_text children errors
    inl o_moneda, errors =
        xml_codec.read_required
            "Moneda"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_moneda_of_string)
            children
            errors
    inl v_fct_conv, errors = xml_codec.read_optional "FctConv" xml_codec.read_text children errors
    inl v_dcto_otr_mnda, errors = xml_codec.read_optional "DctoOtrMnda" xml_codec.read_text children errors
    inl v_recargo_otr_mnda, errors = xml_codec.read_optional "RecargoOtrMnda" xml_codec.read_text children errors
    inl o_monto_item_otr_mnda, errors = xml_codec.read_required "MontoItemOtrMnda" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda {
            prc_otr_mon = xml_codec.present o_prc_otr_mon
            moneda = xml_codec.present o_moneda
            fct_conv = v_fct_conv
            dcto_otr_mnda = v_dcto_otr_mnda
            recargo_otr_mnda = v_recargo_otr_mnda
            monto_item_otr_mnda = xml_codec.present o_monto_item_otr_mnda
        }

and let x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_sub_qty, errors = xml_codec.read_required "SubQty" xml_codec.read_text children errors
    inl o_sub_cod, errors =
        xml_codec.read_required
            "SubCod"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_sub_cod_of_string)
            children
            errors
    inl v_tip_cod_sub_qty, errors =
        xml_codec.read_optional
            "TipCodSubQty"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_tip_cod_sub_qty_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad {
            sub_qty = xml_codec.present o_sub_qty
            sub_cod = xml_codec.present o_sub_cod
            tip_cod_sub_qty = v_tip_cod_sub_qty
        }

and let x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tpo_codigo, errors =
        xml_codec.read_required
            "TpoCodigo"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_tpo_codigo_of_string)
            children
            errors
    inl o_vlr_codigo, errors =
        xml_codec.read_required
            "VlrCodigo"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_vlr_codigo_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item {
            tpo_codigo = xml_codec.present o_tpo_codigo
            vlr_codigo = xml_codec.present o_vlr_codigo
        }

and let x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_id_doc, errors =
        xml_codec.read_required
            "IdDoc"
            x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_of_xml
            children
            errors
    inl o_emisor, errors =
        xml_codec.read_required
            "Emisor"
            x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_of_xml
            children
            errors
    inl o_receptor, errors =
        xml_codec.read_required
            "Receptor"
            x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_of_xml
            children
            errors
    inl v_transporte, errors =
        xml_codec.read_optional
            "Transporte"
            x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_of_xml
            children
            errors
    inl o_totales, errors =
        xml_codec.read_required
            "Totales"
            x_dte_def_type_choice_documento_exportaciones_encabezado_totales_of_xml
            children
            errors
    inl v_otra_moneda, errors =
        xml_codec.read_optional
            "OtraMoneda"
            x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_of_xml
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado {
            id_doc = xml_codec.present o_id_doc
            emisor = xml_codec.present o_emisor
            receptor = xml_codec.present o_receptor
            transporte = v_transporte
            totales = xml_codec.present o_totales
            otra_moneda = v_otra_moneda
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tpo_moneda, errors =
        xml_codec.read_required "TpoMoneda" (xml_codec.read_value x_tip_mon_type_of_string) children errors
    inl v_tpo_cambio, errors = xml_codec.read_optional "TpoCambio" xml_codec.read_text children errors
    inl v_mnt_exe_otr_mnda, errors = xml_codec.read_optional "MntExeOtrMnda" xml_codec.read_text children errors
    inl o_mnt_tot_otr_mnda, errors = xml_codec.read_required "MntTotOtrMnda" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda {
            tpo_moneda = xml_codec.present o_tpo_moneda
            tpo_cambio = v_tpo_cambio
            mnt_exe_otr_mnda = v_mnt_exe_otr_mnda
            mnt_tot_otr_mnda = xml_codec.present o_mnt_tot_otr_mnda
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_totales_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_totales) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tpo_moneda, errors =
        xml_codec.read_required "TpoMoneda" (xml_codec.read_value x_tip_mon_type_of_string) children errors
    inl o_mnt_exe, errors = xml_codec.read_required "MntExe" xml_codec.read_text children errors
    inl o_mnt_total, errors = xml_codec.read_required "MntTotal" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_totales {
            tpo_moneda = xml_codec.present o_tpo_moneda
            mnt_exe = xml_codec.present o_mnt_exe
            mnt_total = xml_codec.present o_mnt_total
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_totales_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_totales) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_totales_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_patente, errors =
        xml_codec.read_optional
            "Patente"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_patente_of_string)
            children
            errors
    inl v_rut_trans, errors =
        xml_codec.read_optional "RUTTrans" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_chofer, errors =
        xml_codec.read_optional
            "Chofer"
            x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_of_xml
            children
            errors
    inl v_dir_dest, errors =
        xml_codec.read_optional
            "DirDest"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_dir_dest_of_string
            )
            children
            errors
    inl v_cmna_dest, errors =
        xml_codec.read_optional "CmnaDest" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_dest, errors =
        xml_codec.read_optional "CiudadDest" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_aduana, errors =
        xml_codec.read_optional
            "Aduana"
            x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_of_xml
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte {
            patente = v_patente
            rut_trans = v_rut_trans
            chofer = v_chofer
            dir_dest = v_dir_dest
            cmna_dest = v_cmna_dest
            ciudad_dest = v_ciudad_dest
            aduana = v_aduana
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_cod_mod_venta, errors = xml_codec.read_optional "CodModVenta" xml_codec.read_text children errors
    inl v_cod_clau_venta, errors = xml_codec.read_optional "CodClauVenta" xml_codec.read_text children errors
    inl v_tot_clau_venta, errors = xml_codec.read_optional "TotClauVenta" xml_codec.read_text children errors
    inl v_cod_via_transp, errors = xml_codec.read_optional "CodViaTransp" xml_codec.read_text children errors
    inl v_nombre_transp, errors =
        xml_codec.read_optional
            "NombreTransp"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nombre_transp_of_string
            )
            children
            errors
    inl v_rut_cia_transp, errors =
        xml_codec.read_optional "RUTCiaTransp" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_nom_cia_transp, errors =
        xml_codec.read_optional
            "NomCiaTransp"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_nom_cia_transp_of_string
            )
            children
            errors
    inl v_id_adic_transp, errors =
        xml_codec.read_optional
            "IdAdicTransp"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_transp_of_string
            )
            children
            errors
    inl v_booking, errors =
        xml_codec.read_optional
            "Booking"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_booking_of_string
            )
            children
            errors
    inl v_operador, errors =
        xml_codec.read_optional
            "Operador"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_operador_of_string
            )
            children
            errors
    inl v_cod_pto_embarque, errors = xml_codec.read_optional "CodPtoEmbarque" xml_codec.read_text children errors
    inl v_id_adic_pto_emb, errors =
        xml_codec.read_optional
            "IdAdicPtoEmb"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_emb_of_string
            )
            children
            errors
    inl v_cod_pto_desemb, errors = xml_codec.read_optional "CodPtoDesemb" xml_codec.read_text children errors
    inl v_id_adic_pto_desemb, errors =
        xml_codec.read_optional
            "IdAdicPtoDesemb"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_id_adic_pto_desemb_of_string
            )
            children
            errors
    inl v_tara, errors = xml_codec.read_optional "Tara" xml_codec.read_text children errors
    inl v_cod_unid_med_tara, errors = xml_codec.read_optional "CodUnidMedTara" xml_codec.read_text children errors
    inl v_peso_bruto, errors = xml_codec.read_optional "PesoBruto" xml_codec.read_text children errors
    inl v_cod_unid_peso_bruto, errors = xml_codec.read_optional "CodUnidPesoBruto" xml_codec.read_text children errors
    inl v_peso_neto, errors = xml_codec.read_optional "PesoNeto" xml_codec.read_text children errors
    inl v_cod_unid_peso_neto, errors = xml_codec.read_optional "CodUnidPesoNeto" xml_codec.read_text children errors
    inl v_tot_items, errors = xml_codec.read_optional "TotItems" xml_codec.read_text children errors
    inl v_tot_bultos, errors = xml_codec.read_optional "TotBultos" xml_codec.read_text children errors
    inl v_tipo_bultos, errors =
        xml_codec.read_repeated
            "TipoBultos"
            x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_of_xml
            children
            errors
    inl v_mnt_flete, errors = xml_codec.read_optional "MntFlete" xml_codec.read_text children errors
    inl v_mnt_seguro, errors = xml_codec.read_optional "MntSeguro" xml_codec.read_text children errors
    inl v_cod_pais_recep, errors = xml_codec.read_optional "CodPaisRecep" xml_codec.read_text children errors
    inl v_cod_pais_destin, errors = xml_codec.read_optional "CodPaisDestin" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana {
            cod_mod_venta = v_cod_mod_venta
            cod_clau_venta = v_cod_clau_venta
            tot_clau_venta = v_tot_clau_venta
            cod_via_transp = v_cod_via_transp
            nombre_transp = v_nombre_transp
            rut_cia_transp = v_rut_cia_transp
            nom_cia_transp = v_nom_cia_transp
            id_adic_transp = v_id_adic_transp
            booking = v_booking
            operador = v_operador
            cod_pto_embarque = v_cod_pto_embarque
            id_adic_pto_emb = v_id_adic_pto_emb
            cod_pto_desemb = v_cod_pto_desemb
            id_adic_pto_desemb = v_id_adic_pto_desemb
            tara = v_tara
            cod_unid_med_tara = v_cod_unid_med_tara
            peso_bruto = v_peso_bruto
            cod_unid_peso_bruto = v_cod_unid_peso_bruto
            peso_neto = v_peso_neto
            cod_unid_peso_neto = v_cod_unid_peso_neto
            tot_items = v_tot_items
            tot_bultos = v_tot_bultos
            tipo_bultos = v_tipo_bultos
            mnt_flete = v_mnt_flete
            mnt_seguro = v_mnt_seguro
            cod_pais_recep = v_cod_pais_recep
            cod_pais_destin = v_cod_pais_destin
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_cod_tpo_bultos, errors = xml_codec.read_optional "CodTpoBultos" xml_codec.read_text children errors
    inl v_cant_bultos, errors = xml_codec.read_optional "CantBultos" xml_codec.read_text children errors
    inl v_marcas, errors =
        xml_codec.read_optional
            "Marcas"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_marcas_of_string
            )
            children
            errors
    inl v_id_container, errors =
        xml_codec.read_optional
            "IdContainer"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_id_container_of_string
            )
            children
            errors
    inl v_sello, errors =
        xml_codec.read_optional
            "Sello"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_sello_of_string
            )
            children
            errors
    inl v_emisor_sello, errors =
        xml_codec.read_optional
            "EmisorSello"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_emisor_sello_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos {
            cod_tpo_bultos = v_cod_tpo_bultos
            cant_bultos = v_cant_bultos
            marcas = v_marcas
            id_container = v_id_container
            sello = v_sello
            emisor_sello = v_emisor_sello
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_chofer, errors =
        xml_codec.read_required "RUTChofer" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_nombre_chofer, errors =
        xml_codec.read_required
            "NombreChofer"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_nombre_chofer_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer {
            rut_chofer = xml_codec.present o_rut_chofer
            nombre_chofer = xml_codec.present o_nombre_chofer
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_recep, errors =
        xml_codec.read_required "RUTRecep" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_cdg_int_recep, errors =
        xml_codec.read_optional
            "CdgIntRecep"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_cdg_int_recep_of_string
            )
            children
            errors
    inl o_rzn_soc_recep, errors =
        xml_codec.read_required "RznSocRecep" (xml_codec.read_value x_rzn_soc_larga_type_of_string) children errors
    inl v_extranjero, errors =
        xml_codec.read_optional
            "Extranjero"
            x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_of_xml
            children
            errors
    inl v_giro_recep, errors =
        xml_codec.read_optional
            "GiroRecep"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_giro_recep_of_string
            )
            children
            errors
    inl v_contacto, errors =
        xml_codec.read_optional
            "Contacto"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_contacto_of_string)
            children
            errors
    inl v_correo_recep, errors =
        xml_codec.read_optional "CorreoRecep" (xml_codec.read_value x_mail_type_of_string) children errors
    inl v_dir_recep, errors =
        xml_codec.read_optional
            "DirRecep"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_recep_of_string)
            children
            errors
    inl v_cmna_recep, errors =
        xml_codec.read_optional "CmnaRecep" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_recep, errors =
        xml_codec.read_optional "CiudadRecep" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_dir_postal, errors =
        xml_codec.read_optional
            "DirPostal"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_dir_postal_of_string
            )
            children
            errors
    inl v_cmna_postal, errors =
        xml_codec.read_optional "CmnaPostal" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_postal, errors =
        xml_codec.read_optional "CiudadPostal" (xml_codec.read_value x_ciudad_type_of_string) children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor {
            rut_recep = xml_codec.present o_rut_recep
            cdg_int_recep = v_cdg_int_recep
            rzn_soc_recep = xml_codec.present o_rzn_soc_recep
            extranjero = v_extranjero
            giro_recep = v_giro_recep
            contacto = v_contacto
            correo_recep = v_correo_recep
            dir_recep = v_dir_recep
            cmna_recep = v_cmna_recep
            ciudad_recep = v_ciudad_recep
            dir_postal = v_dir_postal
            cmna_postal = v_cmna_postal
            ciudad_postal = v_ciudad_postal
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_num_id, errors =
        xml_codec.read_optional
            "NumId"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_num_id_of_string
            )
            children
            errors
    inl v_nacionalidad, errors =
        xml_codec.read_optional
            "Nacionalidad"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_nacionalidad_of_string
            )
            children
            errors
    inl v_id_adic_recep, errors =
        xml_codec.read_optional
            "IdAdicRecep"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_id_adic_recep_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero {
            num_id = v_num_id
            nacionalidad = v_nacionalidad
            id_adic_recep = v_id_adic_recep
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_emisor, errors =
        xml_codec.read_required "RUTEmisor" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rzn_soc, errors =
        xml_codec.read_required
            "RznSoc"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_rzn_soc_of_string)
            children
            errors
    inl o_giro_emis, errors =
        xml_codec.read_required
            "GiroEmis"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_giro_emis_of_string)
            children
            errors
    inl v_telefono, errors =
        xml_codec.read_repeated
            "Telefono"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_telefono_of_string)
            children
            errors
    inl v_correo_emisor, errors =
        xml_codec.read_optional "CorreoEmisor" (xml_codec.read_value x_mail_type_of_string) children errors
    inl v_acteco, errors = xml_codec.read_repeated "Acteco" xml_codec.read_text children errors
    inl v_sucursal, errors =
        xml_codec.read_optional
            "Sucursal"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_sucursal_of_string)
            children
            errors
    inl v_cdg_sii_sucur, errors = xml_codec.read_optional "CdgSIISucur" xml_codec.read_text children errors
    inl v_cod_adic_sucur, errors =
        xml_codec.read_optional
            "CodAdicSucur"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cod_adic_sucur_of_string
            )
            children
            errors
    inl v_dir_origen, errors =
        xml_codec.read_optional
            "DirOrigen"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_dir_origen_of_string)
            children
            errors
    inl v_cmna_origen, errors =
        xml_codec.read_optional "CmnaOrigen" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_origen, errors =
        xml_codec.read_optional "CiudadOrigen" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_cdg_vendedor, errors =
        xml_codec.read_optional
            "CdgVendedor"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_cdg_vendedor_of_string
            )
            children
            errors
    inl v_id_adic_emisor, errors =
        xml_codec.read_optional
            "IdAdicEmisor"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_id_adic_emisor_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_emisor {
            rut_emisor = xml_codec.present o_rut_emisor
            rzn_soc = xml_codec.present o_rzn_soc
            giro_emis = xml_codec.present o_giro_emis
            telefono = v_telefono
            correo_emisor = v_correo_emisor
            acteco = v_acteco
            sucursal = v_sucursal
            cdg_sii_sucur = v_cdg_sii_sucur
            cod_adic_sucur = v_cod_adic_sucur
            dir_origen = v_dir_origen
            cmna_origen = v_cmna_origen
            ciudad_origen = v_ciudad_origen
            cdg_vendedor = v_cdg_vendedor
            id_adic_emisor = v_id_adic_emisor
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_dte, errors =
        xml_codec.read_required "TipoDTE" (xml_codec.read_value x_exp_type_of_string) children errors
    inl o_folio, errors = xml_codec.read_required "Folio" xml_codec.read_text children errors
    inl o_fch_emis, errors = xml_codec.read_required "FchEmis" xml_codec.read_text children errors
    inl v_tipo_despacho, errors =
        xml_codec.read_optional
            "TipoDespacho"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tipo_despacho_of_string
            )
            children
            errors
    inl v_ind_servicio, errors =
        xml_codec.read_optional
            "IndServicio"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_ind_servicio_of_string
            )
            children
            errors
    inl v_fma_pago, errors =
        xml_codec.read_optional
            "FmaPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_fma_pago_of_string)
            children
            errors
    inl v_fma_pag_exp, errors = xml_codec.read_optional "FmaPagExp" xml_codec.read_text children errors
    inl v_fch_cancel, errors = xml_codec.read_optional "FchCancel" xml_codec.read_text children errors
    inl v_mnt_cancel, errors = xml_codec.read_optional "MntCancel" xml_codec.read_text children errors
    inl v_saldo_insol, errors = xml_codec.read_optional "SaldoInsol" xml_codec.read_text children errors
    inl v_mnt_pagos, errors =
        xml_codec.read_repeated
            "MntPagos"
            x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_of_xml
            children
            errors
    inl v_periodo_desde, errors = xml_codec.read_optional "PeriodoDesde" xml_codec.read_text children errors
    inl v_periodo_hasta, errors = xml_codec.read_optional "PeriodoHasta" xml_codec.read_text children errors
    inl v_medio_pago, errors =
        xml_codec.read_optional "MedioPago" (xml_codec.read_value x_medio_pago_type_of_string) children errors
    inl v_tpo_cta_pago, errors =
        xml_codec.read_optional
            "TpoCtaPago"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_tpo_cta_pago_of_string
            )
            children
            errors
    inl v_num_cta_pago, errors =
        xml_codec.read_optional
            "NumCtaPago"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_num_cta_pago_of_string
            )
            children
            errors
    inl v_bco_pago, errors =
        xml_codec.read_optional
            "BcoPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_bco_pago_of_string)
            children
            errors
    inl v_term_pago_cdg, errors =
        xml_codec.read_optional
            "TermPagoCdg"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_cdg_of_string
            )
            children
            errors
    inl v_term_pago_glosa, errors =
        xml_codec.read_optional
            "TermPagoGlosa"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_term_pago_glosa_of_string
            )
            children
            errors
    inl v_term_pago_dias, errors = xml_codec.read_optional "TermPagoDias" xml_codec.read_text children errors
    inl v_fch_venc, errors = xml_codec.read_optional "FchVenc" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc {
            tipo_dte = xml_codec.present o_tipo_dte
            folio = xml_codec.present o_folio
            fch_emis = xml_codec.present o_fch_emis
            tipo_despacho = v_tipo_despacho
            ind_servicio = v_ind_servicio
            fma_pago = v_fma_pago
            fma_pag_exp = v_fma_pag_exp
            fch_cancel = v_fch_cancel
            mnt_cancel = v_mnt_cancel
            saldo_insol = v_saldo_insol
            mnt_pagos = v_mnt_pagos
            periodo_desde = v_periodo_desde
            periodo_hasta = v_periodo_hasta
            medio_pago = v_medio_pago
            tpo_cta_pago = v_tpo_cta_pago
            num_cta_pago = v_num_cta_pago
            bco_pago = v_bco_pago
            term_pago_cdg = v_term_pago_cdg
            term_pago_glosa = v_term_pago_glosa
            term_pago_dias = v_term_pago_dias
            fch_venc = v_fch_venc
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_fch_pago, errors = xml_codec.read_required "FchPago" xml_codec.read_text children errors
    inl o_mnt_pago, errors = xml_codec.read_required "MntPago" xml_codec.read_text children errors
    inl v_glosa_pagos, errors =
        xml_codec.read_optional
            "GlosaPagos"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_glosa_pagos_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos {
            fch_pago = xml_codec.present o_fch_pago
            mnt_pago = xml_codec.present o_mnt_pago
            glosa_pagos = v_glosa_pagos
        }

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_encabezado, errors =
        xml_codec.read_required
            "Encabezado"
            x_dte_def_type_choice_documento_liquidacion_encabezado_of_xml
            children
            errors
    inl v_detalle, errors =
        xml_codec.read_repeated "Detalle" x_dte_def_type_choice_documento_liquidacion_detalle_of_xml children errors
    inl v_sub_tot_info, errors =
        xml_codec.read_repeated
            "SubTotInfo"
            x_dte_def_type_choice_documento_liquidacion_sub_tot_info_of_xml
            children
            errors
    inl v_referencia, errors =
        xml_codec.read_repeated
            "Referencia"
            x_dte_def_type_choice_documento_liquidacion_referencia_of_xml
            children
            errors
    inl v_comisiones, errors =
        xml_codec.read_repeated
            "Comisiones"
            x_dte_def_type_choice_documento_liquidacion_comisiones_of_xml
            children
            errors
    inl o_ted, errors =
        xml_codec.read_required "TED" x_dte_def_type_choice_documento_liquidacion_ted_of_xml children errors
    inl o_tmst_firma, errors = xml_codec.read_required "TmstFirma" xml_codec.read_text children errors
    inl v_id, errors = xml_codec.attribute_required "ID" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion {
            encabezado = xml_codec.present o_encabezado
            detalle = v_detalle
            sub_tot_info = v_sub_tot_info
            referencia = v_referencia
            comisiones = v_comisiones
            ted = xml_codec.present o_ted
            tmst_firma = xml_codec.present o_tmst_firma
            id = v_id
        }

and let x_dte_def_type_choice_documento_liquidacion_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_dd, errors =
        xml_codec.read_required "DD" x_dte_def_type_choice_documento_liquidacion_ted_dd_of_xml children errors
    inl o_frmt, errors =
        xml_codec.read_required "FRMT" x_dte_def_type_choice_documento_liquidacion_ted_frmt_of_xml children errors
    inl v_version, errors = xml_codec.attribute_required "version" attributes errors
    inl errors = xml_codec.fixed_checked "version" "1.0" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted {
            dd = xml_codec.present o_dd
            frmt = xml_codec.present o_frmt
            version = v_version
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_frmt_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_frmt) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_base, errors = xml.text_in children, errors
    inl o_algoritmo, errors =
        xml_codec.attribute_parsed x_dte_def_type_choice_documento_liquidacion_ted_frmt_algoritmo_of_string
            "algoritmo"
            attributes
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_frmt {
            base = v_base
            algoritmo = xml_codec.present o_algoritmo
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_frmt_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_frmt) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_frmt_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_re, errors = xml_codec.read_required "RE" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_td, errors = xml_codec.read_required "TD" (xml_codec.read_value x_liq_type_of_string) children errors
    inl o_f, errors = xml_codec.read_required "F" xml_codec.read_text children errors
    inl o_fe, errors = xml_codec.read_required "FE" xml_codec.read_text children errors
    inl o_rr, errors = xml_codec.read_required "RR" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rsr, errors =
        xml_codec.read_required
            "RSR"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_ted_dd_rsr_of_string)
            children
            errors
    inl o_mnt, errors = xml_codec.read_required "MNT" xml_codec.read_text children errors
    inl o_it1, errors =
        xml_codec.read_required
            "IT1"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_ted_dd_it1_of_string)
            children
            errors
    inl o_caf, errors =
        xml_codec.read_required "CAF" x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_of_xml children errors
    inl o_tsted, errors = xml_codec.read_required "TSTED" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_dd {
            re = xml_codec.present o_re
            td = xml_codec.present o_td
            f = xml_codec.present o_f
            fe = xml_codec.present o_fe
            rr = xml_codec.present o_rr
            rsr = xml_codec.present o_rsr
            mnt = xml_codec.present o_mnt
            it1 = xml_codec.present o_it1
            caf = xml_codec.present o_caf
            tsted = xml_codec.present o_tsted
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_da, errors =
        xml_codec.read_required "DA" x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_of_xml children errors
    inl o_frma, errors =
        xml_codec.read_required
            "FRMA"
            x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_of_xml
            children
            errors
    inl v_version, errors = xml_codec.attribute_required "version" attributes errors
    inl errors = xml_codec.fixed_checked "version" "1.0" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf {
            da = xml_codec.present o_da
            frma = xml_codec.present o_frma
            version = v_version
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_base, errors = xml.text_in children, errors
    inl v_algoritmo, errors = xml_codec.attribute_required "algoritmo" attributes errors
    inl errors = xml_codec.fixed_checked "algoritmo" "SHA1withRSA" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma { base = v_base; algoritmo = v_algoritmo }

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_re, errors = xml_codec.read_required "RE" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rs, errors =
        xml_codec.read_required "RS" (xml_codec.read_value x_rzn_soc_corta_type_of_string) children errors
    inl o_td, errors = xml_codec.read_required "TD" (xml_codec.read_value x_liq_type_of_string) children errors
    inl o_rng, errors =
        xml_codec.read_required
            "RNG"
            x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_of_xml
            children
            errors
    inl o_fa, errors = xml_codec.read_required "FA" xml_codec.read_text children errors
    inl o_choice_rsapk, errors =
        xml_codec.taken
            (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_of_children attributes children)
            errors
    inl o_idk, errors = xml_codec.read_required "IDK" (xml_codec.read_value xsd_integer_of_string) children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da {
            re = xml_codec.present o_re
            rs = xml_codec.present o_rs
            td = xml_codec.present o_td
            rng = xml_codec.present o_rng
            fa = xml_codec.present o_fa
            choice_rsapk = xml_codec.present o_choice_rsapk
            idk = xml_codec.present o_idk
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk) (list xml_codec.decode_error)
    =
    if false
    then xml_codec.no_alternative "x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk"
    elif xml.has_child "RSAPK" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK v
            (
                xml_codec.read_child
                    "RSAPK"
                    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
                    children
            )
    elif xml.has_child "DSAPK" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK v
            (
                xml_codec.read_child
                    "DSAPK"
                    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
                    children
            )
    else xml_codec.no_alternative "x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk"

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_p, errors = xml_codec.read_required "P" xml_codec.read_text children errors
    inl o_q, errors = xml_codec.read_required "Q" xml_codec.read_text children errors
    inl o_g, errors = xml_codec.read_required "G" xml_codec.read_text children errors
    inl o_y, errors = xml_codec.read_required "Y" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk {
            p = xml_codec.present o_p
            q = xml_codec.present o_q
            g = xml_codec.present o_g
            y = xml_codec.present o_y
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_m, errors = xml_codec.read_required "M" xml_codec.read_text children errors
    inl o_e, errors = xml_codec.read_required "E" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk {
            m = xml_codec.present o_m
            e = xml_codec.present o_e
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_d, errors = xml_codec.read_required "D" xml_codec.read_text children errors
    inl o_h, errors = xml_codec.read_required "H" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng {
            d = xml_codec.present o_d
            h = xml_codec.present o_h
        }

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_comisiones_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_comisiones) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_com, errors = xml_codec.read_required "NroLinCom" xml_codec.read_text children errors
    inl o_tipo_movim, errors =
        xml_codec.read_required
            "TipoMovim"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_comisiones_tipo_movim_of_string)
            children
            errors
    inl o_glosa, errors =
        xml_codec.read_required
            "Glosa"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_comisiones_glosa_of_string)
            children
            errors
    inl v_tasa_comision, errors = xml_codec.read_optional "TasaComision" xml_codec.read_text children errors
    inl o_val_com_neto, errors = xml_codec.read_required "ValComNeto" xml_codec.read_text children errors
    inl o_val_com_exe, errors = xml_codec.read_required "ValComExe" xml_codec.read_text children errors
    inl v_val_com_iva, errors = xml_codec.read_optional "ValComIVA" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_comisiones {
            nro_lin_com = xml_codec.present o_nro_lin_com
            tipo_movim = xml_codec.present o_tipo_movim
            glosa = xml_codec.present o_glosa
            tasa_comision = v_tasa_comision
            val_com_neto = xml_codec.present o_val_com_neto
            val_com_exe = xml_codec.present o_val_com_exe
            val_com_iva = v_val_com_iva
        }

and let x_dte_def_type_choice_documento_liquidacion_comisiones_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_comisiones) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_comisiones_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_referencia_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_referencia) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_ref, errors = xml_codec.read_required "NroLinRef" xml_codec.read_text children errors
    inl o_tpo_doc_ref, errors =
        xml_codec.read_required
            "TpoDocRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_referencia_tpo_doc_ref_of_string)
            children
            errors
    inl v_ind_global, errors =
        xml_codec.read_optional
            "IndGlobal"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_referencia_ind_global_of_string)
            children
            errors
    inl o_folio_ref, errors =
        xml_codec.read_required "FolioRef" (xml_codec.read_value x_folio_r_type_of_string) children errors
    inl o_fch_ref, errors = xml_codec.read_required "FchRef" xml_codec.read_text children errors
    inl v_cod_ref, errors =
        xml_codec.read_optional
            "CodRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_referencia_cod_ref_of_string)
            children
            errors
    inl v_razon_ref, errors =
        xml_codec.read_optional
            "RazonRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_referencia_razon_ref_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_referencia {
            nro_lin_ref = xml_codec.present o_nro_lin_ref
            tpo_doc_ref = xml_codec.present o_tpo_doc_ref
            ind_global = v_ind_global
            folio_ref = xml_codec.present o_folio_ref
            fch_ref = xml_codec.present o_fch_ref
            cod_ref = v_cod_ref
            razon_ref = v_razon_ref
        }

and let x_dte_def_type_choice_documento_liquidacion_referencia_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_referencia) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_referencia_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_sub_tot_info_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_sub_tot_info) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_sti, errors = xml_codec.read_required "NroSTI" xml_codec.read_text children errors
    inl o_glosa_sti, errors =
        xml_codec.read_required
            "GlosaSTI"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_sub_tot_info_glosa_sti_of_string)
            children
            errors
    inl v_orden_sti, errors = xml_codec.read_optional "OrdenSTI" xml_codec.read_text children errors
    inl v_sub_tot_neto_sti, errors = xml_codec.read_optional "SubTotNetoSTI" xml_codec.read_text children errors
    inl v_sub_tot_ivasti, errors = xml_codec.read_optional "SubTotIVASTI" xml_codec.read_text children errors
    inl v_sub_tot_adic_sti, errors = xml_codec.read_optional "SubTotAdicSTI" xml_codec.read_text children errors
    inl v_sub_tot_exe_sti, errors = xml_codec.read_optional "SubTotExeSTI" xml_codec.read_text children errors
    inl v_val_subtot_sti, errors = xml_codec.read_optional "ValSubtotSTI" xml_codec.read_text children errors
    inl v_lineas_deta, errors = xml_codec.read_repeated "LineasDeta" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_sub_tot_info {
            nro_sti = xml_codec.present o_nro_sti
            glosa_sti = xml_codec.present o_glosa_sti
            orden_sti = v_orden_sti
            sub_tot_neto_sti = v_sub_tot_neto_sti
            sub_tot_ivasti = v_sub_tot_ivasti
            sub_tot_adic_sti = v_sub_tot_adic_sti
            sub_tot_exe_sti = v_sub_tot_exe_sti
            val_subtot_sti = v_val_subtot_sti
            lineas_deta = v_lineas_deta
        }

and let x_dte_def_type_choice_documento_liquidacion_sub_tot_info_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_sub_tot_info) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_detalle_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_detalle) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_det, errors = xml_codec.read_required "NroLinDet" xml_codec.read_text children errors
    inl v_cdg_item, errors =
        xml_codec.read_repeated
            "CdgItem"
            x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_of_xml
            children
            errors
    inl o_tpo_doc_liq, errors =
        xml_codec.read_required
            "TpoDocLiq"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_tpo_doc_liq_of_string)
            children
            errors
    inl v_ind_exe, errors =
        xml_codec.read_optional
            "IndExe"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_ind_exe_of_string)
            children
            errors
    inl o_nmb_item, errors =
        xml_codec.read_required
            "NmbItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_nmb_item_of_string)
            children
            errors
    inl v_dsc_item, errors =
        xml_codec.read_optional
            "DscItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_dsc_item_of_string)
            children
            errors
    inl v_qty_ref, errors = xml_codec.read_optional "QtyRef" xml_codec.read_text children errors
    inl v_unmd_ref, errors =
        xml_codec.read_optional
            "UnmdRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_unmd_ref_of_string)
            children
            errors
    inl v_prc_ref, errors = xml_codec.read_optional "PrcRef" xml_codec.read_text children errors
    inl v_qty_item, errors = xml_codec.read_optional "QtyItem" xml_codec.read_text children errors
    inl v_subcantidad, errors =
        xml_codec.read_repeated
            "Subcantidad"
            x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_of_xml
            children
            errors
    inl v_fch_elabor, errors = xml_codec.read_optional "FchElabor" xml_codec.read_text children errors
    inl v_fch_vencim, errors = xml_codec.read_optional "FchVencim" xml_codec.read_text children errors
    inl v_unmd_item, errors =
        xml_codec.read_optional
            "UnmdItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_unmd_item_of_string)
            children
            errors
    inl v_prc_item, errors = xml_codec.read_optional "PrcItem" xml_codec.read_text children errors
    inl v_cod_imp_adic, errors =
        xml_codec.read_repeated "CodImpAdic" (xml_codec.read_value x_imp_adic_dte_type_of_string) children errors
    inl o_monto_item, errors = xml_codec.read_required "MontoItem" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_detalle {
            nro_lin_det = xml_codec.present o_nro_lin_det
            cdg_item = v_cdg_item
            tpo_doc_liq = xml_codec.present o_tpo_doc_liq
            ind_exe = v_ind_exe
            nmb_item = xml_codec.present o_nmb_item
            dsc_item = v_dsc_item
            qty_ref = v_qty_ref
            unmd_ref = v_unmd_ref
            prc_ref = v_prc_ref
            qty_item = v_qty_item
            subcantidad = v_subcantidad
            fch_elabor = v_fch_elabor
            fch_vencim = v_fch_vencim
            unmd_item = v_unmd_item
            prc_item = v_prc_item
            cod_imp_adic = v_cod_imp_adic
            monto_item = xml_codec.present o_monto_item
        }

and let x_dte_def_type_choice_documento_liquidacion_detalle_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_detalle) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_sub_qty, errors = xml_codec.read_required "SubQty" xml_codec.read_text children errors
    inl o_sub_cod, errors =
        xml_codec.read_required
            "SubCod"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_sub_cod_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad {
            sub_qty = xml_codec.present o_sub_qty
            sub_cod = xml_codec.present o_sub_cod
        }

and let x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tpo_codigo, errors =
        xml_codec.read_required
            "TpoCodigo"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_tpo_codigo_of_string)
            children
            errors
    inl o_vlr_codigo, errors =
        xml_codec.read_required
            "VlrCodigo"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_vlr_codigo_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item {
            tpo_codigo = xml_codec.present o_tpo_codigo
            vlr_codigo = xml_codec.present o_vlr_codigo
        }

and let x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_id_doc, errors =
        xml_codec.read_required
            "IdDoc"
            x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_of_xml
            children
            errors
    inl o_emisor, errors =
        xml_codec.read_required
            "Emisor"
            x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_of_xml
            children
            errors
    inl o_receptor, errors =
        xml_codec.read_required
            "Receptor"
            x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_of_xml
            children
            errors
    inl o_totales, errors =
        xml_codec.read_required
            "Totales"
            x_dte_def_type_choice_documento_liquidacion_encabezado_totales_of_xml
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado {
            id_doc = xml_codec.present o_id_doc
            emisor = xml_codec.present o_emisor
            receptor = xml_codec.present o_receptor
            totales = xml_codec.present o_totales
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_totales) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_mnt_neto, errors = xml_codec.read_optional "MntNeto" xml_codec.read_text children errors
    inl v_mnt_exe, errors = xml_codec.read_optional "MntExe" xml_codec.read_text children errors
    inl v_tasa_iva, errors = xml_codec.read_optional "TasaIVA" xml_codec.read_text children errors
    inl v_iva, errors = xml_codec.read_optional "IVA" xml_codec.read_text children errors
    inl v_iva_prop, errors = xml_codec.read_optional "IVAProp" xml_codec.read_text children errors
    inl v_iva_terc, errors = xml_codec.read_optional "IVATerc" xml_codec.read_text children errors
    inl v_impto_reten, errors =
        xml_codec.read_repeated
            "ImptoReten"
            x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_of_xml
            children
            errors
    inl v_comisiones, errors =
        xml_codec.read_optional
            "Comisiones"
            x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_of_xml
            children
            errors
    inl o_mnt_total, errors = xml_codec.read_required "MntTotal" xml_codec.read_text children errors
    inl v_monto_periodo, errors = xml_codec.read_optional "MontoPeriodo" xml_codec.read_text children errors
    inl v_saldo_anterior, errors = xml_codec.read_optional "SaldoAnterior" xml_codec.read_text children errors
    inl v_vlr_pagar, errors = xml_codec.read_optional "VlrPagar" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales {
            mnt_neto = v_mnt_neto
            mnt_exe = v_mnt_exe
            tasa_iva = v_tasa_iva
            iva = v_iva
            iva_prop = v_iva_prop
            iva_terc = v_iva_terc
            impto_reten = v_impto_reten
            comisiones = v_comisiones
            mnt_total = xml_codec.present o_mnt_total
            monto_periodo = v_monto_periodo
            saldo_anterior = v_saldo_anterior
            vlr_pagar = v_vlr_pagar
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_totales) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_val_com_neto, errors = xml_codec.read_optional "ValComNeto" xml_codec.read_text children errors
    inl v_val_com_exe, errors = xml_codec.read_optional "ValComExe" xml_codec.read_text children errors
    inl v_val_com_iva, errors = xml_codec.read_optional "ValComIVA" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones {
            val_com_neto = v_val_com_neto
            val_com_exe = v_val_com_exe
            val_com_iva = v_val_com_iva
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_imp, errors =
        xml_codec.read_required "TipoImp" (xml_codec.read_value x_imp_adic_dte_type_of_string) children errors
    inl v_tasa_imp, errors = xml_codec.read_optional "TasaImp" xml_codec.read_text children errors
    inl o_monto_imp, errors = xml_codec.read_required "MontoImp" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten {
            tipo_imp = xml_codec.present o_tipo_imp
            tasa_imp = v_tasa_imp
            monto_imp = xml_codec.present o_monto_imp
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_recep, errors =
        xml_codec.read_required "RUTRecep" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_cdg_int_recep, errors =
        xml_codec.read_optional
            "CdgIntRecep"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_cdg_int_recep_of_string
            )
            children
            errors
    inl o_rzn_soc_recep, errors =
        xml_codec.read_required "RznSocRecep" (xml_codec.read_value x_rzn_soc_larga_type_of_string) children errors
    inl o_giro_recep, errors =
        xml_codec.read_required
            "GiroRecep"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_giro_recep_of_string)
            children
            errors
    inl v_contacto, errors =
        xml_codec.read_optional
            "Contacto"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_contacto_of_string)
            children
            errors
    inl v_correo_recep, errors =
        xml_codec.read_optional "CorreoRecep" (xml_codec.read_value x_mail_type_of_string) children errors
    inl o_dir_recep, errors =
        xml_codec.read_required
            "DirRecep"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_recep_of_string)
            children
            errors
    inl o_cmna_recep, errors =
        xml_codec.read_required "CmnaRecep" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_recep, errors =
        xml_codec.read_optional "CiudadRecep" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_dir_postal, errors =
        xml_codec.read_optional
            "DirPostal"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_dir_postal_of_string)
            children
            errors
    inl v_cmna_postal, errors =
        xml_codec.read_optional "CmnaPostal" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_postal, errors =
        xml_codec.read_optional "CiudadPostal" (xml_codec.read_value x_ciudad_type_of_string) children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado_receptor {
            rut_recep = xml_codec.present o_rut_recep
            cdg_int_recep = v_cdg_int_recep
            rzn_soc_recep = xml_codec.present o_rzn_soc_recep
            giro_recep = xml_codec.present o_giro_recep
            contacto = v_contacto
            correo_recep = v_correo_recep
            dir_recep = xml_codec.present o_dir_recep
            cmna_recep = xml_codec.present o_cmna_recep
            ciudad_recep = v_ciudad_recep
            dir_postal = v_dir_postal
            cmna_postal = v_cmna_postal
            ciudad_postal = v_ciudad_postal
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_emisor, errors =
        xml_codec.read_required "RUTEmisor" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rzn_soc, errors =
        xml_codec.read_required
            "RznSoc"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_rzn_soc_of_string)
            children
            errors
    inl o_giro_emis, errors =
        xml_codec.read_required
            "GiroEmis"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_giro_emis_of_string)
            children
            errors
    inl v_telefono, errors =
        xml_codec.read_repeated
            "Telefono"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_telefono_of_string)
            children
            errors
    inl v_correo_emisor, errors =
        xml_codec.read_optional "CorreoEmisor" (xml_codec.read_value x_mail_type_of_string) children errors
    inl v_acteco, errors = xml_codec.read_repeated "Acteco" xml_codec.read_text children errors
    inl v_sucursal, errors =
        xml_codec.read_optional
            "Sucursal"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_sucursal_of_string)
            children
            errors
    inl v_cdg_sii_sucur, errors = xml_codec.read_optional "CdgSIISucur" xml_codec.read_text children errors
    inl o_dir_origen, errors =
        xml_codec.read_required
            "DirOrigen"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_dir_origen_of_string)
            children
            errors
    inl v_cmna_origen, errors =
        xml_codec.read_optional "CmnaOrigen" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_origen, errors =
        xml_codec.read_optional "CiudadOrigen" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_cdg_vendedor, errors =
        xml_codec.read_optional
            "CdgVendedor"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_cdg_vendedor_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado_emisor {
            rut_emisor = xml_codec.present o_rut_emisor
            rzn_soc = xml_codec.present o_rzn_soc
            giro_emis = xml_codec.present o_giro_emis
            telefono = v_telefono
            correo_emisor = v_correo_emisor
            acteco = v_acteco
            sucursal = v_sucursal
            cdg_sii_sucur = v_cdg_sii_sucur
            dir_origen = xml_codec.present o_dir_origen
            cmna_origen = v_cmna_origen
            ciudad_origen = v_ciudad_origen
            cdg_vendedor = v_cdg_vendedor
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_dte, errors =
        xml_codec.read_required "TipoDTE" (xml_codec.read_value x_liq_type_of_string) children errors
    inl o_folio, errors = xml_codec.read_required "Folio" xml_codec.read_text children errors
    inl o_fch_emis, errors = xml_codec.read_required "FchEmis" xml_codec.read_text children errors
    inl v_ind_servicio, errors =
        xml_codec.read_optional
            "IndServicio"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_ind_servicio_of_string)
            children
            errors
    inl v_mnt_bruto, errors =
        xml_codec.read_optional
            "MntBruto"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_bruto_of_string)
            children
            errors
    inl v_fma_pago, errors =
        xml_codec.read_optional
            "FmaPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_fma_pago_of_string)
            children
            errors
    inl v_fch_cancel, errors = xml_codec.read_optional "FchCancel" xml_codec.read_text children errors
    inl v_mnt_cancel, errors = xml_codec.read_optional "MntCancel" xml_codec.read_text children errors
    inl v_saldo_insol, errors = xml_codec.read_optional "SaldoInsol" xml_codec.read_text children errors
    inl v_mnt_pagos, errors =
        xml_codec.read_repeated
            "MntPagos"
            x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_of_xml
            children
            errors
    inl v_periodo_desde, errors = xml_codec.read_optional "PeriodoDesde" xml_codec.read_text children errors
    inl v_periodo_hasta, errors = xml_codec.read_optional "PeriodoHasta" xml_codec.read_text children errors
    inl v_medio_pago, errors =
        xml_codec.read_optional "MedioPago" (xml_codec.read_value x_medio_pago_type_of_string) children errors
    inl v_tpo_cta_pago, errors =
        xml_codec.read_optional
            "TpoCtaPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_tpo_cta_pago_of_string)
            children
            errors
    inl v_num_cta_pago, errors =
        xml_codec.read_optional
            "NumCtaPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_num_cta_pago_of_string)
            children
            errors
    inl v_bco_pago, errors =
        xml_codec.read_optional
            "BcoPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_bco_pago_of_string)
            children
            errors
    inl v_term_pago_cdg, errors =
        xml_codec.read_optional
            "TermPagoCdg"
            (xml_codec.read_value x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_cdg_of_string)
            children
            errors
    inl v_term_pago_glosa, errors =
        xml_codec.read_optional
            "TermPagoGlosa"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_term_pago_glosa_of_string
            )
            children
            errors
    inl v_term_pago_dias, errors = xml_codec.read_optional "TermPagoDias" xml_codec.read_text children errors
    inl v_fch_venc, errors = xml_codec.read_optional "FchVenc" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc {
            tipo_dte = xml_codec.present o_tipo_dte
            folio = xml_codec.present o_folio
            fch_emis = xml_codec.present o_fch_emis
            ind_servicio = v_ind_servicio
            mnt_bruto = v_mnt_bruto
            fma_pago = v_fma_pago
            fch_cancel = v_fch_cancel
            mnt_cancel = v_mnt_cancel
            saldo_insol = v_saldo_insol
            mnt_pagos = v_mnt_pagos
            periodo_desde = v_periodo_desde
            periodo_hasta = v_periodo_hasta
            medio_pago = v_medio_pago
            tpo_cta_pago = v_tpo_cta_pago
            num_cta_pago = v_num_cta_pago
            bco_pago = v_bco_pago
            term_pago_cdg = v_term_pago_cdg
            term_pago_glosa = v_term_pago_glosa
            term_pago_dias = v_term_pago_dias
            fch_venc = v_fch_venc
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_fch_pago, errors = xml_codec.read_required "FchPago" xml_codec.read_text children errors
    inl o_mnt_pago, errors = xml_codec.read_required "MntPago" xml_codec.read_text children errors
    inl v_glosa_pagos, errors =
        xml_codec.read_optional
            "GlosaPagos"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_glosa_pagos_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos {
            fch_pago = xml_codec.present o_fch_pago
            mnt_pago = xml_codec.present o_mnt_pago
            glosa_pagos = v_glosa_pagos
        }

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_encabezado, errors =
        xml_codec.read_required "Encabezado" x_dte_def_type_choice_documento_documento_encabezado_of_xml children errors
    inl v_detalle, errors =
        xml_codec.read_repeated "Detalle" x_dte_def_type_choice_documento_documento_detalle_of_xml children errors
    inl v_sub_tot_info, errors =
        xml_codec.read_repeated
            "SubTotInfo"
            x_dte_def_type_choice_documento_documento_sub_tot_info_of_xml
            children
            errors
    inl v_dsc_rcg_global, errors =
        xml_codec.read_repeated
            "DscRcgGlobal"
            x_dte_def_type_choice_documento_documento_dsc_rcg_global_of_xml
            children
            errors
    inl v_referencia, errors =
        xml_codec.read_repeated "Referencia" x_dte_def_type_choice_documento_documento_referencia_of_xml children errors
    inl v_comisiones, errors =
        xml_codec.read_repeated "Comisiones" x_dte_def_type_choice_documento_documento_comisiones_of_xml children errors
    inl o_ted, errors =
        xml_codec.read_required "TED" x_dte_def_type_choice_documento_documento_ted_of_xml children errors
    inl o_tmst_firma, errors = xml_codec.read_required "TmstFirma" xml_codec.read_text children errors
    inl v_id, errors = xml_codec.attribute_required "ID" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento {
            encabezado = xml_codec.present o_encabezado
            detalle = v_detalle
            sub_tot_info = v_sub_tot_info
            dsc_rcg_global = v_dsc_rcg_global
            referencia = v_referencia
            comisiones = v_comisiones
            ted = xml_codec.present o_ted
            tmst_firma = xml_codec.present o_tmst_firma
            id = v_id
        }

and let x_dte_def_type_choice_documento_documento_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_dd, errors =
        xml_codec.read_required "DD" x_dte_def_type_choice_documento_documento_ted_dd_of_xml children errors
    inl o_frmt, errors =
        xml_codec.read_required "FRMT" x_dte_def_type_choice_documento_documento_ted_frmt_of_xml children errors
    inl v_version, errors = xml_codec.attribute_required "version" attributes errors
    inl errors = xml_codec.fixed_checked "version" "1.0" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted {
            dd = xml_codec.present o_dd
            frmt = xml_codec.present o_frmt
            version = v_version
        }

and let x_dte_def_type_choice_documento_documento_ted_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_frmt_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_frmt) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_base, errors = xml.text_in children, errors
    inl o_algoritmo, errors =
        xml_codec.attribute_parsed x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_of_string
            "algoritmo"
            attributes
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_frmt { base = v_base; algoritmo = xml_codec.present o_algoritmo }

and let x_dte_def_type_choice_documento_documento_ted_frmt_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_frmt) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_frmt_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_dd_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_re, errors = xml_codec.read_required "RE" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_td, errors = xml_codec.read_required "TD" (xml_codec.read_value x_dte_type_of_string) children errors
    inl o_f, errors = xml_codec.read_required "F" xml_codec.read_text children errors
    inl o_fe, errors = xml_codec.read_required "FE" xml_codec.read_text children errors
    inl o_rr, errors = xml_codec.read_required "RR" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rsr, errors =
        xml_codec.read_required
            "RSR"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_ted_dd_rsr_of_string)
            children
            errors
    inl o_mnt, errors = xml_codec.read_required "MNT" xml_codec.read_text children errors
    inl o_it1, errors =
        xml_codec.read_required
            "IT1"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_ted_dd_it1_of_string)
            children
            errors
    inl o_caf, errors =
        xml_codec.read_required "CAF" x_dte_def_type_choice_documento_documento_ted_dd_caf_of_xml children errors
    inl o_tsted, errors = xml_codec.read_required "TSTED" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_dd {
            re = xml_codec.present o_re
            td = xml_codec.present o_td
            f = xml_codec.present o_f
            fe = xml_codec.present o_fe
            rr = xml_codec.present o_rr
            rsr = xml_codec.present o_rsr
            mnt = xml_codec.present o_mnt
            it1 = xml_codec.present o_it1
            caf = xml_codec.present o_caf
            tsted = xml_codec.present o_tsted
        }

and let x_dte_def_type_choice_documento_documento_ted_dd_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_dd_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_da, errors =
        xml_codec.read_required "DA" x_dte_def_type_choice_documento_documento_ted_dd_caf_da_of_xml children errors
    inl o_frma, errors =
        xml_codec.read_required "FRMA" x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_of_xml children errors
    inl v_version, errors = xml_codec.attribute_required "version" attributes errors
    inl errors = xml_codec.fixed_checked "version" "1.0" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_dd_caf {
            da = xml_codec.present o_da
            frma = xml_codec.present o_frma
            version = v_version
        }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_frma) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_base, errors = xml.text_in children, errors
    inl v_algoritmo, errors = xml_codec.attribute_required "algoritmo" attributes errors
    inl errors = xml_codec.fixed_checked "algoritmo" "SHA1withRSA" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_dd_caf_frma { base = v_base; algoritmo = v_algoritmo }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_frma) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_re, errors = xml_codec.read_required "RE" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rs, errors =
        xml_codec.read_required
            "RS"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_of_string)
            children
            errors
    inl o_td, errors = xml_codec.read_required "TD" (xml_codec.read_value x_dte_type_of_string) children errors
    inl o_rng, errors =
        xml_codec.read_required "RNG" x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_of_xml children errors
    inl o_fa, errors = xml_codec.read_required "FA" xml_codec.read_text children errors
    inl o_choice_rsapk, errors =
        xml_codec.taken
            (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_of_children attributes children)
            errors
    inl o_idk, errors = xml_codec.read_required "IDK" (xml_codec.read_value xsd_integer_of_string) children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da {
            re = xml_codec.present o_re
            rs = xml_codec.present o_rs
            td = xml_codec.present o_td
            rng = xml_codec.present o_rng
            fa = xml_codec.present o_fa
            choice_rsapk = xml_codec.present o_choice_rsapk
            idk = xml_codec.present o_idk
        }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk) (list xml_codec.decode_error)
    =
    if false
    then xml_codec.no_alternative "x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk"
    elif xml.has_child "RSAPK" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK v
            (
                xml_codec.read_child
                    "RSAPK"
                    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
                    children
            )
    elif xml.has_child "DSAPK" children then
        xml_codec.mapped
            fun v => DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK v
            (
                xml_codec.read_child
                    "DSAPK"
                    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
                    children
            )
    else xml_codec.no_alternative "x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk"

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_p, errors = xml_codec.read_required "P" xml_codec.read_text children errors
    inl o_q, errors = xml_codec.read_required "Q" xml_codec.read_text children errors
    inl o_g, errors = xml_codec.read_required "G" xml_codec.read_text children errors
    inl o_y, errors = xml_codec.read_required "Y" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk {
            p = xml_codec.present o_p
            q = xml_codec.present o_q
            g = xml_codec.present o_g
            y = xml_codec.present o_y
        }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_m, errors = xml_codec.read_required "M" xml_codec.read_text children errors
    inl o_e, errors = xml_codec.read_required "E" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk {
            m = xml_codec.present o_m
            e = xml_codec.present o_e
        }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_d, errors = xml_codec.read_required "D" xml_codec.read_text children errors
    inl o_h, errors = xml_codec.read_required "H" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng {
            d = xml_codec.present o_d
            h = xml_codec.present o_h
        }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_comisiones_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_comisiones) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_com, errors = xml_codec.read_required "NroLinCom" xml_codec.read_text children errors
    inl o_tipo_movim, errors =
        xml_codec.read_required
            "TipoMovim"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_of_string)
            children
            errors
    inl o_glosa, errors =
        xml_codec.read_required
            "Glosa"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_comisiones_glosa_of_string)
            children
            errors
    inl v_tasa_comision, errors = xml_codec.read_optional "TasaComision" xml_codec.read_text children errors
    inl o_val_com_neto, errors = xml_codec.read_required "ValComNeto" xml_codec.read_text children errors
    inl o_val_com_exe, errors = xml_codec.read_required "ValComExe" xml_codec.read_text children errors
    inl v_val_com_iva, errors = xml_codec.read_optional "ValComIVA" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_comisiones {
            nro_lin_com = xml_codec.present o_nro_lin_com
            tipo_movim = xml_codec.present o_tipo_movim
            glosa = xml_codec.present o_glosa
            tasa_comision = v_tasa_comision
            val_com_neto = xml_codec.present o_val_com_neto
            val_com_exe = xml_codec.present o_val_com_exe
            val_com_iva = v_val_com_iva
        }

and let x_dte_def_type_choice_documento_documento_comisiones_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_comisiones) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_comisiones_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_referencia_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_referencia) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_ref, errors = xml_codec.read_required "NroLinRef" xml_codec.read_text children errors
    inl o_tpo_doc_ref, errors =
        xml_codec.read_required
            "TpoDocRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_of_string)
            children
            errors
    inl v_ind_global, errors =
        xml_codec.read_optional
            "IndGlobal"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_referencia_ind_global_of_string)
            children
            errors
    inl o_folio_ref, errors =
        xml_codec.read_required "FolioRef" (xml_codec.read_value x_folio_r_type_of_string) children errors
    inl v_rut_otr, errors = xml_codec.read_optional "RUTOtr" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_fch_ref, errors = xml_codec.read_required "FchRef" xml_codec.read_text children errors
    inl v_cod_ref, errors =
        xml_codec.read_optional
            "CodRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_referencia_cod_ref_of_string)
            children
            errors
    inl v_razon_ref, errors =
        xml_codec.read_optional
            "RazonRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_referencia_razon_ref_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_referencia {
            nro_lin_ref = xml_codec.present o_nro_lin_ref
            tpo_doc_ref = xml_codec.present o_tpo_doc_ref
            ind_global = v_ind_global
            folio_ref = xml_codec.present o_folio_ref
            rut_otr = v_rut_otr
            fch_ref = xml_codec.present o_fch_ref
            cod_ref = v_cod_ref
            razon_ref = v_razon_ref
        }

and let x_dte_def_type_choice_documento_documento_referencia_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_referencia) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_referencia_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_dsc_rcg_global_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_dsc_rcg_global) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_dr, errors =
        xml_codec.read_required "NroLinDR" (xml_codec.read_value xsd_positive_integer_of_string) children errors
    inl o_tpo_mov, errors =
        xml_codec.read_required
            "TpoMov"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_of_string)
            children
            errors
    inl v_glosa_dr, errors =
        xml_codec.read_optional
            "GlosaDR"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_dsc_rcg_global_glosa_dr_of_string)
            children
            errors
    inl o_tpo_valor, errors =
        xml_codec.read_required "TpoValor" (xml_codec.read_value x_dinero_porcentaje_type_of_string) children errors
    inl o_valor_dr, errors = xml_codec.read_required "ValorDR" xml_codec.read_text children errors
    inl v_valor_dr_otr_mnda, errors = xml_codec.read_optional "ValorDROtrMnda" xml_codec.read_text children errors
    inl v_ind_exe_dr, errors =
        xml_codec.read_optional
            "IndExeDR"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_dsc_rcg_global_ind_exe_dr_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_dsc_rcg_global {
            nro_lin_dr = xml_codec.present o_nro_lin_dr
            tpo_mov = xml_codec.present o_tpo_mov
            glosa_dr = v_glosa_dr
            tpo_valor = xml_codec.present o_tpo_valor
            valor_dr = xml_codec.present o_valor_dr
            valor_dr_otr_mnda = v_valor_dr_otr_mnda
            ind_exe_dr = v_ind_exe_dr
        }

and let x_dte_def_type_choice_documento_documento_dsc_rcg_global_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_dsc_rcg_global) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_sub_tot_info_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_sub_tot_info) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_sti, errors = xml_codec.read_required "NroSTI" xml_codec.read_text children errors
    inl o_glosa_sti, errors =
        xml_codec.read_required
            "GlosaSTI"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_of_string)
            children
            errors
    inl v_orden_sti, errors = xml_codec.read_optional "OrdenSTI" xml_codec.read_text children errors
    inl v_sub_tot_neto_sti, errors = xml_codec.read_optional "SubTotNetoSTI" xml_codec.read_text children errors
    inl v_sub_tot_ivasti, errors = xml_codec.read_optional "SubTotIVASTI" xml_codec.read_text children errors
    inl v_sub_tot_adic_sti, errors = xml_codec.read_optional "SubTotAdicSTI" xml_codec.read_text children errors
    inl v_sub_tot_exe_sti, errors = xml_codec.read_optional "SubTotExeSTI" xml_codec.read_text children errors
    inl v_val_subtot_sti, errors = xml_codec.read_optional "ValSubtotSTI" xml_codec.read_text children errors
    inl v_lineas_deta, errors = xml_codec.read_repeated "LineasDeta" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_sub_tot_info {
            nro_sti = xml_codec.present o_nro_sti
            glosa_sti = xml_codec.present o_glosa_sti
            orden_sti = v_orden_sti
            sub_tot_neto_sti = v_sub_tot_neto_sti
            sub_tot_ivasti = v_sub_tot_ivasti
            sub_tot_adic_sti = v_sub_tot_adic_sti
            sub_tot_exe_sti = v_sub_tot_exe_sti
            val_subtot_sti = v_val_subtot_sti
            lineas_deta = v_lineas_deta
        }

and let x_dte_def_type_choice_documento_documento_sub_tot_info_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_sub_tot_info) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_sub_tot_info_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_detalle_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_nro_lin_det, errors = xml_codec.read_required "NroLinDet" xml_codec.read_text children errors
    inl v_cdg_item, errors =
        xml_codec.read_repeated
            "CdgItem"
            x_dte_def_type_choice_documento_documento_detalle_cdg_item_of_xml
            children
            errors
    inl v_ind_exe, errors =
        xml_codec.read_optional
            "IndExe"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_ind_exe_of_string)
            children
            errors
    inl v_retenedor, errors =
        xml_codec.read_optional
            "Retenedor"
            x_dte_def_type_choice_documento_documento_detalle_retenedor_of_xml
            children
            errors
    inl o_nmb_item, errors =
        xml_codec.read_required
            "NmbItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_nmb_item_of_string)
            children
            errors
    inl v_dsc_item, errors =
        xml_codec.read_optional
            "DscItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_dsc_item_of_string)
            children
            errors
    inl v_qty_ref, errors = xml_codec.read_optional "QtyRef" xml_codec.read_text children errors
    inl v_unmd_ref, errors =
        xml_codec.read_optional
            "UnmdRef"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_unmd_ref_of_string)
            children
            errors
    inl v_prc_ref, errors = xml_codec.read_optional "PrcRef" xml_codec.read_text children errors
    inl v_qty_item, errors = xml_codec.read_optional "QtyItem" xml_codec.read_text children errors
    inl v_subcantidad, errors =
        xml_codec.read_repeated
            "Subcantidad"
            x_dte_def_type_choice_documento_documento_detalle_subcantidad_of_xml
            children
            errors
    inl v_fch_elabor, errors = xml_codec.read_optional "FchElabor" xml_codec.read_text children errors
    inl v_fch_vencim, errors = xml_codec.read_optional "FchVencim" xml_codec.read_text children errors
    inl v_unmd_item, errors =
        xml_codec.read_optional
            "UnmdItem"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_unmd_item_of_string)
            children
            errors
    inl v_prc_item, errors = xml_codec.read_optional "PrcItem" xml_codec.read_text children errors
    inl v_otr_mnda, errors =
        xml_codec.read_optional
            "OtrMnda"
            x_dte_def_type_choice_documento_documento_detalle_otr_mnda_of_xml
            children
            errors
    inl v_descuento_pct, errors = xml_codec.read_optional "DescuentoPct" xml_codec.read_text children errors
    inl v_descuento_monto, errors = xml_codec.read_optional "DescuentoMonto" xml_codec.read_text children errors
    inl v_sub_dscto, errors =
        xml_codec.read_repeated
            "SubDscto"
            x_dte_def_type_choice_documento_documento_detalle_sub_dscto_of_xml
            children
            errors
    inl v_recargo_pct, errors = xml_codec.read_optional "RecargoPct" xml_codec.read_text children errors
    inl v_recargo_monto, errors = xml_codec.read_optional "RecargoMonto" xml_codec.read_text children errors
    inl v_sub_recargo, errors =
        xml_codec.read_repeated
            "SubRecargo"
            x_dte_def_type_choice_documento_documento_detalle_sub_recargo_of_xml
            children
            errors
    inl v_cod_imp_adic, errors =
        xml_codec.read_repeated "CodImpAdic" (xml_codec.read_value x_imp_adic_dte_type_of_string) children errors
    inl o_monto_item, errors = xml_codec.read_required "MontoItem" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_detalle {
            nro_lin_det = xml_codec.present o_nro_lin_det
            cdg_item = v_cdg_item
            ind_exe = v_ind_exe
            retenedor = v_retenedor
            nmb_item = xml_codec.present o_nmb_item
            dsc_item = v_dsc_item
            qty_ref = v_qty_ref
            unmd_ref = v_unmd_ref
            prc_ref = v_prc_ref
            qty_item = v_qty_item
            subcantidad = v_subcantidad
            fch_elabor = v_fch_elabor
            fch_vencim = v_fch_vencim
            unmd_item = v_unmd_item
            prc_item = v_prc_item
            otr_mnda = v_otr_mnda
            descuento_pct = v_descuento_pct
            descuento_monto = v_descuento_monto
            sub_dscto = v_sub_dscto
            recargo_pct = v_recargo_pct
            recargo_monto = v_recargo_monto
            sub_recargo = v_sub_recargo
            cod_imp_adic = v_cod_imp_adic
            monto_item = xml_codec.present o_monto_item
        }

and let x_dte_def_type_choice_documento_documento_detalle_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_detalle_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_detalle_sub_recargo_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_sub_recargo) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_recargo, errors =
        xml_codec.read_required "TipoRecargo" (xml_codec.read_value x_dinero_porcentaje_type_of_string) children errors
    inl o_valor_recargo, errors = xml_codec.read_required "ValorRecargo" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_detalle_sub_recargo {
            tipo_recargo = xml_codec.present o_tipo_recargo
            valor_recargo = xml_codec.present o_valor_recargo
        }

and let x_dte_def_type_choice_documento_documento_detalle_sub_recargo_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_sub_recargo) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_detalle_sub_recargo_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_detalle_sub_dscto_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_sub_dscto) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_dscto, errors =
        xml_codec.read_required "TipoDscto" (xml_codec.read_value x_dinero_porcentaje_type_of_string) children errors
    inl o_valor_dscto, errors = xml_codec.read_required "ValorDscto" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_detalle_sub_dscto {
            tipo_dscto = xml_codec.present o_tipo_dscto
            valor_dscto = xml_codec.present o_valor_dscto
        }

and let x_dte_def_type_choice_documento_documento_detalle_sub_dscto_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_sub_dscto) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_detalle_sub_dscto_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_detalle_otr_mnda_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_otr_mnda) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_prc_otr_mon, errors = xml_codec.read_required "PrcOtrMon" xml_codec.read_text children errors
    inl o_moneda, errors =
        xml_codec.read_required
            "Moneda"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_of_string)
            children
            errors
    inl v_fct_conv, errors = xml_codec.read_optional "FctConv" xml_codec.read_text children errors
    inl v_dcto_otr_mnda, errors = xml_codec.read_optional "DctoOtrMnda" xml_codec.read_text children errors
    inl v_recargo_otr_mnda, errors = xml_codec.read_optional "RecargoOtrMnda" xml_codec.read_text children errors
    inl v_monto_item_otr_mnda, errors = xml_codec.read_optional "MontoItemOtrMnda" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_detalle_otr_mnda {
            prc_otr_mon = xml_codec.present o_prc_otr_mon
            moneda = xml_codec.present o_moneda
            fct_conv = v_fct_conv
            dcto_otr_mnda = v_dcto_otr_mnda
            recargo_otr_mnda = v_recargo_otr_mnda
            monto_item_otr_mnda = v_monto_item_otr_mnda
        }

and let x_dte_def_type_choice_documento_documento_detalle_otr_mnda_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_otr_mnda) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_detalle_otr_mnda_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_detalle_subcantidad_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_subcantidad) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_sub_qty, errors = xml_codec.read_required "SubQty" xml_codec.read_text children errors
    inl o_sub_cod, errors =
        xml_codec.read_required
            "SubCod"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_detalle_subcantidad {
            sub_qty = xml_codec.present o_sub_qty
            sub_cod = xml_codec.present o_sub_cod
        }

and let x_dte_def_type_choice_documento_documento_detalle_subcantidad_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_subcantidad) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_detalle_subcantidad_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_detalle_retenedor_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_retenedor) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_ind_agente, errors =
        xml_codec.read_required
            "IndAgente"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_of_string)
            children
            errors
    inl v_mnt_base_faena, errors = xml_codec.read_optional "MntBaseFaena" xml_codec.read_text children errors
    inl v_mnt_marg_comer, errors = xml_codec.read_optional "MntMargComer" xml_codec.read_text children errors
    inl v_prc_cons_final, errors = xml_codec.read_optional "PrcConsFinal" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_detalle_retenedor {
            ind_agente = xml_codec.present o_ind_agente
            mnt_base_faena = v_mnt_base_faena
            mnt_marg_comer = v_mnt_marg_comer
            prc_cons_final = v_prc_cons_final
        }

and let x_dte_def_type_choice_documento_documento_detalle_retenedor_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_retenedor) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_detalle_retenedor_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_detalle_cdg_item_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_cdg_item) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tpo_codigo, errors =
        xml_codec.read_required
            "TpoCodigo"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_of_string)
            children
            errors
    inl o_vlr_codigo, errors =
        xml_codec.read_required
            "VlrCodigo"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_detalle_cdg_item {
            tpo_codigo = xml_codec.present o_tpo_codigo
            vlr_codigo = xml_codec.present o_vlr_codigo
        }

and let x_dte_def_type_choice_documento_documento_detalle_cdg_item_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_detalle_cdg_item) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_id_doc, errors =
        xml_codec.read_required
            "IdDoc"
            x_dte_def_type_choice_documento_documento_encabezado_id_doc_of_xml
            children
            errors
    inl o_emisor, errors =
        xml_codec.read_required
            "Emisor"
            x_dte_def_type_choice_documento_documento_encabezado_emisor_of_xml
            children
            errors
    inl v_rut_mandante, errors =
        xml_codec.read_optional "RUTMandante" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_receptor, errors =
        xml_codec.read_required
            "Receptor"
            x_dte_def_type_choice_documento_documento_encabezado_receptor_of_xml
            children
            errors
    inl v_rut_solicita, errors =
        xml_codec.read_optional "RUTSolicita" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_transporte, errors =
        xml_codec.read_optional
            "Transporte"
            x_dte_def_type_choice_documento_documento_encabezado_transporte_of_xml
            children
            errors
    inl o_totales, errors =
        xml_codec.read_required
            "Totales"
            x_dte_def_type_choice_documento_documento_encabezado_totales_of_xml
            children
            errors
    inl v_otra_moneda, errors =
        xml_codec.read_optional
            "OtraMoneda"
            x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_of_xml
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado {
            id_doc = xml_codec.present o_id_doc
            emisor = xml_codec.present o_emisor
            rut_mandante = v_rut_mandante
            receptor = xml_codec.present o_receptor
            rut_solicita = v_rut_solicita
            transporte = v_transporte
            totales = xml_codec.present o_totales
            otra_moneda = v_otra_moneda
        }

and let x_dte_def_type_choice_documento_documento_encabezado_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tpo_moneda, errors =
        xml_codec.read_required "TpoMoneda" (xml_codec.read_value x_tip_mon_type_of_string) children errors
    inl v_tpo_cambio, errors = xml_codec.read_optional "TpoCambio" xml_codec.read_text children errors
    inl v_mnt_neto_otr_mnda, errors = xml_codec.read_optional "MntNetoOtrMnda" xml_codec.read_text children errors
    inl v_mnt_exe_otr_mnda, errors = xml_codec.read_optional "MntExeOtrMnda" xml_codec.read_text children errors
    inl v_mnt_fae_carne_otr_mnda, errors =
        xml_codec.read_optional "MntFaeCarneOtrMnda" xml_codec.read_text children errors
    inl v_mnt_marg_com_otr_mnda, errors =
        xml_codec.read_optional "MntMargComOtrMnda" xml_codec.read_text children errors
    inl v_iva_otr_mnda, errors = xml_codec.read_optional "IVAOtrMnda" xml_codec.read_text children errors
    inl v_imp_ret_otr_mnda, errors =
        xml_codec.read_repeated
            "ImpRetOtrMnda"
            x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_of_xml
            children
            errors
    inl v_iva_no_ret_otr_mnda, errors = xml_codec.read_optional "IVANoRetOtrMnda" xml_codec.read_text children errors
    inl o_mnt_tot_otr_mnda, errors = xml_codec.read_required "MntTotOtrMnda" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda {
            tpo_moneda = xml_codec.present o_tpo_moneda
            tpo_cambio = v_tpo_cambio
            mnt_neto_otr_mnda = v_mnt_neto_otr_mnda
            mnt_exe_otr_mnda = v_mnt_exe_otr_mnda
            mnt_fae_carne_otr_mnda = v_mnt_fae_carne_otr_mnda
            mnt_marg_com_otr_mnda = v_mnt_marg_com_otr_mnda
            iva_otr_mnda = v_iva_otr_mnda
            imp_ret_otr_mnda = v_imp_ret_otr_mnda
            iva_no_ret_otr_mnda = v_iva_no_ret_otr_mnda
            mnt_tot_otr_mnda = xml_codec.present o_mnt_tot_otr_mnda
        }

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_imp_otr_mnda, errors =
        xml_codec.read_required "TipoImpOtrMnda" (xml_codec.read_value x_imp_adic_dte_type_of_string) children errors
    inl v_tasa_imp_otr_mnda, errors = xml_codec.read_optional "TasaImpOtrMnda" xml_codec.read_text children errors
    inl o_vlr_imp_otr_mnda, errors = xml_codec.read_required "VlrImpOtrMnda" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda {
            tipo_imp_otr_mnda = xml_codec.present o_tipo_imp_otr_mnda
            tasa_imp_otr_mnda = v_tasa_imp_otr_mnda
            vlr_imp_otr_mnda = xml_codec.present o_vlr_imp_otr_mnda
        }

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_totales_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_totales) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_mnt_neto, errors = xml_codec.read_optional "MntNeto" xml_codec.read_text children errors
    inl v_mnt_exe, errors = xml_codec.read_optional "MntExe" xml_codec.read_text children errors
    inl v_mnt_base, errors = xml_codec.read_optional "MntBase" xml_codec.read_text children errors
    inl v_mnt_margen_com, errors = xml_codec.read_optional "MntMargenCom" xml_codec.read_text children errors
    inl v_tasa_iva, errors = xml_codec.read_optional "TasaIVA" xml_codec.read_text children errors
    inl v_iva, errors = xml_codec.read_optional "IVA" xml_codec.read_text children errors
    inl v_iva_prop, errors = xml_codec.read_optional "IVAProp" xml_codec.read_text children errors
    inl v_iva_terc, errors = xml_codec.read_optional "IVATerc" xml_codec.read_text children errors
    inl v_impto_reten, errors =
        xml_codec.read_repeated
            "ImptoReten"
            x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_of_xml
            children
            errors
    inl v_iva_no_ret, errors = xml_codec.read_optional "IVANoRet" xml_codec.read_text children errors
    inl v_cred_ec, errors = xml_codec.read_optional "CredEC" xml_codec.read_text children errors
    inl v_grnt_dep, errors = xml_codec.read_optional "GrntDep" xml_codec.read_text children errors
    inl v_comisiones, errors =
        xml_codec.read_optional
            "Comisiones"
            x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_of_xml
            children
            errors
    inl o_mnt_total, errors = xml_codec.read_required "MntTotal" xml_codec.read_text children errors
    inl v_monto_nf, errors = xml_codec.read_optional "MontoNF" xml_codec.read_text children errors
    inl v_monto_periodo, errors = xml_codec.read_optional "MontoPeriodo" xml_codec.read_text children errors
    inl v_saldo_anterior, errors = xml_codec.read_optional "SaldoAnterior" xml_codec.read_text children errors
    inl v_vlr_pagar, errors = xml_codec.read_optional "VlrPagar" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_totales {
            mnt_neto = v_mnt_neto
            mnt_exe = v_mnt_exe
            mnt_base = v_mnt_base
            mnt_margen_com = v_mnt_margen_com
            tasa_iva = v_tasa_iva
            iva = v_iva
            iva_prop = v_iva_prop
            iva_terc = v_iva_terc
            impto_reten = v_impto_reten
            iva_no_ret = v_iva_no_ret
            cred_ec = v_cred_ec
            grnt_dep = v_grnt_dep
            comisiones = v_comisiones
            mnt_total = xml_codec.present o_mnt_total
            monto_nf = v_monto_nf
            monto_periodo = v_monto_periodo
            saldo_anterior = v_saldo_anterior
            vlr_pagar = v_vlr_pagar
        }

and let x_dte_def_type_choice_documento_documento_encabezado_totales_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_totales) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_totales_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_val_com_neto, errors = xml_codec.read_optional "ValComNeto" xml_codec.read_text children errors
    inl v_val_com_exe, errors = xml_codec.read_optional "ValComExe" xml_codec.read_text children errors
    inl v_val_com_iva, errors = xml_codec.read_optional "ValComIVA" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones {
            val_com_neto = v_val_com_neto
            val_com_exe = v_val_com_exe
            val_com_iva = v_val_com_iva
        }

and let x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_imp, errors =
        xml_codec.read_required "TipoImp" (xml_codec.read_value x_imp_adic_dte_type_of_string) children errors
    inl v_tasa_imp, errors = xml_codec.read_optional "TasaImp" xml_codec.read_text children errors
    inl o_monto_imp, errors = xml_codec.read_required "MontoImp" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten {
            tipo_imp = xml_codec.present o_tipo_imp
            tasa_imp = v_tasa_imp
            monto_imp = xml_codec.present o_monto_imp
        }

and let x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_patente, errors =
        xml_codec.read_optional
            "Patente"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_transporte_patente_of_string)
            children
            errors
    inl v_rut_trans, errors =
        xml_codec.read_optional "RUTTrans" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_chofer, errors =
        xml_codec.read_optional
            "Chofer"
            x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_of_xml
            children
            errors
    inl v_dir_dest, errors =
        xml_codec.read_optional
            "DirDest"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_transporte_dir_dest_of_string)
            children
            errors
    inl v_cmna_dest, errors =
        xml_codec.read_optional "CmnaDest" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_dest, errors =
        xml_codec.read_optional "CiudadDest" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_aduana, errors =
        xml_codec.read_optional
            "Aduana"
            x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_of_xml
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_transporte {
            patente = v_patente
            rut_trans = v_rut_trans
            chofer = v_chofer
            dir_dest = v_dir_dest
            cmna_dest = v_cmna_dest
            ciudad_dest = v_ciudad_dest
            aduana = v_aduana
        }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_cod_mod_venta, errors = xml_codec.read_optional "CodModVenta" xml_codec.read_text children errors
    inl v_cod_clau_venta, errors = xml_codec.read_optional "CodClauVenta" xml_codec.read_text children errors
    inl v_tot_clau_venta, errors = xml_codec.read_optional "TotClauVenta" xml_codec.read_text children errors
    inl v_cod_via_transp, errors = xml_codec.read_optional "CodViaTransp" xml_codec.read_text children errors
    inl v_nombre_transp, errors =
        xml_codec.read_optional
            "NombreTransp"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nombre_transp_of_string
            )
            children
            errors
    inl v_rut_cia_transp, errors =
        xml_codec.read_optional "RUTCiaTransp" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_nom_cia_transp, errors =
        xml_codec.read_optional
            "NomCiaTransp"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_nom_cia_transp_of_string
            )
            children
            errors
    inl v_id_adic_transp, errors =
        xml_codec.read_optional
            "IdAdicTransp"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_transp_of_string
            )
            children
            errors
    inl v_booking, errors =
        xml_codec.read_optional
            "Booking"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_booking_of_string
            )
            children
            errors
    inl v_operador, errors =
        xml_codec.read_optional
            "Operador"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_operador_of_string
            )
            children
            errors
    inl v_cod_pto_embarque, errors = xml_codec.read_optional "CodPtoEmbarque" xml_codec.read_text children errors
    inl v_id_adic_pto_emb, errors =
        xml_codec.read_optional
            "IdAdicPtoEmb"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_emb_of_string
            )
            children
            errors
    inl v_cod_pto_desemb, errors = xml_codec.read_optional "CodPtoDesemb" xml_codec.read_text children errors
    inl v_id_adic_pto_desemb, errors =
        xml_codec.read_optional
            "IdAdicPtoDesemb"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_id_adic_pto_desemb_of_string
            )
            children
            errors
    inl v_tara, errors = xml_codec.read_optional "Tara" xml_codec.read_text children errors
    inl v_cod_unid_med_tara, errors = xml_codec.read_optional "CodUnidMedTara" xml_codec.read_text children errors
    inl v_peso_bruto, errors = xml_codec.read_optional "PesoBruto" xml_codec.read_text children errors
    inl v_cod_unid_peso_bruto, errors = xml_codec.read_optional "CodUnidPesoBruto" xml_codec.read_text children errors
    inl v_peso_neto, errors = xml_codec.read_optional "PesoNeto" xml_codec.read_text children errors
    inl v_cod_unid_peso_neto, errors = xml_codec.read_optional "CodUnidPesoNeto" xml_codec.read_text children errors
    inl v_tot_items, errors = xml_codec.read_optional "TotItems" xml_codec.read_text children errors
    inl v_tot_bultos, errors = xml_codec.read_optional "TotBultos" xml_codec.read_text children errors
    inl v_tipo_bultos, errors =
        xml_codec.read_repeated
            "TipoBultos"
            x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_of_xml
            children
            errors
    inl v_mnt_flete, errors = xml_codec.read_optional "MntFlete" xml_codec.read_text children errors
    inl v_mnt_seguro, errors = xml_codec.read_optional "MntSeguro" xml_codec.read_text children errors
    inl v_cod_pais_recep, errors = xml_codec.read_optional "CodPaisRecep" xml_codec.read_text children errors
    inl v_cod_pais_destin, errors = xml_codec.read_optional "CodPaisDestin" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana {
            cod_mod_venta = v_cod_mod_venta
            cod_clau_venta = v_cod_clau_venta
            tot_clau_venta = v_tot_clau_venta
            cod_via_transp = v_cod_via_transp
            nombre_transp = v_nombre_transp
            rut_cia_transp = v_rut_cia_transp
            nom_cia_transp = v_nom_cia_transp
            id_adic_transp = v_id_adic_transp
            booking = v_booking
            operador = v_operador
            cod_pto_embarque = v_cod_pto_embarque
            id_adic_pto_emb = v_id_adic_pto_emb
            cod_pto_desemb = v_cod_pto_desemb
            id_adic_pto_desemb = v_id_adic_pto_desemb
            tara = v_tara
            cod_unid_med_tara = v_cod_unid_med_tara
            peso_bruto = v_peso_bruto
            cod_unid_peso_bruto = v_cod_unid_peso_bruto
            peso_neto = v_peso_neto
            cod_unid_peso_neto = v_cod_unid_peso_neto
            tot_items = v_tot_items
            tot_bultos = v_tot_bultos
            tipo_bultos = v_tipo_bultos
            mnt_flete = v_mnt_flete
            mnt_seguro = v_mnt_seguro
            cod_pais_recep = v_cod_pais_recep
            cod_pais_destin = v_cod_pais_destin
        }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos)
        (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_cod_tpo_bultos, errors = xml_codec.read_optional "CodTpoBultos" xml_codec.read_text children errors
    inl v_cant_bultos, errors = xml_codec.read_optional "CantBultos" xml_codec.read_text children errors
    inl v_marcas, errors =
        xml_codec.read_optional
            "Marcas"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_marcas_of_string
            )
            children
            errors
    inl v_id_container, errors =
        xml_codec.read_optional
            "IdContainer"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_id_container_of_string
            )
            children
            errors
    inl v_sello, errors =
        xml_codec.read_optional
            "Sello"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_sello_of_string
            )
            children
            errors
    inl v_emisor_sello, errors =
        xml_codec.read_optional
            "EmisorSello"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_emisor_sello_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos {
            cod_tpo_bultos = v_cod_tpo_bultos
            cant_bultos = v_cant_bultos
            marcas = v_marcas
            id_container = v_id_container
            sello = v_sello
            emisor_sello = v_emisor_sello
        }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos)
        (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_chofer, errors =
        xml_codec.read_required "RUTChofer" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_nombre_chofer, errors =
        xml_codec.read_required
            "NombreChofer"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer {
            rut_chofer = xml_codec.present o_rut_chofer
            nombre_chofer = xml_codec.present o_nombre_chofer
        }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_receptor) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_recep, errors =
        xml_codec.read_required "RUTRecep" (xml_codec.read_value x_rut_type_of_string) children errors
    inl v_cdg_int_recep, errors =
        xml_codec.read_optional
            "CdgIntRecep"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_receptor_cdg_int_recep_of_string)
            children
            errors
    inl o_rzn_soc_recep, errors =
        xml_codec.read_required "RznSocRecep" (xml_codec.read_value x_rzn_soc_larga_type_of_string) children errors
    inl v_extranjero, errors =
        xml_codec.read_optional
            "Extranjero"
            x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_of_xml
            children
            errors
    inl v_giro_recep, errors =
        xml_codec.read_optional
            "GiroRecep"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_receptor_giro_recep_of_string)
            children
            errors
    inl v_contacto, errors =
        xml_codec.read_optional
            "Contacto"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_receptor_contacto_of_string)
            children
            errors
    inl v_correo_recep, errors =
        xml_codec.read_optional "CorreoRecep" (xml_codec.read_value x_mail_type_of_string) children errors
    inl v_dir_recep, errors =
        xml_codec.read_optional
            "DirRecep"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_recep_of_string)
            children
            errors
    inl v_cmna_recep, errors =
        xml_codec.read_optional "CmnaRecep" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_recep, errors =
        xml_codec.read_optional "CiudadRecep" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_dir_postal, errors =
        xml_codec.read_optional
            "DirPostal"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_receptor_dir_postal_of_string)
            children
            errors
    inl v_cmna_postal, errors =
        xml_codec.read_optional "CmnaPostal" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_postal, errors =
        xml_codec.read_optional "CiudadPostal" (xml_codec.read_value x_ciudad_type_of_string) children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_receptor {
            rut_recep = xml_codec.present o_rut_recep
            cdg_int_recep = v_cdg_int_recep
            rzn_soc_recep = xml_codec.present o_rzn_soc_recep
            extranjero = v_extranjero
            giro_recep = v_giro_recep
            contacto = v_contacto
            correo_recep = v_correo_recep
            dir_recep = v_dir_recep
            cmna_recep = v_cmna_recep
            ciudad_recep = v_ciudad_recep
            dir_postal = v_dir_postal
            cmna_postal = v_cmna_postal
            ciudad_postal = v_ciudad_postal
        }

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_receptor) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_num_id, errors =
        xml_codec.read_optional
            "NumId"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_num_id_of_string
            )
            children
            errors
    inl v_nacionalidad, errors =
        xml_codec.read_optional
            "Nacionalidad"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_nacionalidad_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero {
            num_id = v_num_id
            nacionalidad = v_nacionalidad
        }

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_emisor) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_rut_emisor, errors =
        xml_codec.read_required "RUTEmisor" (xml_codec.read_value x_rut_type_of_string) children errors
    inl o_rzn_soc, errors = xml_codec.read_required "RznSoc" xml_codec.read_text children errors
    inl o_giro_emis, errors =
        xml_codec.read_required
            "GiroEmis"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_of_string)
            children
            errors
    inl v_telefono, errors =
        xml_codec.read_repeated
            "Telefono"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_emisor_telefono_of_string)
            children
            errors
    inl v_correo_emisor, errors =
        xml_codec.read_optional "CorreoEmisor" (xml_codec.read_value x_mail_type_of_string) children errors
    inl v_acteco, errors = xml_codec.read_repeated "Acteco" xml_codec.read_text children errors
    inl v_guia_export, errors =
        xml_codec.read_optional
            "GuiaExport"
            x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_of_xml
            children
            errors
    inl v_sucursal, errors =
        xml_codec.read_optional
            "Sucursal"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_emisor_sucursal_of_string)
            children
            errors
    inl v_cdg_sii_sucur, errors = xml_codec.read_optional "CdgSIISucur" xml_codec.read_text children errors
    inl v_dir_origen, errors =
        xml_codec.read_optional
            "DirOrigen"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_emisor_dir_origen_of_string)
            children
            errors
    inl v_cmna_origen, errors =
        xml_codec.read_optional "CmnaOrigen" (xml_codec.read_value x_comuna_type_of_string) children errors
    inl v_ciudad_origen, errors =
        xml_codec.read_optional "CiudadOrigen" (xml_codec.read_value x_ciudad_type_of_string) children errors
    inl v_cdg_vendedor, errors =
        xml_codec.read_optional
            "CdgVendedor"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_emisor_cdg_vendedor_of_string)
            children
            errors
    inl v_id_adic_emisor, errors =
        xml_codec.read_optional
            "IdAdicEmisor"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_emisor_id_adic_emisor_of_string)
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_emisor {
            rut_emisor = xml_codec.present o_rut_emisor
            rzn_soc = xml_codec.present o_rzn_soc
            giro_emis = xml_codec.present o_giro_emis
            telefono = v_telefono
            correo_emisor = v_correo_emisor
            acteco = v_acteco
            guia_export = v_guia_export
            sucursal = v_sucursal
            cdg_sii_sucur = v_cdg_sii_sucur
            dir_origen = v_dir_origen
            cmna_origen = v_cmna_origen
            ciudad_origen = v_ciudad_origen
            cdg_vendedor = v_cdg_vendedor
            id_adic_emisor = v_id_adic_emisor
        }

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_emisor) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_cdg_traslado, errors =
        xml_codec.read_optional
            "CdgTraslado"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_cdg_traslado_of_string
            )
            children
            errors
    inl v_folio_aut, errors = xml_codec.read_optional "FolioAut" xml_codec.read_text children errors
    inl v_fch_aut, errors = xml_codec.read_optional "FchAut" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export {
            cdg_traslado = v_cdg_traslado
            folio_aut = v_folio_aut
            fch_aut = v_fch_aut
        }

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_id_doc) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_tipo_dte, errors =
        xml_codec.read_required "TipoDTE" (xml_codec.read_value x_dte_type_of_string) children errors
    inl o_folio, errors = xml_codec.read_required "Folio" xml_codec.read_text children errors
    inl o_fch_emis, errors = xml_codec.read_required "FchEmis" xml_codec.read_text children errors
    inl v_ind_no_rebaja, errors =
        xml_codec.read_optional
            "IndNoRebaja"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_no_rebaja_of_string)
            children
            errors
    inl v_tipo_despacho, errors =
        xml_codec.read_optional
            "TipoDespacho"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_tipo_despacho_of_string)
            children
            errors
    inl v_ind_traslado, errors =
        xml_codec.read_optional
            "IndTraslado"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_traslado_of_string)
            children
            errors
    inl v_tpo_impresion, errors =
        xml_codec.read_optional
            "TpoImpresion"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_impresion_of_string)
            children
            errors
    inl v_ind_servicio, errors =
        xml_codec.read_optional
            "IndServicio"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_ind_servicio_of_string)
            children
            errors
    inl v_mnt_bruto, errors =
        xml_codec.read_optional
            "MntBruto"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_bruto_of_string)
            children
            errors
    inl v_fma_pago, errors =
        xml_codec.read_optional
            "FmaPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_fma_pago_of_string)
            children
            errors
    inl v_fma_pag_exp, errors = xml_codec.read_optional "FmaPagExp" xml_codec.read_text children errors
    inl v_fch_cancel, errors = xml_codec.read_optional "FchCancel" xml_codec.read_text children errors
    inl v_mnt_cancel, errors = xml_codec.read_optional "MntCancel" xml_codec.read_text children errors
    inl v_saldo_insol, errors = xml_codec.read_optional "SaldoInsol" xml_codec.read_text children errors
    inl v_mnt_pagos, errors =
        xml_codec.read_repeated
            "MntPagos"
            x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_of_xml
            children
            errors
    inl v_periodo_desde, errors = xml_codec.read_optional "PeriodoDesde" xml_codec.read_text children errors
    inl v_periodo_hasta, errors = xml_codec.read_optional "PeriodoHasta" xml_codec.read_text children errors
    inl v_medio_pago, errors =
        xml_codec.read_optional "MedioPago" (xml_codec.read_value x_medio_pago_type_of_string) children errors
    inl v_tpo_cta_pago, errors =
        xml_codec.read_optional
            "TpoCtaPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_tpo_cta_pago_of_string)
            children
            errors
    inl v_num_cta_pago, errors =
        xml_codec.read_optional
            "NumCtaPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_num_cta_pago_of_string)
            children
            errors
    inl v_bco_pago, errors =
        xml_codec.read_optional
            "BcoPago"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_bco_pago_of_string)
            children
            errors
    inl v_term_pago_cdg, errors =
        xml_codec.read_optional
            "TermPagoCdg"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_cdg_of_string)
            children
            errors
    inl v_term_pago_glosa, errors =
        xml_codec.read_optional
            "TermPagoGlosa"
            (xml_codec.read_value x_dte_def_type_choice_documento_documento_encabezado_id_doc_term_pago_glosa_of_string)
            children
            errors
    inl v_term_pago_dias, errors = xml_codec.read_optional "TermPagoDias" xml_codec.read_text children errors
    inl v_fch_venc, errors = xml_codec.read_optional "FchVenc" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_id_doc {
            tipo_dte = xml_codec.present o_tipo_dte
            folio = xml_codec.present o_folio
            fch_emis = xml_codec.present o_fch_emis
            ind_no_rebaja = v_ind_no_rebaja
            tipo_despacho = v_tipo_despacho
            ind_traslado = v_ind_traslado
            tpo_impresion = v_tpo_impresion
            ind_servicio = v_ind_servicio
            mnt_bruto = v_mnt_bruto
            fma_pago = v_fma_pago
            fma_pag_exp = v_fma_pag_exp
            fch_cancel = v_fch_cancel
            mnt_cancel = v_mnt_cancel
            saldo_insol = v_saldo_insol
            mnt_pagos = v_mnt_pagos
            periodo_desde = v_periodo_desde
            periodo_hasta = v_periodo_hasta
            medio_pago = v_medio_pago
            tpo_cta_pago = v_tpo_cta_pago
            num_cta_pago = v_num_cta_pago
            bco_pago = v_bco_pago
            term_pago_cdg = v_term_pago_cdg
            term_pago_glosa = v_term_pago_glosa
            term_pago_dias = v_term_pago_dias
            fch_venc = v_fch_venc
        }

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_id_doc) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_fch_pago, errors = xml_codec.read_required "FchPago" xml_codec.read_text children errors
    inl o_mnt_pago, errors = xml_codec.read_required "MntPago" xml_codec.read_text children errors
    inl v_glosa_pagos, errors =
        xml_codec.read_optional
            "GlosaPagos"
            (
                xml_codec.read_value
                    x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_glosa_pagos_of_string
            )
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos {
            fch_pago = xml_codec.present o_fch_pago
            mnt_pago = xml_codec.present o_mnt_pago
            glosa_pagos = v_glosa_pagos
        }

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_of_xml
    (c : xml.xml_node)
    : result (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos) (list xml_codec.decode_error)
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_signature_type_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_signed_info, errors = xml_codec.read_required "SignedInfo" x_signature_type_signed_info_of_xml children errors
    inl o_signature_value, errors = xml_codec.read_required "SignatureValue" xml_codec.read_text children errors
    inl o_key_info, errors = xml_codec.read_required "KeyInfo" x_signature_type_key_info_of_xml children errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type {
            signed_info = xml_codec.present o_signed_info
            signature_value = xml_codec.present o_signature_value
            key_info = xml_codec.present o_key_info
        }

and let x_signature_type_of_xml (c : xml.xml_node) : result (x_signature_type) (list xml_codec.decode_error) =
    x_signature_type_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_key_info_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_key_info) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_key_value, errors =
        xml_codec.read_required "KeyValue" x_signature_type_key_info_key_value_of_xml children errors
    inl o_x509_data, errors =
        xml_codec.read_required "X509Data" x_signature_type_key_info_x509_data_of_xml children errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_key_info {
            key_value = xml_codec.present o_key_value
            x509_data = xml_codec.present o_x509_data
        }

and let x_signature_type_key_info_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_key_info) (list xml_codec.decode_error)
    =
    x_signature_type_key_info_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_key_info_x509_data_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_key_info_x509_data) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_x509_certificate, errors = xml_codec.read_required "X509Certificate" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_key_info_x509_data { x509_certificate = xml_codec.present o_x509_certificate }

and let x_signature_type_key_info_x509_data_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_key_info_x509_data) (list xml_codec.decode_error)
    =
    x_signature_type_key_info_x509_data_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_key_info_key_value_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_key_info_key_value) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_choice_rsa_key_value, errors =
        xml_codec.taken
            (x_signature_type_key_info_key_value_choice_rsa_key_value_of_children attributes children)
            errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_key_info_key_value { choice_rsa_key_value = xml_codec.present o_choice_rsa_key_value }

and let x_signature_type_key_info_key_value_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_key_info_key_value) (list xml_codec.decode_error)
    =
    x_signature_type_key_info_key_value_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_key_info_key_value_choice_rsa_key_value_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_key_info_key_value_choice_rsa_key_value) (list xml_codec.decode_error)
    =
    if false
    then xml_codec.no_alternative "x_signature_type_key_info_key_value_choice_rsa_key_value"
    elif xml.has_child "RSAKeyValue" children then
        xml_codec.mapped
            fun v => SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue v
            (
                xml_codec.read_child
                    "RSAKeyValue"
                    x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_of_xml
                    children
            )
    elif xml.has_child "DSAKeyValue" children then
        xml_codec.mapped
            fun v => SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue v
            (
                xml_codec.read_child
                    "DSAKeyValue"
                    x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_of_xml
                    children
            )
    else xml_codec.no_alternative "x_signature_type_key_info_key_value_choice_rsa_key_value"

and let x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_p, errors = xml_codec.read_required "P" xml_codec.read_text children errors
    inl o_q, errors = xml_codec.read_required "Q" xml_codec.read_text children errors
    inl o_g, errors = xml_codec.read_required "G" xml_codec.read_text children errors
    inl o_y, errors = xml_codec.read_required "Y" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value {
            p = xml_codec.present o_p
            q = xml_codec.present o_q
            g = xml_codec.present o_g
            y = xml_codec.present o_y
        }

and let x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value) (list xml_codec.decode_error)
    =
    x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_modulus, errors = xml_codec.read_required "Modulus" xml_codec.read_text children errors
    inl o_exponent, errors = xml_codec.read_required "Exponent" xml_codec.read_text children errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value {
            modulus = xml_codec.present o_modulus
            exponent = xml_codec.present o_exponent
        }

and let x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value) (list xml_codec.decode_error)
    =
    x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_signature_type_signed_info_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_signed_info) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_canonicalization_method, errors =
        xml_codec.read_required
            "CanonicalizationMethod"
            x_signature_type_signed_info_canonicalization_method_of_xml
            children
            errors
    inl o_signature_method, errors =
        xml_codec.read_required "SignatureMethod" x_signature_type_signed_info_signature_method_of_xml children errors
    inl o_reference, errors =
        xml_codec.read_required "Reference" x_signature_type_signed_info_reference_of_xml children errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_signed_info {
            canonicalization_method = xml_codec.present o_canonicalization_method
            signature_method = xml_codec.present o_signature_method
            reference = xml_codec.present o_reference
        }

and let x_signature_type_signed_info_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_signed_info) (list xml_codec.decode_error)
    =
    x_signature_type_signed_info_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_signed_info_reference_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_signed_info_reference) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_transforms, errors =
        xml_codec.read_optional "Transforms" x_signature_type_signed_info_reference_transforms_of_xml children errors
    inl o_digest_method, errors =
        xml_codec.read_required
            "DigestMethod"
            x_signature_type_signed_info_reference_digest_method_of_xml
            children
            errors
    inl o_digest_value, errors = xml_codec.read_required "DigestValue" xml_codec.read_text children errors
    inl v_uri, errors = xml_codec.attribute_required "URI" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_signed_info_reference {
            transforms = v_transforms
            digest_method = xml_codec.present o_digest_method
            digest_value = xml_codec.present o_digest_value
            uri = v_uri
        }

and let x_signature_type_signed_info_reference_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_signed_info_reference) (list xml_codec.decode_error)
    =
    x_signature_type_signed_info_reference_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_signed_info_reference_digest_method_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_signed_info_reference_digest_method) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_algorithm, errors = xml_codec.attribute_required "Algorithm" attributes errors
    inl errors = xml_codec.fixed_checked "Algorithm" "http://www.w3.org/2000/09/xmldsig#sha1" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_signed_info_reference_digest_method { algorithm = v_algorithm }

and let x_signature_type_signed_info_reference_digest_method_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_signed_info_reference_digest_method) (list xml_codec.decode_error)
    =
    x_signature_type_signed_info_reference_digest_method_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_signed_info_reference_transforms_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_signed_info_reference_transforms) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_transform, errors =
        xml_codec.read_required
            "Transform"
            x_signature_type_signed_info_reference_transforms_transform_of_xml
            children
            errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_signed_info_reference_transforms { transform = xml_codec.present o_transform }

and let x_signature_type_signed_info_reference_transforms_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_signed_info_reference_transforms) (list xml_codec.decode_error)
    =
    x_signature_type_signed_info_reference_transforms_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_signed_info_reference_transforms_transform_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_signed_info_reference_transforms_transform) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_algorithm, errors = xml_codec.attribute_required "Algorithm" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_signed_info_reference_transforms_transform { algorithm = v_algorithm }

and let x_signature_type_signed_info_reference_transforms_transform_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_signed_info_reference_transforms_transform) (list xml_codec.decode_error)
    =
    x_signature_type_signed_info_reference_transforms_transform_of_children
        (xml.element_attributes c)
        (xml.element_children c)

and let x_signature_type_signed_info_signature_method_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_signed_info_signature_method) (list xml_codec.decode_error)
    =
    inl errors = []
    inl o_algorithm, errors =
        xml_codec.attribute_parsed x_signature_type_signed_info_signature_method_algorithm_of_string
            "Algorithm"
            attributes
            errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_signed_info_signature_method { algorithm = xml_codec.present o_algorithm }

and let x_signature_type_signed_info_signature_method_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_signed_info_signature_method) (list xml_codec.decode_error)
    =
    x_signature_type_signed_info_signature_method_of_children (xml.element_attributes c) (xml.element_children c)

and let x_signature_type_signed_info_canonicalization_method_of_children
    (attributes : list (string * string)) (children : list xml.xml_node)
    : result (x_signature_type_signed_info_canonicalization_method) (list xml_codec.decode_error)
    =
    inl errors = []
    inl v_algorithm, errors = xml_codec.attribute_required "Algorithm" attributes errors
    inl errors = xml_codec.fixed_checked "Algorithm" "http://www.w3.org/TR/2001/REC-xml-c14n-20010315" attributes errors
    xml_codec.decoded_by errors fun () =>
        x_signature_type_signed_info_canonicalization_method { algorithm = v_algorithm }

and let x_signature_type_signed_info_canonicalization_method_of_xml
    (c : xml.xml_node)
    : result (x_signature_type_signed_info_canonicalization_method) (list xml_codec.decode_error)
    =
    x_signature_type_signed_info_canonicalization_method_of_children (xml.element_attributes c) (xml.element_children c)

and let x_dte_def_type_children forall s. (x : x_dte_def_type s) : list xml.xml_node =
    listm.concat [
        x_dte_def_type_choice_documento_children x.choice_documento
        staged_list (fun v => x_signature_type_to_xml "ds:Signature" v) x.signature
    ]

and let x_dte_def_type_attributes forall s. (x : x_dte_def_type s) : list (string * string) =
    [ ("version", to_text x.version) ]

and let x_dte_def_type_to_xml forall s. (name : string) (x : x_dte_def_type s) : xml.xml_node =
    xml.element name (x_dte_def_type_attributes x) (x_dte_def_type_children x)

and let x_dte_def_type_choice_documento_children (x : x_dte_def_type_choice_documento) : list xml.xml_node =
    match x with
    | DTEDefType_choice_Documento_Documento v => [ x_dte_def_type_choice_documento_documento_to_xml "Documento" v ]
    | DTEDefType_choice_Documento_Liquidacion v =>
        [ x_dte_def_type_choice_documento_liquidacion_to_xml "Liquidacion" v ]
    | DTEDefType_choice_Documento_Exportaciones v =>
        [ x_dte_def_type_choice_documento_exportaciones_to_xml "Exportaciones" v ]

and let x_dte_def_type_choice_documento_exportaciones_children
    (x : x_dte_def_type_choice_documento_exportaciones)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_exportaciones_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_field_Encabezado =>
            [ x_dte_def_type_choice_documento_exportaciones_encabezado_to_xml "Encabezado" v ]
        | DTEDefType_choice_Documento_Exportaciones_field_Detalle =>
            listm.map (fun v => x_dte_def_type_choice_documento_exportaciones_detalle_to_xml "Detalle" v) v
        | DTEDefType_choice_Documento_Exportaciones_field_SubTotInfo =>
            listm.map (fun v => x_dte_def_type_choice_documento_exportaciones_sub_tot_info_to_xml "SubTotInfo" v) v
        | DTEDefType_choice_Documento_Exportaciones_field_DscRcgGlobal =>
            listm.map (fun v => x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_to_xml "DscRcgGlobal" v) v
        | DTEDefType_choice_Documento_Exportaciones_field_Referencia =>
            listm.map (fun v => x_dte_def_type_choice_documento_exportaciones_referencia_to_xml "Referencia" v) v
        | DTEDefType_choice_Documento_Exportaciones_field_TED =>
            [ x_dte_def_type_choice_documento_exportaciones_ted_to_xml "TED" v ]
        | DTEDefType_choice_Documento_Exportaciones_field_TmstFirma =>
            [ xml.element "TmstFirma" [] (xml.text_children v) ]
    inl rec walk (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_field_Encabezado
        exists DTEDefType_choice_Documento_Exportaciones_field_Detalle
        exists DTEDefType_choice_Documento_Exportaciones_field_SubTotInfo
        exists DTEDefType_choice_Documento_Exportaciones_field_DscRcgGlobal
        exists DTEDefType_choice_Documento_Exportaciones_field_Referencia
        exists DTEDefType_choice_Documento_Exportaciones_field_TED
        exists DTEDefType_choice_Documento_Exportaciones_field_TmstFirma
    ]

and let x_dte_def_type_choice_documento_exportaciones_attributes
    (x : x_dte_def_type_choice_documento_exportaciones)
    : list (string * string)
    =
    [ ("ID", x.id) ]

and let x_dte_def_type_choice_documento_exportaciones_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_exportaciones_ted_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_TED_field_DD =>
            [ x_dte_def_type_choice_documento_exportaciones_ted_dd_to_xml "DD" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_field_FRMT =>
            [ x_dte_def_type_choice_documento_exportaciones_ted_frmt_to_xml "FRMT" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_ted_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_TED_field_DD
        exists DTEDefType_choice_Documento_Exportaciones_TED_field_FRMT
    ]

and let x_dte_def_type_choice_documento_exportaciones_ted_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted)
    : list (string * string)
    =
    [ ("version", x.version) ]

and let x_dte_def_type_choice_documento_exportaciones_ted_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_frmt_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_frmt)
    : list xml.xml_node
    =
    xml.text_children x.base

and let x_dte_def_type_choice_documento_exportaciones_ted_frmt_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_frmt)
    : list (string * string)
    =
    [ ("algoritmo", to_text x.algoritmo) ]

and let x_dte_def_type_choice_documento_exportaciones_ted_frmt_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_frmt)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_frmt_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_frmt_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RE => [ text_node "RE" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TD => [ text_node "TD" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_F => [ xml.element "F" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_FE => [ xml.element "FE" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RR => [ text_node "RR" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RSR => [ text_node "RSR" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_MNT => [ xml.element "MNT" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_IT1 => [ text_node "IT1" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_CAF =>
            [ x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_to_xml "CAF" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TSTED =>
            [ xml.element "TSTED" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_ted_dd_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RE
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TD
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_F
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_FE
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RR
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_RSR
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_MNT
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_IT1
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_CAF
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_field_TSTED
    ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_dd)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_DA =>
            [ x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_to_xml "DA" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_FRMA =>
            [ x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_to_xml "FRMA" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_DA
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_field_FRMA
    ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf)
    : list (string * string)
    =
    [ ("version", x.version) ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma)
    : list xml.xml_node
    =
    xml.text_children x.base

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma)
    : list (string * string)
    =
    [ ("algoritmo", x.algoritmo) ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RE => [ text_node "RE" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RS => [ text_node "RS" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_TD => [ text_node "TD" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RNG =>
            [ x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_to_xml "RNG" v ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_FA =>
            [ xml.element "FA" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_choice_rsapk =>
            x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_children v
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_IDK => [ text_node "IDK" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RE
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RS
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_TD
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_RNG
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_FA
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_choice_rsapk
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_field_IDK
    ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk)
    : list xml.xml_node
    =
    match x with
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK v =>
        [ x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_to_xml "RSAPK" v ]
    | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK v =>
        [ x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_to_xml "DSAPK" v ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P =>
            [ xml.element "P" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q =>
            [ xml.element "Q" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G =>
            [ xml.element "G" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y =>
            [ xml.element "Y" [] (xml.text_children v) ]
    inl rec walk
        (fields :
            list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y
    ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M =>
            [ xml.element "M" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E =>
            [ xml.element "E" [] (xml.text_children v) ]
    inl rec walk
        (fields :
            list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E
    ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_children x)

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_children
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_D =>
            [ xml.element "D" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_H =>
            [ xml.element "H" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_D
        exists DTEDefType_choice_Documento_Exportaciones_TED_DD_CAF_DA_RNG_field_H
    ]

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_children x)

and let x_dte_def_type_choice_documento_exportaciones_referencia_children
    (x : x_dte_def_type_choice_documento_exportaciones_referencia)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_referencia_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_NroLinRef =>
            [ xml.element "NroLinRef" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_TpoDocRef => [ text_node "TpoDocRef" v ]
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IndGlobal => optional_text_nodes "IndGlobal" v
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FolioRef => [ text_node "FolioRef" v ]
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RUTOtr => optional_text_nodes "RUTOtr" v
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_IdAdicOtr => optional_text_nodes "IdAdicOtr" v
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_FchRef =>
            [ xml.element "FchRef" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_CodRef => optional_text_nodes "CodRef" v
        | DTEDefType_choice_Documento_Exportaciones_Referencia_field_RazonRef => optional_text_nodes "RazonRef" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_referencia_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_referencia_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_NroLinRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_TpoDocRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_IndGlobal
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_FolioRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_RUTOtr
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_IdAdicOtr
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_FchRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_CodRef
        exists DTEDefType_choice_Documento_Exportaciones_Referencia_field_RazonRef
    ]

and let x_dte_def_type_choice_documento_exportaciones_referencia_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_referencia)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_referencia_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_referencia)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_referencia_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_referencia_children x)

and let x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_children
    (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_NroLinDR => [ text_node "NroLinDR" v ]
        | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoMov => [ text_node "TpoMov" v ]
        | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_GlosaDR => optional_text_nodes "GlosaDR" v
        | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoValor => [ text_node "TpoValor" v ]
        | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDR =>
            [ xml.element "ValorDR" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDROtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "ValorDROtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_IndExeDR => optional_text_nodes "IndExeDR" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_NroLinDR
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoMov
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_GlosaDR
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_TpoValor
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDR
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_ValorDROtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_DscRcgGlobal_field_IndExeDR
    ]

and let x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_children x)

and let x_dte_def_type_choice_documento_exportaciones_sub_tot_info_children
    (x : x_dte_def_type_choice_documento_exportaciones_sub_tot_info)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_NroSTI =>
            [ xml.element "NroSTI" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_GlosaSTI => [ text_node "GlosaSTI" v ]
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_OrdenSTI =>
            xml.option_list (optionm.map (fun v => xml.element "OrdenSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotNetoSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotNetoSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotIVASTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotIVASTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotAdicSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotAdicSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotExeSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotExeSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_ValSubtotSTI =>
            xml.option_list (optionm.map (fun v => xml.element "ValSubtotSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_LineasDeta =>
            listm.map (fun v => xml.element "LineasDeta" [] (xml.text_children v)) v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_sub_tot_info_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_NroSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_GlosaSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_OrdenSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotNetoSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotIVASTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotAdicSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_SubTotExeSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_ValSubtotSTI
        exists DTEDefType_choice_Documento_Exportaciones_SubTotInfo_field_LineasDeta
    ]

and let x_dte_def_type_choice_documento_exportaciones_sub_tot_info_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_sub_tot_info)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_sub_tot_info_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_sub_tot_info)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_sub_tot_info_children x)

and let x_dte_def_type_choice_documento_exportaciones_detalle_children
    (x : x_dte_def_type_choice_documento_exportaciones_detalle)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_detalle_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NroLinDet =>
            [ xml.element "NroLinDet" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_CdgItem =>
            listm.map (fun v => x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_to_xml "CdgItem" v) v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_IndExe => optional_text_nodes "IndExe" v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_NmbItem => [ text_node "NmbItem" v ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DscItem => optional_text_nodes "DscItem" v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyRef =>
            xml.option_list (optionm.map (fun v => xml.element "QtyRef" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdRef => optional_text_nodes "UnmdRef" v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcRef =>
            xml.option_list (optionm.map (fun v => xml.element "PrcRef" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyItem =>
            xml.option_list (optionm.map (fun v => xml.element "QtyItem" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_Subcantidad =>
            listm.map
                fun v => x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_to_xml "Subcantidad" v
                v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchElabor =>
            xml.option_list (optionm.map (fun v => xml.element "FchElabor" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchVencim =>
            xml.option_list (optionm.map (fun v => xml.element "FchVencim" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdItem => optional_text_nodes "UnmdItem" v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcItem =>
            xml.option_list (optionm.map (fun v => xml.element "PrcItem" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_OtrMnda =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_to_xml "OtrMnda" v
                    v
            )
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoPct =>
            xml.option_list (optionm.map (fun v => xml.element "DescuentoPct" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoMonto =>
            xml.option_list (optionm.map (fun v => xml.element "DescuentoMonto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubDscto =>
            listm.map (fun v => x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_to_xml "SubDscto" v) v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoPct =>
            xml.option_list (optionm.map (fun v => xml.element "RecargoPct" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoMonto =>
            xml.option_list (optionm.map (fun v => xml.element "RecargoMonto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubRecargo =>
            listm.map
                fun v => x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_to_xml "SubRecargo" v
                v
        | DTEDefType_choice_Documento_Exportaciones_Detalle_field_MontoItem =>
            [ xml.element "MontoItem" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_detalle_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_NroLinDet
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_CdgItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_IndExe
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_NmbItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_DscItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyRef
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdRef
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcRef
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_QtyItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_Subcantidad
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchElabor
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_FchVencim
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_UnmdItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_PrcItem
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_OtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoPct
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_DescuentoMonto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubDscto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoPct
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_RecargoMonto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_SubRecargo
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_field_MontoItem
    ]

and let x_dte_def_type_choice_documento_exportaciones_detalle_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_detalle)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_detalle_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_detalle)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_detalle_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_detalle_children x)

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_children
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_TipoRecargo =>
            [ text_node "TipoRecargo" v ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_ValorRecargo =>
            [ xml.element "ValorRecargo" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_TipoRecargo
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubRecargo_field_ValorRecargo
    ]

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_children x)

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_children
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_TipoDscto => [ text_node "TipoDscto" v ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_ValorDscto =>
            [ xml.element "ValorDscto" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_TipoDscto
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_SubDscto_field_ValorDscto
    ]

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_children x)

and let x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_children
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_PrcOtrMon =>
            [ xml.element "PrcOtrMon" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_Moneda => [ text_node "Moneda" v ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_FctConv =>
            xml.option_list (optionm.map (fun v => xml.element "FctConv" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_DctoOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "DctoOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_RecargoOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "RecargoOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_MontoItemOtrMnda =>
            [ xml.element "MontoItemOtrMnda" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_PrcOtrMon
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_Moneda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_FctConv
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_DctoOtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_RecargoOtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_OtrMnda_field_MontoItemOtrMnda
    ]

and let x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_children x)

and let x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_children
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubQty =>
            [ xml.element "SubQty" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubCod => [ text_node "SubCod" v ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_TipCodSubQty =>
            optional_text_nodes "TipCodSubQty" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubQty
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_SubCod
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_Subcantidad_field_TipCodSubQty
    ]

and let x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_children x)

and let x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_children
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_TpoCodigo => [ text_node "TpoCodigo" v ]
        | DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_VlrCodigo => [ text_node "VlrCodigo" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_TpoCodigo
        exists DTEDefType_choice_Documento_Exportaciones_Detalle_CdgItem_field_VlrCodigo
    ]

and let x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_IdDoc =>
            [ x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_to_xml "IdDoc" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Emisor =>
            [ x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_to_xml "Emisor" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Receptor =>
            [ x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_to_xml "Receptor" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Transporte =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_to_xml "Transporte" v
                    v
            )
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Totales =>
            [ x_dte_def_type_choice_documento_exportaciones_encabezado_totales_to_xml "Totales" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_field_OtraMoneda =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_to_xml "OtraMoneda" v
                    v
            )
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_IdDoc
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Emisor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Receptor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Transporte
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_Totales
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_field_OtraMoneda
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoMoneda => [ text_node "TpoMoneda" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoCambio =>
            xml.option_list (optionm.map (fun v => xml.element "TpoCambio" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntExeOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "MntExeOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntTotOtrMnda =>
            [ xml.element "MntTotOtrMnda" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoMoneda
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_TpoCambio
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntExeOtrMnda
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_OtraMoneda_field_MntTotOtrMnda
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_totales_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_totales)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_TpoMoneda => [ text_node "TpoMoneda" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntExe =>
            [ xml.element "MntExe" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntTotal =>
            [ xml.element "MntTotal" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_totales_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_totales_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_TpoMoneda
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntExe
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Totales_field_MntTotal
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_totales_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_totales)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_totales_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_totales)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_totales_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_totales_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Patente =>
            optional_text_nodes "Patente" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_RUTTrans =>
            optional_text_nodes "RUTTrans" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Chofer =>
            xml.option_list (
                optionm.map
                    fun v =>
                        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_to_xml "Chofer" v
                    v
            )
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_DirDest =>
            optional_text_nodes "DirDest" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CmnaDest =>
            optional_text_nodes "CmnaDest" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CiudadDest =>
            optional_text_nodes "CiudadDest" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Aduana =>
            xml.option_list (
                optionm.map
                    fun v =>
                        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_to_xml "Aduana" v
                    v
            )
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Patente
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_RUTTrans
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Chofer
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_DirDest
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CmnaDest
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_CiudadDest
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_field_Aduana
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodModVenta =>
            xml.option_list (optionm.map (fun v => xml.element "CodModVenta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodClauVenta =>
            xml.option_list (optionm.map (fun v => xml.element "CodClauVenta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotClauVenta =>
            xml.option_list (optionm.map (fun v => xml.element "TotClauVenta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodViaTransp =>
            xml.option_list (optionm.map (fun v => xml.element "CodViaTransp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NombreTransp =>
            optional_text_nodes "NombreTransp" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_RUTCiaTransp =>
            optional_text_nodes "RUTCiaTransp" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NomCiaTransp =>
            optional_text_nodes "NomCiaTransp" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicTransp =>
            optional_text_nodes "IdAdicTransp" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Booking =>
            optional_text_nodes "Booking" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Operador =>
            optional_text_nodes "Operador" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoEmbarque =>
            xml.option_list (optionm.map (fun v => xml.element "CodPtoEmbarque" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb =>
            optional_text_nodes "IdAdicPtoEmb" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoDesemb =>
            xml.option_list (optionm.map (fun v => xml.element "CodPtoDesemb" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb =>
            optional_text_nodes "IdAdicPtoDesemb" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Tara =>
            xml.option_list (optionm.map (fun v => xml.element "Tara" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidMedTara =>
            xml.option_list (optionm.map (fun v => xml.element "CodUnidMedTara" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoBruto =>
            xml.option_list (optionm.map (fun v => xml.element "PesoBruto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto =>
            xml.option_list (optionm.map (fun v => xml.element "CodUnidPesoBruto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoNeto =>
            xml.option_list (optionm.map (fun v => xml.element "PesoNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto =>
            xml.option_list (optionm.map (fun v => xml.element "CodUnidPesoNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotItems =>
            xml.option_list (optionm.map (fun v => xml.element "TotItems" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotBultos =>
            xml.option_list (optionm.map (fun v => xml.element "TotBultos" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TipoBultos =>
            listm.map
                fun v =>
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_to_xml
                        "TipoBultos"
                        v
                v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntFlete =>
            xml.option_list (optionm.map (fun v => xml.element "MntFlete" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntSeguro =>
            xml.option_list (optionm.map (fun v => xml.element "MntSeguro" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisRecep =>
            xml.option_list (optionm.map (fun v => xml.element "CodPaisRecep" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisDestin =>
            xml.option_list (optionm.map (fun v => xml.element "CodPaisDestin" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodModVenta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodClauVenta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotClauVenta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodViaTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NombreTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_RUTCiaTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_NomCiaTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicTransp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Booking
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Operador
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoEmbarque
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPtoDesemb
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_Tara
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidMedTara
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoBruto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_PesoNeto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotItems
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TotBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_TipoBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntFlete
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_MntSeguro
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_field_CodPaisDestin
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos =>
            xml.option_list (optionm.map (fun v => xml.element "CodTpoBultos" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos =>
            xml.option_list (optionm.map (fun v => xml.element "CantBultos" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas =>
            optional_text_nodes "Marcas" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer =>
            optional_text_nodes "IdContainer" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Sello =>
            optional_text_nodes "Sello" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello =>
            optional_text_nodes "EmisorSello" v
    inl rec walk
        (fields :
            list
                (exists r.
                    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (
                    nodes f (
                        x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_get f x
                    )
                )
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_Sello
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_RUTChofer =>
            [ text_node "RUTChofer" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_NombreChofer =>
            [ text_node "NombreChofer" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_RUTChofer
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Transporte_Chofer_field_NombreChofer
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RUTRecep => [ text_node "RUTRecep" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CdgIntRecep =>
            optional_text_nodes "CdgIntRecep" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RznSocRecep =>
            [ text_node "RznSocRecep" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Extranjero =>
            xml.option_list (
                optionm.map
                    fun v =>
                        x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_to_xml
                            "Extranjero"
                            v
                    v
            )
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_GiroRecep =>
            optional_text_nodes "GiroRecep" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Contacto =>
            optional_text_nodes "Contacto" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CorreoRecep =>
            optional_text_nodes "CorreoRecep" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirRecep =>
            optional_text_nodes "DirRecep" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaRecep =>
            optional_text_nodes "CmnaRecep" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadRecep =>
            optional_text_nodes "CiudadRecep" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirPostal =>
            optional_text_nodes "DirPostal" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaPostal =>
            optional_text_nodes "CmnaPostal" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadPostal =>
            optional_text_nodes "CiudadPostal" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RUTRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CdgIntRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_RznSocRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Extranjero
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_GiroRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_Contacto
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CorreoRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadRecep
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_DirPostal
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CmnaPostal
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_field_CiudadPostal
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_NumId =>
            optional_text_nodes "NumId" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_Nacionalidad =>
            optional_text_nodes "Nacionalidad" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_IdAdicRecep =>
            optional_text_nodes "IdAdicRecep" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_NumId
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_Nacionalidad
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Receptor_Extranjero_field_IdAdicRecep
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RUTEmisor => [ text_node "RUTEmisor" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RznSoc => [ text_node "RznSoc" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_GiroEmis => [ text_node "GiroEmis" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Telefono => repeated_text_nodes "Telefono" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CorreoEmisor =>
            optional_text_nodes "CorreoEmisor" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Acteco =>
            listm.map (fun v => xml.element "Acteco" [] (xml.text_children v)) v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Sucursal => optional_text_nodes "Sucursal" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgSIISucur =>
            xml.option_list (optionm.map (fun v => xml.element "CdgSIISucur" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CodAdicSucur =>
            optional_text_nodes "CodAdicSucur" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_DirOrigen =>
            optional_text_nodes "DirOrigen" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CmnaOrigen =>
            optional_text_nodes "CmnaOrigen" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CiudadOrigen =>
            optional_text_nodes "CiudadOrigen" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgVendedor =>
            optional_text_nodes "CdgVendedor" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_IdAdicEmisor =>
            optional_text_nodes "IdAdicEmisor" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RUTEmisor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_RznSoc
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_GiroEmis
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Telefono
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CorreoEmisor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Acteco
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_Sucursal
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgSIISucur
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CodAdicSucur
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_DirOrigen
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CmnaOrigen
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CiudadOrigen
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_CdgVendedor
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_Emisor_field_IdAdicEmisor
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_emisor)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDTE => [ text_node "TipoDTE" v ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_Folio =>
            [ xml.element "Folio" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchEmis =>
            [ xml.element "FchEmis" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDespacho =>
            optional_text_nodes "TipoDespacho" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_IndServicio =>
            optional_text_nodes "IndServicio" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPago => optional_text_nodes "FmaPago" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPagExp =>
            xml.option_list (optionm.map (fun v => xml.element "FmaPagExp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchCancel =>
            xml.option_list (optionm.map (fun v => xml.element "FchCancel" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntCancel =>
            xml.option_list (optionm.map (fun v => xml.element "MntCancel" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_SaldoInsol =>
            xml.option_list (optionm.map (fun v => xml.element "SaldoInsol" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntPagos =>
            listm.map
                fun v => x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_to_xml "MntPagos" v
                v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoDesde =>
            xml.option_list (optionm.map (fun v => xml.element "PeriodoDesde" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoHasta =>
            xml.option_list (optionm.map (fun v => xml.element "PeriodoHasta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MedioPago =>
            optional_text_nodes "MedioPago" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TpoCtaPago =>
            optional_text_nodes "TpoCtaPago" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_NumCtaPago =>
            optional_text_nodes "NumCtaPago" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_BcoPago => optional_text_nodes "BcoPago" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoCdg =>
            optional_text_nodes "TermPagoCdg" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoGlosa =>
            optional_text_nodes "TermPagoGlosa" v
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoDias =>
            xml.option_list (optionm.map (fun v => xml.element "TermPagoDias" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchVenc =>
            xml.option_list (optionm.map (fun v => xml.element "FchVenc" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDTE
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_Folio
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchEmis
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TipoDespacho
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_IndServicio
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FmaPagExp
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchCancel
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntCancel
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_SaldoInsol
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MntPagos
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoDesde
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_PeriodoHasta
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_MedioPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TpoCtaPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_NumCtaPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_BcoPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoCdg
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoGlosa
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_TermPagoDias
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_field_FchVenc
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_children x)

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_children
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_FchPago =>
            [ xml.element "FchPago" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_MntPago =>
            [ xml.element "MntPago" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_GlosaPagos =>
            optional_text_nodes "GlosaPagos" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_FchPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_MntPago
        exists DTEDefType_choice_Documento_Exportaciones_Encabezado_IdDoc_MntPagos_field_GlosaPagos
    ]

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_attributes
    (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_attributes x)
        (x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_children x)

and let x_dte_def_type_choice_documento_liquidacion_children
    (x : x_dte_def_type_choice_documento_liquidacion)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_liquidacion_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_field_Encabezado =>
            [ x_dte_def_type_choice_documento_liquidacion_encabezado_to_xml "Encabezado" v ]
        | DTEDefType_choice_Documento_Liquidacion_field_Detalle =>
            listm.map (fun v => x_dte_def_type_choice_documento_liquidacion_detalle_to_xml "Detalle" v) v
        | DTEDefType_choice_Documento_Liquidacion_field_SubTotInfo =>
            listm.map (fun v => x_dte_def_type_choice_documento_liquidacion_sub_tot_info_to_xml "SubTotInfo" v) v
        | DTEDefType_choice_Documento_Liquidacion_field_Referencia =>
            listm.map (fun v => x_dte_def_type_choice_documento_liquidacion_referencia_to_xml "Referencia" v) v
        | DTEDefType_choice_Documento_Liquidacion_field_Comisiones =>
            listm.map (fun v => x_dte_def_type_choice_documento_liquidacion_comisiones_to_xml "Comisiones" v) v
        | DTEDefType_choice_Documento_Liquidacion_field_TED =>
            [ x_dte_def_type_choice_documento_liquidacion_ted_to_xml "TED" v ]
        | DTEDefType_choice_Documento_Liquidacion_field_TmstFirma =>
            [ xml.element "TmstFirma" [] (xml.text_children v) ]
    inl rec walk (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_field_Encabezado
        exists DTEDefType_choice_Documento_Liquidacion_field_Detalle
        exists DTEDefType_choice_Documento_Liquidacion_field_SubTotInfo
        exists DTEDefType_choice_Documento_Liquidacion_field_Referencia
        exists DTEDefType_choice_Documento_Liquidacion_field_Comisiones
        exists DTEDefType_choice_Documento_Liquidacion_field_TED
        exists DTEDefType_choice_Documento_Liquidacion_field_TmstFirma
    ]

and let x_dte_def_type_choice_documento_liquidacion_attributes
    (x : x_dte_def_type_choice_documento_liquidacion)
    : list (string * string)
    =
    [ ("ID", x.id) ]

and let x_dte_def_type_choice_documento_liquidacion_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_liquidacion_ted_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_TED_field_DD =>
            [ x_dte_def_type_choice_documento_liquidacion_ted_dd_to_xml "DD" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_field_FRMT =>
            [ x_dte_def_type_choice_documento_liquidacion_ted_frmt_to_xml "FRMT" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_ted_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_TED_field_DD
        exists DTEDefType_choice_Documento_Liquidacion_TED_field_FRMT
    ]

and let x_dte_def_type_choice_documento_liquidacion_ted_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted)
    : list (string * string)
    =
    [ ("version", x.version) ]

and let x_dte_def_type_choice_documento_liquidacion_ted_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_frmt_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_frmt)
    : list xml.xml_node
    =
    xml.text_children x.base

and let x_dte_def_type_choice_documento_liquidacion_ted_frmt_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_frmt)
    : list (string * string)
    =
    [ ("algoritmo", to_text x.algoritmo) ]

and let x_dte_def_type_choice_documento_liquidacion_ted_frmt_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_frmt)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_frmt_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_frmt_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RE => [ text_node "RE" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TD => [ text_node "TD" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_F => [ xml.element "F" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_FE => [ xml.element "FE" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RR => [ text_node "RR" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RSR => [ text_node "RSR" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_MNT => [ xml.element "MNT" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_IT1 => [ text_node "IT1" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_CAF =>
            [ x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_to_xml "CAF" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TSTED => [ xml.element "TSTED" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_ted_dd_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RE
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TD
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_F
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_FE
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RR
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_RSR
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_MNT
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_IT1
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_CAF
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_field_TSTED
    ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_dd)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_DA =>
            [ x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_to_xml "DA" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_FRMA =>
            [ x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_to_xml "FRMA" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_DA
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_field_FRMA
    ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf)
    : list (string * string)
    =
    [ ("version", x.version) ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma)
    : list xml.xml_node
    =
    xml.text_children x.base

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma)
    : list (string * string)
    =
    [ ("algoritmo", x.algoritmo) ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RE => [ text_node "RE" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RS => [ text_node "RS" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_TD => [ text_node "TD" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RNG =>
            [ x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_to_xml "RNG" v ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_FA =>
            [ xml.element "FA" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_choice_rsapk =>
            x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_children v
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_IDK => [ text_node "IDK" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RE
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RS
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_TD
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_RNG
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_FA
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_choice_rsapk
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_field_IDK
    ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk)
    : list xml.xml_node
    =
    match x with
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK v =>
        [ x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_to_xml "RSAPK" v ]
    | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK v =>
        [ x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_to_xml "DSAPK" v ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P =>
            [ xml.element "P" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q =>
            [ xml.element "Q" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G =>
            [ xml.element "G" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y =>
            [ xml.element "Y" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y
    ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M =>
            [ xml.element "M" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E =>
            [ xml.element "E" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E
    ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_children x)

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_children
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_D =>
            [ xml.element "D" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_H =>
            [ xml.element "H" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_D
        exists DTEDefType_choice_Documento_Liquidacion_TED_DD_CAF_DA_RNG_field_H
    ]

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_children x)

and let x_dte_def_type_choice_documento_liquidacion_comisiones_children
    (x : x_dte_def_type_choice_documento_liquidacion_comisiones)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_comisiones_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_NroLinCom =>
            [ xml.element "NroLinCom" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TipoMovim => [ text_node "TipoMovim" v ]
        | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_Glosa => [ text_node "Glosa" v ]
        | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TasaComision =>
            xml.option_list (optionm.map (fun v => xml.element "TasaComision" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComNeto =>
            [ xml.element "ValComNeto" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComExe =>
            [ xml.element "ValComExe" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComIVA =>
            xml.option_list (optionm.map (fun v => xml.element "ValComIVA" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_comisiones_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_comisiones_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_NroLinCom
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TipoMovim
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_Glosa
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_TasaComision
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Liquidacion_Comisiones_field_ValComIVA
    ]

and let x_dte_def_type_choice_documento_liquidacion_comisiones_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_comisiones)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_comisiones_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_comisiones)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_comisiones_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_comisiones_children x)

and let x_dte_def_type_choice_documento_liquidacion_referencia_children
    (x : x_dte_def_type_choice_documento_liquidacion_referencia)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_referencia_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Referencia_field_NroLinRef =>
            [ xml.element "NroLinRef" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Referencia_field_TpoDocRef => [ text_node "TpoDocRef" v ]
        | DTEDefType_choice_Documento_Liquidacion_Referencia_field_IndGlobal => optional_text_nodes "IndGlobal" v
        | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FolioRef => [ text_node "FolioRef" v ]
        | DTEDefType_choice_Documento_Liquidacion_Referencia_field_FchRef =>
            [ xml.element "FchRef" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Referencia_field_CodRef => optional_text_nodes "CodRef" v
        | DTEDefType_choice_Documento_Liquidacion_Referencia_field_RazonRef => optional_text_nodes "RazonRef" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_referencia_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_referencia_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_NroLinRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_TpoDocRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_IndGlobal
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_FolioRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_FchRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_CodRef
        exists DTEDefType_choice_Documento_Liquidacion_Referencia_field_RazonRef
    ]

and let x_dte_def_type_choice_documento_liquidacion_referencia_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_referencia)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_referencia_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_referencia)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_referencia_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_referencia_children x)

and let x_dte_def_type_choice_documento_liquidacion_sub_tot_info_children
    (x : x_dte_def_type_choice_documento_liquidacion_sub_tot_info)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_NroSTI =>
            [ xml.element "NroSTI" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_GlosaSTI => [ text_node "GlosaSTI" v ]
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_OrdenSTI =>
            xml.option_list (optionm.map (fun v => xml.element "OrdenSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotNetoSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotNetoSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotIVASTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotIVASTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotAdicSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotAdicSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotExeSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotExeSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_ValSubtotSTI =>
            xml.option_list (optionm.map (fun v => xml.element "ValSubtotSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_LineasDeta =>
            listm.map (fun v => xml.element "LineasDeta" [] (xml.text_children v)) v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_sub_tot_info_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_NroSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_GlosaSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_OrdenSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotNetoSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotIVASTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotAdicSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_SubTotExeSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_ValSubtotSTI
        exists DTEDefType_choice_Documento_Liquidacion_SubTotInfo_field_LineasDeta
    ]

and let x_dte_def_type_choice_documento_liquidacion_sub_tot_info_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_sub_tot_info)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_sub_tot_info_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_sub_tot_info)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_sub_tot_info_children x)

and let x_dte_def_type_choice_documento_liquidacion_detalle_children
    (x : x_dte_def_type_choice_documento_liquidacion_detalle)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_liquidacion_detalle_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NroLinDet =>
            [ xml.element "NroLinDet" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CdgItem =>
            listm.map (fun v => x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_to_xml "CdgItem" v) v
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_TpoDocLiq => [ text_node "TpoDocLiq" v ]
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_IndExe => optional_text_nodes "IndExe" v
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_NmbItem => [ text_node "NmbItem" v ]
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_DscItem => optional_text_nodes "DscItem" v
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyRef =>
            xml.option_list (optionm.map (fun v => xml.element "QtyRef" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdRef => optional_text_nodes "UnmdRef" v
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcRef =>
            xml.option_list (optionm.map (fun v => xml.element "PrcRef" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyItem =>
            xml.option_list (optionm.map (fun v => xml.element "QtyItem" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_Subcantidad =>
            listm.map
                fun v => x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_to_xml "Subcantidad" v
                v
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchElabor =>
            xml.option_list (optionm.map (fun v => xml.element "FchElabor" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchVencim =>
            xml.option_list (optionm.map (fun v => xml.element "FchVencim" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdItem => optional_text_nodes "UnmdItem" v
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcItem =>
            xml.option_list (optionm.map (fun v => xml.element "PrcItem" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_CodImpAdic => repeated_text_nodes "CodImpAdic" v
        | DTEDefType_choice_Documento_Liquidacion_Detalle_field_MontoItem =>
            [ xml.element "MontoItem" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_detalle_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_detalle_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_NroLinDet
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_CdgItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_TpoDocLiq
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_IndExe
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_NmbItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_DscItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyRef
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdRef
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcRef
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_QtyItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_Subcantidad
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchElabor
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_FchVencim
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_UnmdItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_PrcItem
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_CodImpAdic
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_field_MontoItem
    ]

and let x_dte_def_type_choice_documento_liquidacion_detalle_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_detalle)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_detalle_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_detalle)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_detalle_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_detalle_children x)

and let x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_children
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubQty =>
            [ xml.element "SubQty" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubCod => [ text_node "SubCod" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubQty
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_Subcantidad_field_SubCod
    ]

and let x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_children x)

and let x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_children
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_TpoCodigo => [ text_node "TpoCodigo" v ]
        | DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_VlrCodigo => [ text_node "VlrCodigo" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_TpoCodigo
        exists DTEDefType_choice_Documento_Liquidacion_Detalle_CdgItem_field_VlrCodigo
    ]

and let x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_IdDoc =>
            [ x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_to_xml "IdDoc" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Emisor =>
            [ x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_to_xml "Emisor" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Receptor =>
            [ x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_to_xml "Receptor" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Totales =>
            [ x_dte_def_type_choice_documento_liquidacion_encabezado_totales_to_xml "Totales" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_IdDoc
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Emisor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Receptor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_field_Totales
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntNeto =>
            xml.option_list (optionm.map (fun v => xml.element "MntNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntExe =>
            xml.option_list (optionm.map (fun v => xml.element "MntExe" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_TasaIVA =>
            xml.option_list (optionm.map (fun v => xml.element "TasaIVA" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVA =>
            xml.option_list (optionm.map (fun v => xml.element "IVA" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVAProp =>
            xml.option_list (optionm.map (fun v => xml.element "IVAProp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVATerc =>
            xml.option_list (optionm.map (fun v => xml.element "IVATerc" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_ImptoReten =>
            listm.map
                fun v =>
                    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_to_xml "ImptoReten" v
                v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_Comisiones =>
            xml.option_list (
                optionm.map
                    fun v =>
                        x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_to_xml "Comisiones" v
                    v
            )
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntTotal =>
            [ xml.element "MntTotal" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MontoPeriodo =>
            xml.option_list (optionm.map (fun v => xml.element "MontoPeriodo" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_SaldoAnterior =>
            xml.option_list (optionm.map (fun v => xml.element "SaldoAnterior" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_VlrPagar =>
            xml.option_list (optionm.map (fun v => xml.element "VlrPagar" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_totales_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntNeto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntExe
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_TasaIVA
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVA
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVAProp
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_IVATerc
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_ImptoReten
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_Comisiones
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MntTotal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_MontoPeriodo
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_SaldoAnterior
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_field_VlrPagar
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComNeto =>
            xml.option_list (optionm.map (fun v => xml.element "ValComNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComExe =>
            xml.option_list (optionm.map (fun v => xml.element "ValComExe" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComIVA =>
            xml.option_list (optionm.map (fun v => xml.element "ValComIVA" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_Comisiones_field_ValComIVA
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TipoImp =>
            [ text_node "TipoImp" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TasaImp =>
            xml.option_list (optionm.map (fun v => xml.element "TasaImp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_MontoImp =>
            [ xml.element "MontoImp" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TipoImp
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_TasaImp
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Totales_ImptoReten_field_MontoImp
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RUTRecep => [ text_node "RUTRecep" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CdgIntRecep =>
            optional_text_nodes "CdgIntRecep" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RznSocRecep => [ text_node "RznSocRecep" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_GiroRecep => [ text_node "GiroRecep" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_Contacto => optional_text_nodes "Contacto" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CorreoRecep =>
            optional_text_nodes "CorreoRecep" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirRecep => [ text_node "DirRecep" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaRecep => [ text_node "CmnaRecep" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadRecep =>
            optional_text_nodes "CiudadRecep" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirPostal =>
            optional_text_nodes "DirPostal" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaPostal =>
            optional_text_nodes "CmnaPostal" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadPostal =>
            optional_text_nodes "CiudadPostal" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RUTRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CdgIntRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_RznSocRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_GiroRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_Contacto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CorreoRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadRecep
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_DirPostal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CmnaPostal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Receptor_field_CiudadPostal
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado_receptor)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RUTEmisor => [ text_node "RUTEmisor" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RznSoc => [ text_node "RznSoc" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_GiroEmis => [ text_node "GiroEmis" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Telefono => repeated_text_nodes "Telefono" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CorreoEmisor =>
            optional_text_nodes "CorreoEmisor" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Acteco =>
            listm.map (fun v => xml.element "Acteco" [] (xml.text_children v)) v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Sucursal => optional_text_nodes "Sucursal" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgSIISucur =>
            xml.option_list (optionm.map (fun v => xml.element "CdgSIISucur" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_DirOrigen => [ text_node "DirOrigen" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CmnaOrigen =>
            optional_text_nodes "CmnaOrigen" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CiudadOrigen =>
            optional_text_nodes "CiudadOrigen" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgVendedor =>
            optional_text_nodes "CdgVendedor" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RUTEmisor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_RznSoc
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_GiroEmis
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Telefono
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CorreoEmisor
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Acteco
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_Sucursal
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgSIISucur
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_DirOrigen
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CmnaOrigen
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CiudadOrigen
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_Emisor_field_CdgVendedor
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado_emisor)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TipoDTE => [ text_node "TipoDTE" v ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_Folio =>
            [ xml.element "Folio" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchEmis =>
            [ xml.element "FchEmis" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_IndServicio =>
            optional_text_nodes "IndServicio" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntBruto => optional_text_nodes "MntBruto" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FmaPago => optional_text_nodes "FmaPago" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchCancel =>
            xml.option_list (optionm.map (fun v => xml.element "FchCancel" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntCancel =>
            xml.option_list (optionm.map (fun v => xml.element "MntCancel" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_SaldoInsol =>
            xml.option_list (optionm.map (fun v => xml.element "SaldoInsol" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntPagos =>
            listm.map
                fun v => x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_to_xml "MntPagos" v
                v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoDesde =>
            xml.option_list (optionm.map (fun v => xml.element "PeriodoDesde" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoHasta =>
            xml.option_list (optionm.map (fun v => xml.element "PeriodoHasta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MedioPago => optional_text_nodes "MedioPago" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TpoCtaPago =>
            optional_text_nodes "TpoCtaPago" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_NumCtaPago =>
            optional_text_nodes "NumCtaPago" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_BcoPago => optional_text_nodes "BcoPago" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoCdg =>
            optional_text_nodes "TermPagoCdg" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoGlosa =>
            optional_text_nodes "TermPagoGlosa" v
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoDias =>
            xml.option_list (optionm.map (fun v => xml.element "TermPagoDias" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchVenc =>
            xml.option_list (optionm.map (fun v => xml.element "FchVenc" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TipoDTE
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_Folio
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchEmis
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_IndServicio
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntBruto
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FmaPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchCancel
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntCancel
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_SaldoInsol
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MntPagos
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoDesde
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_PeriodoHasta
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_MedioPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TpoCtaPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_NumCtaPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_BcoPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoCdg
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoGlosa
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_TermPagoDias
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_field_FchVenc
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_children x)

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_children
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_FchPago =>
            [ xml.element "FchPago" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_MntPago =>
            [ xml.element "MntPago" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_GlosaPagos =>
            optional_text_nodes "GlosaPagos" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_FchPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_MntPago
        exists DTEDefType_choice_Documento_Liquidacion_Encabezado_IdDoc_MntPagos_field_GlosaPagos
    ]

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_attributes
    (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_attributes x)
        (x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_children x)

and let x_dte_def_type_choice_documento_documento_children
    (x : x_dte_def_type_choice_documento_documento)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_field_Encabezado =>
            [ x_dte_def_type_choice_documento_documento_encabezado_to_xml "Encabezado" v ]
        | DTEDefType_choice_Documento_Documento_field_Detalle =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_detalle_to_xml "Detalle" v) v
        | DTEDefType_choice_Documento_Documento_field_SubTotInfo =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_sub_tot_info_to_xml "SubTotInfo" v) v
        | DTEDefType_choice_Documento_Documento_field_DscRcgGlobal =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_dsc_rcg_global_to_xml "DscRcgGlobal" v) v
        | DTEDefType_choice_Documento_Documento_field_Referencia =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_referencia_to_xml "Referencia" v) v
        | DTEDefType_choice_Documento_Documento_field_Comisiones =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_comisiones_to_xml "Comisiones" v) v
        | DTEDefType_choice_Documento_Documento_field_TED =>
            [ x_dte_def_type_choice_documento_documento_ted_to_xml "TED" v ]
        | DTEDefType_choice_Documento_Documento_field_TmstFirma => [ xml.element "TmstFirma" [] (xml.text_children v) ]
    inl rec walk (fields : list (exists r. x_dte_def_type_choice_documento_documento_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_field_Encabezado
        exists DTEDefType_choice_Documento_Documento_field_Detalle
        exists DTEDefType_choice_Documento_Documento_field_SubTotInfo
        exists DTEDefType_choice_Documento_Documento_field_DscRcgGlobal
        exists DTEDefType_choice_Documento_Documento_field_Referencia
        exists DTEDefType_choice_Documento_Documento_field_Comisiones
        exists DTEDefType_choice_Documento_Documento_field_TED
        exists DTEDefType_choice_Documento_Documento_field_TmstFirma
    ]

and let x_dte_def_type_choice_documento_documento_attributes
    (x : x_dte_def_type_choice_documento_documento)
    : list (string * string)
    =
    [ ("ID", x.id) ]

and let x_dte_def_type_choice_documento_documento_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_attributes x)
        (x_dte_def_type_choice_documento_documento_children x)

and let x_dte_def_type_choice_documento_documento_ted_children
    (x : x_dte_def_type_choice_documento_documento_ted)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_ted_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_TED_field_DD =>
            [ x_dte_def_type_choice_documento_documento_ted_dd_to_xml "DD" v ]
        | DTEDefType_choice_Documento_Documento_TED_field_FRMT =>
            [ x_dte_def_type_choice_documento_documento_ted_frmt_to_xml "FRMT" v ]
    inl rec walk (fields : list (exists r. x_dte_def_type_choice_documento_documento_ted_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_ted_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_TED_field_DD
        exists DTEDefType_choice_Documento_Documento_TED_field_FRMT
    ]

and let x_dte_def_type_choice_documento_documento_ted_attributes
    (x : x_dte_def_type_choice_documento_documento_ted)
    : list (string * string)
    =
    [ ("version", x.version) ]

and let x_dte_def_type_choice_documento_documento_ted_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_children x)

and let x_dte_def_type_choice_documento_documento_ted_frmt_children
    (x : x_dte_def_type_choice_documento_documento_ted_frmt)
    : list xml.xml_node
    =
    xml.text_children x.base

and let x_dte_def_type_choice_documento_documento_ted_frmt_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_frmt)
    : list (string * string)
    =
    [ ("algoritmo", to_text x.algoritmo) ]

and let x_dte_def_type_choice_documento_documento_ted_frmt_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_frmt)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_frmt_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_frmt_children x)

and let x_dte_def_type_choice_documento_documento_ted_dd_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_ted_dd_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_TED_DD_field_RE => [ text_node "RE" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_TD => [ text_node "TD" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_F => [ xml.element "F" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_FE => [ xml.element "FE" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_RR => [ text_node "RR" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_RSR => [ text_node "RSR" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_MNT => [ xml.element "MNT" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_IT1 => [ text_node "IT1" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_CAF =>
            [ x_dte_def_type_choice_documento_documento_ted_dd_caf_to_xml "CAF" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_field_TSTED => [ xml.element "TSTED" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_ted_dd_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_RE
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_TD
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_F
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_FE
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_RR
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_RSR
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_MNT
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_IT1
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_CAF
        exists DTEDefType_choice_Documento_Documento_TED_DD_field_TSTED
    ]

and let x_dte_def_type_choice_documento_documento_ted_dd_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_dd)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_ted_dd_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_dd)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_dd_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_dd_children x)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_DA =>
            [ x_dte_def_type_choice_documento_documento_ted_dd_caf_da_to_xml "DA" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_FRMA =>
            [ x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_to_xml "FRMA" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_ted_dd_caf_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_DA
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_field_FRMA
    ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf)
    : list (string * string)
    =
    [ ("version", x.version) ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_dd_caf)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_children x)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_frma)
    : list xml.xml_node
    =
    xml.text_children x.base

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_frma)
    : list (string * string)
    =
    [ ("algoritmo", x.algoritmo) ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_frma)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_children x)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RE => [ text_node "RE" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RS => [ text_node "RS" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_TD => [ text_node "TD" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RNG =>
            [ x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_to_xml "RNG" v ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_FA => [ xml.element "FA" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_choice_rsapk =>
            x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_children v
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_IDK => [ text_node "IDK" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RE
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RS
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_TD
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_RNG
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_FA
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_choice_rsapk
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_field_IDK
    ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_children x)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk)
    : list xml.xml_node
    =
    match x with
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK v =>
        [ x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_to_xml "RSAPK" v ]
    | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK v =>
        [ x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_to_xml "DSAPK" v ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P =>
            [ xml.element "P" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q =>
            [ xml.element "Q" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G =>
            [ xml.element "G" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y =>
            [ xml.element "Y" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_P
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Q
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_G
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_DSAPK_field_Y
    ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_children x)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M =>
            [ xml.element "M" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E =>
            [ xml.element "E" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_M
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK_field_E
    ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_children x)

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_children
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_D =>
            [ xml.element "D" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_H =>
            [ xml.element "H" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_D
        exists DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_RNG_field_H
    ]

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_attributes
    (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_attributes x)
        (x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_children x)

and let x_dte_def_type_choice_documento_documento_comisiones_children
    (x : x_dte_def_type_choice_documento_documento_comisiones)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_comisiones_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_Comisiones_field_NroLinCom =>
            [ xml.element "NroLinCom" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Comisiones_field_TipoMovim => [ text_node "TipoMovim" v ]
        | DTEDefType_choice_Documento_Documento_Comisiones_field_Glosa => [ text_node "Glosa" v ]
        | DTEDefType_choice_Documento_Documento_Comisiones_field_TasaComision =>
            xml.option_list (optionm.map (fun v => xml.element "TasaComision" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComNeto =>
            [ xml.element "ValComNeto" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComExe =>
            [ xml.element "ValComExe" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Comisiones_field_ValComIVA =>
            xml.option_list (optionm.map (fun v => xml.element "ValComIVA" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_comisiones_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_comisiones_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_NroLinCom
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_TipoMovim
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_Glosa
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_TasaComision
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Documento_Comisiones_field_ValComIVA
    ]

and let x_dte_def_type_choice_documento_documento_comisiones_attributes
    (x : x_dte_def_type_choice_documento_documento_comisiones)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_comisiones_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_comisiones)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_comisiones_attributes x)
        (x_dte_def_type_choice_documento_documento_comisiones_children x)

and let x_dte_def_type_choice_documento_documento_referencia_children
    (x : x_dte_def_type_choice_documento_documento_referencia)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_referencia_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_Referencia_field_NroLinRef =>
            [ xml.element "NroLinRef" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Referencia_field_TpoDocRef => [ text_node "TpoDocRef" v ]
        | DTEDefType_choice_Documento_Documento_Referencia_field_IndGlobal => optional_text_nodes "IndGlobal" v
        | DTEDefType_choice_Documento_Documento_Referencia_field_FolioRef => [ text_node "FolioRef" v ]
        | DTEDefType_choice_Documento_Documento_Referencia_field_RUTOtr => optional_text_nodes "RUTOtr" v
        | DTEDefType_choice_Documento_Documento_Referencia_field_FchRef =>
            [ xml.element "FchRef" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Referencia_field_CodRef => optional_text_nodes "CodRef" v
        | DTEDefType_choice_Documento_Documento_Referencia_field_RazonRef => optional_text_nodes "RazonRef" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_referencia_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_referencia_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Referencia_field_NroLinRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_TpoDocRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_IndGlobal
        exists DTEDefType_choice_Documento_Documento_Referencia_field_FolioRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_RUTOtr
        exists DTEDefType_choice_Documento_Documento_Referencia_field_FchRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_CodRef
        exists DTEDefType_choice_Documento_Documento_Referencia_field_RazonRef
    ]

and let x_dte_def_type_choice_documento_documento_referencia_attributes
    (x : x_dte_def_type_choice_documento_documento_referencia)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_referencia_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_referencia)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_referencia_attributes x)
        (x_dte_def_type_choice_documento_documento_referencia_children x)

and let x_dte_def_type_choice_documento_documento_dsc_rcg_global_children
    (x : x_dte_def_type_choice_documento_documento_dsc_rcg_global)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_dsc_rcg_global_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_NroLinDR => [ text_node "NroLinDR" v ]
        | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoMov => [ text_node "TpoMov" v ]
        | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_GlosaDR => optional_text_nodes "GlosaDR" v
        | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoValor => [ text_node "TpoValor" v ]
        | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDR =>
            [ xml.element "ValorDR" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDROtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "ValorDROtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_IndExeDR => optional_text_nodes "IndExeDR" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_dsc_rcg_global_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_dsc_rcg_global_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_NroLinDR
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoMov
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_GlosaDR
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_TpoValor
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDR
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_ValorDROtrMnda
        exists DTEDefType_choice_Documento_Documento_DscRcgGlobal_field_IndExeDR
    ]

and let x_dte_def_type_choice_documento_documento_dsc_rcg_global_attributes
    (x : x_dte_def_type_choice_documento_documento_dsc_rcg_global)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_dsc_rcg_global_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_dsc_rcg_global)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_dsc_rcg_global_attributes x)
        (x_dte_def_type_choice_documento_documento_dsc_rcg_global_children x)

and let x_dte_def_type_choice_documento_documento_sub_tot_info_children
    (x : x_dte_def_type_choice_documento_documento_sub_tot_info)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_sub_tot_info_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_NroSTI =>
            [ xml.element "NroSTI" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_GlosaSTI => [ text_node "GlosaSTI" v ]
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_OrdenSTI =>
            xml.option_list (optionm.map (fun v => xml.element "OrdenSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotNetoSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotNetoSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotIVASTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotIVASTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotAdicSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotAdicSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotExeSTI =>
            xml.option_list (optionm.map (fun v => xml.element "SubTotExeSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_ValSubtotSTI =>
            xml.option_list (optionm.map (fun v => xml.element "ValSubtotSTI" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_SubTotInfo_field_LineasDeta =>
            listm.map (fun v => xml.element "LineasDeta" [] (xml.text_children v)) v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_sub_tot_info_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_sub_tot_info_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_NroSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_GlosaSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_OrdenSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotNetoSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotIVASTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotAdicSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_SubTotExeSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_ValSubtotSTI
        exists DTEDefType_choice_Documento_Documento_SubTotInfo_field_LineasDeta
    ]

and let x_dte_def_type_choice_documento_documento_sub_tot_info_attributes
    (x : x_dte_def_type_choice_documento_documento_sub_tot_info)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_sub_tot_info_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_sub_tot_info)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_sub_tot_info_attributes x)
        (x_dte_def_type_choice_documento_documento_sub_tot_info_children x)

and let x_dte_def_type_choice_documento_documento_detalle_children
    (x : x_dte_def_type_choice_documento_documento_detalle)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_detalle_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_Detalle_field_NroLinDet =>
            [ xml.element "NroLinDet" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Detalle_field_CdgItem =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_detalle_cdg_item_to_xml "CdgItem" v) v
        | DTEDefType_choice_Documento_Documento_Detalle_field_IndExe => optional_text_nodes "IndExe" v
        | DTEDefType_choice_Documento_Documento_Detalle_field_Retenedor =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_documento_detalle_retenedor_to_xml "Retenedor" v
                    v
            )
        | DTEDefType_choice_Documento_Documento_Detalle_field_NmbItem => [ text_node "NmbItem" v ]
        | DTEDefType_choice_Documento_Documento_Detalle_field_DscItem => optional_text_nodes "DscItem" v
        | DTEDefType_choice_Documento_Documento_Detalle_field_QtyRef =>
            xml.option_list (optionm.map (fun v => xml.element "QtyRef" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdRef => optional_text_nodes "UnmdRef" v
        | DTEDefType_choice_Documento_Documento_Detalle_field_PrcRef =>
            xml.option_list (optionm.map (fun v => xml.element "PrcRef" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_QtyItem =>
            xml.option_list (optionm.map (fun v => xml.element "QtyItem" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_Subcantidad =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_detalle_subcantidad_to_xml "Subcantidad" v) v
        | DTEDefType_choice_Documento_Documento_Detalle_field_FchElabor =>
            xml.option_list (optionm.map (fun v => xml.element "FchElabor" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_FchVencim =>
            xml.option_list (optionm.map (fun v => xml.element "FchVencim" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_UnmdItem => optional_text_nodes "UnmdItem" v
        | DTEDefType_choice_Documento_Documento_Detalle_field_PrcItem =>
            xml.option_list (optionm.map (fun v => xml.element "PrcItem" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_OtrMnda =>
            xml.option_list (
                optionm.map (fun v => x_dte_def_type_choice_documento_documento_detalle_otr_mnda_to_xml "OtrMnda" v) v
            )
        | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoPct =>
            xml.option_list (optionm.map (fun v => xml.element "DescuentoPct" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoMonto =>
            xml.option_list (optionm.map (fun v => xml.element "DescuentoMonto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_SubDscto =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_detalle_sub_dscto_to_xml "SubDscto" v) v
        | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoPct =>
            xml.option_list (optionm.map (fun v => xml.element "RecargoPct" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_RecargoMonto =>
            xml.option_list (optionm.map (fun v => xml.element "RecargoMonto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_field_SubRecargo =>
            listm.map (fun v => x_dte_def_type_choice_documento_documento_detalle_sub_recargo_to_xml "SubRecargo" v) v
        | DTEDefType_choice_Documento_Documento_Detalle_field_CodImpAdic => repeated_text_nodes "CodImpAdic" v
        | DTEDefType_choice_Documento_Documento_Detalle_field_MontoItem =>
            [ xml.element "MontoItem" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_detalle_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_detalle_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Detalle_field_NroLinDet
        exists DTEDefType_choice_Documento_Documento_Detalle_field_CdgItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_IndExe
        exists DTEDefType_choice_Documento_Documento_Detalle_field_Retenedor
        exists DTEDefType_choice_Documento_Documento_Detalle_field_NmbItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_DscItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_QtyRef
        exists DTEDefType_choice_Documento_Documento_Detalle_field_UnmdRef
        exists DTEDefType_choice_Documento_Documento_Detalle_field_PrcRef
        exists DTEDefType_choice_Documento_Documento_Detalle_field_QtyItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_Subcantidad
        exists DTEDefType_choice_Documento_Documento_Detalle_field_FchElabor
        exists DTEDefType_choice_Documento_Documento_Detalle_field_FchVencim
        exists DTEDefType_choice_Documento_Documento_Detalle_field_UnmdItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_PrcItem
        exists DTEDefType_choice_Documento_Documento_Detalle_field_OtrMnda
        exists DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoPct
        exists DTEDefType_choice_Documento_Documento_Detalle_field_DescuentoMonto
        exists DTEDefType_choice_Documento_Documento_Detalle_field_SubDscto
        exists DTEDefType_choice_Documento_Documento_Detalle_field_RecargoPct
        exists DTEDefType_choice_Documento_Documento_Detalle_field_RecargoMonto
        exists DTEDefType_choice_Documento_Documento_Detalle_field_SubRecargo
        exists DTEDefType_choice_Documento_Documento_Detalle_field_CodImpAdic
        exists DTEDefType_choice_Documento_Documento_Detalle_field_MontoItem
    ]

and let x_dte_def_type_choice_documento_documento_detalle_attributes
    (x : x_dte_def_type_choice_documento_documento_detalle)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_detalle_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_detalle)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_detalle_attributes x)
        (x_dte_def_type_choice_documento_documento_detalle_children x)

and let x_dte_def_type_choice_documento_documento_detalle_sub_recargo_children
    (x : x_dte_def_type_choice_documento_documento_detalle_sub_recargo)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_TipoRecargo => [ text_node "TipoRecargo" v ]
        | DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_ValorRecargo =>
            [ xml.element "ValorRecargo" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_detalle_sub_recargo_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_detalle_sub_recargo_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_TipoRecargo
        exists DTEDefType_choice_Documento_Documento_Detalle_SubRecargo_field_ValorRecargo
    ]

and let x_dte_def_type_choice_documento_documento_detalle_sub_recargo_attributes
    (x : x_dte_def_type_choice_documento_documento_detalle_sub_recargo)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_detalle_sub_recargo_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_detalle_sub_recargo)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_detalle_sub_recargo_attributes x)
        (x_dte_def_type_choice_documento_documento_detalle_sub_recargo_children x)

and let x_dte_def_type_choice_documento_documento_detalle_sub_dscto_children
    (x : x_dte_def_type_choice_documento_documento_detalle_sub_dscto)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_TipoDscto => [ text_node "TipoDscto" v ]
        | DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_ValorDscto =>
            [ xml.element "ValorDscto" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_detalle_sub_dscto_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_detalle_sub_dscto_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_TipoDscto
        exists DTEDefType_choice_Documento_Documento_Detalle_SubDscto_field_ValorDscto
    ]

and let x_dte_def_type_choice_documento_documento_detalle_sub_dscto_attributes
    (x : x_dte_def_type_choice_documento_documento_detalle_sub_dscto)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_detalle_sub_dscto_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_detalle_sub_dscto)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_detalle_sub_dscto_attributes x)
        (x_dte_def_type_choice_documento_documento_detalle_sub_dscto_children x)

and let x_dte_def_type_choice_documento_documento_detalle_otr_mnda_children
    (x : x_dte_def_type_choice_documento_documento_detalle_otr_mnda)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_PrcOtrMon =>
            [ xml.element "PrcOtrMon" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_Moneda => [ text_node "Moneda" v ]
        | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_FctConv =>
            xml.option_list (optionm.map (fun v => xml.element "FctConv" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_DctoOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "DctoOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_RecargoOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "RecargoOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_MontoItemOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "MontoItemOtrMnda" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_detalle_otr_mnda_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_detalle_otr_mnda_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_PrcOtrMon
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_Moneda
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_FctConv
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_DctoOtrMnda
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_RecargoOtrMnda
        exists DTEDefType_choice_Documento_Documento_Detalle_OtrMnda_field_MontoItemOtrMnda
    ]

and let x_dte_def_type_choice_documento_documento_detalle_otr_mnda_attributes
    (x : x_dte_def_type_choice_documento_documento_detalle_otr_mnda)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_detalle_otr_mnda_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_detalle_otr_mnda)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_detalle_otr_mnda_attributes x)
        (x_dte_def_type_choice_documento_documento_detalle_otr_mnda_children x)

and let x_dte_def_type_choice_documento_documento_detalle_subcantidad_children
    (x : x_dte_def_type_choice_documento_documento_detalle_subcantidad)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_detalle_subcantidad_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubQty =>
            [ xml.element "SubQty" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubCod => [ text_node "SubCod" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_detalle_subcantidad_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_detalle_subcantidad_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubQty
        exists DTEDefType_choice_Documento_Documento_Detalle_Subcantidad_field_SubCod
    ]

and let x_dte_def_type_choice_documento_documento_detalle_subcantidad_attributes
    (x : x_dte_def_type_choice_documento_documento_detalle_subcantidad)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_detalle_subcantidad_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_detalle_subcantidad)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_detalle_subcantidad_attributes x)
        (x_dte_def_type_choice_documento_documento_detalle_subcantidad_children x)

and let x_dte_def_type_choice_documento_documento_detalle_retenedor_children
    (x : x_dte_def_type_choice_documento_documento_detalle_retenedor)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_detalle_retenedor_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_IndAgente => [ text_node "IndAgente" v ]
        | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntBaseFaena =>
            xml.option_list (optionm.map (fun v => xml.element "MntBaseFaena" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntMargComer =>
            xml.option_list (optionm.map (fun v => xml.element "MntMargComer" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_PrcConsFinal =>
            xml.option_list (optionm.map (fun v => xml.element "PrcConsFinal" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_detalle_retenedor_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_detalle_retenedor_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_IndAgente
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntBaseFaena
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_MntMargComer
        exists DTEDefType_choice_Documento_Documento_Detalle_Retenedor_field_PrcConsFinal
    ]

and let x_dte_def_type_choice_documento_documento_detalle_retenedor_attributes
    (x : x_dte_def_type_choice_documento_documento_detalle_retenedor)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_detalle_retenedor_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_detalle_retenedor)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_detalle_retenedor_attributes x)
        (x_dte_def_type_choice_documento_documento_detalle_retenedor_children x)

and let x_dte_def_type_choice_documento_documento_detalle_cdg_item_children
    (x : x_dte_def_type_choice_documento_documento_detalle_cdg_item)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_detalle_cdg_item_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_TpoCodigo => [ text_node "TpoCodigo" v ]
        | DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_VlrCodigo => [ text_node "VlrCodigo" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_detalle_cdg_item_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_detalle_cdg_item_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_TpoCodigo
        exists DTEDefType_choice_Documento_Documento_Detalle_CdgItem_field_VlrCodigo
    ]

and let x_dte_def_type_choice_documento_documento_detalle_cdg_item_attributes
    (x : x_dte_def_type_choice_documento_documento_detalle_cdg_item)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_detalle_cdg_item_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_detalle_cdg_item)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_detalle_cdg_item_attributes x)
        (x_dte_def_type_choice_documento_documento_detalle_cdg_item_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_children
    (x : x_dte_def_type_choice_documento_documento_encabezado)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_dte_def_type_choice_documento_documento_encabezado_field r) (v : r) : list xml.xml_node =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_field_IdDoc =>
            [ x_dte_def_type_choice_documento_documento_encabezado_id_doc_to_xml "IdDoc" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_field_Emisor =>
            [ x_dte_def_type_choice_documento_documento_encabezado_emisor_to_xml "Emisor" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTMandante => optional_text_nodes "RUTMandante" v
        | DTEDefType_choice_Documento_Documento_Encabezado_field_Receptor =>
            [ x_dte_def_type_choice_documento_documento_encabezado_receptor_to_xml "Receptor" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_field_RUTSolicita => optional_text_nodes "RUTSolicita" v
        | DTEDefType_choice_Documento_Documento_Encabezado_field_Transporte =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_documento_encabezado_transporte_to_xml "Transporte" v
                    v
            )
        | DTEDefType_choice_Documento_Documento_Encabezado_field_Totales =>
            [ x_dte_def_type_choice_documento_documento_encabezado_totales_to_xml "Totales" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_field_OtraMoneda =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_to_xml "OtraMoneda" v
                    v
            )
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_encabezado_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_IdDoc
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Emisor
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_RUTMandante
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Receptor
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_RUTSolicita
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Transporte
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_Totales
        exists DTEDefType_choice_Documento_Documento_Encabezado_field_OtraMoneda
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoMoneda => [ text_node "TpoMoneda" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoCambio =>
            xml.option_list (optionm.map (fun v => xml.element "TpoCambio" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntNetoOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "MntNetoOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntExeOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "MntExeOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntFaeCarneOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "MntFaeCarneOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntMargComOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "MntMargComOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVAOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "IVAOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_ImpRetOtrMnda =>
            listm.map
                fun v =>
                    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_to_xml
                        "ImpRetOtrMnda"
                        v
                v
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVANoRetOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "IVANoRetOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntTotOtrMnda =>
            [ xml.element "MntTotOtrMnda" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoMoneda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_TpoCambio
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntNetoOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntExeOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntFaeCarneOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntMargComOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVAOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_ImpRetOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_IVANoRetOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_field_MntTotOtrMnda
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TipoImpOtrMnda =>
            [ text_node "TipoImpOtrMnda" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TasaImpOtrMnda =>
            xml.option_list (optionm.map (fun v => xml.element "TasaImpOtrMnda" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_VlrImpOtrMnda =>
            [ xml.element "VlrImpOtrMnda" [] (xml.text_children v) ]
    inl rec walk
        (fields :
            list (exists r. x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TipoImpOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_TasaImpOtrMnda
        exists DTEDefType_choice_Documento_Documento_Encabezado_OtraMoneda_ImpRetOtrMnda_field_VlrImpOtrMnda
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_totales_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_totales_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntNeto =>
            xml.option_list (optionm.map (fun v => xml.element "MntNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntExe =>
            xml.option_list (optionm.map (fun v => xml.element "MntExe" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntBase =>
            xml.option_list (optionm.map (fun v => xml.element "MntBase" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntMargenCom =>
            xml.option_list (optionm.map (fun v => xml.element "MntMargenCom" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_TasaIVA =>
            xml.option_list (optionm.map (fun v => xml.element "TasaIVA" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVA =>
            xml.option_list (optionm.map (fun v => xml.element "IVA" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVAProp =>
            xml.option_list (optionm.map (fun v => xml.element "IVAProp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVATerc =>
            xml.option_list (optionm.map (fun v => xml.element "IVATerc" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_ImptoReten =>
            listm.map
                fun v => x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_to_xml "ImptoReten" v
                v
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVANoRet =>
            xml.option_list (optionm.map (fun v => xml.element "IVANoRet" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_CredEC =>
            xml.option_list (optionm.map (fun v => xml.element "CredEC" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_GrntDep =>
            xml.option_list (optionm.map (fun v => xml.element "GrntDep" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_Comisiones =>
            xml.option_list (
                optionm.map
                    fun v =>
                        x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_to_xml "Comisiones" v
                    v
            )
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntTotal =>
            [ xml.element "MntTotal" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoNF =>
            xml.option_list (optionm.map (fun v => xml.element "MontoNF" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoPeriodo =>
            xml.option_list (optionm.map (fun v => xml.element "MontoPeriodo" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_SaldoAnterior =>
            xml.option_list (optionm.map (fun v => xml.element "SaldoAnterior" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_VlrPagar =>
            xml.option_list (optionm.map (fun v => xml.element "VlrPagar" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_totales_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_encabezado_totales_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntExe
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntBase
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntMargenCom
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_TasaIVA
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVA
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVAProp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVATerc
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_ImptoReten
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_IVANoRet
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_CredEC
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_GrntDep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_Comisiones
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MntTotal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoNF
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_MontoPeriodo
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_SaldoAnterior
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_field_VlrPagar
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_totales_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_totales_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_totales)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_totales_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_totales_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComNeto =>
            xml.option_list (optionm.map (fun v => xml.element "ValComNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComExe =>
            xml.option_list (optionm.map (fun v => xml.element "ValComExe" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComIVA =>
            xml.option_list (optionm.map (fun v => xml.element "ValComIVA" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComExe
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_Comisiones_field_ValComIVA
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TipoImp => [ text_node "TipoImp" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TasaImp =>
            xml.option_list (optionm.map (fun v => xml.element "TasaImp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_MontoImp =>
            [ xml.element "MontoImp" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TipoImp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_TasaImp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Totales_ImptoReten_field_MontoImp
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Patente => optional_text_nodes "Patente" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_RUTTrans => optional_text_nodes "RUTTrans" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Chofer =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_to_xml "Chofer" v
                    v
            )
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_DirDest => optional_text_nodes "DirDest" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CmnaDest => optional_text_nodes "CmnaDest" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CiudadDest =>
            optional_text_nodes "CiudadDest" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Aduana =>
            xml.option_list (
                optionm.map
                    fun v => x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_to_xml "Aduana" v
                    v
            )
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_encabezado_transporte_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Patente
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_RUTTrans
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Chofer
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_DirDest
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CmnaDest
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_CiudadDest
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_field_Aduana
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_transporte)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodModVenta =>
            xml.option_list (optionm.map (fun v => xml.element "CodModVenta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodClauVenta =>
            xml.option_list (optionm.map (fun v => xml.element "CodClauVenta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotClauVenta =>
            xml.option_list (optionm.map (fun v => xml.element "TotClauVenta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodViaTransp =>
            xml.option_list (optionm.map (fun v => xml.element "CodViaTransp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NombreTransp =>
            optional_text_nodes "NombreTransp" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_RUTCiaTransp =>
            optional_text_nodes "RUTCiaTransp" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NomCiaTransp =>
            optional_text_nodes "NomCiaTransp" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicTransp =>
            optional_text_nodes "IdAdicTransp" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Booking =>
            optional_text_nodes "Booking" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Operador =>
            optional_text_nodes "Operador" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoEmbarque =>
            xml.option_list (optionm.map (fun v => xml.element "CodPtoEmbarque" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb =>
            optional_text_nodes "IdAdicPtoEmb" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoDesemb =>
            xml.option_list (optionm.map (fun v => xml.element "CodPtoDesemb" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb =>
            optional_text_nodes "IdAdicPtoDesemb" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Tara =>
            xml.option_list (optionm.map (fun v => xml.element "Tara" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidMedTara =>
            xml.option_list (optionm.map (fun v => xml.element "CodUnidMedTara" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoBruto =>
            xml.option_list (optionm.map (fun v => xml.element "PesoBruto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto =>
            xml.option_list (optionm.map (fun v => xml.element "CodUnidPesoBruto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoNeto =>
            xml.option_list (optionm.map (fun v => xml.element "PesoNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto =>
            xml.option_list (optionm.map (fun v => xml.element "CodUnidPesoNeto" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotItems =>
            xml.option_list (optionm.map (fun v => xml.element "TotItems" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotBultos =>
            xml.option_list (optionm.map (fun v => xml.element "TotBultos" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TipoBultos =>
            listm.map
                fun v =>
                    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_to_xml
                        "TipoBultos"
                        v
                v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntFlete =>
            xml.option_list (optionm.map (fun v => xml.element "MntFlete" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntSeguro =>
            xml.option_list (optionm.map (fun v => xml.element "MntSeguro" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisRecep =>
            xml.option_list (optionm.map (fun v => xml.element "CodPaisRecep" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisDestin =>
            xml.option_list (optionm.map (fun v => xml.element "CodPaisDestin" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodModVenta
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodClauVenta
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotClauVenta
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodViaTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NombreTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_RUTCiaTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_NomCiaTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicTransp
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Booking
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Operador
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoEmbarque
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoEmb
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPtoDesemb
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_IdAdicPtoDesemb
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_Tara
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidMedTara
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoBruto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoBruto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_PesoNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodUnidPesoNeto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotItems
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TotBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_TipoBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntFlete
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_MntSeguro
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_field_CodPaisDestin
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos =>
            xml.option_list (optionm.map (fun v => xml.element "CodTpoBultos" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos =>
            xml.option_list (optionm.map (fun v => xml.element "CantBultos" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas =>
            optional_text_nodes "Marcas" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer =>
            optional_text_nodes "IdContainer" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Sello =>
            optional_text_nodes "Sello" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello =>
            optional_text_nodes "EmisorSello" v
    inl rec walk
        (fields :
            list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CodTpoBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_CantBultos
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Marcas
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_IdContainer
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_Sello
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Aduana_TipoBultos_field_EmisorSello
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_RUTChofer =>
            [ text_node "RUTChofer" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_NombreChofer =>
            [ text_node "NombreChofer" v ]
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_RUTChofer
        exists DTEDefType_choice_Documento_Documento_Encabezado_Transporte_Chofer_field_NombreChofer
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_receptor_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RUTRecep => [ text_node "RUTRecep" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CdgIntRecep =>
            optional_text_nodes "CdgIntRecep" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RznSocRecep => [ text_node "RznSocRecep" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Extranjero =>
            xml.option_list (
                optionm.map
                    fun v =>
                        x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_to_xml "Extranjero" v
                    v
            )
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_GiroRecep => optional_text_nodes "GiroRecep" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Contacto => optional_text_nodes "Contacto" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CorreoRecep =>
            optional_text_nodes "CorreoRecep" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirRecep => optional_text_nodes "DirRecep" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaRecep => optional_text_nodes "CmnaRecep" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadRecep =>
            optional_text_nodes "CiudadRecep" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirPostal => optional_text_nodes "DirPostal" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaPostal =>
            optional_text_nodes "CmnaPostal" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadPostal =>
            optional_text_nodes "CiudadPostal" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_receptor_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_encabezado_receptor_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RUTRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CdgIntRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_RznSocRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Extranjero
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_GiroRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_Contacto
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CorreoRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadRecep
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_DirPostal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CmnaPostal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_field_CiudadPostal
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_receptor)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_receptor_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_receptor_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_NumId =>
            optional_text_nodes "NumId" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_Nacionalidad =>
            optional_text_nodes "Nacionalidad" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_NumId
        exists DTEDefType_choice_Documento_Documento_Encabezado_Receptor_Extranjero_field_Nacionalidad
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_emisor_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RUTEmisor => [ text_node "RUTEmisor" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RznSoc =>
            [ xml.element "RznSoc" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GiroEmis => [ text_node "GiroEmis" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Telefono => repeated_text_nodes "Telefono" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CorreoEmisor =>
            optional_text_nodes "CorreoEmisor" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Acteco =>
            listm.map (fun v => xml.element "Acteco" [] (xml.text_children v)) v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GuiaExport =>
            xml.option_list (
                optionm.map
                    fun v =>
                        x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_to_xml "GuiaExport" v
                    v
            )
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Sucursal => optional_text_nodes "Sucursal" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgSIISucur =>
            xml.option_list (optionm.map (fun v => xml.element "CdgSIISucur" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_DirOrigen => optional_text_nodes "DirOrigen" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CmnaOrigen => optional_text_nodes "CmnaOrigen" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CiudadOrigen =>
            optional_text_nodes "CiudadOrigen" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgVendedor =>
            optional_text_nodes "CdgVendedor" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_IdAdicEmisor =>
            optional_text_nodes "IdAdicEmisor" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_emisor_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_encabezado_emisor_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RUTEmisor
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_RznSoc
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GiroEmis
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Telefono
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CorreoEmisor
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Acteco
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_GuiaExport
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_Sucursal
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgSIISucur
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_DirOrigen
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CmnaOrigen
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CiudadOrigen
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_CdgVendedor
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_field_IdAdicEmisor
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_emisor)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_emisor_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_emisor_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_CdgTraslado =>
            optional_text_nodes "CdgTraslado" v
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FolioAut =>
            xml.option_list (optionm.map (fun v => xml.element "FolioAut" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FchAut =>
            xml.option_list (optionm.map (fun v => xml.element "FchAut" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_CdgTraslado
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FolioAut
        exists DTEDefType_choice_Documento_Documento_Encabezado_Emisor_GuiaExport_field_FchAut
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_id_doc_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDTE => [ text_node "TipoDTE" v ]
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_Folio =>
            [ xml.element "Folio" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchEmis =>
            [ xml.element "FchEmis" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndNoRebaja =>
            optional_text_nodes "IndNoRebaja" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDespacho =>
            optional_text_nodes "TipoDespacho" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndTraslado =>
            optional_text_nodes "IndTraslado" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoImpresion =>
            optional_text_nodes "TpoImpresion" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndServicio =>
            optional_text_nodes "IndServicio" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntBruto => optional_text_nodes "MntBruto" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPago => optional_text_nodes "FmaPago" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPagExp =>
            xml.option_list (optionm.map (fun v => xml.element "FmaPagExp" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchCancel =>
            xml.option_list (optionm.map (fun v => xml.element "FchCancel" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntCancel =>
            xml.option_list (optionm.map (fun v => xml.element "MntCancel" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_SaldoInsol =>
            xml.option_list (optionm.map (fun v => xml.element "SaldoInsol" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntPagos =>
            listm.map
                fun v => x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_to_xml "MntPagos" v
                v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoDesde =>
            xml.option_list (optionm.map (fun v => xml.element "PeriodoDesde" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoHasta =>
            xml.option_list (optionm.map (fun v => xml.element "PeriodoHasta" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MedioPago => optional_text_nodes "MedioPago" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoCtaPago => optional_text_nodes "TpoCtaPago" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_NumCtaPago => optional_text_nodes "NumCtaPago" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_BcoPago => optional_text_nodes "BcoPago" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoCdg =>
            optional_text_nodes "TermPagoCdg" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoGlosa =>
            optional_text_nodes "TermPagoGlosa" v
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoDias =>
            xml.option_list (optionm.map (fun v => xml.element "TermPagoDias" [] (xml.text_children v)) v)
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchVenc =>
            xml.option_list (optionm.map (fun v => xml.element "FchVenc" [] (xml.text_children v)) v)
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_id_doc_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_dte_def_type_choice_documento_documento_encabezado_id_doc_get f x)) (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDTE
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_Folio
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchEmis
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndNoRebaja
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TipoDespacho
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndTraslado
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoImpresion
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_IndServicio
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntBruto
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FmaPagExp
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchCancel
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntCancel
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_SaldoInsol
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MntPagos
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoDesde
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_PeriodoHasta
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_MedioPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TpoCtaPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_NumCtaPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_BcoPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoCdg
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoGlosa
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_TermPagoDias
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_field_FchVenc
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_id_doc_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_id_doc_children x)

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_children
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_FchPago =>
            [ xml.element "FchPago" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_MntPago =>
            [ xml.element "MntPago" [] (xml.text_children v) ]
        | DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_GlosaPagos =>
            optional_text_nodes "GlosaPagos" v
    inl rec walk
        (fields : list (exists r. x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_get f x))
                (walk rest)
        | [] => []
    walk [
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_FchPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_MntPago
        exists DTEDefType_choice_Documento_Documento_Encabezado_IdDoc_MntPagos_field_GlosaPagos
    ]

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_attributes
    (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos)
    : list (string * string)
    =
    []

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_to_xml
    (name : string) (x : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos)
    : xml.xml_node
    =
    xml.element name
        (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_attributes x)
        (x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_children x)

and let x_signature_type_children (x : x_signature_type) : list xml.xml_node =
    inl nodes forall r. (f : x_signature_type_field r) (v : r) : list xml.xml_node =
        match f with
        | SignatureType_field_SignedInfo => [ x_signature_type_signed_info_to_xml "SignedInfo" v ]
        | SignatureType_field_SignatureValue => [ xml.element "SignatureValue" [] (xml.text_children v) ]
        | SignatureType_field_KeyInfo => [ x_signature_type_key_info_to_xml "KeyInfo" v ]
    inl rec walk (fields : list (exists r. x_signature_type_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_signature_type_get f x)) (walk rest)
        | [] => []
    walk [
        exists SignatureType_field_SignedInfo
        exists SignatureType_field_SignatureValue
        exists SignatureType_field_KeyInfo
    ]

and let x_signature_type_attributes (x : x_signature_type) : list (string * string) = []

and let x_signature_type_to_xml (name : string) (x : x_signature_type) : xml.xml_node =
    xml.element name (x_signature_type_attributes x) (x_signature_type_children x)

and let x_signature_type_key_info_children (x : x_signature_type_key_info) : list xml.xml_node =
    inl nodes forall r. (f : x_signature_type_key_info_field r) (v : r) : list xml.xml_node =
        match f with
        | SignatureType_KeyInfo_field_KeyValue => [ x_signature_type_key_info_key_value_to_xml "KeyValue" v ]
        | SignatureType_KeyInfo_field_X509Data => [ x_signature_type_key_info_x509_data_to_xml "X509Data" v ]
    inl rec walk (fields : list (exists r. x_signature_type_key_info_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_signature_type_key_info_get f x)) (walk rest)
        | [] => []
    walk [ exists SignatureType_KeyInfo_field_KeyValue; exists SignatureType_KeyInfo_field_X509Data ]

and let x_signature_type_key_info_attributes (x : x_signature_type_key_info) : list (string * string) = []

and let x_signature_type_key_info_to_xml (name : string) (x : x_signature_type_key_info) : xml.xml_node =
    xml.element name (x_signature_type_key_info_attributes x) (x_signature_type_key_info_children x)

and let x_signature_type_key_info_x509_data_children (x : x_signature_type_key_info_x509_data) : list xml.xml_node =
    inl nodes forall r. (f : x_signature_type_key_info_x509_data_field r) (v : r) : list xml.xml_node =
        match f with
        | SignatureType_KeyInfo_X509Data_field_X509Certificate =>
            [ xml.element "X509Certificate" [] (xml.text_children v) ]
    inl rec walk (fields : list (exists r. x_signature_type_key_info_x509_data_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_signature_type_key_info_x509_data_get f x)) (walk rest)
        | [] => []
    walk [ exists SignatureType_KeyInfo_X509Data_field_X509Certificate ]

and let x_signature_type_key_info_x509_data_attributes (x : x_signature_type_key_info_x509_data) : list (string * string) =
    []

and let x_signature_type_key_info_x509_data_to_xml
    (name : string) (x : x_signature_type_key_info_x509_data)
    : xml.xml_node
    =
    xml.element name (x_signature_type_key_info_x509_data_attributes x) (x_signature_type_key_info_x509_data_children x)

and let x_signature_type_key_info_key_value_children (x : x_signature_type_key_info_key_value) : list xml.xml_node =
    inl nodes forall r. (f : x_signature_type_key_info_key_value_field r) (v : r) : list xml.xml_node =
        match f with
        | SignatureType_KeyInfo_KeyValue_field_choice_rsa_key_value =>
            x_signature_type_key_info_key_value_choice_rsa_key_value_children v
    inl rec walk (fields : list (exists r. x_signature_type_key_info_key_value_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_signature_type_key_info_key_value_get f x)) (walk rest)
        | [] => []
    walk [ exists SignatureType_KeyInfo_KeyValue_field_choice_rsa_key_value ]

and let x_signature_type_key_info_key_value_attributes (x : x_signature_type_key_info_key_value) : list (string * string) =
    []

and let x_signature_type_key_info_key_value_to_xml
    (name : string) (x : x_signature_type_key_info_key_value)
    : xml.xml_node
    =
    xml.element name (x_signature_type_key_info_key_value_attributes x) (x_signature_type_key_info_key_value_children x)

and let x_signature_type_key_info_key_value_choice_rsa_key_value_children
    (x : x_signature_type_key_info_key_value_choice_rsa_key_value)
    : list xml.xml_node
    =
    match x with
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue v =>
        [ x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_to_xml "RSAKeyValue" v ]
    | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue v =>
        [ x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_to_xml "DSAKeyValue" v ]

and let x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_children
    (x : x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_P =>
            [ xml.element "P" [] (xml.text_children v) ]
        | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Q =>
            [ xml.element "Q" [] (xml.text_children v) ]
        | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_G =>
            [ xml.element "G" [] (xml.text_children v) ]
        | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Y =>
            [ xml.element "Y" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_get f x))
                (walk rest)
        | [] => []
    walk [
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_P
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Q
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_G
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_DSAKeyValue_field_Y
    ]

and let x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_attributes
    (x : x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value)
    : list (string * string)
    =
    []

and let x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_to_xml
    (name : string) (x : x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_attributes x)
        (x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_children x)

and let x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_children
    (x : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value)
    : list xml.xml_node
    =
    inl nodes forall r.
        (f : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field r) (v : r)
        : list xml.xml_node
        =
        match f with
        | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Modulus =>
            [ xml.element "Modulus" [] (xml.text_children v) ]
        | SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Exponent =>
            [ xml.element "Exponent" [] (xml.text_children v) ]
    inl rec walk
        (fields : list (exists r. x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append
                (nodes f (x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_get f x))
                (walk rest)
        | [] => []
    walk [
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Modulus
        exists SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue_field_Exponent
    ]

and let x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_attributes
    (x : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value)
    : list (string * string)
    =
    []

and let x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_to_xml
    (name : string) (x : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_attributes x)
        (x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_children x)

and let x_signature_type_signed_info_children (x : x_signature_type_signed_info) : list xml.xml_node =
    inl nodes forall r. (f : x_signature_type_signed_info_field r) (v : r) : list xml.xml_node =
        match f with
        | SignatureType_SignedInfo_field_CanonicalizationMethod =>
            [ x_signature_type_signed_info_canonicalization_method_to_xml "CanonicalizationMethod" v ]
        | SignatureType_SignedInfo_field_SignatureMethod =>
            [ x_signature_type_signed_info_signature_method_to_xml "SignatureMethod" v ]
        | SignatureType_SignedInfo_field_Reference => [ x_signature_type_signed_info_reference_to_xml "Reference" v ]
    inl rec walk (fields : list (exists r. x_signature_type_signed_info_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_signature_type_signed_info_get f x)) (walk rest)
        | [] => []
    walk [
        exists SignatureType_SignedInfo_field_CanonicalizationMethod
        exists SignatureType_SignedInfo_field_SignatureMethod
        exists SignatureType_SignedInfo_field_Reference
    ]

and let x_signature_type_signed_info_attributes (x : x_signature_type_signed_info) : list (string * string) = []

and let x_signature_type_signed_info_to_xml (name : string) (x : x_signature_type_signed_info) : xml.xml_node =
    xml.element name (x_signature_type_signed_info_attributes x) (x_signature_type_signed_info_children x)

and let x_signature_type_signed_info_reference_children (x : x_signature_type_signed_info_reference) : list xml.xml_node =
    inl nodes forall r. (f : x_signature_type_signed_info_reference_field r) (v : r) : list xml.xml_node =
        match f with
        | SignatureType_SignedInfo_Reference_field_Transforms =>
            xml.option_list (
                optionm.map (fun v => x_signature_type_signed_info_reference_transforms_to_xml "Transforms" v) v
            )
        | SignatureType_SignedInfo_Reference_field_DigestMethod =>
            [ x_signature_type_signed_info_reference_digest_method_to_xml "DigestMethod" v ]
        | SignatureType_SignedInfo_Reference_field_DigestValue => [ xml.element "DigestValue" [] (xml.text_children v) ]
    inl rec walk (fields : list (exists r. x_signature_type_signed_info_reference_field r)) : list xml.xml_node =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_signature_type_signed_info_reference_get f x)) (walk rest)
        | [] => []
    walk [
        exists SignatureType_SignedInfo_Reference_field_Transforms
        exists SignatureType_SignedInfo_Reference_field_DigestMethod
        exists SignatureType_SignedInfo_Reference_field_DigestValue
    ]

and let x_signature_type_signed_info_reference_attributes
    (x : x_signature_type_signed_info_reference)
    : list (string * string)
    =
    [ ("URI", x.uri) ]

and let x_signature_type_signed_info_reference_to_xml
    (name : string) (x : x_signature_type_signed_info_reference)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_signed_info_reference_attributes x)
        (x_signature_type_signed_info_reference_children x)

and let x_signature_type_signed_info_reference_digest_method_children
    (x : x_signature_type_signed_info_reference_digest_method)
    : list xml.xml_node
    =
    []

and let x_signature_type_signed_info_reference_digest_method_attributes
    (x : x_signature_type_signed_info_reference_digest_method)
    : list (string * string)
    =
    [ ("Algorithm", x.algorithm) ]

and let x_signature_type_signed_info_reference_digest_method_to_xml
    (name : string) (x : x_signature_type_signed_info_reference_digest_method)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_signed_info_reference_digest_method_attributes x)
        (x_signature_type_signed_info_reference_digest_method_children x)

and let x_signature_type_signed_info_reference_transforms_children
    (x : x_signature_type_signed_info_reference_transforms)
    : list xml.xml_node
    =
    inl nodes forall r. (f : x_signature_type_signed_info_reference_transforms_field r) (v : r) : list xml.xml_node =
        match f with
        | SignatureType_SignedInfo_Reference_Transforms_field_Transform =>
            [ x_signature_type_signed_info_reference_transforms_transform_to_xml "Transform" v ]
    inl rec walk
        (fields : list (exists r. x_signature_type_signed_info_reference_transforms_field r))
        : list xml.xml_node
        =
        match fields with
        | packed :: rest =>
            inl exists r. f = packed
            listm.append (nodes f (x_signature_type_signed_info_reference_transforms_get f x)) (walk rest)
        | [] => []
    walk [ exists SignatureType_SignedInfo_Reference_Transforms_field_Transform ]

and let x_signature_type_signed_info_reference_transforms_attributes
    (x : x_signature_type_signed_info_reference_transforms)
    : list (string * string)
    =
    []

and let x_signature_type_signed_info_reference_transforms_to_xml
    (name : string) (x : x_signature_type_signed_info_reference_transforms)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_signed_info_reference_transforms_attributes x)
        (x_signature_type_signed_info_reference_transforms_children x)

and let x_signature_type_signed_info_reference_transforms_transform_children
    (x : x_signature_type_signed_info_reference_transforms_transform)
    : list xml.xml_node
    =
    []

and let x_signature_type_signed_info_reference_transforms_transform_attributes
    (x : x_signature_type_signed_info_reference_transforms_transform)
    : list (string * string)
    =
    [ ("Algorithm", x.algorithm) ]

and let x_signature_type_signed_info_reference_transforms_transform_to_xml
    (name : string) (x : x_signature_type_signed_info_reference_transforms_transform)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_signed_info_reference_transforms_transform_attributes x)
        (x_signature_type_signed_info_reference_transforms_transform_children x)

and let x_signature_type_signed_info_signature_method_children
    (x : x_signature_type_signed_info_signature_method)
    : list xml.xml_node
    =
    []

and let x_signature_type_signed_info_signature_method_attributes
    (x : x_signature_type_signed_info_signature_method)
    : list (string * string)
    =
    [ ("Algorithm", to_text x.algorithm) ]

and let x_signature_type_signed_info_signature_method_to_xml
    (name : string) (x : x_signature_type_signed_info_signature_method)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_signed_info_signature_method_attributes x)
        (x_signature_type_signed_info_signature_method_children x)

and let x_signature_type_signed_info_canonicalization_method_children
    (x : x_signature_type_signed_info_canonicalization_method)
    : list xml.xml_node
    =
    []

and let x_signature_type_signed_info_canonicalization_method_attributes
    (x : x_signature_type_signed_info_canonicalization_method)
    : list (string * string)
    =
    [ ("Algorithm", x.algorithm) ]

and let x_signature_type_signed_info_canonicalization_method_to_xml
    (name : string) (x : x_signature_type_signed_info_canonicalization_method)
    : xml.xml_node
    =
    xml.element name
        (x_signature_type_signed_info_canonicalization_method_attributes x)
        (x_signature_type_signed_info_canonicalization_method_children x)

and let x_dte_def_type_example (k : i32) : x_dte_def_type unsigned =
    x_dte_def_type {
        choice_documento = x_dte_def_type_choice_documento_example k
        signature = Pending "an unsigned example"
        version = xml.required_value "xsd_decimal" (xsd_decimal_of_string "1.0")
    }

and let x_dte_def_type_choice_documento_example (k : i32) : x_dte_def_type_choice_documento =
    DTEDefType_choice_Documento_Documento (x_dte_def_type_choice_documento_documento_example k)

and let x_dte_def_type_choice_documento_documento_example (k : i32) : x_dte_def_type_choice_documento_documento =
    x_dte_def_type_choice_documento_documento {
        encabezado = x_dte_def_type_choice_documento_documento_encabezado_example k
        detalle = [ x_dte_def_type_choice_documento_documento_detalle_example (k + 0i32) ]
        sub_tot_info = []
        dsc_rcg_global = []
        referencia = []
        comisiones = []
        ted = x_dte_def_type_choice_documento_documento_ted_example k
        tmst_firma = x_fecha_hora_type_example k
        id = "ID_x_dte_def_type_choice_documento_documento"
    }

and let x_dte_def_type_choice_documento_documento_ted_example (k : i32) : x_dte_def_type_choice_documento_documento_ted =
    x_dte_def_type_choice_documento_documento_ted {
        dd = x_dte_def_type_choice_documento_documento_ted_dd_example k
        frmt = x_dte_def_type_choice_documento_documento_ted_frmt_example k
        version = "1.0"
    }

and let x_dte_def_type_choice_documento_documento_ted_frmt_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_frmt
    =
    x_dte_def_type_choice_documento_documento_ted_frmt {
        base = "AAAA"
        algoritmo = x_dte_def_type_choice_documento_documento_ted_frmt_algoritmo_example k
    }

and let x_dte_def_type_choice_documento_documento_ted_dd_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd
    =
    x_dte_def_type_choice_documento_documento_ted_dd {
        re = x_rut_type_example k
        td = x_dte_type_example k
        f = x_folio_type_example k
        fe = x_fecha_type_example k
        rr = x_rut_type_example k
        rsr = x_dte_def_type_choice_documento_documento_ted_dd_rsr_example k
        mnt = "1"
        it1 = x_dte_def_type_choice_documento_documento_ted_dd_it1_example k
        caf = x_dte_def_type_choice_documento_documento_ted_dd_caf_example k
        tsted = x_fecha_hora_type_example k
    }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_caf
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf {
        da = x_dte_def_type_choice_documento_documento_ted_dd_caf_da_example k
        frma = x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_example k
        version = "1.0"
    }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_caf_frma
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_frma {
        base = "AAAA"
        algoritmo = "SHA1withRSA"
    }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_caf_da
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da {
        re = x_rut_type_example k
        rs = x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rs_example k
        td = x_dte_type_example k
        rng = x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_example k
        fa = x_fecha_type_example k
        choice_rsapk = x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_example k
        idk = xsd_integer_example k
    }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk
    =
    DTEDefType_choice_Documento_Documento_TED_DD_CAF_DA_choice_RSAPK_RSAPK (
        x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_example k
    )

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk {
        m = "AAAA"
        e = "AAAA"
    }

and let x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng
    =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng {
        d = x_folio_type_example k
        h = x_folio_type_example k
    }

and let x_dte_def_type_choice_documento_documento_comisiones_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_comisiones
    =
    x_dte_def_type_choice_documento_documento_comisiones {
        nro_lin_com = x_dte_def_type_choice_documento_documento_comisiones_nro_lin_com_example k
        tipo_movim = x_dte_def_type_choice_documento_documento_comisiones_tipo_movim_example k
        glosa = x_dte_def_type_choice_documento_documento_comisiones_glosa_example k
        tasa_comision = None
        val_com_neto = x_monto_type_example k
        val_com_exe = x_monto_type_example k
        val_com_iva = None
    }

and let x_dte_def_type_choice_documento_documento_referencia_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_referencia
    =
    x_dte_def_type_choice_documento_documento_referencia {
        nro_lin_ref = x_dte_def_type_choice_documento_documento_referencia_nro_lin_ref_example k
        tpo_doc_ref = x_dte_def_type_choice_documento_documento_referencia_tpo_doc_ref_example k
        ind_global = None
        folio_ref = x_folio_r_type_example k
        rut_otr = None
        fch_ref = x_fecha_type_example k
        cod_ref = None
        razon_ref = None
    }

and let x_dte_def_type_choice_documento_documento_dsc_rcg_global_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_dsc_rcg_global
    =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global {
        nro_lin_dr = xsd_positive_integer_example k
        tpo_mov = x_dte_def_type_choice_documento_documento_dsc_rcg_global_tpo_mov_example k
        glosa_dr = None
        tpo_valor = x_dinero_porcentaje_type_example k
        valor_dr = x_dec16_2_type_example k
        valor_dr_otr_mnda = None
        ind_exe_dr = None
    }

and let x_dte_def_type_choice_documento_documento_sub_tot_info_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_sub_tot_info
    =
    x_dte_def_type_choice_documento_documento_sub_tot_info {
        nro_sti = x_dte_def_type_choice_documento_documento_sub_tot_info_nro_sti_example k
        glosa_sti = x_dte_def_type_choice_documento_documento_sub_tot_info_glosa_sti_example k
        orden_sti = None
        sub_tot_neto_sti = None
        sub_tot_ivasti = None
        sub_tot_adic_sti = None
        sub_tot_exe_sti = None
        val_subtot_sti = None
        lineas_deta = []
    }

and let x_dte_def_type_choice_documento_documento_detalle_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle
    =
    x_dte_def_type_choice_documento_documento_detalle {
        nro_lin_det = x_dte_def_type_choice_documento_documento_detalle_nro_lin_det_example k
        cdg_item = []
        ind_exe = None
        retenedor = None
        nmb_item = x_dte_def_type_choice_documento_documento_detalle_nmb_item_example k
        dsc_item = None
        qty_ref = None
        unmd_ref = None
        prc_ref = None
        qty_item = None
        subcantidad = []
        fch_elabor = None
        fch_vencim = None
        unmd_item = None
        prc_item = None
        otr_mnda = None
        descuento_pct = None
        descuento_monto = None
        sub_dscto = []
        recargo_pct = None
        recargo_monto = None
        sub_recargo = []
        cod_imp_adic = []
        monto_item = x_monto_type_example k
    }

and let x_dte_def_type_choice_documento_documento_detalle_sub_recargo_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_sub_recargo
    =
    x_dte_def_type_choice_documento_documento_detalle_sub_recargo {
        tipo_recargo = x_dinero_porcentaje_type_example k
        valor_recargo = x_dec16_2_type_example k
    }

and let x_dte_def_type_choice_documento_documento_detalle_sub_dscto_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_sub_dscto
    =
    x_dte_def_type_choice_documento_documento_detalle_sub_dscto {
        tipo_dscto = x_dinero_porcentaje_type_example k
        valor_dscto = x_dec16_2_type_example k
    }

and let x_dte_def_type_choice_documento_documento_detalle_otr_mnda_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_otr_mnda
    =
    x_dte_def_type_choice_documento_documento_detalle_otr_mnda {
        prc_otr_mon = x_dec12_6_type_example k
        moneda = x_dte_def_type_choice_documento_documento_detalle_otr_mnda_moneda_example k
        fct_conv = None
        dcto_otr_mnda = None
        recargo_otr_mnda = None
        monto_item_otr_mnda = None
    }

and let x_dte_def_type_choice_documento_documento_detalle_subcantidad_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_subcantidad
    =
    x_dte_def_type_choice_documento_documento_detalle_subcantidad {
        sub_qty = x_dec12_6_type_example k
        sub_cod = x_dte_def_type_choice_documento_documento_detalle_subcantidad_sub_cod_example k
    }

and let x_dte_def_type_choice_documento_documento_detalle_retenedor_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_retenedor
    =
    x_dte_def_type_choice_documento_documento_detalle_retenedor {
        ind_agente = x_dte_def_type_choice_documento_documento_detalle_retenedor_ind_agente_example k
        mnt_base_faena = None
        mnt_marg_comer = None
        prc_cons_final = None
    }

and let x_dte_def_type_choice_documento_documento_detalle_cdg_item_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_detalle_cdg_item
    =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item {
        tpo_codigo = x_dte_def_type_choice_documento_documento_detalle_cdg_item_tpo_codigo_example k
        vlr_codigo = x_dte_def_type_choice_documento_documento_detalle_cdg_item_vlr_codigo_example k
    }

and let x_dte_def_type_choice_documento_documento_encabezado_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado
    =
    x_dte_def_type_choice_documento_documento_encabezado {
        id_doc = x_dte_def_type_choice_documento_documento_encabezado_id_doc_example k
        emisor = x_dte_def_type_choice_documento_documento_encabezado_emisor_example k
        rut_mandante = None
        receptor = x_dte_def_type_choice_documento_documento_encabezado_receptor_example k
        rut_solicita = None
        transporte = None
        totales = x_dte_def_type_choice_documento_documento_encabezado_totales_example k
        otra_moneda = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda
    =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda {
        tpo_moneda = x_tip_mon_type_example k
        tpo_cambio = None
        mnt_neto_otr_mnda = None
        mnt_exe_otr_mnda = None
        mnt_fae_carne_otr_mnda = None
        mnt_marg_com_otr_mnda = None
        iva_otr_mnda = None
        imp_ret_otr_mnda = []
        iva_no_ret_otr_mnda = None
        mnt_tot_otr_mnda = x_dec14_4_type_example k
    }

and let x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda
    =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda {
        tipo_imp_otr_mnda = x_imp_adic_dte_type_example k
        tasa_imp_otr_mnda = None
        vlr_imp_otr_mnda = x_dec14_4_type_example k
    }

and let x_dte_def_type_choice_documento_documento_encabezado_totales_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_totales
    =
    x_dte_def_type_choice_documento_documento_encabezado_totales {
        mnt_neto = None
        mnt_exe = None
        mnt_base = None
        mnt_margen_com = None
        tasa_iva = None
        iva = None
        iva_prop = None
        iva_terc = None
        impto_reten = []
        iva_no_ret = None
        cred_ec = None
        grnt_dep = None
        comisiones = None
        mnt_total = x_monto_type_example k
        monto_nf = None
        monto_periodo = None
        saldo_anterior = None
        vlr_pagar = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones
    =
    x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones {
        val_com_neto = None
        val_com_exe = None
        val_com_iva = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten
    =
    x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten {
        tipo_imp = x_imp_adic_dte_type_example k
        tasa_imp = None
        monto_imp = x_mnt_imp_type_example k
    }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte {
        patente = None
        rut_trans = None
        chofer = None
        dir_dest = None
        cmna_dest = None
        ciudad_dest = None
        aduana = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana {
        cod_mod_venta = None
        cod_clau_venta = None
        tot_clau_venta = None
        cod_via_transp = None
        nombre_transp = None
        rut_cia_transp = None
        nom_cia_transp = None
        id_adic_transp = None
        booking = None
        operador = None
        cod_pto_embarque = None
        id_adic_pto_emb = None
        cod_pto_desemb = None
        id_adic_pto_desemb = None
        tara = None
        cod_unid_med_tara = None
        peso_bruto = None
        cod_unid_peso_bruto = None
        peso_neto = None
        cod_unid_peso_neto = None
        tot_items = None
        tot_bultos = None
        tipo_bultos = []
        mnt_flete = None
        mnt_seguro = None
        cod_pais_recep = None
        cod_pais_destin = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos {
        cod_tpo_bultos = None
        cant_bultos = None
        marcas = None
        id_container = None
        sello = None
        emisor_sello = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer
    =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer {
        rut_chofer = x_rut_type_example k
        nombre_chofer = x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_nombre_chofer_example k
    }

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor {
        rut_recep = x_rut_type_example k
        cdg_int_recep = None
        rzn_soc_recep = x_rzn_soc_larga_type_example k
        extranjero = None
        giro_recep = None
        contacto = None
        correo_recep = None
        dir_recep = None
        cmna_recep = None
        ciudad_recep = None
        dir_postal = None
        cmna_postal = None
        ciudad_postal = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero
    =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero {
        num_id = None
        nacionalidad = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor {
        rut_emisor = x_rut_type_example k
        rzn_soc = x_dte_def_type_choice_documento_documento_encabezado_emisor_rzn_soc_example k
        giro_emis = x_dte_def_type_choice_documento_documento_encabezado_emisor_giro_emis_example k
        telefono = []
        correo_emisor = None
        acteco = [ x_dte_def_type_choice_documento_documento_encabezado_emisor_acteco_example (k + 0i32) ]
        guia_export = None
        sucursal = None
        cdg_sii_sucur = None
        dir_origen = None
        cmna_origen = None
        ciudad_origen = None
        cdg_vendedor = None
        id_adic_emisor = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export
    =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export {
        cdg_traslado = None
        folio_aut = None
        fch_aut = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc {
        tipo_dte = x_dte_type_example k
        folio = x_folio_type_example k
        fch_emis = x_fecha_type_example k
        ind_no_rebaja = None
        tipo_despacho = None
        ind_traslado = None
        tpo_impresion = None
        ind_servicio = None
        mnt_bruto = None
        fma_pago = None
        fma_pag_exp = None
        fch_cancel = None
        mnt_cancel = None
        saldo_insol = None
        mnt_pagos = []
        periodo_desde = None
        periodo_hasta = None
        medio_pago = None
        tpo_cta_pago = None
        num_cta_pago = None
        bco_pago = None
        term_pago_cdg = None
        term_pago_glosa = None
        term_pago_dias = None
        fch_venc = None
    }

and let x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_example
    (k : i32)
    : x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos
    =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos {
        fch_pago = x_fecha_type_example k
        mnt_pago = x_monto_type_example k
        glosa_pagos = None
    }

and let x_signature_type_example (k : i32) : x_signature_type =
    x_signature_type {
        signed_info = x_signature_type_signed_info_example k
        signature_value = "AAAA"
        key_info = x_signature_type_key_info_example k
    }

and let x_signature_type_key_info_example (k : i32) : x_signature_type_key_info =
    x_signature_type_key_info {
        key_value = x_signature_type_key_info_key_value_example k
        x509_data = x_signature_type_key_info_x509_data_example k
    }

and let x_signature_type_key_info_x509_data_example (k : i32) : x_signature_type_key_info_x509_data =
    x_signature_type_key_info_x509_data {
        x509_certificate = "AAAA"
    }

and let x_signature_type_key_info_key_value_example (k : i32) : x_signature_type_key_info_key_value =
    x_signature_type_key_info_key_value {
        choice_rsa_key_value = x_signature_type_key_info_key_value_choice_rsa_key_value_example k
    }

and let x_signature_type_key_info_key_value_choice_rsa_key_value_example
    (k : i32)
    : x_signature_type_key_info_key_value_choice_rsa_key_value
    =
    SignatureType_KeyInfo_KeyValue_choice_RSAKeyValue_RSAKeyValue (
        x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_example k
    )

and let x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_example
    (k : i32)
    : x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value
    =
    x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value {
        modulus = "AAAA"
        exponent = "AAAA"
    }

and let x_signature_type_signed_info_example (k : i32) : x_signature_type_signed_info =
    x_signature_type_signed_info {
        canonicalization_method = x_signature_type_signed_info_canonicalization_method_example k
        signature_method = x_signature_type_signed_info_signature_method_example k
        reference = x_signature_type_signed_info_reference_example k
    }

and let x_signature_type_signed_info_reference_example (k : i32) : x_signature_type_signed_info_reference =
    x_signature_type_signed_info_reference {
        transforms = None
        digest_method = x_signature_type_signed_info_reference_digest_method_example k
        digest_value = "AAAA"
        uri = "http://example.com"
    }

and let x_signature_type_signed_info_reference_digest_method_example
    (k : i32)
    : x_signature_type_signed_info_reference_digest_method
    =
    x_signature_type_signed_info_reference_digest_method {
        algorithm = "http://www.w3.org/2000/09/xmldsig#sha1"
    }

and let x_signature_type_signed_info_reference_transforms_example
    (k : i32)
    : x_signature_type_signed_info_reference_transforms
    =
    x_signature_type_signed_info_reference_transforms {
        transform = x_signature_type_signed_info_reference_transforms_transform_example k
    }

and let x_signature_type_signed_info_reference_transforms_transform_example
    (k : i32)
    : x_signature_type_signed_info_reference_transforms_transform
    =
    x_signature_type_signed_info_reference_transforms_transform {
        algorithm = "http://example.com"
    }

and let x_signature_type_signed_info_signature_method_example (k : i32) : x_signature_type_signed_info_signature_method =
    x_signature_type_signed_info_signature_method {
        algorithm = x_signature_type_signed_info_signature_method_algorithm_example k
    }

and let x_signature_type_signed_info_canonicalization_method_example
    (k : i32)
    : x_signature_type_signed_info_canonicalization_method
    =
    x_signature_type_signed_info_canonicalization_method {
        algorithm = "http://www.w3.org/TR/2001/REC-xml-c14n-20010315"
    }


instance of_xml x_dte_def_type s {stage_of} = fun c => x_dte_def_type_of_xml (stage_of ()) c
instance to_xml x_dte_def_type s = x_dte_def_type_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones = x_dte_def_type_choice_documento_exportaciones_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones = x_dte_def_type_choice_documento_exportaciones_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted =
    x_dte_def_type_choice_documento_exportaciones_ted_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted =
    x_dte_def_type_choice_documento_exportaciones_ted_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_frmt =
    x_dte_def_type_choice_documento_exportaciones_ted_frmt_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_frmt =
    x_dte_def_type_choice_documento_exportaciones_ted_frmt_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_dd =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_dd =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_frma_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_dsapk_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_choice_rsapk_rsapk_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng =
    x_dte_def_type_choice_documento_exportaciones_ted_dd_caf_da_rng_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_referencia =
    x_dte_def_type_choice_documento_exportaciones_referencia_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_referencia =
    x_dte_def_type_choice_documento_exportaciones_referencia_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global =
    x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global =
    x_dte_def_type_choice_documento_exportaciones_dsc_rcg_global_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_sub_tot_info =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_sub_tot_info =
    x_dte_def_type_choice_documento_exportaciones_sub_tot_info_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_detalle =
    x_dte_def_type_choice_documento_exportaciones_detalle_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_detalle =
    x_dte_def_type_choice_documento_exportaciones_detalle_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_recargo_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto =
    x_dte_def_type_choice_documento_exportaciones_detalle_sub_dscto_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda =
    x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda =
    x_dte_def_type_choice_documento_exportaciones_detalle_otr_mnda_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad =
    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad =
    x_dte_def_type_choice_documento_exportaciones_detalle_subcantidad_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item =
    x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item =
    x_dte_def_type_choice_documento_exportaciones_detalle_cdg_item_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado =
    x_dte_def_type_choice_documento_exportaciones_encabezado_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado =
    x_dte_def_type_choice_documento_exportaciones_encabezado_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda =
    x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda =
    x_dte_def_type_choice_documento_exportaciones_encabezado_otra_moneda_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_totales =
    x_dte_def_type_choice_documento_exportaciones_encabezado_totales_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_totales =
    x_dte_def_type_choice_documento_exportaciones_encabezado_totales_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_aduana_tipo_bultos_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer =
    x_dte_def_type_choice_documento_exportaciones_encabezado_transporte_chofer_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_receptor =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_receptor =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero =
    x_dte_def_type_choice_documento_exportaciones_encabezado_receptor_extranjero_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_emisor =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_emisor =
    x_dte_def_type_choice_documento_exportaciones_encabezado_emisor_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_to_xml

instance of_xml x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_of_xml
instance to_xml x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos =
    x_dte_def_type_choice_documento_exportaciones_encabezado_id_doc_mnt_pagos_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion = x_dte_def_type_choice_documento_liquidacion_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion = x_dte_def_type_choice_documento_liquidacion_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted = x_dte_def_type_choice_documento_liquidacion_ted_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted = x_dte_def_type_choice_documento_liquidacion_ted_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_frmt =
    x_dte_def_type_choice_documento_liquidacion_ted_frmt_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_frmt =
    x_dte_def_type_choice_documento_liquidacion_ted_frmt_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_dd =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_dd =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_frma_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_dsapk_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_choice_rsapk_rsapk_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng =
    x_dte_def_type_choice_documento_liquidacion_ted_dd_caf_da_rng_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_comisiones =
    x_dte_def_type_choice_documento_liquidacion_comisiones_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_comisiones =
    x_dte_def_type_choice_documento_liquidacion_comisiones_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_referencia =
    x_dte_def_type_choice_documento_liquidacion_referencia_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_referencia =
    x_dte_def_type_choice_documento_liquidacion_referencia_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_sub_tot_info =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_sub_tot_info =
    x_dte_def_type_choice_documento_liquidacion_sub_tot_info_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_detalle =
    x_dte_def_type_choice_documento_liquidacion_detalle_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_detalle =
    x_dte_def_type_choice_documento_liquidacion_detalle_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad =
    x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad =
    x_dte_def_type_choice_documento_liquidacion_detalle_subcantidad_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item =
    x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item =
    x_dte_def_type_choice_documento_liquidacion_detalle_cdg_item_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado =
    x_dte_def_type_choice_documento_liquidacion_encabezado_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado =
    x_dte_def_type_choice_documento_liquidacion_encabezado_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado_totales =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado_totales =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_comisiones_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten =
    x_dte_def_type_choice_documento_liquidacion_encabezado_totales_impto_reten_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado_receptor =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado_receptor =
    x_dte_def_type_choice_documento_liquidacion_encabezado_receptor_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado_emisor =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado_emisor =
    x_dte_def_type_choice_documento_liquidacion_encabezado_emisor_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_to_xml

instance of_xml x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_of_xml
instance to_xml x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos =
    x_dte_def_type_choice_documento_liquidacion_encabezado_id_doc_mnt_pagos_to_xml

instance of_xml x_dte_def_type_choice_documento_documento = x_dte_def_type_choice_documento_documento_of_xml
instance to_xml x_dte_def_type_choice_documento_documento = x_dte_def_type_choice_documento_documento_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted = x_dte_def_type_choice_documento_documento_ted_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted = x_dte_def_type_choice_documento_documento_ted_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_frmt =
    x_dte_def_type_choice_documento_documento_ted_frmt_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_frmt =
    x_dte_def_type_choice_documento_documento_ted_frmt_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_dd =
    x_dte_def_type_choice_documento_documento_ted_dd_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_dd =
    x_dte_def_type_choice_documento_documento_ted_dd_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_dd_caf =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_dd_caf =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_frma =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_frma =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_frma_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_dsapk_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_choice_rsapk_rsapk_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng =
    x_dte_def_type_choice_documento_documento_ted_dd_caf_da_rng_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_comisiones =
    x_dte_def_type_choice_documento_documento_comisiones_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_comisiones =
    x_dte_def_type_choice_documento_documento_comisiones_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_referencia =
    x_dte_def_type_choice_documento_documento_referencia_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_referencia =
    x_dte_def_type_choice_documento_documento_referencia_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_dsc_rcg_global =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_dsc_rcg_global =
    x_dte_def_type_choice_documento_documento_dsc_rcg_global_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_sub_tot_info =
    x_dte_def_type_choice_documento_documento_sub_tot_info_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_sub_tot_info =
    x_dte_def_type_choice_documento_documento_sub_tot_info_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_detalle =
    x_dte_def_type_choice_documento_documento_detalle_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_detalle =
    x_dte_def_type_choice_documento_documento_detalle_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_detalle_sub_recargo =
    x_dte_def_type_choice_documento_documento_detalle_sub_recargo_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_detalle_sub_recargo =
    x_dte_def_type_choice_documento_documento_detalle_sub_recargo_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_detalle_sub_dscto =
    x_dte_def_type_choice_documento_documento_detalle_sub_dscto_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_detalle_sub_dscto =
    x_dte_def_type_choice_documento_documento_detalle_sub_dscto_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_detalle_otr_mnda =
    x_dte_def_type_choice_documento_documento_detalle_otr_mnda_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_detalle_otr_mnda =
    x_dte_def_type_choice_documento_documento_detalle_otr_mnda_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_detalle_subcantidad =
    x_dte_def_type_choice_documento_documento_detalle_subcantidad_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_detalle_subcantidad =
    x_dte_def_type_choice_documento_documento_detalle_subcantidad_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_detalle_retenedor =
    x_dte_def_type_choice_documento_documento_detalle_retenedor_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_detalle_retenedor =
    x_dte_def_type_choice_documento_documento_detalle_retenedor_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_detalle_cdg_item =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_detalle_cdg_item =
    x_dte_def_type_choice_documento_documento_detalle_cdg_item_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado =
    x_dte_def_type_choice_documento_documento_encabezado_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado =
    x_dte_def_type_choice_documento_documento_encabezado_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_otra_moneda =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_otra_moneda =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda =
    x_dte_def_type_choice_documento_documento_encabezado_otra_moneda_imp_ret_otr_mnda_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_totales =
    x_dte_def_type_choice_documento_documento_encabezado_totales_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_totales =
    x_dte_def_type_choice_documento_documento_encabezado_totales_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones =
    x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones =
    x_dte_def_type_choice_documento_documento_encabezado_totales_comisiones_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten =
    x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten =
    x_dte_def_type_choice_documento_documento_encabezado_totales_impto_reten_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_transporte =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_transporte =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_aduana_tipo_bultos_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer =
    x_dte_def_type_choice_documento_documento_encabezado_transporte_chofer_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_receptor =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_receptor =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero =
    x_dte_def_type_choice_documento_documento_encabezado_receptor_extranjero_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_emisor =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_emisor =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export =
    x_dte_def_type_choice_documento_documento_encabezado_emisor_guia_export_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_id_doc =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_id_doc =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_to_xml

instance of_xml x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_of_xml
instance to_xml x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos =
    x_dte_def_type_choice_documento_documento_encabezado_id_doc_mnt_pagos_to_xml

instance of_xml x_signature_type = x_signature_type_of_xml
instance to_xml x_signature_type = x_signature_type_to_xml

instance of_xml x_signature_type_key_info = x_signature_type_key_info_of_xml
instance to_xml x_signature_type_key_info = x_signature_type_key_info_to_xml

instance of_xml x_signature_type_key_info_x509_data = x_signature_type_key_info_x509_data_of_xml
instance to_xml x_signature_type_key_info_x509_data = x_signature_type_key_info_x509_data_to_xml

instance of_xml x_signature_type_key_info_key_value = x_signature_type_key_info_key_value_of_xml
instance to_xml x_signature_type_key_info_key_value = x_signature_type_key_info_key_value_to_xml

instance of_xml x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value =
    x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_of_xml
instance to_xml x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value =
    x_signature_type_key_info_key_value_choice_rsa_key_value_dsa_key_value_to_xml

instance of_xml x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value =
    x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_of_xml
instance to_xml x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value =
    x_signature_type_key_info_key_value_choice_rsa_key_value_rsa_key_value_to_xml

instance of_xml x_signature_type_signed_info = x_signature_type_signed_info_of_xml
instance to_xml x_signature_type_signed_info = x_signature_type_signed_info_to_xml

instance of_xml x_signature_type_signed_info_reference = x_signature_type_signed_info_reference_of_xml
instance to_xml x_signature_type_signed_info_reference = x_signature_type_signed_info_reference_to_xml

instance of_xml x_signature_type_signed_info_reference_digest_method =
    x_signature_type_signed_info_reference_digest_method_of_xml
instance to_xml x_signature_type_signed_info_reference_digest_method =
    x_signature_type_signed_info_reference_digest_method_to_xml

instance of_xml x_signature_type_signed_info_reference_transforms =
    x_signature_type_signed_info_reference_transforms_of_xml
instance to_xml x_signature_type_signed_info_reference_transforms =
    x_signature_type_signed_info_reference_transforms_to_xml

instance of_xml x_signature_type_signed_info_reference_transforms_transform =
    x_signature_type_signed_info_reference_transforms_transform_of_xml
instance to_xml x_signature_type_signed_info_reference_transforms_transform =
    x_signature_type_signed_info_reference_transforms_transform_to_xml

instance of_xml x_signature_type_signed_info_signature_method = x_signature_type_signed_info_signature_method_of_xml
instance to_xml x_signature_type_signed_info_signature_method = x_signature_type_signed_info_signature_method_to_xml

instance of_xml x_signature_type_signed_info_canonicalization_method =
    x_signature_type_signed_info_canonicalization_method_of_xml
instance to_xml x_signature_type_signed_info_canonicalization_method =
    x_signature_type_signed_info_canonicalization_method_to_xml

#### generated_cl_dte_v10_writes_its_example (test)

In [1]:
///- --test
///> rust

inl example = x_dte_def_type_example 0i32
inl document =
    xml.element
        "DTE"
        (("xmlns", "http://www.sii.cl/SiiDte") :: x_dte_def_type_attributes example)
        (x_dte_def_type_children example)
file_system.directory_create_all (generated_fiscal_runtime.sample_path "") |> ignore
file_system.write_all_text
    (generated_fiscal_runtime.sample_path "generated_cl_dte_v10_example.xml")
    (xml.write_node (xml.with_namespace "Signature" "http://www.w3.org/2000/09/xmldsig#" document))
inl text = file_system.read_all_text (generated_fiscal_runtime.sample_path "generated_cl_dte_v10_example.xml")
let rec runs (left : i32) (parses : list i64) (writes : list i64) (bytes : i64) : list i64 * list i64 * i64 =
    if left = 0i32
    then parses, writes, bytes
    else
        inl watch = date_time.stopwatch ()
        date_time.stopwatch_start watch
        inl start = date_time.stopwatch_elapsed_microseconds watch
        match xml.read_xml text with
        | node :: [] =>
            inl parsed = xml_codec.expect_decoded (x_dte_def_type_of_xml Unsigned node)
            inl parsed_at = date_time.stopwatch_elapsed_microseconds watch
            inl written = xml.write_node (x_dte_def_type_to_xml "DTE" parsed)
            inl written_at = date_time.stopwatch_elapsed_microseconds watch
            inl parses = generated_fiscal_runtime.insert_run (parsed_at - start) parses
            inl writes = generated_fiscal_runtime.insert_run (written_at - parsed_at) writes
            runs (left - 1i32) parses writes (bytes + sm'.length written)
        | _ => failwith "one DTE element"
inl _ = runs (dyn 5i32) [] [] 0i64
inl parses, writes, bytes = runs (dyn 31i32) [] [] 0i64
inl row = sm'.(++#) "generated_cl_dte_v10 parse_us " (generated_fiscal_runtime.median parses)
inl row = sm'.(++#) row (sm'.(++#) " write_us " (generated_fiscal_runtime.median writes))
inl row = sm'.(++#) row (sm'.(++#) " runs 31 bytes " (sm'.obj_to_string bytes))
console.write_line row
match xml.read_xml text with
| node :: [] =>
    inl parsed = xml_codec.expect_decoded (x_dte_def_type_of_xml Unsigned node)
    inl document =
        xml.element
            "DTE"
            (("xmlns", "http://www.sii.cl/SiiDte") :: x_dte_def_type_attributes parsed)
            (x_dte_def_type_children parsed)
    inl folder = generated_fiscal_runtime.sample_path "../target/bench"
    match file_system.directory_create_all folder with
    | Ok _ =>
        file_system.write_all_text
            (sm'.(++#) folder "/ours-generated_cl_dte_v10_example.xml")
            (xml.write_node (xml.with_namespace "Signature" "http://www.w3.org/2000/09/xmldsig#" document))
    | Error _ => failwith "cannot create target/bench"
| _ => failwith "one DTE element"

generated_cl_dte_v10 parse_us 183 write_us 50 runs 31 bytes 29698

0

#### generated_cl_dte_v10_keeps_raw_strings_out (test)

In [2]:
///- --test --fails "Unification failure"
///> rust

x_rut_type_to_string "raw"

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/generated/generated_cl_dte_v10/generated_cl_dte_v10.spi:25716:26 (main.spi:25660:26): ETY-TermError: Unification failure. Got:      (x_rut_type : *) Expected: string In the definition: spiral_cells_test.
